# DENGU HUNTER — FINAL GitHub Master (CoU AI Hackathon 2026)
**Original training/research preserved • One working inference workflow • T4/CPU safe • FastAPI + Gradio**

## Before you run
1. In **Google Colab**, choose **Runtime → Change runtime type → T4 GPU** if available. GPU capacity is controlled by Colab. **CPU inference works** if T4 is not allocated.
2. Select **Runtime → Run all**. Only the **six code cells at the top** execute. The historical **118 original training/quality-upgrade cells** are preserved **as labelled source archives at the bottom**, preventing accidental retraining and Drive modifications.
3. This uses the **already trained real F2 YOLO `best.pt`** from your Google Drive; there is no hard-coded inference fallback. Without real model weights the preflight stops clearly.
4. Use **non-sensitive demo photographs**, provide optional real user-entered GPS, and mark cases `DEMO_CASE_STAGED` or `FIELD_REPORT_UNVERIFIED`.
5. The live app supports **Citizen upload → real F2 bounding boxes/confidence → rule-based inspection priority → human review → worker assignment → before/after YOLO & human decision → map/dashboard → drone route simulation → labelled evaluation → audit**. Optional F3 model processes close-up images without claiming Aedes species confirmation.
6. FastAPI endpoints share the **same running Gradio/YOLO/case store** and require a second private bearer token. This meets the proposal's FastAPI integration requirement while retaining a Colab-friendly Gradio demonstration UI.

**Scientific honesty:** This prioritizes **potential breeding-site inspections**. It is **not** dengue incidence prediction, larval *Aedes* species confirmation, autonomous flight, independent field cleanup verification, or demonstrated reduction of dengue infections. **15 older intervention entries with pending evidence are not automatically declared successful.**

**Testing boundary:** Static/synthetic tests are *software tests*, not live YOLO/Colab/field validation. Real-image inference and end-to-end Colab testing remain necessary before claiming demo readiness.

In [1]:
# 1/6 — SETUP (install missing dependencies, mount your existing Drive, use T4 if allocated).
import importlib.util, importlib.metadata, os, sys, subprocess
from pathlib import Path
import torch

DH_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DH_GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('PyTorch:', torch.__version__)
print('Inference device:', DH_GPU_NAME if DH_GPU_NAME else 'CPU FALLBACK — no Colab GPU allocation')

packages = {
    'ultralytics': 'ultralytics>=8.3,<9',
    'gradio': 'gradio>=5,<7',
    'folium': 'folium>=0.17,<1',
    'cv2': 'opencv-python-headless>=4.8',
    'PIL': 'pillow>=10',
    'pandas': 'pandas>=2',
    'yaml': 'pyyaml>=6',
    'fastapi': 'fastapi>=0.115,<1',
    'multipart': 'python-multipart>=0.0.20',
}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec('gradio') is not None:
    version = int(importlib.metadata.version('gradio').split('.')[0])
    if version not in (5,6): missing.append('gradio>=5,<7')
if missing:
    print('Installing missing requirements:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

DRIVE = Path(os.environ.get('DH_DRIVE_ROOT','/content/drive/MyDrive'))
if not DRIVE.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if not DRIVE.is_dir():
    raise FileNotFoundError('Google Drive is not accessible: '+str(DRIVE))
SHOWCASE = DRIVE/'DENGU_HUNTER_FINAL_SHOWCASE'
print('Drive root:', DRIVE)
print('Training is DISABLED in this notebook. Original datasets and training runs are read-only.')

PyTorch: 2.14.1+cpu
Inference device: CPU FALLBACK — no Colab GPU allocation
Installing missing requirements: ['ultralytics>=8.3,<9']
Mounted at /content/drive
Drive root: /content/drive/MyDrive
Training is DISABLED in this notebook. Original datasets and training runs are read-only.


In [36]:
"""DENGU HUNTER unified hackathon prototype engine.

Real inference is mandatory for live submissions.  No synthetic results enter the live store.
Only writes to a new Unified_Prototype/App_Data folder, never to old source datasets.
"""
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from threading import RLock
from typing import Optional
import hashlib
import html
import json
import math
import os
import re
import secrets
import statistics
import tempfile
import uuid

import cv2
import folium
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

UTC = timezone.utc
CASE_PATTERN = re.compile(r"^DH-[A-F0-9]{12}$")
MAX_PIXELS = 20_000_000


def utc_now():
    return datetime.now(UTC).isoformat(timespec="seconds")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def image_rgb(image):
    if image is None:
        raise ValueError("An actual image is required. No placeholder detections are allowed.")
    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.asarray(image))
    if image.width * image.height > MAX_PIXELS:
        raise ValueError("Image exceeds 20 megapixels. Please resize it.")
    if image.width < 32 or image.height < 32:
        raise ValueError("Image is too small for a meaningful inspection.")
    return image.convert("RGB")


def check_gps(lat, lon):
    if lat in ("", None) and lon in ("", None):
        return None, None
    if lat in ("", None) or lon in ("", None):
        raise ValueError("Provide BOTH latitude and longitude, or leave BOTH blank.")
    a, b = float(lat), float(lon)
    if not (math.isfinite(a) and math.isfinite(b) and -90 <= a <= 90 and -180 <= b <= 180):
        raise ValueError("GPS coordinates must be finite and in valid ranges.")
    return a, b


def draw_detections(image, detections):
    annotated = image_rgb(image).copy()
    drawer = ImageDraw.Draw(annotated)
    for det in detections:
        x1, y1, x2, y2 = (int(round(float(n))) for n in det["bbox_xyxy"])
        x1 = max(0, min(annotated.width, x1))
        x2 = max(0, min(annotated.width, x2))
        y1 = max(0, min(annotated.height, y1))
        y2 = max(0, min(annotated.height, y2))
        if x2 <= x1 or y2 <= y1:
            continue
        drawer.rectangle((x1, y1, x2, y2), outline="#D82E2E", width=4)
        label = f'{det["class_name"]}: {float(det["confidence"]):.2f}'
        text_y = max(0, y1 - 17)
        drawer.text((x1 + 2, text_y), label, fill="#D82E2E", stroke_width=1, stroke_fill="white")
    return annotated


class RealYoloDetector:
    """Live detector: loads the team's existing trained weights; never substitutes a dummy."""
    def __init__(self, weights, device=None):
        path = Path(weights)
        if not path.is_file():
            raise FileNotFoundError(f"Trained YOLO weights not found: {path}")
        import torch
        from ultralytics import YOLO
        self.weights = str(path)
        self.device = device if device is not None else (0 if torch.cuda.is_available() else "cpu")
        self.model = YOLO(self.weights)

    def predict(self, image):
        rgb = image_rgb(image)
        result = self.model.predict(source=np.asarray(rgb), imgsz=640, conf=0.25,
                                    device=self.device, verbose=False)[0]
        names = result.names
        found = []
        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)
            for xyxy, score, class_id in zip(boxes, scores, classes):
                if isinstance(names, dict):
                    label = names.get(int(class_id), f"class_{class_id}")
                else:
                    label = names[int(class_id)]
                found.append({
                    "class_name": str(label),
                    "class_id": int(class_id),
                    "confidence": round(float(score), 5),
                    "bbox_xyxy": [round(float(v), 2) for v in xyxy],
                })
        return found


def heuristic_priority(detections, rain_context="Unknown", previous_reports=0):
    """Explainable inspection triage, NOT a trained dengue prediction model."""
    if rain_context not in {"Unknown", "Dry", "Recent rain", "Heavy rain"}:
        raise ValueError("Invalid rain context")
    count = max(0, int(previous_reports))
    if not detections:
        return {"score": 0, "level": "UNCONFIRMED", "explanation": "No visible model detection; human inspection may still be needed.",
                "method": "HEURISTIC_INSPECTION_PRIORITY_NOT_DENGUE_FORECAST"}
    max_conf = max(float(d["confidence"]) for d in detections)
    labels = " ".join(str(d["class_name"]).lower() for d in detections)
    bonus = 12 if any(w in labels for w in ("tire", "tyre", "vase", "container", "tank", "drain")) else 6
    rain_bonus = {"Unknown": 0, "Dry": 0, "Recent rain": 8, "Heavy rain": 14}[rain_context]
    history_bonus = min(15, count * 3)
    score = min(100, round(15 + 38 * max_conf + bonus + rain_bonus + history_bonus))
    level = "HIGH" if score >= 70 else "MEDIUM" if score >= 45 else "LOW"
    return {"score": score, "level": level,
            "explanation": f"Detection confidence contribution={round(38 * max_conf)}; site type={bonus}; rain context={rain_bonus}; prior reports={history_bonus}; base=15.",
            "method": "HEURISTIC_INSPECTION_PRIORITY_NOT_DENGUE_FORECAST"}


def recommend_action(detections):
    labels = " ".join(str(d["class_name"]).lower() for d in detections)
    if not detections:
        return "Request a closer human inspection before assigning any intervention."
    if "tire" in labels or "tyre" in labels:
        return "Inspect discarded tyres; safely remove collected water and dispose or store tyres under cover."
    if "drain" in labels:
        return "Inspect drainage; clear safe, accessible blockages under local authority guidance."
    if "tank" in labels:
        return "Inspect water tank; cover properly and refer to authorized workers if needed."
    return "Inspect potential water-holding containers; empty or cover them where safe and authorized."


def alignment_evidence(before, after):
    """Image alignment is only a visual quality indicator, NOT same-site proof."""
    try:
        def gray(im):
            arr = np.asarray(image_rgb(im))
            h, w = arr.shape[:2]
            scale = min(1.0, 960 / max(h, w))
            if scale < 1:
                arr = cv2.resize(arr, (max(32, round(w * scale)), max(32, round(h * scale))))
            return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        a, b = gray(before), gray(after)
        orb = cv2.ORB_create(nfeatures=600)
        k1, d1 = orb.detectAndCompute(a, None)
        k2, d2 = orb.detectAndCompute(b, None)
        if d1 is None or d2 is None or len(k1) < 10 or len(k2) < 10:
            return {"status": "INCONCLUSIVE", "good_matches": 0, "homography_inlier_ratio": None, "same_site_proven": False}
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
        pairs = matcher.knnMatch(d1, d2, k=2)
        good = [m for pair in pairs if len(pair) == 2 for m, n in [pair] if m.distance < 0.75 * n.distance]
        if len(good) < 12:
            return {"status": "INCONCLUSIVE", "good_matches": len(good), "homography_inlier_ratio": None, "same_site_proven": False}
        p1 = np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        p2 = np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        _, mask = cv2.findHomography(p1, p2, cv2.RANSAC, 5.0)
        ratio = round(float(mask.mean()), 3) if mask is not None else None
        status = "GEOMETRIC_MATCH_CANDIDATE" if ratio is not None and ratio >= 0.45 else "INCONCLUSIVE"
        return {"status": status, "good_matches": len(good), "homography_inlier_ratio": ratio, "same_site_proven": False}
    except (cv2.error, ValueError, TypeError):
        return {"status": "INCONCLUSIVE", "good_matches": 0, "homography_inlier_ratio": None, "same_site_proven": False}


class DenguHunterEngine:
    def __init__(self, data_root, f2_detector, f3_detector=None, f2_model_label="EXISTING_TRAINED_F2_YOLO", f3_model_label=None):
        if f2_detector is None:
            raise ValueError("Live F2 detector is mandatory; no fake fallback allowed.")
        self.root = Path(data_root)
        self.cases_dir = self.root / "cases"
        self.audit_dir = self.root / "audits"
        self.cases_dir.mkdir(parents=True, exist_ok=True)
        self.audit_dir.mkdir(parents=True, exist_ok=True)
        self.f2 = f2_detector
        self.f3 = f3_detector
        self.f2_label = f2_model_label
        self.f3_label = f3_model_label
        self.f2_weights_sha256 = sha256_file(f2_detector.weights) if isinstance(f2_detector, RealYoloDetector) else None
        self.f3_weights_sha256 = sha256_file(f3_detector.weights) if isinstance(f3_detector, RealYoloDetector) else None
        self.lock = RLock()

    def _case_path(self, case_id):
        if not CASE_PATTERN.fullmatch(str(case_id or "")):
            raise ValueError("Invalid Case ID. Use the ID shown in the dashboard.")
        return self.cases_dir / case_id / "case.json"

    def _read(self, case_id):
        path = self._case_path(case_id)
        if not path.is_file():
            raise ValueError(f"Unknown Case ID: {case_id}")
        return json.loads(path.read_text(encoding="utf-8"))

    def _save(self, case):
        path = self._case_path(case["case_id"])
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(f".case_{uuid.uuid4().hex}.tmp")
        temporary.write_text(json.dumps(case, indent=2, ensure_ascii=False), encoding="utf-8")
        temporary.replace(path)

    @staticmethod
    def _event(case, action, actor, note=""):
        case["events"].append({"at_utc": utc_now(), "action": action, "actor": str(actor), "note": str(note or "")[:500]})
        case["updated_at_utc"] = utc_now()

    def submit(self, image, source="Ground photo", latitude=None, longitude=None,
               rain_context="Unknown", previous_reports=0, closeup=None, notes="",
               evidence_mode="DEMO_CASE_STAGED"):
        with self.lock:
            if evidence_mode not in {"DEMO_CASE_STAGED", "FIELD_REPORT_UNVERIFIED"}:
                raise ValueError("Choose staged demo or unverified field report explicitly")
            if source not in {"Ground photo", "Aerial photo (user-provided)", "Drone simulation frame"}:
                raise ValueError("Unknown image source")
            before = image_rgb(image)
            previous_reports = int(previous_reports or 0)
            if previous_reports < 0:
                raise ValueError("Prior report count cannot be negative")
            lat, lon = check_gps(latitude, longitude)
            f2_detections = self.f2.predict(before)  # ALWAYS real YOLO in live mode
            if not isinstance(f2_detections, list):
                raise RuntimeError("YOLO returned an unexpected detection format")
            closeup_image = image_rgb(closeup) if closeup is not None else None
            if closeup_image is not None and self.f3 is None:
                raise ValueError("F3 close-up model weights are unavailable; remove the close-up image or restore weights.")
            f3_detections = self.f3.predict(closeup_image) if closeup_image is not None else []
            priority = heuristic_priority(f2_detections, rain_context, previous_reports)
            case_id = "DH-" + uuid.uuid4().hex[:12].upper()
            folder = self._case_path(case_id).parent
            folder.mkdir(parents=True, exist_ok=False)
            before_path = folder / "before.png"
            overlay_path = folder / "before_overlay.png"
            before.save(before_path)
            draw_detections(before, f2_detections).save(overlay_path)
            closeup_path, closeup_sha = None, None
            if closeup_image is not None:
                cp = folder / "larvae_closeup.png"
                closeup_image.save(cp)
                closeup_path, closeup_sha = str(cp), sha256_file(cp)
            case = {
                "case_id": case_id, "created_at_utc": utc_now(), "updated_at_utc": utc_now(),
                "source": source, "evidence_mode": evidence_mode,
                "image_kind": "USER_UPLOADED_REAL_IMAGE", "latitude": lat, "longitude": lon,
                "gps_provided": lat is not None, "gps_verified_by_device": False,
                "rain_context": rain_context, "previous_reports": int(previous_reports),
                "notes": str(notes or "")[:1000],
                "before_path": str(before_path), "before_sha256": sha256_file(before_path),
                "before_overlay_path": str(overlay_path), "before_overlay_sha256": sha256_file(overlay_path),
                "f2_model": self.f2_label, "f2_weights_sha256": self.f2_weights_sha256,
                "f2_detections": f2_detections,
                "f3_model": self.f3_label if closeup_image is not None else None,
                "f3_weights_sha256": self.f3_weights_sha256 if closeup_image is not None else None,
                "f3_detections": f3_detections,
                "f3_closeup_path": closeup_path, "f3_closeup_sha256": closeup_sha,
                "aedes_species_confirmed": False,
                "priority": priority, "recommendation": recommend_action(f2_detections),
                "review_status": "PENDING", "reviewer": None, "reviewed_at_utc": None,
                "task_id": None, "task_status": "NOT_CREATED", "assigned_worker": None,
                "task_created_at_utc": None, "after_path": None, "after_sha256": None,
                "after_overlay_path": None, "after_overlay_sha256": None,
                "after_detections": None, "alignment": None, "ai_change_assessment": None,
                "final_review": "NOT_REVIEWED", "final_reviewer": None,
                "real_drone_flight": False, "real_field_verified": False,
                "verified_cleanup_success": False,
                "stage": "AWAITING_HUMAN_REVIEW", "events": [],
            }
            self._event(case, "CITIZEN_SUBMITTED_REAL_F2_INFERENCE", "citizen", "Human review required before any worker task")
            self._save(case)
            return case, draw_detections(before, f2_detections)

    def review(self, case_id, reviewer, decision, notes=""):
        with self.lock:
            case = self._read(case_id)
            reviewer = str(reviewer or "").strip()
            if not reviewer:
                raise ValueError("Reviewer name is required")
            if decision not in {"APPROVE", "REJECT", "REQUEST_MORE_EVIDENCE"}:
                raise ValueError("Invalid human review decision")
            if case["stage"] != "AWAITING_HUMAN_REVIEW":
                raise ValueError("This case is no longer awaiting initial review")
            case["review_status"] = decision
            case["reviewer"] = reviewer
            case["reviewed_at_utc"] = utc_now()
            if decision == "APPROVE":
                case["task_id"] = "DH-TASK-" + case_id[3:]
                case["task_created_at_utc"] = utc_now()
                case["task_status"] = "READY_TO_ASSIGN"
                case["stage"] = "TASK_READY"
            else:
                case["stage"] = "REJECTED" if decision == "REJECT" else "MORE_EVIDENCE_REQUIRED"
            self._event(case, "HUMAN_REVIEW_" + decision, reviewer, notes)
            self._save(case)
            return case

    def assign(self, case_id, worker, notes=""):
        with self.lock:
            case = self._read(case_id)
            worker = str(worker or "").strip()
            if not worker:
                raise ValueError("Worker name is required")
            if case["stage"] != "TASK_READY" or case["review_status"] != "APPROVE":
                raise ValueError("An approved, unassigned task is required")
            case["assigned_worker"] = worker
            case["task_status"] = "ASSIGNED"
            case["stage"] = "TASK_ASSIGNED"
            self._event(case, "TASK_ASSIGNED", worker, notes)
            self._save(case)
            return case

    def start(self, case_id, worker):
        with self.lock:
            case = self._read(case_id)
            if case["stage"] != "TASK_ASSIGNED" or case["assigned_worker"] != str(worker or "").strip():
                raise ValueError("Only the assigned worker may start an assigned task")
            case["task_status"] = "IN_PROGRESS"
            case["stage"] = "WORK_IN_PROGRESS"
            self._event(case, "WORK_STARTED", worker)
            self._save(case)
            return case

    def submit_after(self, case_id, worker, image, notes="", same_site_attested=False):
        with self.lock:
            case = self._read(case_id)
            if case["stage"] != "WORK_IN_PROGRESS" or case["assigned_worker"] != str(worker or "").strip():
                raise ValueError("Only the assigned worker can submit after-action evidence for a started task")
            if not same_site_attested:
                raise ValueError("Worker must attest that the photo is from the same site (not independently verified).")
            after = image_rgb(image)
            after_detections = self.f2.predict(after)  # Real inference again
            before = Image.open(case["before_path"]).convert("RGB")
            alignment = alignment_evidence(before, after)
            before_count, after_count = len(case["f2_detections"]), len(after_detections)
            if before_count == 0:
                change = "INCONCLUSIVE_NO_ORIGINAL_DETECTION"
            elif alignment["status"] != "GEOMETRIC_MATCH_CANDIDATE":
                change = "INCONCLUSIVE_ALIGNMENT_REVIEW_REQUIRED"
            elif after_count < before_count:
                change = "POSSIBLE_REDUCTION_HUMAN_REVIEW_REQUIRED"
            else:
                change = "POSSIBLE_PERSISTENCE_HUMAN_REVIEW_REQUIRED"
            folder = self._case_path(case_id).parent
            path = folder / "after.png"
            overlay = folder / "after_overlay.png"
            if path.exists():
                raise ValueError("After evidence already exists; do not overwrite saved evidence")
            after.save(path)
            draw_detections(after, after_detections).save(overlay)
            case["after_path"] = str(path)
            case["after_sha256"] = sha256_file(path)
            case["after_overlay_path"] = str(overlay)
            case["after_overlay_sha256"] = sha256_file(overlay)
            case["after_detections"] = after_detections
            case["alignment"] = alignment
            case["ai_change_assessment"] = change
            case["worker_same_site_attestation"] = True
            case["task_status"] = "AFTER_PHOTO_SUBMITTED"
            case["stage"] = "AWAITING_FINAL_HUMAN_REVIEW"
            self._event(case, "AFTER_EVIDENCE_SUBMITTED_REAL_F2_INFERENCE", worker, notes)
            self._save(case)
            return case, draw_detections(after, after_detections)

    def finalize(self, case_id, reviewer, decision, notes=""):
        with self.lock:
            case = self._read(case_id)
            reviewer = str(reviewer or "").strip()
            if not reviewer:
                raise ValueError("Final reviewer name is required")
            if case["stage"] != "AWAITING_FINAL_HUMAN_REVIEW":
                raise ValueError("After-action photo and human review are required")
            if decision not in {"RESOLVED_EVIDENCE_REVIEWED", "UNRESOLVED", "REINSPECTION_REQUIRED"}:
                raise ValueError("Invalid final review decision")
            case["final_review"] = decision
            case["final_reviewer"] = reviewer
            case["task_status"] = decision
            case["stage"] = "CLOSED_EVIDENCE_REVIEWED" if decision == "RESOLVED_EVIDENCE_REVIEWED" else decision
            # Evidence-based human decision is NOT independent field verification.
            # A resolved demo remains a staged demo, not verified cleanup success.
            case["real_field_verified"] = False
            case["verified_cleanup_success"] = False
            self._event(case, "FINAL_HUMAN_REVIEW_" + decision, reviewer, notes)
            self._save(case)
            return case

    def inspect_case(self, case_id):
        """Load the actual saved photos so reviewers can inspect evidence before deciding."""
        with self.lock:
            c = self._read(case_id)
            for path_key, sha_key in (("before_path", "before_sha256"),
                                      ("before_overlay_path", "before_overlay_sha256"),
                                      ("after_path", "after_sha256"),
                                      ("after_overlay_path", "after_overlay_sha256")):
                p = c.get(path_key)
                if p and (not Path(p).is_file() or sha256_file(p) != c[sha_key]):
                    raise ValueError(f"Evidence integrity check failed: {path_key}")
            def open_optional(key):
                return Image.open(c[key]).convert("RGB") if c.get(key) else None
            return (open_optional("before_path"), open_optional("before_overlay_path"),
                    open_optional("after_path"), open_optional("after_overlay_path"),
                    safe_case_summary(c))

    def cases(self):
        with self.lock:
            found = []
            for p in sorted(self.cases_dir.glob("DH-*/case.json")):
                try:
                    obj = json.loads(p.read_text(encoding="utf-8"))
                    if CASE_PATTERN.fullmatch(obj.get("case_id", "")):
                        found.append(obj)
                except (OSError, ValueError):
                    continue
            return sorted(found, key=lambda c: c.get("created_at_utc", ""), reverse=True)

    def table(self):
        rows = []
        for c in self.cases():
            gps = (
    f'{c["latitude"]:.5f}, {c["longitude"]:.5f}'
    if c.get("gps_provided")
    and c.get("latitude") is not None
    and c.get("longitude") is not None
    and (float(c["latitude"]), float(c["longitude"])) != (0.0, 0.0)
    else "NOT PROVIDED"
)
            rows.append({"Case ID": c["case_id"], "Created (UTC)": c["created_at_utc"],
                         "Source": c["source"], "GPS": gps,
                         "Detections": len(c["f2_detections"]), "Risk": c["priority"]["level"],
                         "Priority Score": c["priority"]["score"], "Human Review": c["review_status"],
                         "Task ID": c["task_id"] or "—", "Worker": c["assigned_worker"] or "—",
                         "Stage": c["stage"],
                         "Evidence Mode": c.get("evidence_mode", "LEGACY_UNSPECIFIED"),
                         "Field Verified": "NO"})
        return pd.DataFrame(rows, columns=["Case ID", "Created (UTC)", "Source", "GPS", "Detections", "Risk",
                                           "Priority Score", "Human Review", "Task ID", "Worker", "Stage", "Evidence Mode", "Field Verified"])


    def map_html(self):
        cases = [
            c for c in self.cases()
            if c.get("gps_provided")
            and c.get("latitude") is not None
            and c.get("longitude") is not None
            and (float(c["latitude"]), float(c["longitude"])) != (0.0, 0.0)
        ]

        center = [
            statistics.mean(c["latitude"] for c in cases),
            statistics.mean(c["longitude"] for c in cases)
        ] if cases else [23.685, 90.3563]

        fmap = folium.Map(
            location=center,
            zoom_start=12 if cases else 6,
            control_scale=True,
            tiles="OpenStreetMap"
        )

        colors = {
            "HIGH": "red",
            "MEDIUM": "orange",
            "LOW": "green",
            "UNCONFIRMED": "gray"
        }

        for c in cases:
            description = html.escape(
                f'{c["case_id"]} | {c["priority"]["level"]} | '
                f'{c["stage"]} | {c.get("evidence_mode", "LEGACY_UNSPECIFIED")}'
            )

            folium.Marker(
                [c["latitude"], c["longitude"]],
                popup=description,
                tooltip=html.escape(c["case_id"]),
                icon=folium.Icon(
                    color=colors.get(c["priority"]["level"], "blue"),
                    icon="info-sign"
                )
            ).add_to(fmap)

        return fmap._repr_html_()




    def dashboard(self):

        all_cases = self.cases()
        n = len(all_cases)
        tasks = [c for c in all_cases if c["task_id"]]
        afters = [c for c in tasks if c["after_path"]]

        gps = sum(
            bool(c.get("gps_provided"))
            and c.get("latitude") is not None
            and c.get("longitude") is not None
            and (float(c["latitude"]), float(c["longitude"])) != (0.0, 0.0)
            for c in all_cases
        )

        delays = []
        for c in tasks:
            try:
                d = datetime.fromisoformat(c["task_created_at_utc"]) - datetime.fromisoformat(c["created_at_utc"])
                delays.append(d.total_seconds())
            except (ValueError, TypeError):
                pass
        summary = {
            "citizen_reports": n,
            "staged_demo_cases": sum(c.get("evidence_mode") == "DEMO_CASE_STAGED" for c in all_cases),
            "unverified_field_reports": sum(c.get("evidence_mode") == "FIELD_REPORT_UNVERIFIED" for c in all_cases),
            "verified_cleanup_successes": 0,
            "human_review_pending": sum(c["review_status"] == "PENDING" for c in all_cases),
            "human_approved": sum(c["review_status"] == "APPROVE" for c in all_cases),
            "worker_tasks_created": len(tasks), "after_photos_submitted": len(afters),
            "evidence_reviewed_resolved_not_field_verified": sum(c["final_review"] == "RESOLVED_EVIDENCE_REVIEWED" for c in all_cases),
            "gps_completeness_percent": round(100 * gps / n, 1) if n else None,
            "task_followup_evidence_percent": round(100 * len(afters) / len(tasks), 1) if tasks else None,
            "median_submission_to_task_seconds": round(statistics.median(delays), 1) if delays else None,
            "false_alert_rate": evaluation_status(self),
            "real_field_verification": "NOT PERFORMED", "risk_method": "Explainable rule-based inspection priority; not a trained outbreak forecast",
            "drone_status": "Simulation/available imagery only; no autonomous flight claimed",
        }
        return summary, self.table(), self.map_html()

    def audit(self):
        cases = self.cases()
        findings = []
        for c in cases:
            cid = c["case_id"]
            def check(label, passed):
                findings.append({"case_id": cid, "check": label, "pass": bool(passed)})
            before = Path(c["before_path"])
            check("before_image_sha256", before.is_file() and sha256_file(before) == c["before_sha256"])
            check("before_overlay_sha256", Path(c["before_overlay_path"]).is_file() and sha256_file(c["before_overlay_path"]) == c["before_overlay_sha256"])
            check("real_model_provenance_recorded", bool(c.get("f2_model")) and bool(c.get("f2_weights_sha256"))
                  and "FAKE" not in c["f2_model"].upper() and "TEST" not in c["f2_model"].upper())
            check("gps_pair_consistent", (c["latitude"] is None) == (c["longitude"] is None))
            check("task_requires_human_approval", not c["task_id"] or (c["review_status"] == "APPROVE" and c["reviewed_at_utc"] is not None))
            check("no_auto_aedes_claim", c["aedes_species_confirmed"] is False)
            check("no_real_field_claim", c["real_field_verified"] is False)
            check("no_verified_cleanup_success_claim", c.get("verified_cleanup_success",False) is False)
            check("explicit_demo_or_field_label", c.get("evidence_mode") in {"DEMO_CASE_STAGED","FIELD_REPORT_UNVERIFIED"})
            if c["after_path"]:
                check("after_image_sha256", Path(c["after_path"]).is_file() and sha256_file(c["after_path"]) == c["after_sha256"])
                check("after_overlay_sha256", Path(c["after_overlay_path"]).is_file() and sha256_file(c["after_overlay_path"]) == c["after_overlay_sha256"])
                check("after_requires_task_and_worker", bool(c["task_id"] and c["assigned_worker"]))
                check("after_never_auto_resolves", c["ai_change_assessment"] != "RESOLVED")
        passed = sum(f["pass"] for f in findings)
        report = {"generated_at_utc": utc_now(), "case_count": len(cases),
                  "checks_passed": passed, "checks_total": len(findings),
                  "audit_status": "PASS" if findings and passed == len(findings) else "NEEDS_LIVE_CASES_OR_REVIEW",
                  "checks": findings,
                  "proposal_coverage_percent": None,
                  "notes": "PASS checks data integrity and workflow logic; does not prove dengue reduction, real cleanup or full proposal coverage."}
        path = self.audit_dir / f'Audit_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:6]}.json'
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        return report, str(path)


def safe_case_summary(case):
    return {k: case.get(k) for k in (
        "case_id", "stage", "source", "evidence_mode", "verified_cleanup_success", "gps_provided", "latitude", "longitude", "priority",
        "review_status", "task_id", "task_status", "assigned_worker", "recommendation",
        "f2_model", "f2_detections", "f3_model", "f3_detections", "ai_change_assessment",
        "alignment", "final_review", "real_field_verified", "aedes_species_confirmed")}



# ============================================================
# F1 — TRANSPARENT DRONE-ROUTE SIMULATION (NO ACTUAL FLIGHT)
# ============================================================
def _distance_xy(a, b):
    return math.hypot(float(a[0])-float(b[0]), float(a[1])-float(b[1]))


def _mission_xy_html(rows, center_available=False):
    """Offline XY visualization if GPS was not supplied; all points are SIMULATED."""
    width, height, pad = 760, 410, 45
    xs = [float(r['x_m']) for r in rows] + [0.0]
    ys = [float(r['y_m']) for r in rows] + [0.0]
    lo_x, hi_x, lo_y, hi_y = min(xs), max(xs), min(ys), max(ys)
    span_x, span_y = max(1.0, hi_x-lo_x), max(1.0, hi_y-lo_y)
    def px(x): return pad+(float(x)-lo_x)/span_x*(width-2*pad)
    def py(y): return height-pad-(float(y)-lo_y)/span_y*(height-2*pad)
    colors=['#2563eb','#d97706','#16a34a','#7c3aed']
    lines=[]
    groups={}
    for r in rows:
        groups.setdefault((r['drone'],r['sortie']),[]).append(r)
    for (drone,sortie), stops in groups.items():
        points=[(0,0)]+[(s['x_m'],s['y_m']) for s in stops]+[(0,0)]
        coord=' '.join(f'{px(x):.1f},{py(y):.1f}' for x,y in points)
        color=colors[(int(drone.split('-')[-1])-1)%len(colors)]
        lines.append(f'<polyline points="{coord}" fill="none" stroke="{color}" stroke-width="2.5" opacity="0.75" />')
    circles=[]
    for r in rows:
        color=colors[(int(r['drone'].split('-')[-1])-1)%len(colors)]
        circles.append(f'<circle cx="{px(r["x_m"]):.1f}" cy="{py(r["y_m"]):.1f}" r="5" fill="{color}" />')
        circles.append(f'<text x="{px(r["x_m"])+6:.1f}" y="{py(r["y_m"])-6:.1f}" font-size="10" fill="#374151">{html.escape(r["waypoint_id"])}</text>')
    circles.append(f'<circle cx="{px(0):.1f}" cy="{py(0):.1f}" r="8" fill="#dc2626" />')
    return (f'<div style="font:14px sans-serif;padding:10px;border:1px solid #aaa">'
            '<b>SIMULATED MISSION — NO REAL FLIGHT OR CAPTURED PHOTOS</b>'
            f'<svg viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="Simulated XY drone routes">'
            + ''.join(lines+circles) + '</svg><p>Local XY in metres; origin is a simulated base. No GPS coordinates were invented.</p></div>')


def plan_simulated_drone_mission(data_root, center_lat=None, center_lon=None, rows=3, cols=4,
                                  spacing_m=30, drones=2, speed_mps=5, battery_minutes=12,
                                  dwell_seconds=12, manual_revisit_ids=''):
    """Deterministic battery-constrained planned waypoints, NOT a real autonomous flight.

    Photo capture and suspected-site detection are NEVER simulated as observed facts.
    Optional manually flagged waypoints are revisit plans, not AI findings.
    """
    lat, lon = check_gps(center_lat, center_lon)
    rows, cols, drones = int(rows), int(cols), int(drones)
    spacing_m, speed_mps, battery_minutes, dwell_seconds = map(float, (spacing_m,speed_mps,battery_minutes,dwell_seconds))
    if not (2 <= rows <= 12 and 2 <= cols <= 12 and rows*cols <= 100):
        raise ValueError('Choose 2–12 rows and columns, with at most 100 planned waypoints.')
    if not (1 <= drones <= 4 and 5 <= spacing_m <= 300 and 1 <= speed_mps <= 20
            and 3 <= battery_minutes <= 60 and 0 <= dwell_seconds <= 120):
        raise ValueError('Invalid drone/spacing/speed/battery/dwell values.')
    if lat is not None and abs(lat) >= 89:
        raise ValueError('Simulated GPS grid is not supported near the poles.')
    base_waypoints=[]
    for r in range(rows):
        indices=range(cols) if r%2==0 else range(cols-1,-1,-1)
        for c in indices:
            base_waypoints.append({'waypoint_id':f'WP-{len(base_waypoints)+1:03d}',
                                   'x_m':round((c-(cols-1)/2)*spacing_m,2),
                                   'y_m':round(((rows-1)/2-r)*spacing_m,2),
                                   'visit_type':'PLANNED_GRID_SCAN'})
    lookup={w['waypoint_id']:w for w in base_waypoints}
    selected=[v.strip().upper() for v in str(manual_revisit_ids or '').split(',') if v.strip()]
    if len(selected)!=len(set(selected)) or any(v not in lookup for v in selected):
        raise ValueError('Manual revisit IDs must be unique existing IDs such as WP-001,WP-003.')
    all_visits=base_waypoints + [dict(lookup[k],visit_type='MANUALLY_REQUESTED_REINSPECTION') for k in selected]
    budget_seconds=battery_minutes*60*0.8  # 20% battery reserve
    drone_state=[{'last':(0.,0.),'used':0.,'sortie':1,'sequence':0} for _ in range(drones)]
    records=[]
    for i, wp in enumerate(all_visits):
        drone_index=i%drones
        state=drone_state[drone_index]
        point=(wp['x_m'],wp['y_m'])
        single_cost=(_distance_xy((0,0),point)*2/speed_mps)+dwell_seconds
        if single_cost>budget_seconds+1e-8:
            raise ValueError(f'{wp["waypoint_id"]} exceeds 80% battery budget even on its own. Increase battery or speed, or reduce grid spacing.')
        leg=_distance_xy(state['last'],point)/speed_mps+dwell_seconds
        return_seconds=_distance_xy(point,(0,0))/speed_mps
        if state['used']+leg+return_seconds>budget_seconds+1e-8:
            state['sortie']+=1
            state['last']=(0.,0.)
            state['used']=0.
            leg=_distance_xy((0,0),point)/speed_mps+dwell_seconds
        state['used']+=leg
        state['last']=point
        state['sequence']+=1
        if lat is None:
            gps_lat, gps_lon=None,None
        else:
            gps_lat=round(lat+wp['y_m']/111320.0,7)
            gps_lon=round(lon+wp['x_m']/(111320.0*math.cos(math.radians(lat))),7)
        records.append({**wp,'drone':f'DRONE-{drone_index+1}',
                        'sortie':state['sortie'],'sequence':state['sequence'],
                        'latitude':gps_lat,'longitude':gps_lon,
                        'coordinate_source':'DERIVED_SIMULATED_FROM_USER_CENTER' if lat is not None else 'SIMULATED_LOCAL_XY_ONLY',
                        'photo_status':'NOT_CAPTURED_SIMULATION',
                        'real_flight':False,'ai_suspicious_site_confirmed':False})
    sorties=[]
    for (drone,sortie) in sorted({(r['drone'],r['sortie']) for r in records}):
        stops=[r for r in records if r['drone']==drone and r['sortie']==sortie]
        pts=[(0.,0.)]+[(r['x_m'],r['y_m']) for r in stops]+[(0.,0.)]
        distance=sum(_distance_xy(a,b) for a,b in zip(pts,pts[1:]))
        seconds=distance/speed_mps+len(stops)*dwell_seconds
        assert seconds<=budget_seconds+1e-7
        sorties.append({'drone':drone,'sortie':sortie,'waypoints':len(stops),
                        'planned_distance_m':round(distance,1),
                        'planned_duration_s':round(seconds,1),
                        'battery_budget_s_including_20pct_reserve':round(budget_seconds,1)})
    summary={'scope':'PLANNED_SIMULATION_ONLY','real_drone_flights':0,'real_photos_captured':0,
             'waypoint_visits':len(records),'unique_waypoints':len(base_waypoints),
             'manually_flagged_revisits':len(selected),'drones':drones,'sorties':len(sorties),
             'battery_reserve_percent':20,'user_supplied_gps_center':lat is not None,
             'gps_coordinates_are_simulated':lat is not None,
             'not_real_autonomous_capture':True,
             'sortie_details':sorties,'created_at_utc':utc_now()}
    mission_root=Path(data_root)/'drone_simulations'/('Mission_'+datetime.now(UTC).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6])
    mission_root.mkdir(parents=True,exist_ok=False)
    csv_path=mission_root/'simulated_waypoints.csv'
    pd.DataFrame(records).to_csv(csv_path,index=False)
    json_path=mission_root/'mission_summary.json'
    json_path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
    if lat is None:
        visual=_mission_xy_html(records)
    else:
        m=folium.Map(location=[lat,lon],zoom_start=17,tiles='OpenStreetMap')
        folium.Marker([lat,lon],tooltip='USER-ENTERED SIMULATED BASE',icon=folium.Icon(color='red')).add_to(m)
        palette=['blue','orange','green','purple']
        for (drone,sortie) in sorted({(r['drone'],r['sortie']) for r in records}):
            stops=[r for r in records if r['drone']==drone and r['sortie']==sortie]
            path=[[lat,lon]]+[[r['latitude'],r['longitude']] for r in stops]+[[lat,lon]]
            color=palette[(int(drone.split('-')[-1])-1)%len(palette)]
            folium.PolyLine(path,color=color,weight=3,tooltip=f'{drone} sortie {sortie} SIMULATED').add_to(m)
            for r in stops:
                folium.CircleMarker([r['latitude'],r['longitude']],radius=5,color=color,fill=True,
                    tooltip=f'{r["waypoint_id"]} — {r["visit_type"]} — SIMULATED').add_to(m)
        visual='<b>SIMULATED GPS ROUTE — no drone telemetry or photo capture</b>'+m._repr_html_()
    return summary,pd.DataFrame(records),visual,str(csv_path)


def legacy_mission_evidence(legacy_root):
    """Read-only display of actual saved F5 simulation, if present; never writes to source."""
    root=Path(legacy_root)
    paths=sorted(root.glob('Mission_Planner/F5_Optimized_*/F5_Optimized_Mission_Assignments.csv'))
    if not paths:
        return {'status':'NOT FOUND','note':'Use the simulator above; no historic F5 file was found.'},pd.DataFrame()
    p=paths[-1]
    df=pd.read_csv(p)
    return {'status':'READ_ONLY_HISTORIC_SIMULATION','source':str(p),'rows':len(df),
            'no_real_drone_flight_claim':True},df.head(50)


# ============================================================
# EVALUATION — REAL YOLO INFERENCE + HUMAN-LABELLED TEST DATA
# ============================================================
def _safe_eval_image_path(value, image_root):
    root=Path(image_root).expanduser().resolve()
    candidate=Path(str(value).strip()).expanduser()
    if not candidate.is_absolute():
        candidate=root/candidate
    candidate=candidate.resolve()
    if not candidate.is_relative_to(root):
        raise ValueError(f'Image path must stay inside the selected image root: {value}')
    if not candidate.is_file():
        raise FileNotFoundError(f'Labelled evaluation image missing: {candidate}')
    if candidate.suffix.lower() not in {'.jpg','.jpeg','.png','.webp','.bmp'}:
        raise ValueError('Unsupported evaluation image type: '+candidate.suffix)
    return candidate


def evaluate_labelled_image_manifest(engine, manifest_csv, image_root, max_images=300, allowed_drive_root=None):
    """Evaluate *image-level* alerts against independent 0/1 human ground truth.

    Metrics are image-level, NOT bounding-box mAP or species confirmation.
    False alert among flagged images = FP/(TP+FP); false-positive rate = FP/(FP+TN).
    """
    if allowed_drive_root is not None and not Path(image_root).expanduser().resolve().is_relative_to(Path(allowed_drive_root).resolve()):
        raise ValueError('Evaluation image root must be inside the mounted Google Drive.')
    if not manifest_csv:
        raise ValueError('Upload a human-labelled CSV with image_path and ground_truth_present (0 or 1).')
    path=Path(manifest_csv)
    if not path.is_file() or path.suffix.lower()!='.csv':
        raise ValueError('A CSV file is required.')
    if path.stat().st_size>2_000_000:
        raise ValueError('CSV exceeds 2 MB.')
    df=pd.read_csv(path,dtype={'image_path':'string'})
    required={'image_path','ground_truth_present'}
    if not required.issubset(df.columns):
        raise ValueError('CSV must contain image_path,ground_truth_present columns.')
    if not (1<=len(df)<=int(max_images)):
        raise ValueError(f'CSV must contain 1–{max_images} images.')
    labels=[]
    paths=[]
    for _,row in df.iterrows():
        label=str(row['ground_truth_present']).strip()
        if label not in {'0','1'}:
            raise ValueError('ground_truth_present must be exactly 0 or 1 for every row.')
        labels.append(int(label))
        paths.append(_safe_eval_image_path(row['image_path'],image_root))
    if len(set(paths))!=len(paths):
        raise ValueError('Duplicate images in evaluation manifest are not allowed.')
    observations=[]
    for imgpath,actual in zip(paths,labels):
        with Image.open(imgpath) as img:
            predictions=engine.f2.predict(image_rgb(img))  # REAL model on each image
        flagged=bool(predictions)
        observations.append({'image_path':str(imgpath),'ground_truth_present':actual,
                             'ai_alert':int(flagged),'detections':len(predictions),
                             'max_confidence':max([float(x['confidence']) for x in predictions],default=None),
                             'prediction':'TP' if actual and flagged else 'TN' if not actual and not flagged
                             else 'FP' if flagged else 'FN'})
    counts={key:sum(x['prediction']==key for x in observations) for key in ('TP','FP','TN','FN')}
    tp,fp,tn,fn=(counts[k] for k in ('TP','FP','TN','FN'))
    def safe_ratio(a,b): return round(a/b,5) if b else None
    summary={'status':'EVALUATED_FROM_HUMAN_LABELLED_IMAGES','evaluated_at_utc':utc_now(),
             'sample_size':len(observations),'positive_ground_truth':sum(labels),
             'negative_ground_truth':len(labels)-sum(labels),**counts,
             'image_level_precision':safe_ratio(tp,tp+fp),
             'image_level_recall':safe_ratio(tp,tp+fn),
             'false_alert_fraction_among_alerts':safe_ratio(fp,tp+fp),
             'false_positive_rate_among_negative_images':safe_ratio(fp,fp+tn),
             'definitions':{'false_alert_fraction_among_alerts':'FP / (TP + FP)',
                            'false_positive_rate_among_negative_images':'FP / (FP + TN)',
                            'image_level_precision':'TP / (TP + FP)',
                            'image_level_recall':'TP / (TP + FN)'},
             'model_label':engine.f2_label,'model_sha256':engine.f2_weights_sha256,
             'manifest_sha256':sha256_file(path),
             'scope':'IMAGE_LEVEL_ONLY_NOT_BOX_PRECISION_RECALL',
             'ground_truth_source':'USER_SUPPLIED_LABELS_NOT_INDEPENDENTLY_VERIFIED',
             'warning':'No negative images means false-alert rate cannot be measured. Use held-out, human-labelled data.'}
    folder=engine.root/'evaluations'/('Eval_'+datetime.now(UTC).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6])
    folder.mkdir(parents=True,exist_ok=False)
    pd.DataFrame(observations).to_csv(folder/'per_image_predictions.csv',index=False)
    (folder/'image_level_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    return summary,pd.DataFrame(observations),str(folder/'image_level_metrics.json')


def validate_yolo_boxes(engine, dataset_yaml, split='test', drive_root=None):
    """Optional official Ultralytics box metrics on a held-out annotated dataset."""
    if not isinstance(engine.f2,RealYoloDetector):
        raise ValueError('Official YOLO validation requires the actual trained model.')
    yaml_path=Path(str(dataset_yaml or '').strip()).expanduser().resolve()
    if drive_root is not None and not yaml_path.is_relative_to(Path(drive_root).resolve()):
        raise ValueError('Dataset YAML must be under the mounted Google Drive.')
    if not yaml_path.is_file() or yaml_path.suffix.lower() not in {'.yaml','.yml'}:
        raise ValueError('Enter an existing labelled dataset .yaml path from Google Drive.')
    if split not in {'val','test'}:
        raise ValueError('Split must be val or test.')
    import yaml
    dataset=yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
    if not isinstance(dataset,dict) or not dataset.get(split):
        raise ValueError(f'Dataset YAML does not define a {split} split.')
    output=engine.root/'evaluations'/'yolo_validation'
    output.mkdir(parents=True,exist_ok=True)
    result=engine.f2.model.val(data=str(yaml_path),split=split,imgsz=640,
                               device=engine.f2.device,
                               plots=False,save_json=False,project=str(output),
                               name='Validation_'+uuid.uuid4().hex[:8],verbose=False)
    box=getattr(result,'box',None)
    if box is None:
        raise RuntimeError('Model validation returned no bounding-box metrics.')
    def value(attr):
        x=getattr(box,attr,None)
        return round(float(x),5) if x is not None else None
    summary={'status':'EVALUATED_FROM_LABELLED_YOLO_SPLIT','split':split,
             'dataset_yaml':str(yaml_path),'dataset_yaml_sha256':sha256_file(yaml_path),
             'evaluated_at_utc':utc_now(),'box_precision':value('mp'),
             'box_recall':value('mr'),'map50':value('map50'),'map50_95':value('map'),
             'model_sha256':engine.f2_weights_sha256,
             'false_alert_rate':'NOT DERIVED FROM YOLO mAP; use labelled image CSV with negative images',
             'note':'Label quality and true held-out split must be independently confirmed.'}
    report_path=output/('YOLO_Box_Metrics_'+uuid.uuid4().hex[:8]+'.json')
    report_path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
    return summary,str(report_path)


def evaluation_status(engine):
    files=sorted((engine.root/'evaluations').glob('Eval_*/image_level_metrics.json'))
    if not files:
        return {'status':'NOT EVALUATED','false_alert_rate':None,
                'note':'Upload independent human-labelled positive AND negative images.'}
    result=json.loads(files[-1].read_text(encoding='utf-8'))
    return {'status':result['status'],'sample_size':result['sample_size'],
            'false_alert_fraction_among_alerts':result['false_alert_fraction_among_alerts'],
            'false_positive_rate_among_negative_images':result['false_positive_rate_among_negative_images'],
            'image_level_precision':result['image_level_precision'],
            'image_level_recall':result['image_level_recall'],
            'report_path':str(files[-1])}


def build_gradio_app(engine, legacy_root=None):
    import gradio as gr
    legacy_root = Path(legacy_root) if legacy_root else None

    def submit_handler(image, source, lat, lon, rain, history, closeup, notes, evidence_mode):
        case, overlay = engine.submit(image, source, lat, lon, rain, history, closeup, notes,
                                      evidence_mode=evidence_mode)
        return overlay, safe_case_summary(case), (f'Created **{case["case_id"]}** — **{evidence_mode}**. '
            'Next: human review. No worker task until approval; no verified cleanup claim.')

    def inspect_handler(case_id):
        return engine.inspect_case(case_id)

    def review_handler(case_id, reviewer, decision, notes):
        c = engine.review(case_id, reviewer, decision, notes)
        return safe_case_summary(c)

    def assign_handler(case_id, worker, notes):
        return safe_case_summary(engine.assign(case_id, worker, notes))

    def start_handler(case_id, worker):
        return safe_case_summary(engine.start(case_id, worker))

    def after_handler(case_id, worker, image, notes, attested):
        c, overlay = engine.submit_after(case_id, worker, image, notes, attested)
        return overlay, safe_case_summary(c)

    def finalize_handler(case_id, reviewer, decision, notes):
        return safe_case_summary(engine.finalize(case_id, reviewer, decision, notes))

    def refresh_handler():
        summary, table, map_frame = engine.dashboard()
        return summary, table, map_frame

    def audit_handler():
        report, path = engine.audit()
        return report, path

    def mission_handler(lat, lon, rows, cols, spacing, drones, speed, battery, dwell, revisits):
        return plan_simulated_drone_mission(engine.root, lat, lon, rows, cols, spacing, drones, speed, battery, dwell, revisits)

    def legacy_mission_handler():
        return legacy_mission_evidence(legacy_root) if legacy_root else ({'status':'NO_LEGACY_ROOT'},pd.DataFrame())

    def eval_handler(manifest, image_root):
        return evaluate_labelled_image_manifest(engine,manifest,image_root,allowed_drive_root=legacy_root.parent if legacy_root else None)

    def box_eval_handler(yaml_path,split):
        return validate_yolo_boxes(engine,yaml_path,split,drive_root=legacy_root.parent if legacy_root else None)

    def eval_status_handler():
        return evaluation_status(engine)

    legacy_note = ""
    if legacy_root and legacy_root.is_dir():
        registry = sorted(legacy_root.glob("Feature_8_Community_Government/F8_Unified_Data/Integration_*/F8_Unified_Task_Registry.csv"))
        mission = sorted(legacy_root.glob("Mission_Planner/F5_Optimized_*/F5_Optimized_Mission_Assignments.csv"))
        legacy_note = f"Existing read-only evidence: F8 registry {'found' if registry else 'not found'}; F5 simulation {'found' if mission else 'not found'}. These are not new live cases."

    with gr.Blocks(title="DENGU HUNTER | Unified Prototype") as demo:
        gr.Markdown("# DENGU HUNTER — Unified Hackathon Prototype\n**Real F2 YOLO inference • Human-reviewed tasks • GPS map • Before/after evidence**")
        gr.Markdown("**Boundaries:** Inspection-priority rules, **not dengue outbreak prediction**. Close-up detections do **not** confirm Aedes species. **DEMO_CASE_STAGED** and **FIELD_REPORT_UNVERIFIED** are explicitly labelled; no verified cleanup success or autonomous flight. Use non-sensitive imagery/GPS for public demo.")
        if legacy_note:
            gr.Markdown(legacy_note)

        with gr.Tab("1. Citizen + Real AI"):
            with gr.Row():
                with gr.Column():
                    photo = gr.Image(label="Original ground/aerial image", type="pil")
                    closeup = gr.Image(label="Optional larvae close-up (requires existing F3 model)", type="pil")
                    source = gr.Dropdown(["Ground photo", "Aerial photo (user-provided)", "Drone simulation frame"], value="Ground photo", label="Source")
                    with gr.Row():
                        lat = gr.Number(label="Latitude (optional)", value=None)
                        lon = gr.Number(label="Longitude (optional)", value=None)
                    rain = gr.Dropdown(["Unknown", "Dry", "Recent rain", "Heavy rain"], value="Unknown", label="Rain context (user-provided)")
                    history = gr.Number(label="Prior site reports (user-provided)", value=0, precision=0, minimum=0)
                    notes = gr.Textbox(label="Notes (avoid personal data)", lines=2)
                    evidence_mode = gr.Dropdown(
                        ["DEMO_CASE_STAGED", "FIELD_REPORT_UNVERIFIED"],
                        value="DEMO_CASE_STAGED",
                        label="Evidence mode — explicitly distinguish a staged demo from an unverified field report")
                    submit = gr.Button("Run REAL YOLO and submit for review", variant="primary")
                with gr.Column():
                    annotated = gr.Image(label="Real AI bounding boxes", type="pil", interactive=False)
                    submission = gr.JSON(label="Saved case + AI evidence")
                    submitted_msg = gr.Markdown()
            submit.click(submit_handler, [photo, source, lat, lon, rain, history, closeup, notes, evidence_mode],
                         [annotated, submission, submitted_msg], api_name="submit_report")

        with gr.Tab("2. Human Review"):
            gr.Markdown("All new citizen reports require a named human decision before a task is created. Approving a potential site does **not** confirm Aedes larvae.")
            review_id = gr.Textbox(label="Case ID (copy from Citizen tab / Dashboard)")
            inspect_btn = gr.Button("Load original + AI evidence before reviewing")
            with gr.Row():
                review_original = gr.Image(label="Original evidence", type="pil", interactive=False)
                review_overlay = gr.Image(label="Real YOLO detections", type="pil", interactive=False)
            review_details = gr.JSON(label="Case evidence and priority")
            inspect_btn.click(lambda cid: (lambda a,b,c,d,e: (a,b,e))(*inspect_handler(cid)), [review_id],
                              [review_original, review_overlay, review_details], api_name="inspect_case")
            reviewer = gr.Textbox(label="Reviewer name")
            decision = gr.Dropdown(["APPROVE", "REJECT", "REQUEST_MORE_EVIDENCE"], value="APPROVE", label="Decision")
            review_notes = gr.Textbox(label="Review notes")
            review_btn = gr.Button("Save human decision", variant="primary")
            review_result = gr.JSON(label="Case status / task creation")
            review_btn.click(review_handler, [review_id, reviewer, decision, review_notes], review_result, api_name="review_case")

        with gr.Tab("3. Worker Task"):
            worker_id = gr.Textbox(label="Approved Case ID")
            worker_name = gr.Textbox(label="Worker name")
            worker_notes = gr.Textbox(label="Assignment notes")
            with gr.Row():
                assign_btn = gr.Button("Assign worker")
                start_btn = gr.Button("Start task")
            worker_result = gr.JSON(label="Worker task status")
            assign_btn.click(assign_handler, [worker_id, worker_name, worker_notes], worker_result, api_name="assign_worker")
            start_btn.click(start_handler, [worker_id, worker_name], worker_result, api_name="start_task")

        with gr.Tab("4. After Photo + Verification"):
            after_id = gr.Textbox(label="Started Case ID")
            after_worker = gr.Textbox(label="Assigned worker name")
            after_photo = gr.Image(label="Real after-action photo", type="pil")
            after_notes = gr.Textbox(label="Work notes")
            attestation = gr.Checkbox(label="I confirm this is intended to show the same site (worker statement, NOT independent proof)")
            after_btn = gr.Button("Run REAL YOLO on after photo and save evidence", variant="primary")
            after_overlay = gr.Image(label="After image: real YOLO boxes", type="pil", interactive=False)
            after_result = gr.JSON(label="Alignment, SHA-256 evidence and tentative AI comparison")
            after_btn.click(after_handler, [after_id, after_worker, after_photo, after_notes, attestation],
                            [after_overlay, after_result], api_name="submit_after_photo")
            gr.Markdown("### Final human evidence review (never automatic resolution). Staged demo decisions are NOT independently verified physical cleanup; legacy entries missing after photos/human approval remain pending.")
            final_id = gr.Textbox(label="Case ID for final review")
            final_inspect_btn = gr.Button("Load before / after evidence for human comparison")
            with gr.Row():
                final_before = gr.Image(label="Before photo", type="pil", interactive=False)
                final_after = gr.Image(label="After photo", type="pil", interactive=False)
            final_details = gr.JSON(label="AI tentative comparison + alignment (not proof)")
            final_inspect_btn.click(lambda cid: (lambda a,b,c,d,e: (a,c,e))(*inspect_handler(cid)), [final_id],
                                    [final_before, final_after, final_details], api_name="inspect_after_evidence")
            final_reviewer = gr.Textbox(label="Final reviewer")
            final_decision = gr.Dropdown(["RESOLVED_EVIDENCE_REVIEWED", "UNRESOLVED", "REINSPECTION_REQUIRED"],
                                         value="REINSPECTION_REQUIRED", label="Evidence-based decision")
            final_notes = gr.Textbox(label="Why this decision?")
            final_btn = gr.Button("Save final human evidence review")
            final_result = gr.JSON(label="Final case status (not independent field verification)")
            final_btn.click(finalize_handler, [final_id, final_reviewer, final_decision, final_notes],
                            final_result, api_name="finalize_case")

        with gr.Tab("5. GPS Map + Government Dashboard"):
            refresh_btn = gr.Button("Refresh map and dashboard", variant="primary")
            stats = gr.JSON(label="Measured prototype workflow metrics")
            cases_table = gr.Dataframe(label="Unified citizen-to-worker cases", interactive=False, wrap=True)
            gps_map = gr.HTML(label="Interactive GPS map — no fabricated pins")
            refresh_btn.click(refresh_handler, [], [stats, cases_table, gps_map], api_name="get_dashboard")
            demo.load(refresh_handler, [], [stats, cases_table, gps_map])
            gr.Markdown("Map markers appear only when **both real user-provided GPS coordinates** exist. No GPS = no pin. Coordinates are not independently authenticated.")

        with gr.Tab("6. Drone Route Simulation"):
            gr.Markdown("### F1 — Battery-aware planned routes, **SIMULATION ONLY**\n"
                        "No real drone flights, telemetry, photographs or autonomous suspicious-site findings are claimed. "
                        "Optional GPS centre is entered by you; all generated waypoint GPS is **simulated**, not measured. "
                        "Use a real aerial photo in the Citizen tab to run YOLO on it.")
            with gr.Row():
                with gr.Column():
                    with gr.Row():
                        mission_lat=gr.Number(label="Optional user-supplied base latitude",value=None)
                        mission_lon=gr.Number(label="Optional user-supplied base longitude",value=None)
                    with gr.Row():
                        mission_rows=gr.Slider(2,8,value=3,step=1,label="Grid rows")
                        mission_cols=gr.Slider(2,8,value=4,step=1,label="Grid columns")
                    mission_spacing=gr.Slider(10,100,value=30,step=5,label="Waypoint spacing (metres)")
                    mission_drones=gr.Slider(1,4,value=2,step=1,label="Simulated drones")
                    mission_speed=gr.Slider(1,12,value=5,step=1,label="Assumed speed (m/s)")
                    mission_battery=gr.Slider(3,30,value=12,step=1,label="Battery duration (minutes); 20% reserve")
                    mission_dwell=gr.Slider(0,60,value=12,step=1,label="Planned camera dwell per waypoint (seconds)")
                    mission_revisit=gr.Textbox(label="Optional MANUALLY flagged revisit IDs, e.g. WP-001,WP-003",value="")
                    mission_btn=gr.Button("Simulate planned mission (NO REAL FLIGHT)",variant="primary")
                with gr.Column():
                    mission_summary=gr.JSON(label="Honest mission summary")
                    mission_csv=gr.Textbox(label="Saved simulated route CSV (Drive)",interactive=False)
            mission_table=gr.Dataframe(label="Planned waypoints (not real captures)",interactive=False)
            mission_visual=gr.HTML(label="Simulated route map / local XY")
            mission_btn.click(mission_handler,
                [mission_lat,mission_lon,mission_rows,mission_cols,mission_spacing,
                 mission_drones,mission_speed,mission_battery,mission_dwell,mission_revisit],
                [mission_summary,mission_table,mission_visual,mission_csv],api_name="simulate_drone_mission")
            gr.Markdown("### Earlier F5 drone-planning evidence (read-only)")
            legacy_btn=gr.Button("Load saved mission assignments, if available")
            legacy_status=gr.JSON(label="Existing F5 evidence status")
            legacy_table=gr.Dataframe(label="Existing simulated mission assignments",interactive=False)
            legacy_btn.click(legacy_mission_handler,[],[legacy_status,legacy_table],api_name="get_legacy_mission")

        with gr.Tab("7. Detection Evaluation"):
            gr.Markdown("### F2 — Evaluate **real** YOLO predictions on independently human-labelled images\n"
                        "No fabricated metrics. For **image-level** precision/recall and false-alert rates, "
                        "upload a CSV with columns `image_path,ground_truth_present` (labels 0/1). "
                        "Use held-out photos, including **negative** images without target breeding-site objects. "
                        "Paths are relative to the image root folder on Drive, or absolute within it. "
                        "Without labelled data, metrics remain **NOT EVALUATED**.")
            gr.Textbox(value="image_path,ground_truth_present\nEvaluation/images/positive_01.jpg,1\nEvaluation/images/negative_01.jpg,0",
                    lines=3,label="CSV format example (NOT real data)",interactive=False)
            eval_manifest=gr.File(label="Upload YOUR human-labelled evaluation CSV",file_types=[".csv"],type="filepath")
            eval_image_root=gr.Textbox(label="Image root directory on mounted Google Drive",
                    value=str(legacy_root.parent if legacy_root else '/content/drive/MyDrive'))
            eval_btn=gr.Button("Run REAL YOLO on labelled images (no fake scores)",variant="primary")
            eval_result=gr.JSON(label="Measured image-level confusion matrix, false alerts, precision, recall")
            eval_table=gr.Dataframe(label="Per-image real predictions",interactive=False)
            eval_path=gr.Textbox(label="Saved evaluation JSON (Drive)",interactive=False)
            eval_btn.click(eval_handler,[eval_manifest,eval_image_root],[eval_result,eval_table,eval_path],
                           api_name="evaluate_labelled_images")
            gr.Markdown("### Optional: YOLO bounding-box precision, recall and mAP on annotated val/test split")
            yaml_path=gr.Textbox(label="Existing dataset.yaml path on Drive (with val/test annotations)",value="")
            split_choice=gr.Dropdown(["test","val"],value="test",label="True held-out split")
            box_btn=gr.Button("Run official YOLO box validation (optional)")
            box_result=gr.JSON(label="YOLO box precision / recall / mAP")
            box_path=gr.Textbox(label="Saved YOLO evaluation JSON",interactive=False)
            box_btn.click(box_eval_handler,[yaml_path,split_choice],[box_result,box_path],api_name="evaluate_yolo_boxes")
            eval_status_btn=gr.Button("Show latest real evaluation status")
            eval_latest=gr.JSON(label="Most recent labelled evaluation, if any")
            eval_status_btn.click(eval_status_handler,[],eval_latest,api_name="get_evaluation_status")
            gr.Markdown("**Metric definitions:** False-alert fraction = FP/(TP+FP); false-positive rate = FP/(FP+TN). "
                        "These are different. A positive/negative *image-level* evaluation is not bounding-box mAP. "
                        "This software evaluation does not demonstrate fewer dengue cases.")

        with gr.Tab("8. Evidence Audit"):
            audit_btn = gr.Button("Run integrity and workflow audit")
            audit_result = gr.JSON(label="Integrity checks; NOT a proposal-coverage percentage")
            audit_file = gr.Textbox(label="Saved audit JSON path", interactive=False)
            audit_btn.click(audit_handler, [], [audit_result, audit_file], api_name="run_audit")
            gr.Markdown("**Important:** Automated test PASS never means 100% proposal completion, verified real cleanup, or improved public-health outcomes.")
    return demo


# ============================================================
# FASTAPI BACKEND — same engine and persisted case workflow.
# Attach router to Gradio's FastAPI application *after* launch.
# Protected by a separate random Bearer token; never hardcode it.
# ============================================================
from fastapi import APIRouter, Depends, HTTPException, Header, UploadFile, File, Form
from pydantic import BaseModel
import io

class ActionBody(BaseModel):
    reviewer: str = ""
    worker: str = ""
    decision: str = ""
    notes: str = ""

def build_fastapi_router(engine, api_token):

    if not api_token or len(str(api_token)) < 24:
        raise ValueError("Use a strong, randomly generated API token")
    router = APIRouter(prefix="/api/v1", tags=["dengu-hunter"])

    def authorized(authorization: str | None = Header(None)):
        if not secrets.compare_digest(str(authorization or ""), "Bearer " + str(api_token)):
            raise HTTPException(status_code=401, detail="Bearer token required")

    def convert_error(fn, *args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except (ValueError, FileNotFoundError, TypeError) as exc:
            raise HTTPException(status_code=422, detail=str(exc)) from exc

    async def read_image(upload: UploadFile):
        payload = await upload.read(12 * 1024 * 1024 + 1)
        if len(payload) > 12 * 1024 * 1024:
            raise HTTPException(status_code=413, detail="Image exceeds 12 MB")
        try:
            with Image.open(io.BytesIO(payload)) as raw:
                return image_rgb(raw)
        except (OSError, ValueError, Image.DecompressionBombError) as exc:
            raise HTTPException(status_code=422, detail="Invalid/oversized image") from exc

    @router.get("/health", dependencies=[Depends(authorized)])
    def health():
        return {"status": "LIVE_AI_BACKEND", "model": engine.f2_label,
                "model_sha256": engine.f2_weights_sha256,
                "scope": "breeding-site inspection priority; not dengue prediction"}

    @router.get("/cases", dependencies=[Depends(authorized)])
    def cases():
        return [safe_case_summary(c) for c in engine.cases()]

    @router.get("/cases/{case_id}", dependencies=[Depends(authorized)])
    def case_details(case_id: str):
        return convert_error(lambda: safe_case_summary(engine.inspect_case(case_id)[-1]))

    @router.get("/dashboard", dependencies=[Depends(authorized)])
    def dashboard_api():
        summary, table, _ = engine.dashboard()
        return {"summary": summary, "cases": table.to_dict(orient="records")}

    @router.post("/cases", dependencies=[Depends(authorized)])
    async def submit_api(image: UploadFile = File(...),
                         evidence_mode: str = Form("DEMO_CASE_STAGED"),
                         source: str = Form("Ground photo"),
                         latitude: float | None = Form(None),
                         longitude: float | None = Form(None),
                         rain_context: str = Form("Unknown"),
                         previous_reports: int = Form(0), notes: str = Form(""),
                         closeup: UploadFile | None = File(None)):
        img = await read_image(image)
        close = await read_image(closeup) if closeup is not None else None
        case, _ = convert_error(engine.submit, img, source, latitude, longitude,
                                rain_context, previous_reports, close, notes,
                                evidence_mode=evidence_mode)
        return safe_case_summary(case)

    @router.post("/cases/{case_id}/review", dependencies=[Depends(authorized)])
    def review_api(case_id: str, body: ActionBody):
        return safe_case_summary(convert_error(engine.review, case_id, body.reviewer, body.decision, body.notes))

    @router.post("/cases/{case_id}/assign", dependencies=[Depends(authorized)])
    def assign_api(case_id: str, body: ActionBody):
        return safe_case_summary(convert_error(engine.assign, case_id, body.worker, body.notes))

    @router.post("/cases/{case_id}/start", dependencies=[Depends(authorized)])
    def start_api(case_id: str, body: ActionBody):
        return safe_case_summary(convert_error(engine.start, case_id, body.worker))

    @router.post("/cases/{case_id}/after", dependencies=[Depends(authorized)])
    async def after_api(case_id: str, image: UploadFile = File(...),
                        worker: str = Form(...), notes: str = Form(""),
                        same_site_attested: bool = Form(False)):
        img = await read_image(image)
        case, _ = convert_error(engine.submit_after, case_id, worker, img, notes, same_site_attested)
        return safe_case_summary(case)

    @router.post("/cases/{case_id}/finalize", dependencies=[Depends(authorized)])
    def finalize_api(case_id: str, body: ActionBody):
        return safe_case_summary(convert_error(engine.finalize, case_id, body.reviewer, body.decision, body.notes))

    return router

In [37]:
# 3/6 — MODEL PREFLIGHT. Find actual existing F2/F3 weights; do not train or fake YOLO.
# If the F2 auto-detected path is wrong, enter your exact original F2 best.pt below.
F2_WEIGHTS_OVERRIDE = ''  # Example: '/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/DengueHunter_v1/weights/best.pt'
F3_WEIGHTS_OVERRIDE = ''  # optional

F2_CANDIDATES = [
    Path(F2_WEIGHTS_OVERRIDE).expanduser() if F2_WEIGHTS_OVERRIDE.strip() else None,
    DRIVE/'DENGUE_HUNTER_RESULTS/DengueHunter_v1/weights/best.pt',
    DRIVE/'DENGUE_HUNTER_RESULTS/Detection_Model/DengueHunter_v1/weights/best.pt',
    SHOWCASE/'Models/Detection_Model/best.pt',
]
F2_CANDIDATES = [p for p in F2_CANDIDATES if p is not None]
F2_WEIGHTS = next((p for p in F2_CANDIDATES if p.is_file()),None)
if F2_WEIGHTS is None:
    print('Expected F2 paths:')
    for p in F2_CANDIDATES: print('   ',p)
    print('Other discovered best.pt files in known result roots:')
    for base in [DRIVE/'DENGUE_HUNTER_RESULTS',SHOWCASE/'Models']:
        if base.is_dir():
            for p in list(base.rglob('best.pt'))[:25]: print('   ',p)
    raise FileNotFoundError('Trained real F2 best.pt not found. Set F2_WEIGHTS_OVERRIDE to the existing weight path.')

F3_WEIGHTS = (Path(F3_WEIGHTS_OVERRIDE).expanduser() if F3_WEIGHTS_OVERRIDE.strip()
    else DRIVE/'DENGUE_HUNTER_RESULTS/Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt')
real_f2 = RealYoloDetector(F2_WEIGHTS, device=DH_DEVICE)
real_f3 = RealYoloDetector(F3_WEIGHTS, device=DH_DEVICE) if F3_WEIGHTS.is_file() else None

# Reject accidental use of F3 larvae weights as the breeding-site F2 detector.
model_classes = [str(x).lower() for x in real_f2.model.names.values()] if isinstance(real_f2.model.names,dict) else [str(x).lower() for x in real_f2.model.names]
if not any(any(term in label for term in ('bottle','tire','tyre','vase','drain','coconut')) for label in model_classes):
    raise RuntimeError('Loaded model classes do not resemble F2 site indicators. Verify F2_WEIGHTS_OVERRIDE: '+str(model_classes))

APP_DATA = SHOWCASE/'Unified_Prototype'/'App_Data'
engine = DenguHunterEngine(APP_DATA,real_f2,real_f3,
                          f2_model_label='F2_TRAINED_YOLO_BREEDING_SITE_INDICATORS',
                          f3_model_label='F3_TRAINED_CLOSEUP_LARVAE_YOLO_NO_SPECIES_CONFIRMATION' if real_f3 else None)
assert engine.f2_weights_sha256 and len(engine.f2_weights_sha256)==64
print('REAL F2 model:', F2_WEIGHTS)
print('F2 classes:', model_classes)
print('F2 SHA-256:', engine.f2_weights_sha256)
print('F3 optional model:',str(F3_WEIGHTS) if real_f3 else 'UNAVAILABLE — close-up analysis optional')
print('New case/evidence folder only:',APP_DATA)
print('PREFLIGHT: model loaded. Real photo inference and full workflow remain TO BE TESTED.')

REAL F2 model: /content/drive/MyDrive/DENGUE_HUNTER_RESULTS/DengueHunter_v1/weights/best.pt
F2 classes: ['bottle', 'coconut-exocarp', 'drain-inlet', 'tire', 'vase']
F2 SHA-256: 4005471909e8ed82ec2024767a20a4620950f264f7178c73870e2166c826d693
F3 optional model: /content/drive/MyDrive/DENGUE_HUNTER_RESULTS/Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt
New case/evidence folder only: /content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE/Unified_Prototype/App_Data
PREFLIGHT: model loaded. Real photo inference and full workflow remain TO BE TESTED.


In [38]:
# 4/6 — LIVE GRADIO + FASTAPI: one app, same trained detector & case store.
# Colab controls share-link availability; never commit outputs containing passwords/tokens.
import secrets
DEMO_USERNAME='demo'
DEMO_PASSWORD=secrets.token_urlsafe(14)
FASTAPI_TOKEN=secrets.token_urlsafe(28)
app=build_gradio_app(engine,legacy_root=SHOWCASE)
app.queue(default_concurrency_limit=1)
print('Gradio username:',DEMO_USERNAME)
print('TEMPORARY Gradio password:',DEMO_PASSWORD)
print('TEMPORARY FastAPI bearer token:',FASTAPI_TOKEN)
print('Neither token is hard-coded in the repository. Clear outputs before any GitHub upload.')
print('Launching REAL model demo; use a real uploaded image to confirm inference.')
try:
    app.launch(share=True,auth=(DEMO_USERNAME,DEMO_PASSWORD),show_error=True,prevent_thread_lock=True)
except Exception as exc:
    print('Public sharing unavailable:',type(exc).__name__,str(exc)[:200])
    print('Trying local Gradio server. In Colab, public link may not be available.')
    app.launch(share=False,auth=(DEMO_USERNAME,DEMO_PASSWORD),show_error=True,prevent_thread_lock=True)

# Gradio creates a FastAPI ASGI app at launch; attach real protected endpoints afterward.
app.app.include_router(build_fastapi_router(engine,FASTAPI_TOKEN))
print('FastAPI REST routes available on the SAME Gradio host under /api/v1/ (Bearer token required).')
print('Example paths: /api/v1/health, /api/v1/cases, /api/v1/dashboard, /api/v1/cases/{id}/review')

Gradio username: demo
TEMPORARY Gradio password: 9C2UOiJQliGDQNUvsNo
TEMPORARY FastAPI bearer token: XOQGO5NWgnktO-hVwdlUPt8j_ZvE5BCWnqZgVw
Neither token is hard-coded in the repository. Clear outputs before any GitHub upload.
Launching REAL model demo; use a real uploaded image to confirm inference.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://982a6625f45036a69e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


FastAPI REST routes available on the SAME Gradio host under /api/v1/ (Bearer token required).
Example paths: /api/v1/health, /api/v1/cases, /api/v1/dashboard, /api/v1/cases/{id}/review


In [5]:
# 5/6 — NON-DESTRUCTIVE READINESS REPORT: not a fake live-test success.
from datetime import datetime,timezone
startup = {
    'checked_at_utc':datetime.now(timezone.utc).isoformat(),
    'runtime_device': DH_GPU_NAME or 'CPU',
    'f2_real_weights_loaded':True,
    'f2_weights_sha256':engine.f2_weights_sha256,
    'f3_optional_closeup_loaded':bool(real_f3),
    'historic_training_cells_executed':False,
    'live_user_image_inference':'NOT YET CONFIRMED — submit a real photo in Citizen tab',
    'citizen_to_worker_to_after_live_test':'NOT YET CONFIRMED — follow steps in the app',
    'drone_mode':'SIMULATED PLANNED ROUTES, NO REAL FLIGHT',
    'risk_mode':'EXPLAINABLE BREEDING-SITE INSPECTION PRIORITY, NOT OUTBREAK PREDICTION',
    'species_confirmation':'NOT PROVIDED; requires expert verification',
    'independent_field_cleanup_verification':'NOT PROVIDED',
    'evaluation_status':evaluation_status(engine),
    'github_dates':'CHECK ACTUAL REPOSITORY AND COMMITS; NOT VERIFIED HERE',
}
print(json.dumps(startup,indent=2,default=str))
print('Open Gradio link from Cell 4. Use the 8 tabs and save evidence to Drive.')

{
  "checked_at_utc": "2026-11-22T10:20:52.280980+00:00",
  "runtime_device": "CPU",
  "f2_real_weights_loaded": true,
  "f2_weights_sha256": "4005471909e8ed82ec2024767a20a4620950f264f7178c73870e2166c826d693",
  "f3_optional_closeup_loaded": true,
  "historic_training_cells_executed": false,
  "live_user_image_inference": "NOT YET CONFIRMED \u2014 submit a real photo in Citizen tab",
  "citizen_to_worker_to_after_live_test": "NOT YET CONFIRMED \u2014 follow steps in the app",
  "drone_mode": "SIMULATED PLANNED ROUTES, NO REAL FLIGHT",
  "risk_mode": "EXPLAINABLE BREEDING-SITE INSPECTION PRIORITY, NOT OUTBREAK PREDICTION",
  "species_confirmation": "NOT PROVIDED; requires expert verification",
  "independent_field_cleanup_verification": "NOT PROVIDED",
  "evaluation_status": {
    "status": "NOT EVALUATED",
    "false_alert_rate": null,
    "note": "Upload independent human-labelled positive AND negative images."
  },
  "github_dates": "CHECK ACTUAL REPOSITORY AND COMMITS; NOT VERIFIED 

In [6]:
# 6/6 — GITHUB RELEASE CHECK (read-only): does not rerun training/inference.
# The project can be committed now as code, but only claim LIVE DEMO READY after manual real-image test.
from pathlib import Path
import json
status, rows, html_map = engine.dashboard()
checks={
    'real_f2_weights_loaded':bool(engine.f2_weights_sha256),
    'cases_saved':status['citizen_reports'],
    'reviewed_cases':status['human_approved'],
    'worker_tasks':status['worker_tasks_created'],
    'followup_photos':status['after_photos_submitted'],
    'gps_completeness_percent':status['gps_completeness_percent'],
    'median_submission_to_task_seconds':status['median_submission_to_task_seconds'],
    'false_alert_rate':status['false_alert_rate'],
    'verified_cleanup_successes':0,
}
print(json.dumps(checks,indent=2,default=str))
print('FINAL GATE: run a REAL photo and complete a STAGED, clearly labelled review/task/after sequence.')
print('GitHub: use actual commit history; disclose dataset provenance; clear outputs; avoid uploading private case photos/model weights.')

{
  "real_f2_weights_loaded": true,
  "cases_saved": 0,
  "reviewed_cases": 0,
  "worker_tasks": 0,
  "followup_photos": 0,
  "gps_completeness_percent": null,
  "median_submission_to_task_seconds": null,
  "false_alert_rate": {
    "status": "NOT EVALUATED",
    "false_alert_rate": null,
    "note": "Upload independent human-labelled positive AND negative images."
  },
  "verified_cleanup_successes": 0
}
FINAL GATE: run a REAL photo and complete a STAGED, clearly labelled review/task/after sequence.
GitHub: use actual commit history; disclose dataset provenance; clear outputs; avoid uploading private case photos/model weights.


---
## Manual acceptance test — run this in the LIVE app before calling it a final demo

1. **Citizen + Real AI:** upload a genuine photo. Confirm output contains **actual bounding boxes/confidences or an honest zero-detection result**, model ID and stored case ID. Mark test images as `DEMO_CASE_STAGED`.
2. **Human Review:** inspect saved original/overlay, approve only after looking; approval creates a task, **not Aedes confirmation**.
3. **Worker Task:** assign and start the case.
4. **After Photo + Verification:** upload a real after photo for the *same* place or a **clearly stated staged demo pair**, record same-site worker attestation, run real after-image YOLO and perform human final review. Do **not** claim independent field cleanup verification.
5. **Map:** real user-entered paired GPS makes a pin; blank GPS makes **no fabricated pin**. Drone routes are marked **SIMULATED**.
6. **Evaluation:** genuine held-out human-labelled images required for image-level metrics. With no labels, show **NOT EVALUATED**; YOLO box mAP and image-level false-alert rate are distinct.
7. **FastAPI:** protected `/api/v1/health` returns the same model SHA as the citizen demo. API key and Gradio password are NOT committed.
8. **GitHub:** clear outputs, add real image/dataset/model sources and licenses to README, verify repository creation/first commit dates from actual Git history.

**The 118 code histories below are exact source-text archives**, not active execution cells. They are preserved for reproducibility; use the original notebook if you intentionally need retraining. No training runs as part of `Run all` here.

---
## ORIGINAL 118-CELL SOURCE HISTORY — PRESERVED, NOT EXECUTED
These are exact originals from `DENGU_HUNTER_Quality_Upgraded(2).ipynb`. Execution outputs are intentionally excluded for privacy and repository size.

### Original cell 1/118 — PYTHON SOURCE (archived, not run)

````python
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not connected")
````

### Original cell 2/118 — PYTHON SOURCE (archived, not run)

````python
from google.colab import drive
drive.mount('/content/drive')

import os

print("\nSearching for Dataset...")

matches = []

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.lower() == "breeding place detection.zip":
            matches.append(os.path.join(root, file))

if matches:
    print("\nDATASET FOUND!")
    for path in matches:
        print(path)
else:
    print("\nZIP NOT FOUND. Check Google Drive upload.")
````

### Original cell 3/118 — PYTHON SOURCE (archived, not run)

````python
import os
import zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/DENGUE_HUNTER_DATA/Breeding Place Detection.zip"
EXTRACT_DIR = "/content/DENGUE_HUNTER_DATA"

print("Checking ZIP...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Corrupted file: {bad}")

    print("ZIP integrity: PASS")
    print("Total files:", len(z.namelist()))

    z.extractall(EXTRACT_DIR)

print("\nExtraction completed!")

# Locate data.yaml
yaml_files = list(Path(EXTRACT_DIR).rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("data.yaml not found!")

DATA_YAML = str(yaml_files[0])
DATA_ROOT = Path(DATA_YAML).parent

print("\nDataset configuration:", DATA_YAML)

# Count images and labels
for split in ["train", "valid", "test"]:
    image_dir = DATA_ROOT / split / "images"
    label_dir = DATA_ROOT / split / "labels"

    images = list(image_dir.glob("*")) if image_dir.exists() else []
    images = [p for p in images if p.suffix.lower() in
              [".jpg", ".jpeg", ".png", ".webp"]]

    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Matched:", sum((label_dir / (p.stem + ".txt")).exists()
                         for p in images))
````

### Original cell 4/118 — PYTHON SOURCE (archived, not run)

````python
from pathlib import Path
import yaml

ROOT = Path("/content/DENGUE_HUNTER_DATA/Breeding Place Detection")

classes = [
    "Bottle",
    "Coconut-Exocarp",
    "Drain-Inlet",
    "Tire",
    "Vase"
]

# Create corrected dataset configuration
config = {
    "path": str(ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": classes
}

fixed_yaml = ROOT / "data_fixed.yaml"

with open(fixed_yaml, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Validate every annotation
errors = []
total_labels = 0

for split in ["train", "valid", "test"]:
    label_dir = ROOT / split / "labels"

    for file in label_dir.glob("*.txt"):
        for line_no, line in enumerate(
            file.read_text().splitlines(), 1
        ):
            if not line.strip():
                continue

            parts = line.split()

            try:
                assert len(parts) == 5
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]

                assert 0 <= cls < 5
                assert all(0 <= x <= 1 for x in coords)
                assert coords[2] > 0 and coords[3] > 0

                total_labels += 1

            except Exception:
                errors.append(f"{file.name}: line {line_no}")

print("Total valid annotations:", total_labels)
print("Annotation errors:", len(errors))
print("Corrected YAML:", fixed_yaml)

if errors:
    print("First errors:", errors[:10])
else:
    print("FINAL DATASET VALIDATION: PASSED")
````

### Original cell 5/118 — PYTHON SOURCE (archived, not run)

````python
from pathlib import Path
from collections import Counter

ROOT = Path("/content/DENGUE_HUNTER_DATA/Breeding Place Detection")

reasons = Counter()
examples = []

for split in ["train", "valid", "test"]:
    folder = ROOT / split / "labels"

    for file in folder.glob("*.txt"):
        for n, line in enumerate(file.read_text().splitlines(), 1):
            if not line.strip():
                continue

            p = line.split()
            reason = None

            try:
                if len(p) != 5:
                    reason = f"Wrong field count: {len(p)}"
                else:
                    cls = int(p[0])
                    values = [float(x) for x in p[1:]]

                    if not 0 <= cls < 5:
                        reason = f"Invalid class ID: {cls}"
                    elif not all(0 <= x <= 1 for x in values):
                        reason = f"Coordinate outside 0-1: {values}"
                    elif values[2] <= 0 or values[3] <= 0:
                        reason = "Zero/negative box dimension"

            except Exception as e:
                reason = f"Parsing error: {e}"

            if reason:
                reasons[reason] += 1

                if len(examples) < 15:
                    examples.append({
                        "file": file.name,
                        "line": n,
                        "reason": reason,
                        "content": line[:250]
                    })

print("ERROR SUMMARY:")
for reason, count in reasons.items():
    print(count, "->", reason)

print("\nFIRST 15 EXAMPLES:")
for item in examples:
    print(item)
````

### Original cell 6/118 — PYTHON SOURCE (archived, not run)

````python
from pathlib import Path
from collections import Counter

ROOT = Path("/content/DENGUE_HUNTER_DATA/Breeding Place Detection")

formats = Counter()
bad = []
polygon_examples = []

for split in ["train", "valid", "test"]:
    folder = ROOT / split / "labels"

    for file in folder.glob("*.txt"):
        for line_no, line in enumerate(file.read_text().splitlines(), 1):
            if not line.strip():
                continue

            p = line.split()

            try:
                cls = int(p[0])
                coords = [float(x) for x in p[1:]]

                if len(p) == 5:
                    formats["Bounding Box"] += 1
                    if not (0 <= cls < 5):
                        raise ValueError("Invalid class")
                    if not all(0 <= x <= 1 for x in coords):
                        raise ValueError("Invalid coordinates")
                    if coords[2] <= 0 or coords[3] <= 0:
                        raise ValueError("Invalid box dimensions")

                elif len(p) >= 7 and len(coords) % 2 == 0:
                    formats["Polygon"] += 1
                    if not (0 <= cls < 5):
                        raise ValueError("Invalid class")
                    if not all(0 <= x <= 1 for x in coords):
                        raise ValueError("Invalid polygon coordinates")

                    if len(polygon_examples) < 3:
                        polygon_examples.append(file.name)

                else:
                    raise ValueError(f"Unexpected format: {len(p)} fields")

            except Exception as e:
                bad.append((file.name, line_no, str(e)))

print("ANNOTATION FORMAT SUMMARY")
print(formats)
print("\nInvalid annotations:", len(bad))
print("Polygon examples:", polygon_examples)
print("First errors:", bad[:10])
````

### Original cell 7/118 — PYTHON SOURCE (archived, not run)

````python
from pathlib import Path
import shutil
import yaml

SOURCE = Path("/content/DENGUE_HUNTER_DATA/Breeding Place Detection")
OUTPUT = Path("/content/DENGUE_HUNTER_DETECTION")

# Preserve original dataset
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

print("Creating a separate detection dataset...")
shutil.copytree(SOURCE, OUTPUT)

converted = 0
preserved = 0
total = 0

for split in ["train", "valid", "test"]:
    label_dir = OUTPUT / split / "labels"

    for file in label_dir.glob("*.txt"):
        new_lines = []

        for line in file.read_text().splitlines():
            if not line.strip():
                continue

            p = line.split()
            cls = int(p[0])
            values = [float(x) for x in p[1:]]

            if len(p) == 5:
                new_lines.append(line)
                preserved += 1

            else:
                # Convert polygon to enclosing bounding box
                xs = values[0::2]
                ys = values[1::2]

                xmin, xmax = min(xs), max(xs)
                ymin, ymax = min(ys), max(ys)

                x_center = (xmin + xmax) / 2
                y_center = (ymin + ymax) / 2
                width = xmax - xmin
                height = ymax - ymin

                if width <= 0 or height <= 0:
                    raise ValueError(f"Invalid polygon: {file.name}")

                new_lines.append(
                    f"{cls} {x_center:.8f} {y_center:.8f} "
                    f"{width:.8f} {height:.8f}"
                )
                converted += 1

            total += 1

        file.write_text("\n".join(new_lines) + "\n")

# Create independent YOLO detection configuration
config = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": [
        "Bottle",
        "Coconut-Exocarp",
        "Drain-Inlet",
        "Tire",
        "Vase"
    ]
}

with open(OUTPUT / "data.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\nDETECTION DATASET READY")
print("Original bounding boxes:", preserved)
print("Converted polygons:", converted)
print("Total annotations:", total)
print("Output:", OUTPUT)
print("Original dataset preserved:", SOURCE.exists())
````

### Original cell 8/118 — PYTHON SOURCE (archived, not run)

````python

!pip install -q ultralytics

import torch
from ultralytics import YOLO
from pathlib import Path

# Check GPU
assert torch.cuda.is_available(), "GPU is not available!"

print("GPU:", torch.cuda.get_device_name(0))

# Dataset
DATA = "/content/DENGUE_HUNTER_DETECTION/data.yaml"

# Save results permanently in Google Drive
PROJECT = "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS"

# Load pretrained YOLOv8 Small
model = YOLO("yolov8s.pt")

# Start training
results = model.train(
    data=DATA,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    seed=42,
    deterministic=True,
    project=PROJECT,
    name="DengueHunter_v1",
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)

print("\nTRAINING COMPLETED!")
print("Results:", PROJECT + "/DengueHunter_v1")
print("Best Model:", PROJECT + "/DengueHunter_v1/weights/best.pt")

````

### Original cell 9/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

for p in Path("/content").rglob("data.yaml"):
    print(p)

````

### Original cell 10/118 — PYTHON SOURCE (archived, not run)

````python

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
for p in Path('/content/drive/MyDrive').rglob('data.yaml'):
    print(p)

````

### Original cell 11/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

for p in Path('/content/drive/MyDrive').rglob('*.yaml'):
    print(p)

````

### Original cell 12/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

p = Path('/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/DengueHunter_v1/args.yaml')
print(p.read_text())

````

### Original cell 13/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

for p in Path('/content/drive/MyDrive').rglob('*.zip'):
    print(p)

````

### Original cell 14/118 — PYTHON SOURCE (archived, not run)

````python

import zipfile

zip_path = "/content/drive/MyDrive/DENGUE_HUNTER_DATA/Breeding Place Detection.zip"

with zipfile.ZipFile(zip_path) as z:
    files = z.namelist()
    print("Total files:", len(files))
    print("\nDataset configuration:")
    for f in files:
        if f.endswith((".yaml", ".yml")):
            print(f)
    print("\nTest files:")
    for f in files:
        if "/test/" in "/" + f or f.startswith("test/"):
            print(f)
            if sum(("/test/" in "/" + x or x.startswith("test/")) for x in files[:files.index(f)+1]) >= 10:
                break

````

### Original cell 15/118 — PYTHON SOURCE (archived, not run)

````python

import zipfile

zip_path = "/content/drive/MyDrive/DENGUE_HUNTER_DATA/Breeding Place Detection.zip"

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()

    images = [n for n in names if "/test/images/" in n and n.lower().endswith((".jpg", ".jpeg", ".png"))]
    labels = [n for n in names if "/test/labels/" in n and n.endswith(".txt")]

    print("Test images:", len(images))
    print("Test labels:", len(labels))
    print("\nOriginal data.yaml:")
    print(z.read("Breeding Place Detection/data.yaml").decode("utf-8"))

````

### Original cell 16/118 — PYTHON SOURCE (archived, not run)

````python

import zipfile
from pathlib import Path

zip_path = "/content/drive/MyDrive/DENGUE_HUNTER_DATA/Breeding Place Detection.zip"
dataset = Path("/content/Breeding Place Detection")

if not (dataset / "test/images").exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content")

print("Dataset ready:", (dataset / "data.yaml").exists())
print("Test images:", len(list((dataset / "test/images").glob("*.jpg"))))
print("Test labels:", len(list((dataset / "test/labels").glob("*.txt"))))

````

### Original cell 17/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO
from pathlib import Path

model_path = "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/DengueHunter_v1/weights/best.pt"
data_path = "/content/Breeding Place Detection/data.yaml"

assert Path(model_path).exists(), "best.pt not found!"
assert Path(data_path).exists(), "data.yaml not found!"

model = YOLO(model_path)

results = model.val(
    data=data_path,
    split="test",
    imgsz=640,
    batch=16,
    plots=False,
    project="/content/DENGUE_HUNTER_TEST",
    name="Independent_Test",
    exist_ok=True
)

print("\n===== INDEPENDENT TEST RESULTS =====")
print("Precision:", round(results.box.mp, 4))
print("Recall:", round(results.box.mr, 4))
print("mAP50:", round(results.box.map50, 4))
print("mAP50-95:", round(results.box.map, 4))

````

### Original cell 18/118 — PYTHON SOURCE (archived, not run)

````python

%pip install -q ultralytics

````

### Original cell 19/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO
from pathlib import Path
import yaml

base = Path("/content/Breeding Place Detection")

# Temporary test configuration
config = {
    "path": str(base),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": ["Bottle", "Coconut-Exocarp",
              "Drain-Inlet", "Tire", "Vase"]
}

yaml_path = "/content/dengue_test.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f)

# Load existing trained model
model = YOLO(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/DengueHunter_v1/weights/best.pt"
)

# Independent test
results = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    batch=16,
    plots=False,
    project="/content/DENGUE_HUNTER_TEST",
    name="Independent_Test",
    exist_ok=True
)

print("\n===== INDEPENDENT TEST RESULTS =====")
print("Precision:", round(results.box.mp, 4))
print("Recall:", round(results.box.mr, 4))
print("mAP50:", round(results.box.map50, 4))
print("mAP50-95:", round(results.box.map, 4))

````

### Original cell 20/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

root = Path("/content/drive/MyDrive")

keywords = ["integration", "association", "overlay", "audit", "visual"]

for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg", ".csv"]:
        if any(k in str(p).lower() for k in keywords):
            print(p)

````

### Original cell 21/118 — PYTHON SOURCE (archived, not run)

````python

from PIL import Image
from IPython.display import display

path = "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/Dual_AI_Integration/Batch_Integration/batch_visual_audit.jpg"

img = Image.open(path)
print("Image size:", img.size)
display(img)

````

### Original cell 22/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd

path = "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/Dual_AI_Integration/Batch_Integration/Final_AI_Assessment/object_assessment.csv"

df = pd.read_csv(path)

print("Total objects:", len(df))
print("Columns:", df.columns.tolist())
print(df.to_string(index=False))

````

### Original cell 23/118 — PYTHON SOURCE (archived, not run)

````python

!pip -q install ultralytics

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt"
)

dataset = "/content/drive/MyDrive/DENGUE_HUNTER_DATA/LARVAE_MODEL_PREPARATION/YOLO_Dataset/Larvae_Detection_Ready"

results = model.val(
    data=dataset + "/data.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    project="/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/Larvae_Model",
    name="F3_Independent_Test",
    exist_ok=False
)

print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

````

### Original cell 24/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import yaml

dataset = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_DATA/"
    "LARVAE_MODEL_PREPARATION/YOLO_Dataset/"
    "Larvae_Detection_Ready"
)

# Check test images and labels
images = list((dataset / "test/images").glob("*"))
labels = list((dataset / "test/labels").glob("*.txt"))

print("Test images:", len(images))
print("Test labels:", len(labels))

# Create temporary configuration
with open(dataset / "data.yaml", "r") as f:
    config = yaml.safe_load(f)

config["test"] = "test/images"

temp_yaml = "/content/f3_test.yaml"

with open(temp_yaml, "w") as f:
    yaml.safe_dump(config, f)

print("Temporary test YAML created:", temp_yaml)
print("Original data.yaml unchanged")

````

### Original cell 25/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

root = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_DATA/"
    "LARVAE_MODEL_PREPARATION/YOLO_Dataset/"
    "Larvae_Detection_Ready"
)

print("Dataset exists:", root.exists())

for folder in ["train", "valid", "test"]:
    p = root / folder
    print(f"\n{folder} exists:", p.exists())

    if p.exists():
        for item in p.iterdir():
            print(" -", item.name)

print("\nPossible test folders:")
for p in root.rglob("*"):
    if p.is_dir() and "test" in p.name.lower():
        print(p)

````

### Original cell 26/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/DengueHunter_Larvae_v1/"
    "weights/best.pt"
)

results = model.val(
    data="/content/drive/MyDrive/DENGUE_HUNTER_DATA/"
         "LARVAE_MODEL_PREPARATION/YOLO_Dataset/"
         "Larvae_Detection_Ready/data.yaml",
    split="val",
    imgsz=640,
    batch=8,
    device="cpu",
    project="/content/DENGUE_HUNTER_F3_CHECK",
    name="Validation_Check",
    exist_ok=False
)

print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)

````

### Original cell 27/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt"
)

print("MODEL CLASSES:", model.names)

````

### Original cell 28/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO
from pathlib import Path
from IPython.display import display, Image

model_path = (
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt"
)

dataset = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_DATA/"
    "LARVAE_MODEL_PREPARATION/YOLO_Dataset/"
    "Larvae_Detection_Ready"
)

model = YOLO(model_path)

print("Model classes:", model.names)

images = sorted(
    p for p in (dataset / "valid/images").iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

print("Validation images:", len(images))

if not images:
    raise FileNotFoundError("No validation images found")

results = model.predict(
    source=str(images[0]),
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/F3_LARVAE_CHECK",
    name="Prediction",
    exist_ok=True
)

print("Detections:", len(results[0].boxes))
display(Image(filename=str(
    Path(results[0].save_dir) / images[0].name
)))

````

### Original cell 29/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/DengueHunter_Larvae_v1/"
    "weights/best.pt"
)

model.predict(
    source="/content/drive/MyDrive/DENGUE_HUNTER_DATA/"
           "LARVAE_MODEL_PREPARATION/YOLO_Dataset/"
           "Larvae_Detection_Ready/valid/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/F3_LARVAE_CHECK",
    name="All_20_Predictions",
    exist_ok=False
)

print("F3 prediction batch completed!")

````

### Original cell 30/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

folder = Path("/content/F3_LARVAE_CHECK/All_20_Predictions")

files = sorted(
    p for p in folder.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

w, h = 320, 280
cols = 4
rows = (len(files) + cols - 1) // cols

sheet = Image.new("RGB", (cols*w, rows*h), "white")
draw = ImageDraw.Draw(sheet)

for i, file in enumerate(files):
    img = Image.open(file).convert("RGB")
    img.thumbnail((w-12, h-35))

    x = (i % cols)*w
    y = (i // cols)*h

    sheet.paste(img, (x+6, y+6))
    draw.text((x+6, y+h-25), file.name[:28], fill="black")

output = "/content/F3_LARVAE_CHECK/F3_Visual_Audit.jpg"
sheet.save(output, quality=90)

print("Images:", len(files))
print("Saved:", output)

display(sheet)

````

### Original cell 31/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import shutil

source = Path("/content/F3_LARVAE_CHECK/All_20_Predictions")
destination = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/F3_Dashboard_Evidence"
)

if not source.exists():
    print("Prediction folder missing. Tell me before proceeding.")
else:
    destination.mkdir(parents=True, exist_ok=True)
    images = list(source.glob("*.jpg")) + list(source.glob("*.png"))

    for img in images:
        shutil.copy2(img, destination / img.name)

    print("Images copied:", len(images))
    print("Saved to:", destination)

````

### Original cell 32/118 — PYTHON SOURCE (archived, not run)

````python

from ultralytics import YOLO
from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive")

model = YOLO(
    str(base / "DENGUE_HUNTER_RESULTS/Larvae_Model/"
        "DengueHunter_Larvae_v1/weights/best.pt")
)

images = base / (
    "DENGUE_HUNTER_DATA/LARVAE_MODEL_PREPARATION/"
    "YOLO_Dataset/Larvae_Detection_Ready/valid/images"
)

output = base / (
    "DENGUE_HUNTER_RESULTS/Larvae_Model/"
    "F3_Dashboard_Evidence"
)

records = []

for result in model.predict(
    source=str(images),
    imgsz=640,
    conf=0.25,
    device="cpu",
    stream=True,
    verbose=False
):
    filename = Path(result.path).name

    if len(result.boxes) == 0:
        records.append({
            "image": filename,
            "detection_id": "",
            "class_name": "",
            "confidence": "",
            "x1": "", "y1": "",
            "x2": "", "y2": "",
            "review_status": "NEEDS_REVIEW"
        })
        continue

    for i, box in enumerate(result.boxes, 1):
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        records.append({
            "image": filename,
            "detection_id": i,
            "class_name": "mosquito-larvae",
            "confidence": round(float(box.conf[0]), 4),
            "x1": round(x1, 2),
            "y1": round(y1, 2),
            "x2": round(x2, 2),
            "y2": round(y2, 2),
            "review_status": "PENDING"
        })

df = pd.DataFrame(records)

csv_path = output / "F3_Larvae_Predictions.csv"
df.to_csv(csv_path, index=False)

print("Images processed:", df["image"].nunique())
print("Detection boxes:", df["confidence"].notna().sum())
print("CSV saved:", csv_path)

````

### Original cell 33/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

csv_path = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/F3_Dashboard_Evidence/"
    "F3_Larvae_Predictions.csv"
)

print(csv_path.read_text(encoding="utf-8"))

````

### Original cell 34/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

folder = Path(
    "/content/drive/MyDrive/DENGUE_HUNTER_RESULTS/"
    "Larvae_Model/F3_Dashboard_Evidence"
)

pred = pd.read_csv(folder / "F3_Larvae_Predictions.csv")
images = sorted(pred["image"].unique())

review_file = folder / "F3_Human_Review.csv"

if review_file.exists():
    review = pd.read_csv(review_file).fillna("")
else:
    review = pd.DataFrame({
        "image": images,
        "human_decision": ["PENDING"] * len(images)
    })

position = [0]
output = widgets.Output()

def show_image():
    with output:
        clear_output(wait=True)

        name = images[position[0]]
        rows = pred[pred["image"] == name]
        detections = rows["confidence"].dropna()

        print(f"Image {position[0]+1} / {len(images)}")
        print("File:", name)
        print("AI detections:", len(detections))
        print("Confidence:", detections.round(3).tolist())

        path = folder / name
        if path.exists():
            display(Image(filename=str(path), width=600))
        else:
            print("Image not found:", path)

        current = review.loc[
            review["image"] == name, "human_decision"
        ].iloc[0]

        print("Your decision:", current)
        print("Reviewed:",
              (review["human_decision"] != "PENDING").sum(),
              "/", len(images))

def decide(choice):
    name = images[position[0]]

    review.loc[
        review["image"] == name, "human_decision"
    ] = choice

    review.to_csv(review_file, index=False)

    if position[0] < len(images) - 1:
        position[0] += 1

    show_image()

def move(step):
    position[0] = max(
        0, min(len(images)-1, position[0] + step)
    )
    show_image()

agree = widgets.Button(description="Agree with AI",
                       button_style="success")
disagree = widgets.Button(description="Disagree with AI",
                          button_style="danger")
uncertain = widgets.Button(description="Uncertain",
                           button_style="warning")
previous = widgets.Button(description="Previous")
next_image = widgets.Button(description="Next")

agree.on_click(lambda b: decide("AGREE"))
disagree.on_click(lambda b: decide("DISAGREE"))
uncertain.on_click(lambda b: decide("UNCERTAIN"))
previous.on_click(lambda b: move(-1))
next_image.on_click(lambda b: move(1))

display(widgets.HBox([agree, disagree, uncertain]))
display(widgets.HBox([previous, next_image]))
display(output)

show_image()

````

### Original cell 35/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path

root = Path("/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE")

matches = [
    p for p in root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".py", ".ipynb", ".json"]
    and any(word in p.name.lower()
            for word in ["hotspot", "risk", "predict", "feature_4"])
]

print("F4 related code/config files:")
for p in matches:
    print(p.relative_to(root))

if not matches:
    print("No matching F4 code/config file found.")

````

### Original cell 36/118 — PYTHON SOURCE (archived, not run)

````python
# Cell 1 — Setup: mount Drive, no training
from google.colab import drive
drive.mount('/content/drive')

!pip -q install ultralytics

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import pandas as pd, numpy as np, json, hashlib, os, re, ast, zipfile, shutil
from PIL import Image
from ultralytics import YOLO

DRIVE = Path('/content/drive/MyDrive')
SHOWCASE = DRIVE / 'DENGU_HUNTER_FINAL_SHOWCASE'
DATA_ROOT = DRIVE / 'DENGUE_HUNTER_DATA'
F3_EVIDENCE = DRIVE / 'DENGUE_HUNTER_RESULTS/Larvae_Model/F3_Dashboard_Evidence'
F3_WEIGHTS = DRIVE / 'DENGUE_HUNTER_RESULTS/Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt'
F3_DATASET = DATA_ROOT / 'LARVAE_MODEL_PREPARATION/YOLO_Dataset/Larvae_Detection_Ready'

# Only new outputs. Re-running creates a different folder.
OUT = DRIVE / 'DENGUE_HUNTER_RESULTS/Larvae_Model/F3_Completion_Audit_Runs' / ('Run_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
OUT.mkdir(parents=True, exist_ok=False)
SUMMARY = {'run_dir':str(OUT), 'review_audit':'NOT_RUN', 'two_stage':'NOT_RUN', 'independent_test':'NOT_RUN'}
print('New reports only:', OUT)
print('Existing F3 weights:', F3_WEIGHTS.exists())
print('Existing F3 evidence folder:', F3_EVIDENCE.exists())

````

### Original cell 37/118 — PYTHON SOURCE (archived, not run)

````python
expected_text = """011_jpeg.rf.1ce22edc74eeac384042c4f1db0def80.jpg DISAGREE
014_jpg.rf.5010d73d2622dfee5b8b10823a2fd43e.jpg DISAGREE
022_jpg.rf.8a3fc189f675f44b477e2928d49ad4f5.jpg DISAGREE
026_jpg.rf.cd522008ce3dc62558cf5e9d2786b13e.jpg DISAGREE
029_jpg.rf.643565d250755ab34ecc2e9a251d504e.jpg DISAGREE
036_jpg.rf.7e763367a15d61594bf1b639629f70e8.jpg UNCERTAIN
038_jpg.rf.9947f169ff58a4d264e0e5246168dafd.jpg DISAGREE
049_jpg.rf.3a31fa5fa6f9790babd26b2c9f6bb010.jpg DISAGREE
059_jpg.rf.9f29db303d714b455da4c77a45a517a2.jpg AGREE
071_jpg.rf.d87df4f0f5b2129e9d04c9d9891b7b9f.jpg AGREE
075_jpg.rf.6407fbd05d622ab6c563370f90d17669.jpg UNCERTAIN
082_jpg.rf.21104b0e3bbfbc33e90387bb9c7c50af.jpg DISAGREE
090_jpg.rf.0f29ad94249b2cdfe7edd114329efae9.jpg DISAGREE
093_jpg.rf.dc13074964e42726e0c84b8fa8225b66.jpg DISAGREE
096_jpg.rf.33d7f4dd73b2b16520e55f7811386a46.jpg AGREE
097_jpg.rf.bec42e6645fbf81e94fcb0a0ccf8902b.jpg DISAGREE
102_png.rf.f408a67fb2eaaedd0daf60b599bda82c.jpg AGREE
108_png.rf.b023e7e95b4543e43d6546df62f8d405.jpg UNCERTAIN
113_png.rf.dfab62fd8db6ace1d537248936c1d713.jpg AGREE
114_png.rf.4608edec08b59511b8eecdf264f08609.jpg DISAGREE"""

pairs = [line.rsplit(' ',1) for line in expected_text.strip().splitlines()]
expected = pd.DataFrame(pairs, columns=['image','expected_decision'])
assert len(expected)==20 and expected.image.is_unique
assert set(expected.expected_decision).issubset({'AGREE','DISAGREE','UNCERTAIN'})
print('Chat decisions:', expected.expected_decision.value_counts().to_dict())

review_path = F3_EVIDENCE / 'F3_Human_Review.csv'
pred_path = F3_EVIDENCE / 'F3_Larvae_Predictions.csv'

if not review_path.exists() or not pred_path.exists():
    print('BLOCKED — missing Drive file(s):')
    if not review_path.exists(): print(' ',review_path)
    if not pred_path.exists(): print(' ',pred_path)
    SUMMARY['review_audit']='BLOCKED_MISSING_CSV'
else:
    review = pd.read_csv(review_path, dtype=str).fillna('')
    pred = pd.read_csv(pred_path)
    if not {'image','human_decision'}.issubset(review.columns) or 'image' not in pred.columns:
        print('BLOCKED — unexpected CSV columns:', review.columns.tolist(), pred.columns.tolist())
        SUMMARY['review_audit']='BLOCKED_BAD_COLUMNS'
    else:
        review['image'] = review['image'].str.strip()
        review['human_decision'] = review['human_decision'].str.strip().str.upper()
        pred_images = set(pred.image.dropna().astype(str).str.strip())
        duplicated = review.image.duplicated().sum()
        numeric_conf = pd.to_numeric(pred['confidence'], errors='coerce') if 'confidence' in pred.columns else pd.Series(dtype=float)
        detected_rows = int(numeric_conf.notna().sum())
        empty_rows = len(pred) - detected_rows
        SUMMARY['prediction_audit'] = 'PASS' if (len(pred)==23 and detected_rows==18 and empty_rows==5) else 'CHECK_COUNTS'
        print('Prediction records: detections=',detected_rows, 'no-detection=',empty_rows,
              '| prediction audit:', SUMMARY['prediction_audit'])
        audit = expected.merge(review[['image','human_decision']], on='image', how='left', validate='one_to_many')
        audit['matches_drive'] = audit.expected_decision.eq(audit.human_decision)
        audit['exists_in_predictions'] = audit.image.isin(pred_images)
        audit.to_csv(OUT / 'F3_Human_Review_Audit.csv', index=False)
        extra = set(review.image)-set(expected.image)
        missing_pred = set(expected.image)-pred_images
        ok = (len(review)==20 and duplicated==0 and len(pred_images)==20 and
              len(audit)==20 and audit.matches_drive.all() and audit.exists_in_predictions.all() and not extra)
        print('Drive review rows:',len(review),' | unique:',review.image.nunique())
        print('Prediction unique images:',len(pred_images),' | prediction CSV rows:',len(pred))
        print('Matched human decisions:',int(audit.matches_drive.sum()),'/20')
        print('Unexpected review images:',sorted(extra))
        print('Missing prediction images:',sorted(missing_pred))
        if not audit.matches_drive.all():
            print('MISMATCHES:')
            display(audit[~audit.matches_drive][['image','expected_decision','human_decision']])
        SUMMARY['review_audit']='PASS' if ok else 'FAIL_REVIEW_MISMATCH'
        print('HUMAN REVIEW AUDIT:',SUMMARY['review_audit'])

````

### Original cell 38/118 — PYTHON SOURCE (archived, not run)

````python
# F2 precomputed evidence candidates; originals are read-only.
evidence_candidates = [
    SHOWCASE / 'Unified_AI_Integration/unified_ai_evidence_register.csv',
    SHOWCASE / 'association_results.csv',
    DRIVE / 'DENGUE_HUNTER_RESULTS/association_results.csv',
]
evidence_path = next((p for p in evidence_candidates if p.exists()),None)

# Functions for input discovery, no changes to source images.
def colname(df, aliases):
    lookup = {c.lower().strip():c for c in df.columns}
    return next((lookup[x] for x in aliases if x in lookup),None)

def is_matched(x):
    s = str(x).strip().upper()
    return s in {'MATCHED','TRUE','YES','1','WATER_MATCHED','ASSOCIATED'}

def image_index(needed):
    roots = [DATA_ROOT/'Breeding Place Detection', DATA_ROOT/'Breeding_Place_Detection',
             Path('/content/DENGUE_HUNTER_DATA/Breeding Place Detection'),
             Path('/content/DENGUE_HUNTER_DETECTION'), SHOWCASE/'Surveillance_Integration',
             SHOWCASE/'Development_Batch_Features_3_4_6_8']
    found = {}
    for base in roots:
        if not base.is_dir(): continue
        for parent,dirs,files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in {'runs','weights','labels'}]
            for name in files:
                if name in needed and name not in found:
                    found[name]=Path(parent)/name
            if len(found)==len(needed): return found
    # Original F2 ZIP is read-only; extract *only requested images* to the new run folder.
    remaining=needed-set(found)
    zip_path=DATA_ROOT/'Breeding Place Detection.zip'
    if remaining and zip_path.exists():
        cache=OUT/'F2_image_cache'
        with zipfile.ZipFile(zip_path) as z:
            choices=defaultdict(list)
            for member in z.namelist():
                name=Path(member).name
                if name in remaining and not member.endswith('/'):
                    choices[name].append(member)
            for name, members in choices.items():
                members.sort(key=lambda p:(0 if '/test/' in '/'+p.lower() else 1 if '/valid/' in '/'+p.lower() else 2, p))
                cache.mkdir(exist_ok=True)
                target=cache/name
                with z.open(members[0]) as src, target.open('wb') as dst: shutil.copyfileobj(src,dst)
                found[name]=target
                if len(members)>1: print('Note: duplicate ZIP basename; used:',members[0])
    return found

def read_bbox(row, columns, w, h):
    names={k:colname(pd.DataFrame(columns=columns),v) for k,v in {
        'x1':['x1','bbox_x1','box_x1'], 'y1':['y1','bbox_y1','box_y1'],
        'x2':['x2','bbox_x2','box_x2'], 'y2':['y2','bbox_y2','box_y2']}.items()}
    vals=None
    if all(names.values()):
        try: vals=[float(row[names[k]]) for k in ['x1','y1','x2','y2']]
        except (ValueError,TypeError): pass
    if vals is None:
        b=colname(pd.DataFrame(columns=columns),['bbox','box','bounding_box'])
        if b and pd.notna(row[b]):
            try:
                vals=ast.literal_eval(str(row[b]))
                if not isinstance(vals,(tuple,list)) or len(vals)!=4: vals=None
                else: vals=list(map(float,vals))
            except (ValueError,SyntaxError,TypeError): pass
    if vals is None:return None
    x1,y1,x2,y2=vals
    if all(0<=v<=1 for v in vals): x1,x2=x1*w,x2*w; y1,y2=y1*h
    if x2<=x1 or y2<=y1: return None
    pad=0.10*max(x2-x1,y2-y1)
    return (max(0,int(x1-pad)),max(0,int(y1-pad)),min(w,int(x2+pad)),min(h,int(y2+pad)))

if evidence_path is None or not F3_WEIGHTS.exists():
    print('BLOCKED: F2 evidence CSV or F3 model missing.')
    print('F2 CSV candidates:',*[str(p)+':'+str(p.exists()) for p in evidence_candidates],sep='\n  ')
    print('F3 model:',F3_WEIGHTS,F3_WEIGHTS.exists())
    SUMMARY['two_stage']='BLOCKED_MISSING_INPUT'
else:
    ev=pd.read_csv(evidence_path).fillna('')
    print('Using F2 evidence:',evidence_path)
    print('F2 columns:',ev.columns.tolist())
    image_col=colname(ev,['image','image_name','filename','file_name','source_image','associated_image'])
    match_col=colname(ev,['association','association_status','water_association','water_match','water_matched','water_holding_evidence','matched'])
    if not image_col or not match_col:
        print('BLOCKED: F2 CSV needs an image column and a water-association column.')
        SUMMARY['two_stage']='BLOCKED_F2_SCHEMA'
    else:
        ev['_matched']=ev[match_col].map(is_matched)
        ev['_image']=ev[image_col].astype(str).map(lambda x:Path(x).name)
        images=sorted(x for x in ev['_image'].unique() if x and x.lower()!='nan')
        matched=ev[ev['_matched']].copy()
        print('F2 objects:',len(ev),'| water-matched:',len(matched),'| unique images:',len(images))
        found=image_index(set(images))
        print('F2 image files located:',len(found),'/',len(images))
        model=YOLO(str(F3_WEIGHTS))
        rows=[]
        for name in images:
            subset=ev[ev['_image']==name]
            positives=subset[subset['_matched']]
            base={'f2_image':name,'f2_object_count':len(subset),'f2_water_matched_count':len(positives),
                  'f2_source':str(evidence_path),'f3_model':str(F3_WEIGHTS),
                  'f3_candidate_boxes':0,'f3_max_confidence':'','human_review':'PENDING_NEW_F2_IMAGE'}
            if positives.empty:
                rows.append({**base,'input_mode':'SKIPPED','status':'SKIPPED_NO_F2_WATER_MATCH'})
                continue
            if name not in found:
                rows.append({**base,'input_mode':'MISSING_IMAGE','status':'BLOCKED_IMAGE_NOT_FOUND'})
                continue
            with Image.open(found[name]) as src:
                rgb=src.convert('RGB'); w,h=rgb.size
                crops=[]; cropped=0
                for _,e in positives.iterrows():
                    box=read_bbox(e,ev.columns,w,h)
                    if box:
                        crops.append(np.array(rgb.crop(box))[:,:,::-1].copy()) # BGR for YOLO
                        cropped+=1
                if not crops:
                    crops=[np.array(rgb)[:,:,::-1].copy()]
                    mode='FULL_IMAGE_FALLBACK_NO_F2_BOX'
                else: mode='F2_MATCHED_OBJECT_CROPS'
            total=0; confidences=[]
            for crop in crops:
                predictions=model.predict(source=crop,conf=0.25,verbose=False,imgsz=640)
                for p in predictions:
                    if p.boxes is not None:
                        total+=len(p.boxes)
                        confidences.extend(p.boxes.conf.cpu().numpy().tolist())
            rows.append({**base,'input_mode':mode,'f3_candidate_boxes':total,
                         'f3_max_confidence':round(max(confidences),4) if confidences else '',
                         'status':'LARVAE_CANDIDATE_REVIEW_REQUIRED' if total else 'NO_LARVAE_DETECTED_REVIEW_REQUIRED'})
        integrated=pd.DataFrame(rows)
        integrated.to_csv(OUT/'F3_F2_Two_Stage_Candidates.csv',index=False)
        print('Saved NEW candidate report:',OUT/'F3_F2_Two_Stage_Candidates.csv')
        print('Status counts:',integrated.status.value_counts().to_dict())
        print('Input modes:',integrated.input_mode.value_counts().to_dict())
        SUMMARY['two_stage']='EXECUTED_WITH_BLOCKED_IMAGES' if integrated.status.str.startswith('BLOCKED').any() else 'EXECUTED_NEEDS_HUMAN_REVIEW'

````

### Original cell 39/118 — PYTHON SOURCE (archived, not run)

````python
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def canonical_stem(p):
    s=p.stem.lower().split('.rf.')[0]
    return re.sub(r'_(jpg|jpeg|png|webp)$','',s)

def image_files(folder):
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]) if folder.is_dir() else []

# Never substitute valid/ for independent test.
test_candidates=[
    DATA_ROOT/'F3_INDEPENDENT_TEST/test',
    F3_DATASET/'test',
]
test_root=next((p for p in test_candidates if image_files(p/'images') and (p/'labels').is_dir()),None)
if not F3_WEIGHTS.exists():
    print('BLOCKED: existing F3 model missing.')
    SUMMARY['independent_test']='BLOCKED_MODEL_MISSING'
elif test_root is None:
    print('BLOCKED_NO_INDEPENDENT_TEST: No separate labeled test split was found.')
    print('Expected folder:',DATA_ROOT/'F3_INDEPENDENT_TEST/test/images')
    print('And YOLO labels:',DATA_ROOT/'F3_INDEPENDENT_TEST/test/labels')
    SUMMARY['independent_test']='BLOCKED_NO_INDEPENDENT_TEST'
else:
    test_imgs=image_files(test_root/'images')
    missing_labels=[p.name for p in test_imgs if not (test_root/'labels'/(p.stem+'.txt')).exists()]
    if missing_labels:
        print('BLOCKED_UNLABELED_TEST: missing YOLO label files (negative images need empty .txt):',missing_labels[:15])
        SUMMARY['independent_test']='BLOCKED_UNLABELED_TEST'
    else:
        reference=[]
        for split in ['train','valid','val']:
            reference.extend(image_files(F3_DATASET/split/'images'))
        if not reference:
            print('BLOCKED: original F3 training/validation images are unavailable; cannot check leakage.')
            SUMMARY['independent_test']='BLOCKED_REFERENCE_IMAGES_MISSING'
        else:
            known_hashes={sha256(p) for p in reference}
            known_stems={canonical_stem(p) for p in reference}
            exact_overlap=[p.name for p in test_imgs if sha256(p) in known_hashes]
            stem_overlap=[p.name for p in test_imgs if canonical_stem(p) in known_stems]
            # Reject source-level duplicates as well as exact bytes; near-duplicates may still need manual review.
            if exact_overlap or stem_overlap:
                print('BLOCKED_LEAKAGE: test overlaps training/validation.')
                print('Exact duplicate images:',exact_overlap[:20])
                print('Source-filename overlaps:',stem_overlap[:20])
                SUMMARY['independent_test']='BLOCKED_POTENTIAL_LEAKAGE'
            else:
                bad=[]; positive=0
                for p in test_imgs:
                    lab=test_root/'labels'/(p.stem+'.txt')
                    for ln,line in enumerate(lab.read_text().splitlines(),1):
                        parts=line.split()
                        try:
                            values=list(map(float,parts))
                            if len(values)!=5 or values[0]!=0 or any(not 0<=v<=1 for v in values[1:]): raise ValueError()
                            positive+=1
                        except Exception:bad.append(f'{lab.name}:{ln}')
                if bad or positive==0:
                    print('BLOCKED: bad labels or zero annotated larvae.',bad[:10], 'positive objects:',positive)
                    SUMMARY['independent_test']='BLOCKED_BAD_OR_EMPTY_GT'
                else:
                    import yaml
                    test_yaml=OUT/'F3_Independent_Test_data.yaml'
                    with test_yaml.open('w') as f:
                        yaml.safe_dump({'path':str(test_root.parent),'train':'test/images',
                                        'val':'test/images','test':'test/images',
                                        'nc':1,'names':['mosquito-larvae']},f,sort_keys=False)
                    print('Independent images:',len(test_imgs),'| GT larvae boxes:',positive)
                    try:
                        model=YOLO(str(F3_WEIGHTS))
                        metrics=model.val(data=str(test_yaml),split='test',imgsz=640,plots=False,
                                          project=str(OUT),name='F3_Independent_Eval',exist_ok=False,verbose=False)
                        scores={'images':len(test_imgs),'ground_truth_boxes':positive,
                                'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),
                                'map50':float(metrics.box.map50),'map50_95':float(metrics.box.map)}
                        (OUT/'F3_Independent_Test_Metrics.json').write_text(json.dumps(scores,indent=2))
                        print('Independent test results:',scores)
                        SUMMARY['independent_test']='EVALUATED_INDEPENDENT_LABELED_SET'
                    except Exception as ex:
                        print('Evaluation failed:',repr(ex))
                        SUMMARY['independent_test']='EVALUATION_ERROR'

````

### Original cell 40/118 — PYTHON SOURCE (archived, not run)

````python
# Cell 5 — Final run summary (new file only)
summary_path=OUT/'F3_Completion_Audit_Summary.json'
summary_path.write_text(json.dumps(SUMMARY,indent=2))
print(json.dumps(SUMMARY,indent=2))
print('\nPlease copy these 3 status values and send them in ChatGPT:')
print('review_audit:',SUMMARY['review_audit'])
print('two_stage:',SUMMARY['two_stage'])
print('independent_test:',SUMMARY['independent_test'])
print('Reports folder:',OUT)

````

### Original cell 41/118 — PYTHON SOURCE (archived, not run)

````python
# 1/3 — Mount Drive, read original evidence (read-only)
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import os, sys, json, hashlib, io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Google Colab; for local automated testing, DH_DRIVE_ROOT can override the path.
if not os.environ.get('DH_DRIVE_ROOT'):
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE = Path(os.environ.get('DH_DRIVE_ROOT','/content/drive/MyDrive'))
F2_CSV = DRIVE/'DENGU_HUNTER_FINAL_SHOWCASE/Unified_AI_Integration/unified_ai_evidence_register.csv'
F3_DIR = DRIVE/'DENGUE_HUNTER_RESULTS/Larvae_Model/F3_Dashboard_Evidence'
F3_PRED = F3_DIR/'F3_Larvae_Predictions.csv'
F3_REVIEW = F3_DIR/'F3_Human_Review.csv'
F3_MODEL = DRIVE/'DENGUE_HUNTER_RESULTS/Larvae_Model/DengueHunter_Larvae_v1/weights/best.pt'
F3_VALID = DRIVE/'DENGUE_HUNTER_DATA/LARVAE_MODEL_PREPARATION/YOLO_Dataset/Larvae_Detection_Ready/valid/images'
OUT = DRIVE/'DENGUE_HUNTER_RESULTS/Larvae_Model/F3_Hackathon_Final_Runs'/('Run_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))

required = [F2_CSV, F3_PRED, F3_REVIEW, F3_MODEL]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Existing source(s) missing; originals were NOT changed:\n'+'\n'.join(missing))
OUT.mkdir(parents=True,exist_ok=False)
print('All 4 existing F2/F3 sources found. NO retraining.')
print('New outputs only:',OUT)

````

### Original cell 42/118 — PYTHON SOURCE (archived, not run)

````python
# 2/3 — Create final F3 evidence pack and pitch figure
f2 = pd.read_csv(F2_CSV, dtype=str).fillna('')
pred = pd.read_csv(F3_PRED, dtype=str).fillna('')
review = pd.read_csv(F3_REVIEW, dtype=str).fillna('')
for label, df, columns in [
    ('F2', f2, ['image','association']),
    ('F3 predictions',pred,['image','class_name','confidence']),
    ('F3 human review',review,['image','human_decision'])]:
    missing_cols=set(columns)-set(df.columns)
    if missing_cols: raise ValueError(f'{label}: missing CSV columns: {missing_cols}')

# Stage 1: REAL PRECOMPUTED F2 evidence. Original F2 images are not required.
f2['image']=f2['image'].str.strip()
f2['is_water_matched']=f2['association'].str.strip().str.upper().eq('MATCHED')
site_rows=[]
for image_name, group in f2.groupby('image',sort=True):
    matches=group[group.is_water_matched]
    site_rows.append({
        'f2_site_image':image_name,
        'f2_detected_objects':len(group),
        'f2_water_matched_objects':len(matches),
        'f2_water_evidence':'MATCHED' if len(matches) else 'NOT_MATCHED',
        'f3_closeup_requested':bool(len(matches)),
        'f3_closeup_received':'NOT_LINKED_TO_F2_SITE',
        'site_to_sample_pairing':'UNVERIFIED',
        'workflow_next_step':'COLLECT_CLOSEUP_AND_RUN_LARVAE_AI' if len(matches) else 'MANUAL_INSPECTION_IF_NEEDED'
    })
sites=pd.DataFrame(site_rows)
eligible=sites.loc[sites.f3_closeup_requested,'f2_site_image'].tolist()
if not eligible: raise ValueError('F2 CSV has no water-associated site; cannot demonstrate F2 gate.')

# Stage 2: Existing actual F3 predictions and completed human decisions (DIFFERENT images).
pred['image']=pred['image'].str.strip()
pred['numeric_confidence']=pd.to_numeric(pred['confidence'],errors='coerce')
pred['is_ai_box']=(pred.numeric_confidence.notna() &
                   pred.class_name.str.strip().str.lower().eq('mosquito-larvae'))
per_image=[]
for image_name, group in pred.groupby('image',sort=True):
    boxes=group.loc[group.is_ai_box]
    per_image.append({'image':image_name,'ai_boxes':len(boxes),
                      'ai_detected_larvae_candidate':bool(len(boxes)),
                      'ai_max_confidence':round(float(boxes.numeric_confidence.max()),4) if len(boxes) else None,
                      'model_class':'mosquito-larvae',
                      'species_verified':'NO'})
per_image=pd.DataFrame(per_image)
review['image']=review['image'].str.strip()
review['human_decision']=review['human_decision'].str.strip().str.upper()
if review.image.duplicated().any(): raise ValueError('Duplicate F3 review image.')
if not set(review.human_decision).issubset({'AGREE','DISAGREE','UNCERTAIN'}):
    raise ValueError('Unexpected F3 human decision value.')
if set(review.image)!=set(per_image.image):
    raise ValueError('Human Review and Prediction image sets differ; inspect originals.')
stage2=per_image.merge(review[['image','human_decision']],on='image',validate='one_to_one')
stage2['review_interpretation']=stage2.human_decision.map({
    'AGREE':'HUMAN_AGREES_WITH_AI_NOT_SPECIES_PROOF',
    'DISAGREE':'HUMAN_DISAGREES_REQUIRES_REVIEW',
    'UNCERTAIN':'REQUIRES_ADDITIONAL_HUMAN_REVIEW'})
stage2['f2_site_link']='NO_PAIRED_SITE_LINK_IN_EXISTING_DATA'

sites.to_csv(OUT/'F3_Stage1_Water_Matched_Sites.csv',index=False)
stage2.to_csv(OUT/'F3_Stage2_Closeup_AI_Human_Review.csv',index=False)
counts=review.human_decision.value_counts().to_dict()
metrics={
  'feature':'F3 — Two-Stage Larvae Verification',
  'hackathon_status':'READY_EXISTING_EVIDENCE_AND_LIVE_DEMO',
  'f2_sites_total':int(len(sites)),
  'f2_sites_water_matched':int(len(eligible)),
  'f2_matched_object_evidence':int(f2.is_water_matched.sum()),
  'f3_closeup_images_reviewed':int(len(stage2)),
  'f3_images_with_ai_boxes':int((stage2.ai_boxes>0).sum()),
  'f3_ai_boxes':int(stage2.ai_boxes.sum()),
  'f3_images_without_ai_boxes':int((stage2.ai_boxes==0).sum()),
  'human_review_counts':{k:int(counts.get(k,0)) for k in ['AGREE','DISAGREE','UNCERTAIN']},
  'human_review_audit':'PASS_AS_PREVIOUSLY_VERIFIED_IN_COLAB',
  'f2_to_f3_same_site_pairing':'NOT_ESTABLISHED_BY_EXISTING_DATA',
  'independent_test':'NOT_PERFORMED_NOT_REQUIRED_FOR_HACKATHON_DEMO',
  'aedes_species_identification':'NOT_ESTABLISHED',
  'model_weights':str(F3_MODEL),
  'original_files_modified':False,
}
(OUT/'F3_Hackathon_Evidence_Summary.json').write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding='utf-8')

fig,axes=plt.subplots(1,3,figsize=(14,4.5))
fig.suptitle('DENGU HUNTER  |  F3 Two-Stage Verification — Hackathon Evidence',fontsize=15,weight='bold')
axes[0].bar(['F2 sites','Water-match\nsites'],[len(sites),len(eligible)],color=['#294f6b','#16a085'])
axes[0].set_title('Stage 1 · Breeding-site evidence');axes[0].set_ylim(0,max(1,len(sites))*1.2)
for i,n in enumerate([len(sites),len(eligible)]):axes[0].text(i,n+.2,str(n),ha='center',weight='bold')
axes[1].bar(['Close-ups','AI positive\nimages','AI boxes'],[len(stage2),int((stage2.ai_boxes>0).sum()),int(stage2.ai_boxes.sum())],color=['#294f6b','#16a085','#db9833'])
axes[1].set_title('Stage 2 · Existing larvae AI evidence');axes[1].set_ylim(0,max(1,len(stage2))*1.3)
for i,n in enumerate([len(stage2),int((stage2.ai_boxes>0).sum()),int(stage2.ai_boxes.sum())]):axes[1].text(i,n+.2,str(n),ha='center',weight='bold')
axes[2].bar(['Agree','Disagree','Uncertain'],[counts.get('AGREE',0),counts.get('DISAGREE',0),counts.get('UNCERTAIN',0)],color=['#16a085','#b64d55','#db9833'])
axes[2].set_title('Human-in-the-loop · 20 decisions');axes[2].set_ylim(0,max(1,len(stage2))*0.85)
for i,k in enumerate(['AGREE','DISAGREE','UNCERTAIN']):axes[2].text(i,counts.get(k,0)+.2,str(counts.get(k,0)),ha='center',weight='bold')
for ax in axes:ax.spines[['top','right']].set_visible(False)
fig.text(.5,.015,'Important: F2 sites and F3 validation close-ups are NOT paired. Human review does not confirm Aedes species.',ha='center',fontsize=9,color='#8c3131')
fig.tight_layout(rect=[0,.05,1,.92]);fig.savefig(OUT/'F3_Hackathon_Evidence_Board.png',dpi=160,bbox_inches='tight');plt.show();plt.close(fig)

readme=f"""# F3 Hackathon Demo — Generated from Existing Project Data

Stage 1: {len(sites)} F2 sites; {len(eligible)} have water-matched evidence; {int(f2.is_water_matched.sum())} matched objects.
Stage 2: {len(stage2)} independent F3 validation close-up examples; {int(stage2.ai_boxes.sum())} model boxes across {int((stage2.ai_boxes>0).sum())} images.
Human review: AGREE {counts.get('AGREE',0)}, DISAGREE {counts.get('DISAGREE',0)}, UNCERTAIN {counts.get('UNCERTAIN',0)}.

DEMO FLOW: Choose F2 water-matched site -> provide close-up image -> live existing larvae YOLO inference -> human review -> save new CSV.

The 20 historical close-ups are NOT from the 15 F2 sites as far as existing evidence establishes. Using them is an unpaired WORKFLOW DEMO. A new uploaded close-up is not independently site-verified either. Model class is generic mosquito-larvae; no Aedes species identification. This is not an independent evaluation.

All originals are read-only. The new live review file, if any, is F3_Live_Demo_Reviews.csv in this same run folder.
"""
(OUT/'F3_README_LIMITATIONS.md').write_text(readme,encoding='utf-8')
print('F3 HACKATHON EVIDENCE READY')
print('F2 sites:',len(sites),'| water-matched sites:',len(eligible),'| water-matched objects:',int(f2.is_water_matched.sum()))
print('F3 close-ups:',len(stage2),'| boxes:',int(stage2.ai_boxes.sum()),'| review:',counts)
print('All outputs saved to:',OUT)

````

### Original cell 43/118 — PYTHON SOURCE (archived, not run)

````python
# 3/3 — Launch the hackathon interaction: real F3 model + human approval
import importlib.util, subprocess
for package in ['ultralytics','gradio']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
from ultralytics import YOLO
import gradio as gr

model=YOLO(str(F3_MODEL))  # Existing best.pt; no train()

sample_roots=[F3_VALID, F3_VALID.parent.parent/'val/images', F3_DIR]
example_files=[]
for root in sample_roots:
    if root.is_dir():
        example_files=[p for p in sorted(root.rglob('*')) if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
        if example_files: break
# Prefer an existing image whose filename appears in the reviewed F3 CSV.
known=set(stage2.image)
example_files=sorted(example_files,key=lambda p:(p.name not in known,p.name))
default_example=str(example_files[0]) if example_files else None
print('Sample close-up found:',bool(default_example),'| count:',len(example_files))

LIVE_CSV=OUT/'F3_Live_Demo_Reviews.csv'
LIVE_COLS=['timestamp_utc','selected_f2_site','f2_water_match','closeup_sha256','ai_candidate_boxes',
           'ai_max_confidence','model_class','human_decision','site_pairing_claim',
           'aedes_species_verified','model_weights','source_note']

def analyze(site,closeup):
    if site not in eligible: return None,'Choose a water-associated F2 site.',{}
    if closeup is None: return None,'Upload a close-up or use the example already loaded.',{}
    if not isinstance(closeup,Image.Image): closeup=Image.fromarray(np.asarray(closeup).astype('uint8')).convert('RGB')
    rgb=closeup.convert('RGB')
    buf=io.BytesIO();rgb.save(buf,format='PNG')
    digest=hashlib.sha256(buf.getvalue()).hexdigest()
    bgr=np.asarray(rgb)[:,:,::-1].copy()
    result=model.predict(source=bgr,conf=.25,imgsz=640,verbose=False)[0]
    num=int(len(result.boxes)) if result.boxes is not None else 0
    max_conf=float(result.boxes.conf.max().item()) if num else None
    annotated=Image.fromarray(result.plot()[:,:,::-1])
    note=(f'F2 site: water-associated evidence exists. F3 AI: {num} generic mosquito-larvae candidate box(es). '
          + (f'Max confidence: {max_conf:.1%}. ' if max_conf is not None else 'No detection; manual review still required. ')
          + 'Human review required. NOT Aedes species confirmation. Site-to-sample pairing is unverified unless attested.')
    state={'site':site,'sha256':digest,'boxes':num,'max_conf':max_conf}
    return annotated,note,state

def save_review(site,decision,attested,state):
    if not state or not state.get('sha256'):
        return 'Run Analyze first. No review was saved.'
    if site!=state.get('site'):
        return 'Selected site changed. Run Analyze again before saving.'
    if decision not in {'AGREE','DISAGREE','UNCERTAIN'}:
        return 'Choose a Human Review decision first.'
    pairing='USER_ATTESTED_SAME_SITE_NOT_INDEPENDENTLY_VERIFIED' if attested else 'UNVERIFIED_DEMO_PAIR'
    row={'timestamp_utc':datetime.now(timezone.utc).isoformat(),
         'selected_f2_site':site,'f2_water_match':'MATCHED',
         'closeup_sha256':state['sha256'],'ai_candidate_boxes':state['boxes'],
         'ai_max_confidence':state['max_conf'],'model_class':'mosquito-larvae',
         'human_decision':decision,'site_pairing_claim':pairing,
         'aedes_species_verified':'NO','model_weights':str(F3_MODEL),
         'source_note':'USER_UPLOADED_OR_UNPAIRED_VALIDATION_EXAMPLE'}
    pd.DataFrame([row],columns=LIVE_COLS).to_csv(
        LIVE_CSV,mode='a',header=not LIVE_CSV.exists(),index=False)
    return f'SAVED ✓  {decision} | {state["boxes"]} candidate box(es) | new file: {LIVE_CSV.name}'

with gr.Blocks(title='DENGU HUNTER — F3 Live Larvae Verification') as demo:
    gr.Markdown('## 🦟 DENGU HUNTER | F3 — Two-Stage Larvae Verification')
    gr.Markdown('**Stage 1:** F2 water evidence → **Stage 2:** close-up YOLO larvae detection → **Human review**. Hackathon prototype, not Aedes-species diagnosis.')
    with gr.Row():
        site=gr.Dropdown(choices=eligible,value=eligible[0],label='Stage 1 · F2 water-associated site')
        photo=gr.Image(type='pil',value=default_example,label='Stage 2 · Close-up sample (preloaded example is NOT from selected F2 site)')
    run=gr.Button('🔍 Analyze close-up with existing YOLO model',variant='primary')
    with gr.Row():
        annotated=gr.Image(type='pil',label='AI bounding boxes')
        explanation=gr.Textbox(label='AI result and limitations',lines=5)
    state=gr.State({})
    run.click(analyze,inputs=[site,photo],outputs=[annotated,explanation,state])
    gr.Markdown('### Human Verification')
    with gr.Row():
        decision=gr.Radio(['AGREE','DISAGREE','UNCERTAIN'],label='Do you agree with the AI result?')
        pairing=gr.Checkbox(label='I confirm this close-up came from the selected F2 site (user-attested only)',value=False)
    save=gr.Button('💾 Save human review to NEW Drive CSV')
    saved=gr.Textbox(label='Save status')
    save.click(save_review,inputs=[site,decision,pairing,state],outputs=saved)
    gr.Markdown('**Note:** F3 detects generic mosquito-larvae candidates, not confirmed Aedes. Existing F2 site images and F3 sample images are unpaired. Original files stay unchanged.')

print('F3 Hackathon interactive demo starting. Keep this Colab runtime open during presentation.')
demo.launch(share=True,show_error=True,debug=False)

````

### Original cell 44/118 — PYTHON SOURCE (archived, not run)

````python
from google.colab import files

print("Upload 2 images: Overview + Close-up")
uploaded = files.upload()

print("\nUploaded images:")
for name in uploaded:
    print(name)
````

### Original cell 45/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Google Drive
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Existing F4 data — read only
source = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE/"
    "Development_Batch_Features_3_4_6_8/"
    "predictive_hotspot_demo.csv"
)

assert source.exists(), f"F4 CSV not found: {source}"

df = pd.read_csv(source)

required = [
    "location_id",
    "rainfall_mm",
    "temperature_c",
    "water_holding_evidence",
    "demonstration_risk_score",
    "demonstration_risk_category"
]

missing = set(required) - set(df.columns)
assert not missing, f"Missing columns: {missing}"

for c in required[1:5]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert df[required].notna().all().all(), "Some required values are missing"
assert df["location_id"].is_unique, "Duplicate location IDs"
assert df["demonstration_risk_score"].between(0, 1).all()
assert df["water_holding_evidence"].isin([0, 1]).all()

# Risk categories for demonstration
def risk_category(score):
    return np.select(
        [score >= 0.65, score >= 0.35],
        ["HIGH", "MEDIUM"],
        default="LOW"
    )

# Preserve original scores
df["current_risk_score"] = df["demonstration_risk_score"]
df["current_risk_category"] = df["demonstration_risk_category"]

# Illustrative future weather scenarios, NOT real forecasts
scenarios = {
    "3d": (15, 0.5),
    "7d": (30, 1.0)
}

for period, (extra_rain, extra_temp) in scenarios.items():
    df[f"{period}_rainfall_mm"] = df["rainfall_mm"] + extra_rain
    df[f"{period}_temperature_c"] = df["temperature_c"] + extra_temp

    rain_effect = (
        0.35 * (extra_rain / 100)
        * (0.4 + 0.6 * df["water_holding_evidence"])
    )
    temp_effect = 0.25 * (extra_temp / 20)

    df[f"{period}_risk_score"] = (
        df["current_risk_score"] + rain_effect + temp_effect
    ).clip(0, 1).round(4)

    df[f"{period}_risk_category"] = risk_category(
        df[f"{period}_risk_score"]
    )

df["current_priority_rank"] = (
    df["current_risk_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

df["future_priority_rank"] = (
    df["7d_risk_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

df = df.sort_values("future_priority_rank")

# Save only NEW files
output = (
    Path("/content/drive/MyDrive/DENGU_HUNTER_RESULTS")
    / "F4_Hackathon_Runs"
    / datetime.now().strftime("Run_%Y%m%d_%H%M%S_%f")
)
output.mkdir(parents=True, exist_ok=True)

df.to_csv(output / "F4_Future_Risk_Analysis.csv", index=False)

# F5-ready mission priorities
f5 = df[[
    "location_id",
    "future_priority_rank",
    "7d_risk_score",
    "7d_risk_category"
]].rename(columns={
    "future_priority_rank": "priority_rank",
    "7d_risk_score": "risk_score",
    "7d_risk_category": "risk_category"
})

f5.to_csv(output / "F4_To_F5_Priority.csv", index=False)

# Risk comparison chart
plot_df = df.sort_values("current_risk_score")

fig, ax = plt.subplots(figsize=(11, 6))
y = np.arange(len(plot_df))

ax.barh(y - 0.2, plot_df["current_risk_score"],
        height=0.4, label="Current simulated risk")
ax.barh(y + 0.2, plot_df["7d_risk_score"],
        height=0.4, label="7-day what-if scenario")

ax.set_yticks(y)
ax.set_yticklabels(plot_df["location_id"])
ax.set_xlim(0, 1)
ax.set_xlabel("Risk Score")
ax.set_title("DENGU HUNTER F4 — Simulated Hotspot Risk")
ax.legend()
plt.tight_layout()
plt.savefig(output / "F4_Risk_Comparison.png", dpi=150)
plt.show()

print("\nF4 DATA LOADED:", len(df), "areas")
print("Current categories:",
      df["current_risk_category"].value_counts().to_dict())
print("7-day scenario categories:",
      df["7d_risk_category"].value_counts().to_dict())

print("\nTOP 5 FUTURE PRIORITY AREAS:")
print(f5.head(5).to_string(index=False))

print("\nF4 REPORTS SAVED:", output)
print("STATUS: F4 SIMULATION GENERATED")
print("NOTE: Simulated inputs and hypothetical weather scenarios.")
print("NOT a trained or validated dengue forecasting model.")

````

### Original cell 46/118 — PYTHON SOURCE (archived, not run)

````python

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

days = widgets.ToggleButtons(
    options=[("3 Days", 3), ("7 Days", 7)],
    value=7,
    description="Scenario:"
)

rain = widgets.IntSlider(
    value=30, min=0, max=100, step=5,
    description="Extra Rain (mm):",
    style={"description_width": "initial"}
)

temp = widgets.FloatSlider(
    value=1.0, min=-2, max=4, step=0.5,
    description="Temp Change (C):",
    style={"description_width": "initial"}
)

output = widgets.Output()

def change_days(change):
    if change["new"] == 3:
        rain.value = 15
        temp.value = 0.5
    else:
        rain.value = 30
        temp.value = 1.0

days.observe(change_days, names="value")

def simulate(_=None):
    with output:
        clear_output(wait=True)

        result = df.copy()

        result["new_risk"] = (
            result["current_risk_score"]
            + 0.35 * (rain.value / 100) *
              (0.4 + 0.6 * result["water_holding_evidence"])
            + 0.25 * (temp.value / 20)
        ).clip(0, 1)

        result["category"] = np.select(
            [result["new_risk"] >= 0.65,
             result["new_risk"] >= 0.35],
            ["HIGH", "MEDIUM"],
            default="LOW"
        )

        result = result.sort_values(
            ["new_risk", "location_id"],
            ascending=[False, True]
        )

        result["rank"] = range(1, len(result) + 1)

        print(f"{days.value}-DAY WHAT-IF SCENARIO")
        print("Risk counts:",
              result["category"].value_counts().to_dict())
        print("\nTOP 5 PRIORITY AREAS")

        display(result[[
            "rank", "location_id",
            "current_risk_score",
            "new_risk", "category"
        ]].head(5))

        top = result.head(10).iloc[::-1]

        plt.figure(figsize=(9, 5))
        plt.barh(top["location_id"], top["new_risk"])
        plt.xlim(0, 1)
        plt.xlabel("Simulated Risk Score")
        plt.title("DENGU HUNTER — Live Hotspot Ranking")
        plt.tight_layout()
        plt.show()

button = widgets.Button(
    description="Update Risk Simulation",
    button_style="success"
)
button.on_click(simulate)

display(days, rain, temp, button, output)
simulate()

````

### Original cell 47/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE")

f2_path = base / "Unified_AI_Integration" / "unified_ai_evidence_register.csv"
f4_path = base / "Development_Batch_Features_3_4_6_8" / "predictive_hotspot_demo.csv"

assert f2_path.exists(), "F2 evidence CSV not found!"
assert f4_path.exists(), "F4 hotspot CSV not found!"

f2 = pd.read_csv(f2_path)
f4 = pd.read_csv(f4_path)

print("=== F2 EXISTING EVIDENCE ===")
print("Total detection records:", len(f2))
print("Unique images:", f2["image"].nunique())
print("Columns:", f2.columns.tolist())

print("\nAssociation types:")
print(f2["association"].value_counts(dropna=False))

print("\n=== F4 EXISTING AREAS ===")
print("Total areas:", len(f4))
print("Columns:", f4.columns.tolist())

# Check whether the two datasets refer to the same images
def image_name(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\\", "/").split("/")[-1].lower().strip()

f2_names = set(f2["image"].map(image_name)) - {""}

if "associated_image" in f4.columns:
    f4_names = set(f4["associated_image"].map(image_name)) - {""}
    common = sorted(f2_names & f4_names)
    print("\n=== F2-F4 IMAGE MATCH CHECK ===")
    print("Matching image filenames:", len(common))
    print("Examples:", common[:10])
else:
    print("\nF4 has no associated_image column.")

print("\n=== F2 SAMPLE ===")
display(f2[[
    "image", "object", "association",
    "inspection_priority"
]].head(10))

print("\n=== F4 SAMPLE ===")
display(f4[[
    "location_id", "associated_image",
    "water_holding_evidence"
]].head(10))

print("\nSTEP 3 DATA CHECK COMPLETED")

````

### Original cell 48/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE")

f2 = pd.read_csv(
    base / "Unified_AI_Integration" /
    "unified_ai_evidence_register.csv"
)

f4 = pd.read_csv(
    base / "Development_Batch_Features_3_4_6_8" /
    "predictive_hotspot_demo.csv"
)

def image_key(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\\", "/").split("/")[-1].strip().lower()

f2["image_key"] = f2["image"].apply(image_key)
f4["image_key"] = f4["associated_image"].apply(image_key)

f2["is_matched"] = (
    f2["association"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("MATCHED")
)

# Aggregate real F2 detection records by image
summary = f2.groupby("image_key").agg(
    total_f2_objects=("image_key", "size"),
    matched_objects=("is_matched", "sum")
).reset_index()

summary["unmatched_objects"] = (
    summary["total_f2_objects"] -
    summary["matched_objects"]
)

summary["f2_water_flag"] = (
    summary["matched_objects"] > 0
).astype(int)

# Connect F2 evidence to F4 by existing image references
integrated = f4.merge(
    summary,
    on="image_key",
    how="left",
    validate="one_to_one"
)

integrated["flag_match"] = (
    integrated["water_holding_evidence"] ==
    integrated["f2_water_flag"]
)

print("=== F2 + F4 INTEGRATION AUDIT ===")

print("Total F4 areas:", len(integrated))
print("Areas linked to F2 evidence:",
      integrated["total_f2_objects"].notna().sum())

print("Total F2 objects:",
      integrated["total_f2_objects"].sum())

print("Water-matched objects:",
      integrated["matched_objects"].sum())

print("F2 water-positive images:",
      integrated["f2_water_flag"].sum())

print("F4 water-positive flags:",
      integrated["water_holding_evidence"].sum())

print("Flag agreement:",
      integrated["flag_match"].sum(),
      "/", len(integrated))

print("\n=== AREA-WISE EVIDENCE CHECK ===")

display(integrated[[
    "location_id",
    "total_f2_objects",
    "matched_objects",
    "unmatched_objects",
    "f2_water_flag",
    "water_holding_evidence",
    "flag_match"
]])

print("\n=== FLAG MISMATCHES ===")

mismatches = integrated[
    integrated["flag_match"] == False
]

if mismatches.empty:
    print("PASS: All F4 water flags agree with F2 evidence.")
else:
    display(mismatches[[
        "location_id",
        "f2_water_flag",
        "water_holding_evidence"
    ]])

print("\nAUDIT FINISHED — No original files modified.")

````

### Original cell 49/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

base = Path("/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE")

f2 = pd.read_csv(
    base / "Unified_AI_Integration/unified_ai_evidence_register.csv"
)

f4 = pd.read_csv(
    base / "Development_Batch_Features_3_4_6_8/predictive_hotspot_demo.csv"
)

def key(x):
    return str(x).replace("\\", "/").split("/")[-1].lower().strip()

f2["image_key"] = f2["image"].apply(key)
f4["image_key"] = f4["associated_image"].apply(key)

f2["matched"] = (
    f2["association"].astype(str).str.strip().str.upper()
    == "MATCHED"
)

evidence = f2.groupby("image_key").agg(
    total_f2_objects=("matched", "size"),
    matched_objects=("matched", "sum")
).reset_index()

evidence["f2_water_flag"] = (
    evidence["matched_objects"] > 0
).astype(int)

corrected = f4.merge(
    evidence, on="image_key",
    how="left", validate="one_to_one"
)

assert corrected["f2_water_flag"].notna().all()
assert len(corrected) == 15

# Verify the existing demonstration scoring formula
old_flag = corrected["water_holding_evidence"].astype(float)
rainfall = corrected["rainfall_mm"].astype(float)
temperature = corrected["temperature_c"].astype(float)

original_formula = (
    0.40 * old_flag
    + 0.35 * (rainfall / 100)
    + 0.25 * ((temperature - 20) / 20)
).clip(0, 1)

difference = (
    original_formula - corrected["demonstration_risk_score"]
).abs().max()

print("Original formula max difference:", round(difference, 5))

if difference > 0.02:
    raise ValueError(
        "Original risk formula differs. Send this output before recalculating."
    )

# Correct evidence using actual F2 detection records
corrected["original_water_flag"] = old_flag.astype(int)
corrected["water_holding_evidence"] = corrected["f2_water_flag"].astype(int)

# Recalculate current risk
corrected["current_risk_score"] = (
    0.40 * corrected["water_holding_evidence"]
    + 0.35 * (rainfall / 100)
    + 0.25 * ((temperature - 20) / 20)
).clip(0, 1).round(4)

def category(scores):
    return np.select(
        [scores >= 0.65, scores >= 0.35],
        ["HIGH", "MEDIUM"],
        default="LOW"
    )

corrected["current_risk_category"] = category(
    corrected["current_risk_score"]
)

# Hypothetical scenarios, not real weather forecasts
for period, extra_rain, extra_temp in [
    ("3d", 15, 0.5),
    ("7d", 30, 1.0)
]:
    rain_effect = (
        0.35 * extra_rain / 100
        * (0.4 + 0.6 * corrected["water_holding_evidence"])
    )
    temp_effect = 0.25 * extra_temp / 20

    corrected[f"{period}_risk_score"] = (
        corrected["current_risk_score"]
        + rain_effect + temp_effect
    ).clip(0, 1).round(4)

    corrected[f"{period}_risk_category"] = category(
        corrected[f"{period}_risk_score"]
    )

# Future priority ranking
corrected = corrected.sort_values(
    ["7d_risk_score", "location_id"],
    ascending=[False, True]
).reset_index(drop=True)

corrected["priority_rank"] = np.arange(1, len(corrected) + 1)

# Save in NEW folder
folder = (
    Path("/content/drive/MyDrive/DENGU_HUNTER_RESULTS")
    / "F4_Hackathon_Runs"
    / ("Corrected_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
)
folder.mkdir(parents=True, exist_ok=True)

corrected.to_csv(
    folder / "F4_Corrected_Integrated_Risk.csv",
    index=False
)

corrected[[
    "location_id", "priority_rank",
    "7d_risk_score", "7d_risk_category"
]].rename(columns={
    "7d_risk_score": "risk_score",
    "7d_risk_category": "risk_category"
}).to_csv(folder / "F4_To_F5_Priority.csv", index=False)

# Update notebook variable for interactive widget
df = corrected.copy()

print("\n=== CORRECTED F4 RESULTS ===")
print("Areas:", len(df))
print("F2 linked:", df["f2_water_flag"].notna().sum())
print("Water-positive areas:", df["water_holding_evidence"].sum())
print("Current risk:", df["current_risk_category"].value_counts().to_dict())
print("3-day risk:", df["3d_risk_category"].value_counts().to_dict())
print("7-day risk:", df["7d_risk_category"].value_counts().to_dict())

print("\nTOP 5 PRIORITY:")
display(df[[
    "priority_rank", "location_id",
    "matched_objects", "current_risk_score",
    "7d_risk_score", "7d_risk_category"
]].head(5))

print("\nSaved to:", folder)
print("Original files unchanged.")
print("F2 evidence integrated into simulated area scores.")

````

### Original cell 50/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_RESULTS/F4_Hackathon_Runs"
)

runs = sorted(
    p for p in root.glob("Corrected_*")
    if p.is_dir() and (p / "F4_Corrected_Integrated_Risk.csv").exists()
)
assert runs, "Corrected F4 run not found!"

folder = runs[-1]

df = pd.read_csv(folder / "F4_Corrected_Integrated_Risk.csv")
f5 = pd.read_csv(folder / "F4_To_F5_Priority.csv")

# 1. F2 integration check
check_f2 = (
    len(df) == 15
    and df["location_id"].is_unique
    and df["f2_water_flag"].equals(
        df["water_holding_evidence"]
    )
)

# 2. Rainfall and temperature impact
water = df["water_holding_evidence"]
base = df["current_risk_score"]

rain_impact = 0.35 * (30 / 100) * (0.4 + 0.6 * water)
temp_impact = 0.25 * (1.0 / 20)

df["rain_only_risk"] = (base + rain_impact).clip(0, 1)
df["temp_only_risk"] = (base + temp_impact).clip(0, 1)

# 3. Verify both future scenarios
expected_3d = (
    base
    + 0.35 * (15 / 100) * (0.4 + 0.6 * water)
    + 0.25 * (0.5 / 20)
).clip(0, 1)

expected_7d = (
    base + rain_impact + temp_impact
).clip(0, 1)

check_3d = np.allclose(
    df["3d_risk_score"], expected_3d, atol=0.00011
)

check_7d = np.allclose(
    df["7d_risk_score"], expected_7d, atol=0.00011
)

# 4. Risk category check
def category(scores):
    return np.select(
        [scores >= 0.65, scores >= 0.35],
        ["HIGH", "MEDIUM"],
        default="LOW"
    )

check_categories = (
    np.array_equal(
        df["current_risk_category"],
        category(df["current_risk_score"])
    )
    and np.array_equal(
        df["3d_risk_category"],
        category(df["3d_risk_score"])
    )
    and np.array_equal(
        df["7d_risk_category"],
        category(df["7d_risk_score"])
    )
)

# 5. F5 priority handoff check
check_f5 = (
    len(f5) == 15
    and f5["location_id"].tolist() == df["location_id"].tolist()
    and f5["priority_rank"].tolist() == list(range(1, 16))
    and np.allclose(
        f5["risk_score"], df["7d_risk_score"]
    )
    and f5["risk_category"].tolist()
        == df["7d_risk_category"].tolist()
)

tests = {
    "F2 Evidence Integration": check_f2,
    "3-Day Risk Simulation": check_3d,
    "7-Day Risk Simulation": check_7d,
    "Risk Category Consistency": check_categories,
    "F5 Priority CSV Handoff": check_f5
}

print("=== F4 FINAL TEST RESULTS ===")
for name, passed in tests.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

# 6. Weather impact chart
top = df.head(5).iloc[::-1]
y = np.arange(len(top))
w = 0.2

plt.figure(figsize=(10, 5))
plt.barh(y - w, top["current_risk_score"], w,
         label="Current Risk")
plt.barh(y, top["rain_only_risk"], w,
         label="Extra Rain Only")
plt.barh(y + w, top["temp_only_risk"], w,
         label="Temperature Change Only")

plt.yticks(y, top["location_id"])
plt.xlim(0, 1)
plt.xlabel("Simulated Risk Score")
plt.title("F4 — Rainfall vs Temperature Impact")
plt.legend()
plt.tight_layout()
plt.savefig(folder / "F4_Weather_Impact.png", dpi=150)
plt.show()

# Save new analysis, original source untouched
df.to_csv(folder / "F4_Final_Analysis.csv", index=False)

print("\n=== RISK DISTRIBUTION ===")
for label, col in [
    ("Current", "current_risk_category"),
    ("3-Day", "3d_risk_category"),
    ("7-Day", "7d_risk_category")
]:
    print(label, df[col].value_counts().to_dict())

print("\n=== FINAL RESULT ===")
print("Tests passed:", sum(tests.values()), "/", len(tests))
print("STATUS:", "ALL TESTS PASS" if all(tests.values())
      else "REVIEW REQUIRED")
print("Saved folder:", folder)
print("Original CSV files unchanged.")

````

### Original cell 51/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import os

drive = Path("/content/drive/MyDrive")

f2_path = (
    drive / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Unified_AI_Integration"
    / "unified_ai_evidence_register.csv"
)

f4_root = (
    drive / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
)

runs = sorted(
    p for p in f4_root.glob("Corrected_*")
    if (p / "F4_Corrected_Integrated_Risk.csv").exists()
)

assert f2_path.exists(), "F2 CSV missing!"
assert runs, "Corrected F4 CSV missing!"

f2 = pd.read_csv(f2_path)
f4 = pd.read_csv(
    runs[-1] / "F4_Corrected_Integrated_Risk.csv"
)

def image_key(x):
    return str(x).replace("\\", "/").split("/")[-1].lower().strip()

needed = set(f4["associated_image"].map(image_key))
found = {}

# Search only existing project folders
roots = [
    drive / "DENGU_HUNTER_FINAL_SHOWCASE",
    drive / "DENGUE_HUNTER_RESULTS"
]

for root in roots:
    if not root.exists():
        continue

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in [".git", ".ipynb_checkpoints"]
        ]

        for filename in files:
            name = filename.lower()

            if name in needed and name not in found:
                found[name] = str(Path(current) / filename)

        if len(found) == len(needed):
            break

print("=== F4 ENHANCEMENT IMAGE CHECK ===")
print("F2 detection records:", len(f2))
print("F4 areas:", len(f4))
print("Required images:", len(needed))
print("Images found:", len(found))
print("Images missing:", len(needed) - len(found))

print("\n=== SIM-AREA-011 ===")

area = f4[
    f4["location_id"] == "SIM-AREA-011"
].iloc[0]

key = image_key(area["associated_image"])

print("Image name:", key)
print("Image available:", key in found)
print("Image path:", found.get(key, "NOT FOUND"))
print("Matched F2 objects:", area["matched_objects"])
print("Current risk:", area["current_risk_score"])
print("7-day risk:", area["7d_risk_score"])
print("Priority rank:", area["priority_rank"])

print("\nSTEP 1 COMPLETED")

````

### Original cell 52/118 — PYTHON SOURCE (archived, not run)

````python

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Reuse data from previous successful cell
assert "f2" in globals(), "Run previous F2 data cell first"
assert "f4" in globals(), "Run previous F4 data cell first"
assert "found" in globals(), "Run previous image search cell first"

def get_image_key(value):
    return str(value).replace("\\", "/").split("/")[-1].lower().strip()

area_selector = widgets.Dropdown(
    options=sorted(f4["location_id"].unique()),
    value="SIM-AREA-011",
    description="Select Area:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

demo_output = widgets.Output()

def show_area(change=None):
    with demo_output:
        clear_output(wait=True)

        area_id = area_selector.value
        area = f4.loc[
            f4["location_id"] == area_id
        ].iloc[0]

        key = get_image_key(area["associated_image"])
        records = f2.loc[
            f2["image"].map(get_image_key) == key
        ].copy()

        matched = (
            records["association"]
            .astype(str)
            .str.strip()
            .str.upper()
            .eq("MATCHED")
        )

        print("=" * 50)
        print("DENGU HUNTER — F4 VISUAL EVIDENCE")
        print("=" * 50)
        print("Selected Area:", area_id)
        print("F2 Detected Objects:", len(records))
        print("Water-matched Objects:", int(matched.sum()))
        print("Unmatched Objects:", int((~matched).sum()))
        print("Water Evidence Flag:", area["water_holding_evidence"])

        print("\nCURRENT RISK:", area["current_risk_score"])
        print("3-DAY RISK:", area["3d_risk_score"])
        print("7-DAY RISK:", area["7d_risk_score"])
        print("7-DAY CATEGORY:", area["7d_risk_category"])
        print("DRONE PRIORITY RANK:", area["priority_rank"])

        path = found.get(key)

        if path:
            img = Image.open(path).convert("RGB")

            fig, axes = plt.subplots(
                1, 2, figsize=(12, 4.5)
            )

            axes[0].imshow(img)
            axes[0].set_title("F2 Evidence Image")
            axes[0].axis("off")

            labels = ["Current", "3 Days", "7 Days"]
            values = [
                area["current_risk_score"],
                area["3d_risk_score"],
                area["7d_risk_score"]
            ]

            axes[1].bar(labels, values)
            axes[1].set_ylim(0, 1)
            axes[1].set_ylabel("Simulated Risk Score")
            axes[1].set_title("F4 Hotspot Risk")

            for i, value in enumerate(values):
                axes[1].text(
                    i, value + 0.02,
                    f"{value:.4f}",
                    ha="center"
                )

            plt.tight_layout()
            plt.show()
        else:
            print("Image not available for this area.")

        print("\n=== F2 DETECTION EVIDENCE ===")

        columns = [
            "object",
            "object_confidence",
            "association",
            "mask_confidence",
            "inspection_priority"
        ]

        display(records[columns].reset_index(drop=True))

        print("\nEvidence: Existing F2 AI results")
        print("Area and weather: Simulated demonstration")
        print("Water match does NOT confirm mosquito larvae.")

area_selector.observe(show_area, names="value")

display(area_selector, demo_output)
show_area()

````

### Original cell 53/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
from pathlib import Path

results = []

for _, area in f4.iterrows():
    area_id = area["location_id"]
    key = get_image_key(area["associated_image"])

    records = f2[
        f2["image"].map(get_image_key) == key
    ]

    matched = (
        records["association"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("MATCHED")
        .sum()
    )

    image_ok = (
        key in found
        and Path(found[key]).is_file()
    )

    evidence_ok = (
        len(records) > 0
        and int(matched) == int(area["matched_objects"])
        and int(matched > 0) == int(area["water_holding_evidence"])
    )

    risk_ok = all(
        0 <= float(area[col]) <= 1
        for col in [
            "current_risk_score",
            "3d_risk_score",
            "7d_risk_score"
        ]
    )

    results.append({
        "Area": area_id,
        "Image": "PASS" if image_ok else "FAIL",
        "F2 Evidence": "PASS" if evidence_ok else "FAIL",
        "Risk Score": "PASS" if risk_ok else "FAIL"
    })

test_df = pd.DataFrame(results)
display(test_df)

all_pass = (
    (test_df[["Image", "F2 Evidence", "Risk Score"]] == "PASS")
    .all(axis=1)
)

print("\n=== F4 VISUAL INTEGRATION TEST ===")
print("Total Areas:", len(test_df))
print("Fully Passed:", int(all_pass.sum()))
print("Failed:", int((~all_pass).sum()))

if len(test_df) == 15 and all_pass.all():
    print("STATUS: ALL TESTS PASS")
    print("VISUAL EVIDENCE INTEGRATION: 100% DONE")
    print("OVERALL F4 ENHANCEMENT: 40%")
else:
    print("STATUS: REVIEW REQUIRED")
    print("Please share the failed rows.")

print("\nOriginal files unchanged.")

````

### Original cell 54/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
from pathlib import Path

results = []

for _, area in f4.iterrows():
    area_id = area["location_id"]
    key = get_image_key(area["associated_image"])

    records = f2[
        f2["image"].map(get_image_key) == key
    ]

    matched = (
        records["association"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("MATCHED")
        .sum()
    )

    image_ok = (
        key in found
        and Path(found[key]).is_file()
    )

    evidence_ok = (
        len(records) > 0
        and int(matched) == int(area["matched_objects"])
        and int(matched > 0) == int(area["water_holding_evidence"])
    )

    risk_ok = all(
        0 <= float(area[col]) <= 1
        for col in [
            "current_risk_score",
            "3d_risk_score",
            "7d_risk_score"
        ]
    )

    results.append({
        "Area": area_id,
        "Image": "PASS" if image_ok else "FAIL",
        "F2 Evidence": "PASS" if evidence_ok else "FAIL",
        "Risk Score": "PASS" if risk_ok else "FAIL"
    })

test_df = pd.DataFrame(results)
display(test_df)

all_pass = (
    (test_df[["Image", "F2 Evidence", "Risk Score"]] == "PASS")
    .all(axis=1)
)

print("\n=== F4 VISUAL INTEGRATION TEST ===")
print("Total Areas:", len(test_df))
print("Fully Passed:", int(all_pass.sum()))
print("Failed:", int((~all_pass).sum()))

if len(test_df) == 15 and all_pass.all():
    print("STATUS: ALL TESTS PASS")
    print("VISUAL EVIDENCE INTEGRATION: 100% DONE")
    print("OVERALL F4 ENHANCEMENT: 40%")
else:
    print("STATUS: REVIEW REQUIRED")
    print("Please share the failed rows.")

print("\nOriginal files unchanged.")

````

### Original cell 55/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np
from IPython.display import display

# Reuse existing F4 data
required = [
    "location_id",
    "water_holding_evidence",
    "rainfall_mm",
    "temperature_c",
    "current_risk_score"
]

missing = [c for c in required if c not in f4.columns]
assert not missing, f"Missing columns: {missing}"

water = pd.to_numeric(f4["water_holding_evidence"])
rain = pd.to_numeric(f4["rainfall_mm"])
temp = pd.to_numeric(f4["temperature_c"])
saved = pd.to_numeric(f4["current_risk_score"])

# Explainable score components
water_part = 0.40 * water
rain_part = 0.35 * (rain / 100)
temp_part = 0.25 * ((temp - 20) / 20)

raw_score = water_part + rain_part + temp_part
calculated = np.clip(raw_score, 0, 1)

difference = np.abs(calculated - saved)

explain_df = pd.DataFrame({
    "Area": f4["location_id"],
    "Water Contribution": water_part.round(4),
    "Rainfall Contribution": rain_part.round(4),
    "Temperature Contribution": temp_part.round(4),
    "Calculated Risk": calculated.round(4),
    "Saved Risk": saved.round(4),
    "Difference": difference.round(6),
    "Test": np.where(
        difference <= 0.001, "PASS", "FAIL"
    )
})

print("=== F4 EXPLAINABLE RISK SCORE TEST ===")
display(explain_df)

print("\n=== FINAL RESULT ===")
print("Total Areas:", len(explain_df))
print("Passed:", (explain_df["Test"] == "PASS").sum())
print("Failed:", (explain_df["Test"] == "FAIL").sum())
print("Maximum Difference:", round(difference.max(), 6))

print("\n=== SIM-AREA-011 EXPLANATION ===")
display(
    explain_df[
        explain_df["Area"] == "SIM-AREA-011"
    ]
)

if (explain_df["Test"] == "PASS").all():
    print("\nSTATUS: ALL TESTS PASS")
    print("Ready for Explainable Risk Visualization")
else:
    print("\nSTATUS: REVIEW REQUIRED")

print("\nOriginal files unchanged.")

````

### Original cell 56/118 — PYTHON SOURCE (archived, not run)

````python

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Existing verified F4 dataset
assert "f4" in globals(), "F4 data not loaded!"

selector = widgets.Dropdown(
    options=sorted(f4["location_id"].unique()),
    value="SIM-AREA-011",
    description="Select Area:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

output = widgets.Output()

def show_explanation(change=None):
    with output:
        clear_output(wait=True)

        area = f4.loc[
            f4["location_id"] == selector.value
        ].iloc[0]

        water = float(area["water_holding_evidence"])
        rain = float(area["rainfall_mm"])
        temp = float(area["temperature_c"])

        water_score = 0.40 * water
        rain_score = 0.35 * rain / 100
        temp_score = 0.25 * (temp - 20) / 20

        parts = [
            water_score,
            rain_score,
            temp_score
        ]

        total = float(np.clip(sum(parts), 0, 1))
        saved = float(area["current_risk_score"])

        if total >= 0.65:
            category = "HIGH"
        elif total >= 0.35:
            category = "MEDIUM"
        else:
            category = "LOW"

        print("=" * 48)
        print("DENGU HUNTER - EXPLAINABLE RISK")
        print("=" * 48)

        print("Area:", selector.value)
        print("Water Evidence:", int(water))
        print("Rainfall:", rain, "mm")
        print("Temperature:", temp, "C")
        print()
        print("Water Contribution:", round(water_score, 4))
        print("Rainfall Contribution:", round(rain_score, 4))
        print("Temperature Contribution:", round(temp_score, 4))
        print()
        print("Calculated Risk:", round(total, 4))
        print("Saved Risk:", round(saved, 4))
        print("Risk Category:", category)

        labels = [
            "F2 Water Evidence",
            "Rainfall",
            "Temperature"
        ]

        fig, ax = plt.subplots(figsize=(10, 4.5))

        bars = ax.barh(
            labels,
            parts,
            color=["#26957A", "#448FD1", "#E6A14A"]
        )

        ax.invert_yaxis()
        ax.set_xlim(0, max(0.5, max(parts) * 1.5))
        ax.set_xlabel("Contribution to Risk Score")
        ax.set_title(
            f"{selector.value} - Risk Explanation"
        )

        for bar, value in zip(bars, parts):
            share = 100 * value / sum(parts) if sum(parts) else 0

            ax.text(
                bar.get_width() + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.4f} ({share:.1f}%)",
                va="center"
            )

        plt.tight_layout()
        plt.show()

        print(
            f"\nEXPLANATION: {selector.value} is "
            f"{category} RISK under the demo scoring rule."
        )

        print(
            "Formula check:",
            "PASS" if abs(total - saved) <= 0.001
            else "FAIL"
        )

        print("\nNote: Rule-based simulation,")
        print("not a clinically validated dengue forecast.")

selector.observe(show_explanation, names="value")

display(selector, output)
show_explanation()

````

### Original cell 57/118 — PYTHON SOURCE (archived, not run)

````python

import numpy as np
import pandas as pd

results = []

for _, area in f4.iterrows():
    water = float(area["water_holding_evidence"])
    rain = float(area["rainfall_mm"])
    temp = float(area["temperature_c"])
    saved = float(area["current_risk_score"])

    # Three risk contributions
    water_part = 0.40 * water
    rain_part = 0.35 * rain / 100
    temp_part = 0.25 * (temp - 20) / 20

    calculated = float(np.clip(
        water_part + rain_part + temp_part, 0, 1
    ))

    # Risk category
    if calculated >= 0.65:
        category = "HIGH"
    elif calculated >= 0.35:
        category = "MEDIUM"
    else:
        category = "LOW"

    score_ok = abs(calculated - saved) <= 0.001

    category_ok = True
    if "current_risk_category" in f4.columns:
        category_ok = (
            category == str(
                area["current_risk_category"]
            ).strip().upper()
        )

    inputs_ok = (
        water in [0, 1]
        and rain >= 0
        and np.isfinite(temp)
        and 0 <= saved <= 1
    )

    passed = score_ok and category_ok and inputs_ok

    results.append({
        "Area": area["location_id"],
        "Calculated": round(calculated, 4),
        "Saved": round(saved, 4),
        "Category": category,
        "Test": "PASS" if passed else "FAIL"
    })

final_test = pd.DataFrame(results)

print("=== F4 EXPLAINABLE RISK FINAL TEST ===")
display(final_test)

passed_count = (final_test["Test"] == "PASS").sum()
unique_ok = final_test["Area"].nunique() == 15

print("\n=== FINAL SUMMARY ===")
print("Total Areas:", len(final_test))
print("Passed:", passed_count)
print("Failed:", len(final_test) - passed_count)
print("Unique Areas:", final_test["Area"].nunique())
print("Category Distribution:")
print(final_test["Category"].value_counts().to_dict())

if (
    len(final_test) == 15
    and passed_count == 15
    and unique_ok
):
    print("\nSTATUS: ALL TESTS PASS")
    print("EXPLAINABLE RISK SCORE: 100% DONE")
    print("OVERALL F4 ENHANCEMENT: 70%")
else:
    print("\nSTATUS: REVIEW REQUIRED")

print("\nOriginal files unchanged.")

````

### Original cell 58/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np
from IPython.display import display

assert "f4" in globals(), "F4 data not loaded!"

# Existing F4 scenario settings
# These are hypothetical, NOT real forecasts.
water = pd.to_numeric(f4["water_holding_evidence"])
current = pd.to_numeric(f4["current_risk_score"])

def simulate_risk(extra_rain, extra_temp):
    rain_effect = (
        0.35 * (extra_rain / 100)
        * (0.4 + 0.6 * water)
    )
    temp_effect = 0.25 * (extra_temp / 20)

    return np.clip(
        current + rain_effect + temp_effect,
        0, 1
    )

# Original hypothetical scenarios
calculated_3d = simulate_risk(15, 0.5)
calculated_7d = simulate_risk(30, 1.0)

saved_3d = pd.to_numeric(f4["3d_risk_score"])
saved_7d = pd.to_numeric(f4["7d_risk_score"])

check_3d = np.abs(calculated_3d - saved_3d) <= 0.001
check_7d = np.abs(calculated_7d - saved_7d) <= 0.001

comparison = pd.DataFrame({
    "Area": f4["location_id"],
    "Current": current.round(4),
    "3-Day": saved_3d.round(4),
    "7-Day": saved_7d.round(4),
    "3-Day Increase": (saved_3d - current).round(4),
    "7-Day Increase": (saved_7d - current).round(4),
    "Test": np.where(
        check_3d & check_7d, "PASS", "FAIL"
    )
})

print("=== F4 FUTURE RISK COMPARISON ===")
display(comparison)

def risk_category(scores):
    return pd.Series(
        np.select(
            [scores >= 0.65, scores >= 0.35],
            ["HIGH", "MEDIUM"],
            default="LOW"
        )
    ).value_counts().to_dict()

print("\n=== RISK DISTRIBUTION ===")
print("Current:", risk_category(current))
print("3-Day:", risk_category(saved_3d))
print("7-Day:", risk_category(saved_7d))

print("\n=== SIM-AREA-011 ===")
display(
    comparison[
        comparison["Area"] == "SIM-AREA-011"
    ]
)

passed = (comparison["Test"] == "PASS").sum()

print("\n=== FINAL VERIFICATION ===")
print("Total Areas:", len(comparison))
print("Passed:", passed)
print("Failed:", len(comparison) - passed)

if len(comparison) == 15 and passed == 15:
    print("STATUS: ALL TESTS PASS")
    print("Ready for Interactive Weather Scenarios")
else:
    print("STATUS: REVIEW REQUIRED")

print("\nOriginal files unchanged.")

````

### Original cell 59/118 — PYTHON SOURCE (archived, not run)

````python

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

assert "f4" in globals(), "Load F4 data first!"

# Work with a copy; original F4 data stays unchanged
base = f4.copy()

area_widget = widgets.Dropdown(
    options=sorted(base["location_id"].unique()),
    value="SIM-AREA-011",
    description="Select Area:",
    style={"description_width": "initial"}
)

scenario_widget = widgets.Dropdown(
    options=[
        ("Current Weather", "current"),
        ("Wet Scenario", "wet"),
        ("Dry Scenario", "dry"),
        ("Custom Scenario", "custom")
    ],
    value="wet",
    description="Scenario:",
    style={"description_width": "initial"}
)

rain_slider = widgets.FloatSlider(
    value=30, min=-40, max=60, step=5,
    description="Rain Change (mm):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

temp_slider = widgets.FloatSlider(
    value=1, min=-3, max=3, step=0.5,
    description="Temp Change (C):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

out = widgets.Output()

def category(score):
    if score >= 0.65:
        return "HIGH"
    elif score >= 0.35:
        return "MEDIUM"
    return "LOW"

def update_demo(change=None):
    with out:
        clear_output(wait=True)

        df = base.copy()

        water = pd.to_numeric(
            df["water_holding_evidence"]
        )

        current = pd.to_numeric(
            df["current_risk_score"]
        )

        rainfall = pd.to_numeric(
            df["rainfall_mm"]
        )

        # Do not allow negative absolute rainfall
        new_rainfall = np.maximum(
            0, rainfall + rain_slider.value
        )

        effective_change = new_rainfall - rainfall

        rain_effect = (
            0.35 * (effective_change / 100)
            * (0.4 + 0.6 * water)
        )

        temp_effect = (
            0.25 * (temp_slider.value / 20)
        )

        df["Custom Risk"] = np.clip(
            current + rain_effect + temp_effect,
            0, 1
        )

        df["Custom Category"] = (
            df["Custom Risk"].apply(category)
        )

        df = df.sort_values(
            ["Custom Risk", "location_id"],
            ascending=[False, True]
        ).reset_index(drop=True)

        df["New Rank"] = np.arange(1, len(df) + 1)

        original_ranks = base[
            ["location_id", "priority_rank"]
        ].rename(
            columns={"priority_rank": "Previous Rank"}
        )

        df = df.merge(
            original_ranks,
            on="location_id",
            how="left"
        )

        selected = df.loc[
            df["location_id"] == area_widget.value
        ].iloc[0]

        values = [
            float(selected["current_risk_score"]),
            float(selected["3d_risk_score"]),
            float(selected["7d_risk_score"]),
            float(selected["Custom Risk"])
        ]

        labels = [
            "Current", "3 Days", "7 Days", "Custom"
        ]

        fig, ax = plt.subplots(figsize=(10, 4.5))
        bars = ax.bar(labels, values)

        ax.set_ylim(0, 1.1)
        ax.set_ylabel("Simulated Risk Score")
        ax.set_title(
            f"{area_widget.value} - Weather Risk Comparison"
        )

        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.02,
                f"{value:.4f}",
                ha="center"
            )

        plt.tight_layout()
        plt.show()

        print("=== WEATHER SCENARIO ===")
        print("Rainfall Change:", rain_slider.value, "mm")
        print("Temperature Change:", temp_slider.value, "C")
        print()
        print("Selected Area:", area_widget.value)
        print("Custom Risk:", round(values[3], 4))
        print("Custom Category:", selected["Custom Category"])
        print("New Priority Rank:", int(selected["New Rank"]))

        print("\n=== CUSTOM RISK DISTRIBUTION ===")
        print(df["Custom Category"].value_counts().to_dict())

        print("\n=== UPDATED DRONE PRIORITY ===")

        display(
            df[[
                "New Rank",
                "location_id",
                "Previous Rank",
                "Custom Risk",
                "Custom Category"
            ]].round(4)
        )

        print("\nNote: Hypothetical weather simulation.")
        print("Not a real dengue or weather forecast.")

def apply_preset(change):
    presets = {
        "current": (0, 0),
        "wet": (30, 1),
        "dry": (-30, -1)
    }

    if scenario_widget.value in presets:
        r, t = presets[scenario_widget.value]
        rain_slider.value = r
        temp_slider.value = t

    update_demo()

def manual_change(change):
    if scenario_widget.value != "custom":
        scenario_widget.value = "custom"
    else:
        update_demo()

scenario_widget.observe(apply_preset, names="value")
area_widget.observe(update_demo, names="value")
rain_slider.observe(manual_change, names="value")
temp_slider.observe(manual_change, names="value")

display(
    widgets.VBox([
        area_widget,
        scenario_widget,
        rain_slider,
        temp_slider,
        out
    ])
)

update_demo()

````

### Original cell 60/118 — PYTHON SOURCE (archived, not run)

````python

import numpy as np
import pandas as pd

assert "f4" in globals(), "F4 data not loaded!"

df = f4.copy()

water = pd.to_numeric(df["water_holding_evidence"])
rain = pd.to_numeric(df["rainfall_mm"])
current = pd.to_numeric(df["current_risk_score"])

def simulate(rain_change, temp_change):
    actual_change = (
        np.maximum(0, rain + rain_change) - rain
    )

    rain_effect = (
        0.35 * (actual_change / 100)
        * (0.4 + 0.6 * water)
    )

    temp_effect = 0.25 * (temp_change / 20)

    return pd.Series(
        np.clip(current + rain_effect + temp_effect, 0, 1),
        index=df.index
    )

day3 = simulate(15, 0.5)
day7 = simulate(30, 1)
wet = simulate(30, 1)
dry = simulate(-30, -1)

def category(scores):
    return pd.Series(
        np.select(
            [scores >= 0.65, scores >= 0.35],
            ["HIGH", "MEDIUM"],
            default="LOW"
        ),
        index=df.index
    )

# Final checks
tests = {
    "15 Unique Areas": (
        len(df) == 15
        and df["location_id"].nunique() == 15
    ),

    "3-Day Score Verification": np.allclose(
        day3, df["3d_risk_score"], atol=0.001
    ),

    "7-Day Score Verification": np.allclose(
        day7, df["7d_risk_score"], atol=0.001
    ),

    "Wet Scenario Verification": np.allclose(
        wet, day7, atol=0.001
    ),

    "Dry Risk Decreases": bool(
        (dry <= current + 1e-9).all()
    ),

    "Risk Scores Within 0-1": all(
        ((x >= 0) & (x <= 1)).all()
        for x in [current, day3, day7, wet, dry]
    ),

    "Risk Ranking Valid": (
        df["priority_rank"].nunique() == 15
        and set(df["priority_rank"].astype(int))
        == set(range(1, 16))
    )
}

print("=== F4 FINAL ENHANCEMENT TEST ===")

for name, passed in tests.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\n=== SCENARIO RISK DISTRIBUTION ===")
for name, scores in [
    ("Current", current),
    ("3-Day", day3),
    ("7-Day", day7),
    ("Wet", wet),
    ("Dry", dry)
]:
    print(name, category(scores).value_counts().to_dict())

print("\n=== DRY SCENARIO PRIORITY ===")

dry_result = pd.DataFrame({
    "Area": df["location_id"],
    "Dry Risk": dry.round(4),
    "Category": category(dry)
})

dry_result = dry_result.sort_values(
    ["Dry Risk", "Area"],
    ascending=[False, True]
).reset_index(drop=True)

dry_result.insert(
    0, "New Rank",
    range(1, len(dry_result) + 1)
)

display(dry_result)

passed_count = sum(tests.values())

print("\n=== FINAL RESULT ===")
print("Tests Passed:", passed_count, "/", len(tests))

if passed_count == len(tests):
    print("STATUS: ALL TESTS PASS")
    print("F4 ENHANCEMENTS: 100% DONE")
else:
    print("STATUS: REVIEW REQUIRED")

print("Original Files: UNCHANGED")

````

### Original cell 61/118 — PYTHON SOURCE (archived, not run)

````python

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive")

f5_folder = root / "DENGU_HUNTER_FINAL_SHOWCASE" / "Mission_Planner"

priority_file = (
    root / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
    / "Corrected_20261122_052657"
    / "F4_To_F5_Priority.csv"
)

print("========== F5 FILE CHECK ==========")
print("Mission Planner Folder:", f5_folder.exists())
print("F4 Priority File:", priority_file.exists())

if f5_folder.exists():
    files = sorted(f5_folder.rglob("*"))

    print("\n=== EXISTING MISSION PLANNER FILES ===")

    for file in files[:60]:
        if file.is_file():
            print("FILE:", file.relative_to(f5_folder))
        else:
            print("FOLDER:", file.relative_to(f5_folder))

    print("\n=== EXISTING CSV PREVIEW ===")

    csv_files = [
        f for f in files
        if f.is_file() and f.suffix.lower() == ".csv"
    ]

    for file in csv_files[:5]:
        try:
            df = pd.read_csv(file)
            print("\nFilename:", file.name)
            print("Rows:", len(df))
            print("Columns:", df.columns.tolist())
            display(df.head(5))
        except Exception as e:
            print("Read Error:", e)

if priority_file.exists():
    priority_df = pd.read_csv(priority_file)

    print("\n=== F4 TO F5 PRIORITY DATA ===")
    print("Total Areas:", len(priority_df))
    print("Columns:", priority_df.columns.tolist())

    display(priority_df.head(15))

print("\n========== STEP 1 COMPLETE ==========")
print("No files modified.")

````

### Original cell 62/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json

root = Path("/content/drive/MyDrive")

f5 = root / "DENGU_HUNTER_FINAL_SHOWCASE" / "Mission_Planner"

f4_source = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Development_Batch_Features_3_4_6_8"
    / "predictive_hotspot_demo.csv"
)

f4_priority = (
    root / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
    / "Corrected_20261122_052657"
    / "F4_To_F5_Priority.csv"
)

tasks = pd.read_csv(f5 / "balanced_drone_assignments.csv")
areas = pd.read_csv(f4_source)
priority = pd.read_csv(f4_priority)
summary = pd.read_csv(f5 / "balanced_drone_summary.csv")

print("=== STEP 2: F4 + F5 INTEGRATION CHECK ===")

print("\nF5 Tasks:", len(tasks))
print("F4 Areas:", len(areas))
print("F4 Priorities:", len(priority))

def image_key(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\\", "/").split("/")[-1].strip().lower()

tasks["image_key"] = tasks["image"].apply(image_key)
areas["image_key"] = areas["associated_image"].apply(image_key)

print("\n=== DUPLICATE CHECK ===")
print("Duplicate Drone Images:", tasks["image_key"].duplicated().sum())
print("Duplicate F4 Images:", areas["image_key"].duplicated().sum())
print("Duplicate F4 Area IDs:", areas["location_id"].duplicated().sum())

if areas["image_key"].is_unique:
    linked = tasks.merge(
        areas[["location_id", "image_key"]],
        on="image_key",
        how="left",
        validate="many_to_one"
    )

    linked = linked.merge(
        priority,
        on="location_id",
        how="left",
        validate="many_to_one"
    )

    print("\n=== MATCHING RESULT ===")
    print("Matched Areas:", linked["location_id"].notna().sum())
    print("Missing Areas:", linked["location_id"].isna().sum())
    print("Matched F4 Priorities:", linked["priority_rank"].notna().sum())

    print("\n=== DRONE TASKS WITH F4 RANK ===")
    display(
        linked[
            [
                "drone", "sequence", "location_id",
                "priority_rank", "risk_score",
                "risk_category", "priority"
            ]
        ].sort_values(["drone", "sequence"])
    )

    print("\n=== F4 RANK ORDER CHECK ===")
    for drone, group in linked.groupby("drone"):
        group = group.sort_values("sequence")
        ranks = group["priority_rank"].dropna().tolist()
        print(drone, "Ranks:", ranks)
        print("Risk Order Correct:", ranks == sorted(ranks))

    print("\n=== DUPLICATE AREA CHECK ===")
    print("Repeated Area IDs:",
          linked["location_id"].dropna().duplicated().sum())

else:
    print("\nWARNING: Duplicate F4 image names found.")
    print("Matching skipped until duplicates are resolved.")

print("\n=== EXISTING BATTERY SUMMARY ===")
display(summary)

report_file = f5 / "balanced_mission_report.json"

if report_file.exists():
    print("\n=== EXISTING MISSION SETTINGS ===")
    with open(report_file, "r", encoding="utf-8") as file:
        report = json.load(file)

    print(json.dumps(report, indent=2, ensure_ascii=False)[:3500])

print("\nSTEP 2 CHECK COMPLETED")
print("Original files unchanged.")

````

### Original cell 63/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from math import hypot
import pandas as pd

root = Path("/content/drive/MyDrive")

f5 = root / "DENGU_HUNTER_FINAL_SHOWCASE" / "Mission_Planner"

f4_source = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Development_Batch_Features_3_4_6_8"
    / "predictive_hotspot_demo.csv"
)

f4_priority = (
    root / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
    / "Corrected_20261122_052657"
    / "F4_To_F5_Priority.csv"
)

tasks = pd.read_csv(f5 / "balanced_drone_assignments.csv")
areas = pd.read_csv(f4_source)
priority = pd.read_csv(f4_priority)

def image_key(value):
    return str(value).replace("\\", "/").split("/")[-1].strip().lower()

tasks["image_key"] = tasks["image"].apply(image_key)
areas["image_key"] = areas["associated_image"].apply(image_key)

linked = tasks.merge(
    areas[["location_id", "image_key"]],
    on="image_key",
    how="left",
    validate="one_to_one"
)

linked = linked.merge(
    priority,
    on="location_id",
    how="left",
    validate="one_to_one"
)

assert len(linked) == 15
assert linked["priority_rank"].notna().all()
assert linked["location_id"].is_unique

# Simulation settings (not real drone specifications)
DEPOT = (250.0, 250.0)
SPEED_M_PER_MIN = 60.0
INSPECTION_MIN = 3.0
BATTERY_LIMIT_MIN = 45.0

def calculate_route(df):
    points = [DEPOT]

    for _, row in df.iterrows():
        points.append((float(row["x_m"]), float(row["y_m"])))

    points.append(DEPOT)

    distance = sum(
        hypot(
            points[i+1][0] - points[i][0],
            points[i+1][1] - points[i][1]
        )
        for i in range(len(points)-1)
    )

    flight_time = distance / SPEED_M_PER_MIN
    inspection_time = len(df) * INSPECTION_MIN
    total_time = flight_time + inspection_time

    return round(distance, 2), round(total_time, 2)

results = []

print("=== STEP 3: RISK-BASED ROUTE TEST ===")
print("Synthetic Depot:", DEPOT)
print("Simulated Battery Limit:", BATTERY_LIMIT_MIN, "min")

for drone, group in linked.groupby("drone"):

    old_route = group.sort_values("sequence")
    new_route = group.sort_values("priority_rank").copy()

    new_route["new_sequence"] = range(1, len(new_route)+1)

    old_distance, old_time = calculate_route(old_route)
    new_distance, new_time = calculate_route(new_route)

    print("\n==========", drone, "==========")

    print("OLD F4 RANK ORDER:",
          old_route["priority_rank"].tolist())

    print("NEW F4 RANK ORDER:",
          new_route["priority_rank"].tolist())

    print("Old Route Distance:", old_distance, "m")
    print("New Route Distance:", new_distance, "m")

    print("Old Estimated Time:", old_time, "min")
    print("New Estimated Time:", new_time, "min")

    battery_ok = new_time <= BATTERY_LIMIT_MIN

    print("New Route Within Battery Limit:", battery_ok)

    results.append({
        "Drone": drone,
        "Tasks": len(group),
        "Old_Distance_m": old_distance,
        "New_Distance_m": new_distance,
        "Old_Time_min": old_time,
        "New_Time_min": new_time,
        "Battery_PASS": battery_ok
    })

    display(
        new_route[
            [
                "new_sequence",
                "location_id",
                "priority_rank",
                "risk_score",
                "risk_category"
            ]
        ].reset_index(drop=True)
    )

print("\n=== STEP 3 SUMMARY ===")
summary = pd.DataFrame(results)
display(summary)

print("All Areas Assigned:", len(linked) == 15)
print("Duplicate Areas:", linked["location_id"].duplicated().sum())
print("Both Drones Within Limit:", summary["Battery_PASS"].all())

print("\nNOTE: Priority-only sorting is not yet route optimization.")
print("No original files modified.")

````

### Original cell 64/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from math import hypot
import pandas as pd

root = Path("/content/drive/MyDrive")

f5 = root / "DENGU_HUNTER_FINAL_SHOWCASE" / "Mission_Planner"

f4 = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Development_Batch_Features_3_4_6_8"
    / "predictive_hotspot_demo.csv"
)

priority_path = (
    root / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
    / "Corrected_20261122_052657"
    / "F4_To_F5_Priority.csv"
)

tasks = pd.read_csv(f5 / "balanced_drone_assignments.csv")
areas = pd.read_csv(f4)
priority = pd.read_csv(priority_path)

def image_key(x):
    return str(x).replace("\\", "/").split("/")[-1].strip().lower()

tasks["key"] = tasks["image"].apply(image_key)
areas["key"] = areas["associated_image"].apply(image_key)

data = tasks.merge(
    areas[["location_id", "key"]],
    on="key",
    how="left",
    validate="one_to_one"
).merge(
    priority,
    on="location_id",
    how="left",
    validate="one_to_one"
)

assert len(data) == 15
assert data["location_id"].notna().all()
assert data["priority_rank"].notna().all()
assert data["location_id"].is_unique

points = {
    row.location_id: (float(row.x_m), float(row.y_m))
    for row in data.itertuples()
}

ranks = {
    row.location_id: int(row.priority_rank)
    for row in data.itertuples()
}

risks = {
    row.location_id: float(row.risk_score)
    for row in data.itertuples()
}

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY_LIMIT = 45.0

drones = ["Drone 1", "Drone 2"]

original = {
    d: data[data["drone"] == d]
        .sort_values("sequence")["location_id"].tolist()
    for d in drones
}

def route_stats(route):
    previous = DEPOT
    distance = 0.0
    elapsed = 0.0
    visits = []

    for location in route:
        current = points[location]
        leg = hypot(
            current[0] - previous[0],
            current[1] - previous[1]
        )

        distance += leg
        elapsed += leg / SPEED
        visits.append((location, elapsed))

        elapsed += INSPECTION
        previous = current

    return_leg = hypot(
        DEPOT[0] - previous[0],
        DEPOT[1] - previous[1]
    )

    distance += return_leg
    elapsed += return_leg / SPEED

    return distance, elapsed, visits

def priority_wait(plan):
    total = 0.0
    weight_total = 0.0

    for drone in drones:
        _, _, visits = route_stats(plan[drone])

        for location, arrival in visits:
            weight = (16 - ranks[location]) ** 2
            total += weight * arrival
            weight_total += weight

    return total / weight_total

def score(plan):
    if any(len(plan[d]) == 0 for d in drones):
        return float("inf")

    times = [route_stats(plan[d])[1] for d in drones]

    if any(t > BATTERY_LIMIT for t in times):
        return float("inf")

    # Prioritize early visits, while reducing total mission time
    return (
        priority_wait(plan)
        + 0.035 * sum(times)
        + 0.015 * max(times)
    )

def candidates(plan):
    # Reorder tasks within each drone
    for drone in drones:
        route = plan[drone]

        for i in range(len(route)):
            for j in range(len(route)):
                if i == j:
                    continue

                p = {d: plan[d].copy() for d in drones}
                task = p[drone].pop(i)
                p[drone].insert(j, task)
                yield p

        # Reverse route sections
        for i in range(len(route)):
            for j in range(i + 2, len(route)):
                p = {d: plan[d].copy() for d in drones}
                p[drone][i:j+1] = reversed(p[drone][i:j+1])
                yield p

    # Swap tasks between drones
    a, b = drones

    for i in range(len(plan[a])):
        for j in range(len(plan[b])):
            p = {d: plan[d].copy() for d in drones}
            p[a][i], p[b][j] = p[b][j], p[a][i]
            yield p

    # Move a task from one drone to the other
    for source, target in [(a, b), (b, a)]:
        if len(plan[source]) <= 1:
            continue

        for i in range(len(plan[source])):
            for j in range(len(plan[target]) + 1):
                p = {d: plan[d].copy() for d in drones}
                task = p[source].pop(i)
                p[target].insert(j, task)
                yield p

print("=== STEP 4: INTELLIGENT ROUTE OPTIMIZATION ===")

best = {d: original[d].copy() for d in drones}
best_score = score(best)

assert best_score != float("inf"), "Old route exceeds battery limit"

print("Original Score:", round(best_score, 3))

iterations = 0

for step in range(30):
    next_plan = None
    next_score = best_score

    for proposal in candidates(best):
        value = score(proposal)

        if value < next_score - 0.000001:
            next_score = value
            next_plan = proposal

    if next_plan is None:
        break

    best = next_plan
    best_score = next_score
    iterations += 1

print("Optimization Improvements:", iterations)
print("Optimized Score:", round(best_score, 3))

print("\n=== OLD VS OPTIMIZED ===")

comparison = []
schedule = []

for drone in drones:
    old_dist, old_time, _ = route_stats(original[drone])
    new_dist, new_time, visits = route_stats(best[drone])

    comparison.append({
        "Drone": drone,
        "Tasks": len(best[drone]),
        "Old_Distance_m": round(old_dist, 2),
        "New_Distance_m": round(new_dist, 2),
        "Old_Time_min": round(old_time, 2),
        "New_Time_min": round(new_time, 2),
        "Battery_PASS": new_time <= BATTERY_LIMIT
    })

    for sequence, (location, arrival) in enumerate(visits, 1):
        schedule.append({
            "Drone": drone,
            "Sequence": sequence,
            "Area": location,
            "F4_Rank": ranks[location],
            "Risk": risks[location],
            "Arrival_min": round(arrival, 2)
        })

comparison_df = pd.DataFrame(comparison)
schedule_df = pd.DataFrame(schedule)

display(comparison_df)

print("\n=== NEW DRONE MISSION ORDER ===")
display(schedule_df.sort_values(["Drone", "Sequence"]))

print("\n=== TOP 5 HIGH-RISK AREAS ===")
display(
    schedule_df[schedule_df["F4_Rank"] <= 5]
    .sort_values("F4_Rank")
)

print("\n=== FINAL VALIDATION ===")

all_locations = [x for d in drones for x in best[d]]

print("All 15 Areas Assigned:", len(all_locations) == 15)
print("Duplicate Areas:", len(all_locations) - len(set(all_locations)))
print("Both Drones Used:", all(len(best[d]) > 0 for d in drones))
print("Both Within Battery Limit:", comparison_df["Battery_PASS"].all())

print(
    "Old Priority-Weighted Arrival:",
    round(priority_wait(original), 2), "min"
)
print(
    "New Priority-Weighted Arrival:",
    round(priority_wait(best), 2), "min"
)

print("\nSimulation only — no real drone flight.")
print("Original Drive files unchanged.")

````

### Original cell 65/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import json

# Step 4 variables must still be available
required = [
    "best", "original", "data", "points",
    "ranks", "risks", "route_stats",
    "priority_wait", "DEPOT", "BATTERY_LIMIT"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Run Step 4 again first. Missing: " + ", ".join(missing)
    )

drones = ["Drone 1", "Drone 2"]
all_ids = [loc for drone in drones for loc in best[drone]]

# Validate before saving
assert len(all_ids) == 15
assert len(set(all_ids)) == 15
assert set(all_ids) == set(data["location_id"])
assert all(len(best[d]) > 0 for d in drones)
assert all(
    route_stats(best[d])[1] <= BATTERY_LIMIT
    for d in drones
)
assert priority_wait(best) < priority_wait(original)

# Build mission records
lookup = data.set_index("location_id")
records = []
summaries = []

for drone in drones:
    distance, total_time, visits = route_stats(best[drone])
    old_distance, old_time, _ = route_stats(original[drone])

    summaries.append({
        "Drone": drone,
        "Tasks": len(best[drone]),
        "Old_Distance_m": round(old_distance, 2),
        "New_Distance_m": round(distance, 2),
        "Old_Time_min": round(old_time, 2),
        "New_Time_min": round(total_time, 2),
        "Battery_Limit_min": BATTERY_LIMIT,
        "Battery_Remaining_min": round(BATTERY_LIMIT-total_time, 2),
        "Battery_PASS": total_time <= BATTERY_LIMIT
    })

    for sequence, (loc, arrival) in enumerate(visits, 1):
        row = lookup.loc[loc]

        records.append({
            "drone": drone,
            "sequence": sequence,
            "location_id": loc,
            "image": row["image"],
            "priority_rank": ranks[loc],
            "risk_score": risks[loc],
            "risk_category": row["risk_category"],
            "x_m": points[loc][0],
            "y_m": points[loc][1],
            "arrival_min": round(arrival, 2),
            "coordinate_type": "SIMULATED"
        })

mission_df = pd.DataFrame(records)
summary_df = pd.DataFrame(summaries)

# Create a new unique folder: originals stay untouched
base = (
    Path("/content/drive/MyDrive")
    / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
)

folder_name = (
    "F5_Optimized_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)

output_dir = base / folder_name
output_dir.mkdir(parents=True, exist_ok=False)

mission_df.to_csv(
    output_dir / "F5_Optimized_Mission_Assignments.csv",
    index=False
)

summary_df.to_csv(
    output_dir / "F5_Optimized_Drone_Summary.csv",
    index=False
)

# Create a visual map using synthetic XY coordinates
fig, ax = plt.subplots(figsize=(11, 8))

colors = {
    "Drone 1": "royalblue",
    "Drone 2": "darkorange"
}

for drone in drones:
    route = best[drone]
    coords = [DEPOT] + [points[x] for x in route] + [DEPOT]

    xs = [p[0] for p in coords]
    ys = [p[1] for p in coords]

    ax.plot(
        xs, ys,
        marker="o",
        linewidth=2,
        color=colors[drone],
        label=drone
    )

    for sequence, loc in enumerate(route, 1):
        x, y = points[loc]

        ax.annotate(
            f"{loc[-3:]} | R{ranks[loc]} | S{sequence}",
            (x, y),
            xytext=(5, 6),
            textcoords="offset points",
            fontsize=8
        )

ax.scatter(
    DEPOT[0], DEPOT[1],
    marker="*",
    s=250,
    color="red",
    label="Simulated Depot",
    zorder=5
)

ax.set_title(
    "DENGU HUNTER - F5 Optimized Drone Mission\n"
    "Synthetic XY Simulation | Not Real GPS"
)

ax.set_xlabel("Simulated X (m)")
ax.set_ylabel("Simulated Y (m)")
ax.grid(alpha=0.3)
ax.legend()
ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()

map_path = output_dir / "F5_Optimized_Route_Map.png"
plt.savefig(map_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# Save complete simulation report
old_wait = priority_wait(original)
new_wait = priority_wait(best)

report = {
    "feature": "F5 Intelligent Drone Mission Planner",
    "simulation_only": True,
    "real_drone_flight": False,
    "coordinate_type": "Synthetic local XY",
    "algorithm": "Priority-aware local search heuristic",
    "battery_limit_min": BATTERY_LIMIT,
    "simulated_depot": list(DEPOT),
    "total_areas": len(all_ids),
    "duplicate_areas": len(all_ids)-len(set(all_ids)),
    "drone_count": len(drones),
    "old_priority_weighted_arrival_min": round(old_wait, 4),
    "new_priority_weighted_arrival_min": round(new_wait, 4),
    "priority_arrival_improvement_percent": round(
        (old_wait-new_wait)/old_wait*100, 2
    ),
    "missions": summary_df.to_dict(orient="records"),
    "all_battery_tests_pass": bool(
        summary_df["Battery_PASS"].all()
    )
}

report_path = output_dir / "F5_Optimized_Mission_Report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n=== STEP 5 SAVE RESULT ===")
print("Saved Folder:", output_dir)

print("\n=== SAVED FILES ===")
for file in sorted(output_dir.iterdir()):
    print(file.name, "|", file.stat().st_size, "bytes")

print("\n=== FINAL SUMMARY ===")
display(summary_df)

print("Areas:", len(mission_df))
print("Duplicates:", mission_df["location_id"].duplicated().sum())
print("Battery PASS:", summary_df["Battery_PASS"].all())
print("Priority Improvement:", report["priority_arrival_improvement_percent"], "%")
print("Original Files: UNCHANGED")
print("STEP 5 COMPLETE")

````

### Original cell 66/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from math import hypot, isclose
import pandas as pd
import numpy as np
import json

root = Path("/content/drive/MyDrive")

folder = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
    / "F5_Optimized_20261122_055356_483143"
)

priority_path = (
    root / "DENGU_HUNTER_RESULTS"
    / "F4_Hackathon_Runs"
    / "Corrected_20261122_052657"
    / "F4_To_F5_Priority.csv"
)

mission = pd.read_csv(
    folder / "F5_Optimized_Mission_Assignments.csv"
)

summary = pd.read_csv(
    folder / "F5_Optimized_Drone_Summary.csv"
)

priority = pd.read_csv(priority_path)

with open(
    folder / "F5_Optimized_Mission_Report.json",
    encoding="utf-8"
) as f:
    report = json.load(f)

checks = {}

def test(name, result):
    checks[name] = bool(result)
    print(
        ("PASS" if result else "FAIL"),
        "-", name
    )

print("=== F5 FINAL AUTOMATED VALIDATION ===")

# 1. Total tasks
test("15 Tasks Assigned", len(mission) == 15)

# 2. No duplicate area
test(
    "No Duplicate Area",
    mission["location_id"].is_unique
)

# 3. All F4 areas included
test(
    "All F4 Areas Covered",
    set(mission["location_id"])
    == set(priority["location_id"])
)

# 4. F4 risk and rank consistency
merged = mission.merge(
    priority,
    on="location_id",
    how="left",
    suffixes=("", "_f4"),
    validate="one_to_one"
)

risk_ok = (
    merged["priority_rank"].eq(
        merged["priority_rank_f4"]
    ).all()
    and np.allclose(
        merged["risk_score"],
        merged["risk_score_f4"],
        atol=0.00001
    )
    and merged["risk_category"].eq(
        merged["risk_category_f4"]
    ).all()
)

test("F4 Risk Integration Correct", risk_ok)

# 5. Both drones used
test(
    "Two Drones Used",
    mission["drone"].nunique() == 2
    and mission.groupby("drone").size().min() > 0
)

# 6. Correct task sequence
sequence_ok = all(
    sorted(group["sequence"].tolist())
    == list(range(1, len(group) + 1))
    for _, group in mission.groupby("drone")
)

test("Mission Sequence Valid", sequence_ok)

# 7. Coordinates are simulated and valid
coordinate_ok = (
    mission["coordinate_type"].eq("SIMULATED").all()
    and np.isfinite(
        mission[["x_m", "y_m"]].to_numpy(
            dtype=float
        )
    ).all()
)

test("Synthetic Coordinates Valid", coordinate_ok)

# 8. Independently recalculate distance and time
DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
LIMIT = 45.0

distance_ok = True
arrival_ok = True
time_ok = True
battery_ok = True

for drone, group in mission.groupby("drone"):

    group = group.sort_values("sequence")
    previous = DEPOT
    distance = 0.0
    elapsed = 0.0

    for row in group.itertuples():
        current = (float(row.x_m), float(row.y_m))

        leg = hypot(
            current[0] - previous[0],
            current[1] - previous[1]
        )

        distance += leg
        elapsed += leg / SPEED

        if not isclose(
            elapsed, row.arrival_min, abs_tol=0.03
        ):
            arrival_ok = False

        elapsed += INSPECTION
        previous = current

    return_leg = hypot(
        previous[0] - DEPOT[0],
        previous[1] - DEPOT[1]
    )

    distance += return_leg
    elapsed += return_leg / SPEED

    saved = summary[
        summary["Drone"] == drone
    ].iloc[0]

    if not isclose(
        distance,
        saved["New_Distance_m"],
        abs_tol=0.03
    ):
        distance_ok = False

    if not isclose(
        elapsed,
        saved["New_Time_min"],
        abs_tol=0.03
    ):
        time_ok = False

    if elapsed > LIMIT:
        battery_ok = False

test("Route Distance Recalculation", distance_ok)
test("Arrival Time Recalculation", arrival_ok)
test("Mission Time Recalculation", time_ok)
test("Both Battery Limits PASS", battery_ok)

# 9. Highest-risk areas visited first
top_two = mission[
    mission["priority_rank"].isin([1, 2])
]

test(
    "Top 2 Risk Areas First",
    len(top_two) == 2
    and top_two["sequence"].eq(1).all()
)

# 10. Priority-weighted arrival improvement
weights = (
    16 - mission["priority_rank"]
) ** 2

new_weighted_arrival = (
    (weights * mission["arrival_min"]).sum()
    / weights.sum()
)

old_weighted_arrival = report[
    "old_priority_weighted_arrival_min"
]

improvement = (
    (old_weighted_arrival - new_weighted_arrival)
    / old_weighted_arrival * 100
)

test(
    "Priority Arrival Improved",
    new_weighted_arrival < old_weighted_arrival
)

test(
    "Saved Report Consistent",
    isclose(
        improvement,
        report["priority_arrival_improvement_percent"],
        abs_tol=0.1
    )
)

# 11. Visual map exists
map_file = folder / "F5_Optimized_Route_Map.png"

test(
    "Route Map Saved",
    map_file.exists()
    and map_file.stat().st_size > 1000
)

# 12. Simulation correctly identified
test(
    "Simulation Clearly Declared",
    report.get("simulation_only") is True
    and report.get("real_drone_flight") is False
)

passed = sum(checks.values())
total = len(checks)

print("\n=== FINAL TEST SUMMARY ===")
print("Tests Passed:", passed, "/", total)
print(
    "Priority Improvement:",
    round(improvement, 2), "%"
)

if passed == total:
    print("STATUS: ALL TESTS PASS")
    print("F5 HACKATHON SIMULATION: 100% DONE")

    validation_file = (
        folder / "F5_Final_Validation.json"
    )

    if validation_file.exists():
        print("Existing validation file preserved.")
    else:
        with open(
            validation_file, "w", encoding="utf-8"
        ) as f:
            json.dump(
                {
                    "status": "ALL TESTS PASS",
                    "tests_passed": passed,
                    "total_tests": total,
                    "checks": checks,
                    "simulation_only": True
                },
                f,
                indent=2
            )

        print("Validation report saved.")
else:
    print("STATUS: VALIDATION INCOMPLETE")
    print("Failed Tests:")
    for name, ok in checks.items():
        if not ok:
            print("-", name)

print("\nOriginal Mission Files: UNCHANGED")

````

### Original cell 67/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")

f5_folder = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
    / "F5_Optimized_20261122_055356_483143"
)

integration = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Unified_AI_Integration"
)

f7_folder = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Feature_7_Closed_Loop"
)

files = [
    f5_folder / "F5_Optimized_Mission_Assignments.csv",
    integration / "before_after_verification_intake.csv",
    integration / "human_approved_intervention_workflow.csv"
]

print("=== F5 ADVANCED ENHANCEMENT: STEP 1 ===")

for file in files:
    print("\n" + "=" * 45)
    print("FILE:", file.name)
    print("Exists:", file.exists())

    if file.exists():
        try:
            df = pd.read_csv(file)
            print("Rows:", len(df))
            print("Columns:", df.columns.tolist())
            print("Missing Values:")
            print(df.isna().sum().to_dict())
            display(df.head(5))
        except Exception as e:
            print("Read Error:", str(e))

print("\n=== EXISTING F7 FILES ===")
print("F7 Folder Exists:", f7_folder.exists())

if f7_folder.exists():
    for file in sorted(f7_folder.rglob("*"))[:40]:
        if file.is_file():
            print(file.relative_to(f7_folder))

print("\n=== STEP 1 COMPLETED ===")
print("Original Files: UNCHANGED")

````

### Original cell 68/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")

base = root / "DENGU_HUNTER_FINAL_SHOWCASE"

mission_file = (
    base / "Mission_Planner"
    / "F5_Optimized_20261122_055356_483143"
    / "F5_Optimized_Mission_Assignments.csv"
)

integration = base / "Unified_AI_Integration"

before_file = integration / "before_after_verification_intake.csv"
action_file = integration / "human_approved_intervention_workflow.csv"

mission = pd.read_csv(mission_file)
before = pd.read_csv(before_file)
action = pd.read_csv(action_file)

def image_key(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\\", "/").split("/")[-1].strip().lower()

mission["image_key"] = mission["image"].apply(image_key)
before["image_key"] = before["before_image"].apply(image_key)
action["image_key"] = action["image"].apply(image_key)

before = before.rename(
    columns={"location_id": "f7_location_id"}
)

linked = mission.merge(
    before.drop(columns=["before_image"]),
    on="image_key",
    how="left",
    validate="one_to_one"
)

linked = linked.merge(
    action.drop(columns=["image"]),
    on="image_key",
    how="left",
    validate="one_to_one"
)

print("=== F5 SMART REVISIT: STEP 2 ===")
print("Total Areas:", len(linked))
print("F7 Matched:", linked["f7_location_id"].notna().sum())
print("Intervention Matched:", linked["task_id"].notna().sum())

assert len(linked) == 15
assert linked["f7_location_id"].notna().all()
assert linked["task_id"].notna().all()
assert linked["location_id"].is_unique

def status(value):
    if pd.isna(value):
        return "MISSING"
    return str(value).strip().upper()

def decide_revisit(row):
    approval = status(row["human_approval"])
    action_status = status(row["field_action_status"])
    verification = status(row["verification_status"])
    comparison = status(row["comparison_result"])
    confirmation = status(row["human_confirmation"])

    after_available = pd.notna(row["after_image"]) and (
        str(row["after_image"]).strip() != ""
    )

    completed = action_status in {
        "COMPLETED", "DONE", "FINISHED"
    }

    human_confirmed = confirmation in {
        "CONFIRMED", "APPROVED", "VERIFIED", "YES"
    }

    cleared = (
        verification in {"VERIFIED_CLEAR", "CLEARED", "RESOLVED"}
        and comparison in {"CLEARED", "ELIMINATED", "RESOLVED"}
        and human_confirmed
    )

    if cleared:
        return "SKIP_REVISIT"

    if completed:
        if comparison in {"FAILED", "STILL_PRESENT", "PERSISTENT"}:
            return "REVISIT_REQUIRED"

        if not after_available:
            return "REVISIT_REQUIRED"

        return "HUMAN_REVIEW_FIRST"

    if row["risk_category"] == "LOW":
        return "LOW_RISK_MONITOR"

    if approval != "APPROVED" or not completed:
        return "URGENT_FIELD_ACTION_FIRST"

    return "MANUAL_REVIEW"

linked["revisit_decision"] = linked.apply(
    decide_revisit, axis=1
)

print("\n=== SMART REVISIT DECISIONS ===")

display(
    linked[
        [
            "location_id",
            "priority_rank",
            "risk_category",
            "human_approval",
            "field_action_status",
            "verification_status",
            "revisit_decision"
        ]
    ].sort_values("priority_rank")
)

print("\n=== DECISION DISTRIBUTION ===")
print(linked["revisit_decision"].value_counts().to_string())

print("\n=== VALIDATION ===")
print("All Areas Linked:", len(linked) == 15)
print("Duplicate Areas:", linked["location_id"].duplicated().sum())
print("Missing Decisions:", linked["revisit_decision"].isna().sum())
print("Original Files: UNCHANGED")

print("\nSTEP 2 COMPLETE")

````

### Original cell 69/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd

print("=== F5 SMART REVISIT: STEP 3 ===")

# Step 2-এর linked DataFrame ব্যবহার
assert "linked" in globals(), "Run Step 2 first"
assert len(linked) == 15

def clean(value):
    if pd.isna(value):
        return ""
    return str(value).strip().upper()

def smart_revisit(row):
    risk = clean(row["risk_category"])
    approval = clean(row["human_approval"])
    action = clean(row["field_action_status"])
    verification = clean(row["verification_status"])
    comparison = clean(row["comparison_result"])
    confirmation = clean(row["human_confirmation"])

    after_image = row["after_image"]
    has_after = (
        pd.notna(after_image)
        and str(after_image).strip() != ""
    )

    completed = action in {"COMPLETED", "DONE", "FINISHED"}

    # Inconsistent data must never trigger automatic clearance
    if completed and approval != "APPROVED":
        return "DATA_CONFLICT_REVIEW"

    if has_after and not completed:
        return "DATA_CONFLICT_REVIEW"

    if approval in {"REJECTED", "DENIED"}:
        return "NO_ACTION_REJECTED"

    if not completed:
        if approval != "APPROVED":
            if risk == "LOW":
                return "LOW_RISK_MONITOR"
            return "AWAITING_HUMAN_APPROVAL"

        return "FIELD_ACTION_REQUIRED"

    # Intervention completed, but verification image missing
    if not has_after:
        return "REVISIT_REQUIRED"

    # Breeding evidence still exists
    if comparison in {
        "STILL_PRESENT", "FAILED",
        "PERSISTENT", "NOT_CLEARED"
    }:
        return "REVISIT_REQUIRED"

    clear_result = (
        verification in {
            "VERIFIED_CLEAR", "CLEARED", "RESOLVED"
        }
        and comparison in {
            "CLEARED", "ELIMINATED", "RESOLVED"
        }
        and confirmation in {
            "CONFIRMED", "APPROVED", "VERIFIED", "YES"
        }
    )

    if clear_result:
        return "SKIP_REVISIT"

    return "HUMAN_REVIEW_FIRST"


# Recalculate actual decisions using safer rules
linked["revisit_decision"] = linked.apply(
    smart_revisit, axis=1
)

print("\n=== CURRENT REAL WORKFLOW STATUS ===")
print(linked["revisit_decision"].value_counts().to_string())

# Test cases use simulated status changes only
base = linked[
    linked["risk_category"] == "HIGH"
].iloc[0].copy()

scenarios = [
    ("High Risk Pending", {},
     "AWAITING_HUMAN_APPROVAL"),

    ("Low Risk Pending",
     {"risk_category": "LOW"},
     "LOW_RISK_MONITOR"),

    ("Approved Not Started",
     {"human_approval": "APPROVED"},
     "FIELD_ACTION_REQUIRED"),

    ("Completed No After Image",
     {"human_approval": "APPROVED",
      "field_action_status": "COMPLETED"},
     "REVISIT_REQUIRED"),

    ("After Image Pending Review",
     {"human_approval": "APPROVED",
      "field_action_status": "COMPLETED",
      "after_image": "SIMULATED_TEST_AFTER.jpg"},
     "HUMAN_REVIEW_FIRST"),

    ("Verified Clear",
     {"human_approval": "APPROVED",
      "field_action_status": "COMPLETED",
      "after_image": "SIMULATED_TEST_AFTER.jpg",
      "verification_status": "VERIFIED_CLEAR",
      "comparison_result": "CLEARED",
      "human_confirmation": "CONFIRMED"},
     "SKIP_REVISIT"),

    ("Breeding Still Present",
     {"human_approval": "APPROVED",
      "field_action_status": "COMPLETED",
      "after_image": "SIMULATED_TEST_AFTER.jpg",
      "comparison_result": "STILL_PRESENT"},
     "REVISIT_REQUIRED"),

    ("Fake Verification Without Approval",
     {"field_action_status": "COMPLETED",
      "after_image": "SIMULATED_TEST_AFTER.jpg",
      "verification_status": "VERIFIED_CLEAR",
      "comparison_result": "CLEARED",
      "human_confirmation": "CONFIRMED"},
     "DATA_CONFLICT_REVIEW"),

    ("After Image Without Intervention",
     {"human_approval": "APPROVED",
      "after_image": "SIMULATED_TEST_AFTER.jpg"},
     "DATA_CONFLICT_REVIEW"),

    ("Rejected Intervention",
     {"human_approval": "REJECTED"},
     "NO_ACTION_REJECTED")
]

passed = 0

print("\n=== AUTOMATED SCENARIO TESTS ===")

for name, changes, expected in scenarios:
    sample = base.copy()

    for key, value in changes.items():
        sample[key] = value

    actual = smart_revisit(sample)
    ok = actual == expected

    print(
        ("PASS" if ok else "FAIL"),
        "-", name,
        "| Result:", actual
    )

    if ok:
        passed += 1

print("\n=== FINAL STEP 3 SUMMARY ===")
print("Tests Passed:", passed, "/", len(scenarios))

assert passed == len(scenarios)
assert linked["revisit_decision"].notna().all()
assert linked["location_id"].is_unique

# Current data has no verified after images
assert not linked["revisit_decision"].eq(
    "SKIP_REVISIT"
).any()

print("STATUS: ALL TESTS PASS")
print("Original Files: UNCHANGED")
print("Simulated Test Images: NOT SAVED")

````

### Original cell 70/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import json
import hashlib

print("=== F5 SMART REVISIT: STEP 4 ===")

# Ensure previous steps are available
assert "linked" in globals(), "Run Step 2 first"
assert "smart_revisit" in globals(), "Run Step 3 first"
assert "scenarios" in globals(), "Run Step 3 first"
assert "base" in globals(), "Run Step 3 first"

# Verify original sources
source_files = [
    Path(mission_file),
    Path(before_file),
    Path(action_file)
]

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hashes = {
    str(p): file_hash(p) for p in source_files
}

# Recheck all scenario tests
test_results = []

for name, changes, expected in scenarios:
    sample = base.copy()

    for key, value in changes.items():
        sample[key] = value

    actual = smart_revisit(sample)

    test_results.append({
        "scenario": name,
        "expected": expected,
        "actual": actual,
        "result": "PASS" if actual == expected else "FAIL",
        "data_type": "SIMULATED_TEST"
    })

tests_df = pd.DataFrame(test_results)

assert len(tests_df) == 10
assert tests_df["result"].eq("PASS").all()

# Recalculate final decisions
final_df = linked.copy()

final_df["revisit_decision"] = final_df.apply(
    smart_revisit, axis=1
)

next_actions = {
    "AWAITING_HUMAN_APPROVAL":
        "Request human approval before field intervention",
    "FIELD_ACTION_REQUIRED":
        "Assign approved field intervention",
    "REVISIT_REQUIRED":
        "Schedule follow-up drone verification",
    "HUMAN_REVIEW_FIRST":
        "Request human review of after evidence",
    "SKIP_REVISIT":
        "No immediate revisit; continue monitoring",
    "LOW_RISK_MONITOR":
        "Monitor risk; reassess if conditions change",
    "DATA_CONFLICT_REVIEW":
        "Resolve conflicting workflow records",
    "NO_ACTION_REJECTED":
        "Hold intervention; require manual reassessment",
    "MANUAL_REVIEW":
        "Review workflow manually"
}

final_df["recommended_next_action"] = (
    final_df["revisit_decision"].map(next_actions)
)

final_df["after_image_available"] = (
    final_df["after_image"].notna()
    & final_df["after_image"].astype(str).str.strip().ne("")
)

final_df["data_scope"] = "PROJECT_SIMULATION"
final_df["planning_stage"] = "NEXT_CYCLE_FOLLOW_UP"

export_columns = [
    "location_id",
    "f7_location_id",
    "task_id",
    "image",
    "priority_rank",
    "risk_score",
    "risk_category",
    "human_approval",
    "field_action_status",
    "verification_status",
    "after_image_available",
    "revisit_decision",
    "recommended_next_action",
    "planning_stage",
    "data_scope"
]

export_df = (
    final_df[export_columns]
    .sort_values("priority_rank")
    .reset_index(drop=True)
)

# Final checks before saving
assert len(export_df) == 15
assert export_df["location_id"].is_unique
assert export_df["revisit_decision"].notna().all()
assert export_df["recommended_next_action"].notna().all()

# No unverified area can be automatically cleared
assert not export_df["revisit_decision"].eq(
    "SKIP_REVISIT"
).any()

summary = {
    "feature": "F5 Advanced - Smart Revisit Decision",
    "status": "VALIDATED_SIMULATION",
    "total_areas": int(len(export_df)),
    "areas_linked": 15,
    "duplicate_areas": 0,
    "scenario_tests_passed": int(
        tests_df["result"].eq("PASS").sum()
    ),
    "scenario_tests_total": int(len(tests_df)),
    "decision_distribution": {
        str(k): int(v)
        for k, v in export_df[
            "revisit_decision"
        ].value_counts().items()
    },
    "simulation_only": True,
    "real_drone_flight": False,
    "real_field_verification_claimed": False,
    "original_f5_missions_modified": False,
    "purpose": "Next-cycle follow-up planning"
}

# Create a NEW folder; never overwrite originals
parent = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
    / "F5_Advanced_Enhancements"
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = parent / f"Smart_Revisit_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=False)

decision_path = (
    output_dir / "F5_Smart_Revisit_Decisions.csv"
)
test_path = (
    output_dir / "F5_Smart_Revisit_Tests.csv"
)
summary_path = (
    output_dir / "F5_Smart_Revisit_Summary.json"
)

export_df.to_csv(decision_path, index=False)
tests_df.to_csv(test_path, index=False)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Confirm originals unchanged
for path in source_files:
    assert file_hash(path) == original_hashes[str(path)]

# Confirm exported files exist
for path in [decision_path, test_path, summary_path]:
    assert path.exists()
    assert path.stat().st_size > 0

print("\n=== FINAL VALIDATION ===")
print("Areas:", len(export_df))
print("Duplicate Areas:", export_df["location_id"].duplicated().sum())
print("Tests Passed:", summary["scenario_tests_passed"], "/ 10")
print("Original Files: UNCHANGED")
print("Real Drone Flight: NO")
print("Real Field Verification: NO")

print("\n=== SAVED FILES ===")
for path in [decision_path, test_path, summary_path]:
    print("SAVED:", path.name)

print("\nFolder:", output_dir)

print("\n=== DECISION DISTRIBUTION ===")
print(export_df["revisit_decision"].value_counts())

print("\nSTATUS: SMART REVISIT EXPORT COMPLETE")

````

### Original cell 71/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import math

print("=== F5 BATTERY RESERVE: STEP 5 ===")

root = Path("/content/drive/MyDrive")

f5_dir = (
    root / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
    / "F5_Optimized_20261122_055356_483143"
)

assignments = pd.read_csv(
    f5_dir / "F5_Optimized_Mission_Assignments.csv"
)

summary = pd.read_csv(
    f5_dir / "F5_Optimized_Drone_Summary.csv"
)

# Existing F5 simulation assumptions
BATTERY_LIMIT = 45.0
RESERVE_PERCENT = 10.0
RESERVE_MIN = BATTERY_LIMIT * RESERVE_PERCENT / 100
USABLE_TIME = BATTERY_LIMIT - RESERVE_MIN

SPEED_M_PER_MIN = 60.0
INSPECTION_MIN = 3.0
DEPOT = (250.0, 250.0)

print("\n=== BATTERY SETTINGS ===")
print("Battery Limit:", BATTERY_LIMIT, "min")
print("Emergency Reserve:", RESERVE_PERCENT, "%")
print("Reserved Time:", RESERVE_MIN, "min")
print("Usable Mission Time:", USABLE_TIME, "min")

assert len(assignments) == 15
assert assignments["location_id"].is_unique
assert assignments["drone"].nunique() == 2

def distance(a, b):
    return math.hypot(
        a[0] - b[0],
        a[1] - b[1]
    )

results = []

for drone, group in assignments.groupby("drone"):
    group = group.sort_values("sequence")

    current = DEPOT
    route_distance = 0.0

    for _, task in group.iterrows():
        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        route_distance += distance(current, target)
        current = target

    # Include return journey
    return_distance = distance(current, DEPOT)
    route_distance += return_distance

    travel_time = route_distance / SPEED_M_PER_MIN
    inspection_time = len(group) * INSPECTION_MIN
    mission_time = travel_time + inspection_time

    remaining = BATTERY_LIMIT - mission_time
    reserve_gap = remaining - RESERVE_MIN

    results.append({
        "Drone": drone,
        "Tasks": len(group),
        "Distance_m": round(route_distance, 2),
        "Return_Distance_m": round(return_distance, 2),
        "Travel_Time_min": round(travel_time, 2),
        "Inspection_Time_min": round(inspection_time, 2),
        "Mission_Time_min": round(mission_time, 2),
        "Battery_Remaining_min": round(remaining, 2),
        "Reserve_Gap_min": round(reserve_gap, 2),
        "Reserve_Status": (
            "PASS" if reserve_gap >= -1e-9
            else "NEEDS_REPLANNING"
        )
    })

battery_df = pd.DataFrame(results)

print("\n=== CURRENT DRONE BATTERY ANALYSIS ===")
display(battery_df)

print("\n=== CROSS-CHECK ORIGINAL F5 ===")

for _, row in battery_df.iterrows():
    original = summary.loc[
        summary["Drone"] == row["Drone"]
    ]

    assert len(original) == 1

    saved_time = float(
        original.iloc[0]["New_Time_min"]
    )

    difference = abs(
        saved_time - row["Mission_Time_min"]
    )

    print(
        row["Drone"],
        "| Saved:", saved_time,
        "| Recalculated:", row["Mission_Time_min"],
        "| Difference:", round(difference, 3)
    )

    assert difference <= 0.15

print("\n=== BATTERY RESERVE SUMMARY ===")

print(
    battery_df["Reserve_Status"]
    .value_counts()
    .to_string()
)

print("\n=== PLANNING REQUIREMENT ===")

if battery_df["Reserve_Status"].eq("PASS").all():
    print("Both drones already satisfy 10% reserve.")
else:
    print("Replanning required for 10% battery reserve.")
    print("Next: optimize routes or defer tasks safely.")

print("\nOriginal Files: UNCHANGED")
print("No Real Drone Flight")
print("STEP 5 COMPLETE")

````

### Original cell 72/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import math
from pathlib import Path
from functools import lru_cache

print("=== F5 BATTERY RESERVE: STEP 6 ===")

f5_dir = (
    Path("/content/drive/MyDrive")
    / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
    / "F5_Optimized_20261122_055356_483143"
)

tasks = pd.read_csv(
    f5_dir / "F5_Optimized_Mission_Assignments.csv"
).sort_values("priority_rank").reset_index(drop=True)

assert len(tasks) == 15
assert tasks["location_id"].is_unique
assert tasks["priority_rank"].is_unique

LIMIT = 45.0
RESERVE = 4.5
MAX_TIME = LIMIT - RESERVE
SPEED = 60.0
INSPECTION = 3.0
DEPOT = (250.0, 250.0)

points = list(zip(
    tasks["x_m"].astype(float),
    tasks["y_m"].astype(float)
))

n = len(points)
FULL = (1 << n) - 1

def dist(a, b):
    return math.hypot(
        a[0] - b[0],
        a[1] - b[1]
    )

depot_dist = [dist(DEPOT, p) for p in points]

D = [
    [dist(points[i], points[j]) for j in range(n)]
    for i in range(n)
]

def indices(mask):
    while mask:
        bit = mask & -mask
        yield bit.bit_length() - 1
        mask ^= bit

# Exact shortest route for a subset of areas.
# Optional fixed_first keeps a priority area first.
def create_solver(fixed_first=None):

    @lru_cache(maxsize=None)
    def dp(mask, last):
        bit = 1 << last

        if mask == bit:
            if fixed_first is None or last == fixed_first:
                return depot_dist[last]
            return math.inf

        previous = mask ^ bit

        if fixed_first is not None:
            if not (previous & (1 << fixed_first)):
                return math.inf

        return min(
            dp(previous, p) + D[p][last]
            for p in indices(previous)
        )

    @lru_cache(maxsize=None)
    def best(mask):
        if mask == 0:
            return math.inf, None

        return min(
            (
                dp(mask, last) + depot_dist[last],
                last
            )
            for last in indices(mask)
        )

    def reconstruct(mask):
        _, last = best(mask)

        if last is None:
            return []

        reverse_route = []

        while mask:
            reverse_route.append(last)

            previous = mask ^ (1 << last)

            if previous == 0:
                break

            old_last = last

            last = min(
                indices(previous),
                key=lambda p:
                    dp(previous, p) + D[p][old_last]
            )

            mask = previous

        return list(reversed(reverse_route))

    return best, reconstruct


def mission_time(mask, best_function):
    route_distance, _ = best_function(mask)

    if not math.isfinite(route_distance):
        return math.inf

    return (
        route_distance / SPEED
        + mask.bit_count() * INSPECTION
    )


# Original top two risk areas:
# Drone 1 starts with rank 2.
# Drone 2 starts with rank 1.
rank1 = tasks.index[
    tasks["priority_rank"] == 1
][0]

rank2 = tasks.index[
    tasks["priority_rank"] == 2
][0]

best1, route1 = create_solver(rank2)
best2, route2 = create_solver(rank1)

best_plan = None
best_score = None
mode = "HIGH_RISK_FIRST"

print("\nSearching battery-safe routes...")
print("First attempt: keep top 2 risk areas first")

for mask1 in range(1, FULL):
    if not (mask1 & (1 << rank2)):
        continue

    if mask1 & (1 << rank1):
        continue

    mask2 = FULL ^ mask1

    t1 = mission_time(mask1, best1)
    t2 = mission_time(mask2, best2)

    if t1 > MAX_TIME + 1e-9:
        continue

    if t2 > MAX_TIME + 1e-9:
        continue

    score = (
        max(t1, t2),
        t1 + t2
    )

    if best_score is None or score < best_score:
        best_score = score
        best_plan = (
            route1(mask1),
            route2(mask2)
        )


# Fallback: exact battery feasibility without
# forcing the top 2 areas to be first.
if best_plan is None:

    print("Top-2-first constraint not feasible.")
    print("Checking unrestricted exact routes...")

    mode = "BATTERY_FIRST"

    best_any, route_any = create_solver()

    for mask1 in range(1, FULL):

        # Avoid checking symmetric partitions twice
        if not (mask1 & 1):
            continue

        mask2 = FULL ^ mask1

        t1 = mission_time(mask1, best_any)
        t2 = mission_time(mask2, best_any)

        if t1 > MAX_TIME + 1e-9:
            continue

        if t2 > MAX_TIME + 1e-9:
            continue

        score = (
            max(t1, t2),
            t1 + t2
        )

        if best_score is None or score < best_score:
            best_score = score
            best_plan = (
                route_any(mask1),
                route_any(mask2)
            )


if best_plan is None:
    print("\nSTATUS: NO FEASIBLE SINGLE-SORTIE PLAN")
    print("15 tasks cannot all be scheduled")
    print("under the tested reserve constraints.")
    print("Next option: extra sortie or deferred tasks.")

else:
    rows = []
    summary_rows = []

    for drone, route in zip(
        ["Drone 1", "Drone 2"],
        best_plan
    ):
        current = DEPOT
        elapsed = 0.0
        total_distance = 0.0

        for sequence, idx in enumerate(route, 1):

            leg = dist(current, points[idx])
            total_distance += leg
            elapsed += leg / SPEED

            task = tasks.iloc[idx]

            rows.append({
                "drone": drone,
                "sequence": sequence,
                "location_id": task["location_id"],
                "priority_rank": int(
                    task["priority_rank"]
                ),
                "risk_score": float(
                    task["risk_score"]
                ),
                "x_m": points[idx][0],
                "y_m": points[idx][1],
                "arrival_min": round(elapsed, 3)
            })

            elapsed += INSPECTION
            current = points[idx]

        return_distance = dist(current, DEPOT)
        total_distance += return_distance
        elapsed += return_distance / SPEED

        summary_rows.append({
            "Drone": drone,
            "Tasks": len(route),
            "Distance_m": round(total_distance, 2),
            "Mission_Time_min": round(elapsed, 3),
            "Battery_Remaining_min": round(
                LIMIT - elapsed, 3
            ),
            "Reserve_PASS": elapsed <= MAX_TIME + 1e-9
        })

    reserve_routes = pd.DataFrame(rows)
    reserve_summary = pd.DataFrame(summary_rows)

    assert len(reserve_routes) == 15
    assert reserve_routes["location_id"].is_unique
    assert reserve_summary["Reserve_PASS"].all()
    assert reserve_routes["drone"].nunique() == 2

    print("\n=== NEW BATTERY-SAFE PLAN ===")
    display(reserve_summary)

    print("\n=== NEW MISSION SEQUENCE ===")
    display(
        reserve_routes[
            [
                "drone",
                "sequence",
                "location_id",
                "priority_rank",
                "arrival_min"
            ]
        ]
    )

    print("\nOptimization Mode:", mode)
    print("Areas Covered:", len(reserve_routes))
    print("Duplicate Areas:",
          reserve_routes["location_id"].duplicated().sum())
    print("Battery Reserve:",
          RESERVE, "min per drone")

    print("\nSTATUS: FEASIBLE PLAN FOUND")
    print("Next: check priority-arrival performance")

print("\nOriginal Files: UNCHANGED")
print("New Files Saved: NO")
print("STEP 6 COMPLETE")

````

### Original cell 73/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np

print("=== F5 BATTERY RESERVE: STEP 7 ===")

assert "reserve_routes" in globals(), "Run Step 6 first"
assert "reserve_summary" in globals(), "Run Step 6 first"

old = pd.read_csv(
    f5_dir / "F5_Optimized_Mission_Assignments.csv"
)

new = reserve_routes.copy()

# Verify same areas and same risk ranks
assert len(old) == len(new) == 15
assert old["location_id"].is_unique
assert new["location_id"].is_unique

comparison = old[
    ["location_id", "priority_rank", "arrival_min"]
].merge(
    new[
        ["location_id", "priority_rank", "arrival_min"]
    ],
    on="location_id",
    suffixes=("_old", "_new"),
    validate="one_to_one"
)

assert (
    comparison["priority_rank_old"]
    == comparison["priority_rank_new"]
).all()

# Higher risk receives greater importance
comparison["weight"] = (
    16 - comparison["priority_rank_old"]
) ** 2

old_weighted = np.average(
    comparison["arrival_min_old"],
    weights=comparison["weight"]
)

new_weighted = np.average(
    comparison["arrival_min_new"],
    weights=comparison["weight"]
)

priority_change = (
    (old_weighted - new_weighted)
    / old_weighted * 100
)

# Historical pre-optimization benchmark,
# rounded from the earlier F5 analysis
original_baseline = 22.82

new_improvement_from_baseline = (
    (original_baseline - new_weighted)
    / original_baseline * 100
)

print("\n=== PRIORITY ARRIVAL COMPARISON ===")

print(
    "Previous Optimized Weighted Arrival:",
    round(old_weighted, 3), "min"
)

print(
    "New Battery-Safe Weighted Arrival:",
    round(new_weighted, 3), "min"
)

print(
    "Change vs Previous Optimized:",
    round(priority_change, 2), "%"
)

print(
    "Improvement vs Original Baseline:",
    round(new_improvement_from_baseline, 2), "%"
)

print("\n=== TOP 5 HIGH-RISK AREA ARRIVALS ===")

top5 = comparison.sort_values(
    "priority_rank_old"
).head(5)

display(
    top5[
        [
            "location_id",
            "priority_rank_old",
            "arrival_min_old",
            "arrival_min_new"
        ]
    ]
)

print("\n=== BATTERY RESERVE CHECK ===")

display(reserve_summary)

assert reserve_summary["Reserve_PASS"].all()
assert (
    reserve_summary["Battery_Remaining_min"] >= 4.5
).all()

# Check top 2 highest-risk tasks remain first
top2 = new[
    new["priority_rank"].isin([1, 2])
]

assert len(top2) == 2
assert top2["sequence"].eq(1).all()
assert top2["drone"].nunique() == 2

print("\n=== FINAL DECISION ===")

if new_weighted <= old_weighted:
    print("RESULT: BATTERY + PRIORITY BOTH IMPROVED")
    print("Next: Save and validate battery-safe plan")

else:
    print("RESULT: BATTERY IMPROVED, PRIORITY TRADE-OFF")
    print("Next: Optimize priority arrival")
    print("while keeping 10% battery reserve")

print("\nAreas Covered:", len(new))
print("Duplicate Areas:", new["location_id"].duplicated().sum())
print("Original Files: UNCHANGED")
print("STEP 7 COMPLETE")

````

### Original cell 74/118 — PYTHON SOURCE (archived, not run)

````python

import pandas as pd
import numpy as np
import math
import random

print("=== F5 PRIORITY + BATTERY OPTIMIZATION: STEP 8 ===")

assert "reserve_routes" in globals(), "Run Step 6 first"
assert "f5_dir" in globals(), "Previous F5 path missing"

original = pd.read_csv(
    f5_dir / "F5_Optimized_Mission_Assignments.csv"
)

data = {
    row["location_id"]: row.to_dict()
    for _, row in original.iterrows()
}

drone_names = ["Drone 1", "Drone 2"]

initial_routes = [
    reserve_routes[
        reserve_routes["drone"] == drone
    ].sort_values("sequence")["location_id"].tolist()
    for drone in drone_names
]

assert sorted(sum(initial_routes, [])) == sorted(data)

# Fixed high-risk first stops
assert int(data[initial_routes[0][0]]["priority_rank"]) == 2
assert int(data[initial_routes[1][0]]["priority_rank"]) == 1

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY_LIMIT = 45.0
RESERVE_MIN = 4.5
MAX_MISSION = BATTERY_LIMIT - RESERVE_MIN

def calculate_plan(routes):
    weighted_sum = 0.0
    total_weight = 0.0
    mission_times = []
    output_rows = []
    total_distances = []

    for drone_index, route in enumerate(routes):
        current = DEPOT
        elapsed = 0.0
        distance_total = 0.0

        for sequence, location_id in enumerate(route, 1):
            task = data[location_id]

            target = (
                float(task["x_m"]),
                float(task["y_m"])
            )

            leg = math.dist(current, target)

            distance_total += leg
            elapsed += leg / SPEED

            arrival = elapsed
            rank = int(task["priority_rank"])
            weight = (16 - rank) ** 2

            weighted_sum += arrival * weight
            total_weight += weight

            output_rows.append({
                "drone": drone_names[drone_index],
                "sequence": sequence,
                "location_id": location_id,
                "image": task["image"],
                "priority_rank": rank,
                "risk_score": float(task["risk_score"]),
                "risk_category": task["risk_category"],
                "x_m": target[0],
                "y_m": target[1],
                "arrival_min": arrival,
                "coordinate_type": "SIMULATED"
            })

            elapsed += INSPECTION
            current = target

        return_distance = math.dist(current, DEPOT)

        distance_total += return_distance
        elapsed += return_distance / SPEED

        mission_times.append(elapsed)
        total_distances.append(distance_total)

    weighted_arrival = weighted_sum / total_weight

    return (
        weighted_arrival,
        mission_times,
        total_distances,
        output_rows
    )


def evaluate(routes):
    weighted, times, _, _ = calculate_plan(routes)

    violations = [
        max(0.0, t - MAX_MISSION)
        for t in times
    ]

    penalty = (
        7.0 * sum(v * v for v in violations)
        + 3.0 * sum(violations)
    )

    return weighted + penalty, weighted, times


rng = random.Random(42)

def make_neighbor(routes):
    new = [r.copy() for r in routes]

    operation = rng.choice([
        "swap_within",
        "reverse",
        "relocate",
        "swap_between",
        "move_between"
    ])

    if operation == "swap_within":
        d = rng.randrange(2)

        if len(new[d]) > 2:
            i, j = rng.sample(
                range(1, len(new[d])), 2
            )
            new[d][i], new[d][j] = (
                new[d][j], new[d][i]
            )

    elif operation == "reverse":
        d = rng.randrange(2)

        if len(new[d]) > 2:
            i, j = sorted(
                rng.sample(range(1, len(new[d])), 2)
            )
            new[d][i:j+1] = reversed(
                new[d][i:j+1]
            )

    elif operation == "relocate":
        d = rng.randrange(2)

        if len(new[d]) > 2:
            i = rng.randrange(1, len(new[d]))
            item = new[d].pop(i)

            j = rng.randrange(1, len(new[d]) + 1)
            new[d].insert(j, item)

    elif operation == "swap_between":
        if len(new[0]) > 1 and len(new[1]) > 1:
            i = rng.randrange(1, len(new[0]))
            j = rng.randrange(1, len(new[1]))

            new[0][i], new[1][j] = (
                new[1][j], new[0][i]
            )

    elif operation == "move_between":
        source = rng.randrange(2)
        destination = 1 - source

        if len(new[source]) > 1:
            i = rng.randrange(1, len(new[source]))
            item = new[source].pop(i)

            j = rng.randrange(
                1, len(new[destination]) + 1
            )
            new[destination].insert(j, item)

    return new


best_routes = [r.copy() for r in initial_routes]
_, best_weighted, best_times = evaluate(best_routes)

print("\nInitial Weighted Arrival:",
      round(best_weighted, 3), "min")

print("Searching better battery-safe routes...")

RESTARTS = 5
ITERATIONS = 6000

for restart in range(RESTARTS):

    current = [r.copy() for r in best_routes]

    # Diversify each restart
    for _ in range(restart * 3):
        current = make_neighbor(current)

    current_score, _, _ = evaluate(current)

    for step in range(ITERATIONS):

        temperature = 2.5 * (
            (0.03 / 2.5) ** (
                step / (ITERATIONS - 1)
            )
        )

        candidate = make_neighbor(current)

        score, weighted, times = evaluate(candidate)

        difference = score - current_score

        if (
            difference <= 0
            or rng.random() < math.exp(
                -difference / temperature
            )
        ):
            current = candidate
            current_score = score

        battery_ok = all(
            t <= MAX_MISSION + 1e-9
            for t in times
        )

        if battery_ok and weighted < best_weighted - 1e-9:
            best_weighted = weighted
            best_routes = [
                r.copy() for r in candidate
            ]
            best_times = times.copy()


# Final validated result
(
    final_weighted,
    final_times,
    final_distances,
    final_rows
) = calculate_plan(best_routes)

battery_priority_routes = pd.DataFrame(final_rows)

battery_priority_summary = pd.DataFrame([
    {
        "Drone": drone_names[i],
        "Tasks": len(best_routes[i]),
        "Distance_m": round(final_distances[i], 2),
        "Mission_Time_min": round(final_times[i], 3),
        "Battery_Remaining_min": round(
            BATTERY_LIMIT - final_times[i], 3
        ),
        "Reserve_PASS": (
            final_times[i] <= MAX_MISSION + 1e-9
        )
    }
    for i in range(2)
])

# Automated safety validation
assert len(battery_priority_routes) == 15
assert battery_priority_routes["location_id"].is_unique
assert battery_priority_summary["Reserve_PASS"].all()

assert (
    battery_priority_routes.groupby("drone")
    ["sequence"].min().eq(1).all()
)

assert int(
    data[best_routes[0][0]]["priority_rank"]
) == 2

assert int(
    data[best_routes[1][0]]["priority_rank"]
) == 1

initial_weighted = calculate_plan(initial_routes)[0]

assert final_weighted <= initial_weighted + 1e-9

old_weighted = np.average(
    original["arrival_min"],
    weights=(16 - original["priority_rank"]) ** 2
)

baseline = 22.82

print("\n=== OPTIMIZED BATTERY-SAFE PLAN ===")
display(battery_priority_summary)

print("\n=== NEW ROUTE ===")
display(
    battery_priority_routes[
        [
            "drone",
            "sequence",
            "location_id",
            "priority_rank",
            "arrival_min"
        ]
    ].round(3)
)

print("\n=== PERFORMANCE COMPARISON ===")

print(
    "Previous F5 Weighted Arrival:",
    round(old_weighted, 3), "min"
)

print(
    "Step 6 Battery-Safe Arrival:",
    round(initial_weighted, 3), "min"
)

print(
    "Step 8 Optimized Arrival:",
    round(final_weighted, 3), "min"
)

print(
    "Improvement vs Step 6:",
    round(
        (initial_weighted - final_weighted)
        / initial_weighted * 100, 2
    ), "%"
)

print(
    "Improvement vs Original Baseline:",
    round(
        (baseline - final_weighted)
        / baseline * 100, 2
    ), "%"
)

print("\n=== FINAL VALIDATION ===")
print("Areas Covered:", len(battery_priority_routes))
print(
    "Duplicate Areas:",
    battery_priority_routes["location_id"]
    .duplicated().sum()
)
print(
    "Battery Reserve PASS:",
    battery_priority_summary["Reserve_PASS"].sum(),
    "/ 2"
)

print("Top 2 Risk Areas First: PASS")
print("Original Files: UNCHANGED")
print("New Files Saved: NO")
print("Simulation Only: YES")
print("\nSTEP 8 COMPLETE")

````

### Original cell 75/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import math
import json

print("=== F5 BATTERY RESERVE: STEP 9 ===")

assert "battery_priority_routes" in globals(), "Run Step 8 first"

source_dir = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Optimized_20261122_055356_483143"
)

source_file = source_dir / "F5_Optimized_Mission_Assignments.csv"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_hash = sha256(source_file)

original = pd.read_csv(source_file)
routes = battery_priority_routes.copy()

LIMIT = 45.0
RESERVE = 4.5
MAX_TIME = LIMIT - RESERVE
SPEED = 60.0
INSPECTION = 3.0
DEPOT = (250.0, 250.0)

checks = {}

checks["15 Areas Covered"] = (
    len(routes) == 15
    and set(routes["location_id"]) == set(original["location_id"])
)

checks["No Duplicate Areas"] = routes["location_id"].is_unique

checks["Two Drones Used"] = (
    set(routes["drone"]) == {"Drone 1", "Drone 2"}
)

reference = original.set_index("location_id")
candidate = routes.set_index("location_id")

checks["F4 Risk Data Preserved"] = (
    candidate["priority_rank"].sort_index().equals(
        reference["priority_rank"].sort_index()
    )
    and np.allclose(
        candidate["risk_score"].sort_index(),
        reference["risk_score"].sort_index()
    )
)

checks["Synthetic Coordinates Preserved"] = (
    np.allclose(
        candidate[["x_m", "y_m"]].sort_index(),
        reference[["x_m", "y_m"]].sort_index()
    )
    and routes["coordinate_type"].eq("SIMULATED").all()
)

summary_rows = []
arrival_checks = []
sequence_checks = []

for drone in ["Drone 1", "Drone 2"]:
    group = (
        routes[routes["drone"] == drone]
        .sort_values("sequence")
    )

    sequence_checks.append(
        group["sequence"].tolist()
        == list(range(1, len(group) + 1))
    )

    current = DEPOT
    elapsed = 0.0
    total_distance = 0.0

    for _, task in group.iterrows():
        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        leg = math.dist(current, target)

        elapsed += leg / SPEED
        total_distance += leg

        arrival_checks.append(
            abs(elapsed - float(task["arrival_min"])) < 0.002
        )

        elapsed += INSPECTION
        current = target

    return_distance = math.dist(current, DEPOT)
    total_distance += return_distance
    elapsed += return_distance / SPEED

    summary_rows.append({
        "Drone": drone,
        "Tasks": len(group),
        "Distance_m": round(total_distance, 3),
        "Mission_Time_min": round(elapsed, 3),
        "Battery_Limit_min": LIMIT,
        "Required_Reserve_min": RESERVE,
        "Battery_Remaining_min": round(LIMIT - elapsed, 3),
        "Reserve_PASS": elapsed <= MAX_TIME + 1e-9
    })

summary = pd.DataFrame(summary_rows)

checks["Mission Sequence Valid"] = all(sequence_checks)
checks["Arrival Times Recalculated"] = all(arrival_checks)
checks["10 Percent Reserve PASS"] = summary["Reserve_PASS"].all()

top2 = routes[routes["priority_rank"].isin([1, 2])]

checks["Top Two Risks First"] = (
    len(top2) == 2
    and top2["sequence"].eq(1).all()
    and top2["drone"].nunique() == 2
)

weights = (16 - routes["priority_rank"]) ** 2

new_weighted = np.average(
    routes["arrival_min"],
    weights=weights
)

old_weighted = np.average(
    original["arrival_min"],
    weights=(16 - original["priority_rank"]) ** 2
)

checks["Priority Better Than Previous F5"] = (
    new_weighted < old_weighted
)

checks["Return To Base Included"] = (
    summary["Distance_m"].gt(0).all()
    and summary["Mission_Time_min"].gt(
        summary["Tasks"] * INSPECTION
    ).all()
)

print("\n=== AUTOMATED VALIDATION ===")

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

passed_count = sum(bool(v) for v in checks.values())

print("\nTests Passed:", passed_count, "/", len(checks))

assert all(checks.values()), "Validation failed; no files saved"

print("\n=== FINAL DRONE SUMMARY ===")
display(summary)

print("\nWeighted Arrival:", round(new_weighted, 3), "min")
print(
    "Improvement vs Previous F5:",
    round((old_weighted-new_weighted)/old_weighted*100, 2),
    "%"
)

# Save only after validation
parent = (
    source_dir.parent / "F5_Advanced_Enhancements"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

output_dir = parent / f"Battery_Reserve_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=False)

assign_path = output_dir / "F5_Battery_Safe_Assignments.csv"
summary_path = output_dir / "F5_Battery_Safe_Summary.csv"
report_path = output_dir / "F5_Battery_Safe_Report.json"
map_path = output_dir / "F5_Battery_Safe_Route_Map.png"

routes.to_csv(assign_path, index=False)
summary.to_csv(summary_path, index=False)

report = {
    "feature": "F5 Advanced Battery Reserve Planning",
    "status": "VALIDATED_SIMULATION",
    "total_areas": len(routes),
    "drones": 2,
    "battery_limit_min": LIMIT,
    "reserve_min": RESERVE,
    "usable_mission_min": MAX_TIME,
    "previous_weighted_arrival_min": round(float(old_weighted), 4),
    "new_weighted_arrival_min": round(float(new_weighted), 4),
    "improvement_vs_previous_percent": round(
        float((old_weighted-new_weighted)/old_weighted*100), 2
    ),
    "optimization_method": "Simulated Annealing Heuristic",
    "validation": {
        k: bool(v) for k, v in checks.items()
    },
    "simulation_only": True,
    "coordinate_type": "Synthetic local XY",
    "real_drone_flight": False,
    "original_files_modified": False
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# Synthetic XY route visualization
fig, ax = plt.subplots(figsize=(11, 8))

for drone in ["Drone 1", "Drone 2"]:
    group = (
        routes[routes["drone"] == drone]
        .sort_values("sequence")
    )

    xs = [DEPOT[0]] + group["x_m"].tolist() + [DEPOT[0]]
    ys = [DEPOT[1]] + group["y_m"].tolist() + [DEPOT[1]]

    ax.plot(xs, ys, marker="o", linewidth=2, label=drone)

    for _, task in group.iterrows():
        ax.annotate(
            f"R{int(task['priority_rank'])} / S{int(task['sequence'])}",
            (task["x_m"], task["y_m"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8
        )

ax.scatter(
    DEPOT[0], DEPOT[1],
    marker="*", s=250,
    color="red", label="Simulated Depot",
    zorder=5
)

ax.set_title(
    "DENGU HUNTER - Battery-Safe Priority Mission\n"
    "Synthetic XY Simulation | 10% Battery Reserve"
)
ax.set_xlabel("Synthetic X (m)")
ax.set_ylabel("Synthetic Y (m)")
ax.grid(alpha=0.3)
ax.legend()
ax.set_aspect("equal", adjustable="datalim")

plt.tight_layout()
plt.savefig(map_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# Final file integrity checks
assert sha256(source_file) == source_hash

saved_files = [
    assign_path, summary_path, report_path, map_path
]

assert all(p.exists() and p.stat().st_size > 0 for p in saved_files)

assert len(pd.read_csv(assign_path)) == 15

print("\n=== SAVED FILES ===")

for path in saved_files:
    print("SAVED:", path.name)

print("\nOutput Folder:", output_dir)

print("\nOriginal F5 File: UNCHANGED")
print("Simulation Only: YES")
print("STATUS: BATTERY RESERVE VALIDATION COMPLETE")

````

### Original cell 76/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from itertools import permutations
import pandas as pd
import math

print("=== F5 DRONE FAILURE RECOVERY: STEP 10 ===")

source = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670/"
    "F5_Battery_Safe_Assignments.csv"
)

tasks = pd.read_csv(source)

assert len(tasks) == 15
assert tasks["location_id"].is_unique

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY_LIMIT = 45.0
RESERVE = 4.5
USABLE = BATTERY_LIMIT - RESERVE

def point(row):
    return (float(row["x_m"]), float(row["y_m"]))

def route_metrics(route):
    current = DEPOT
    elapsed = 0.0
    distance = 0.0
    arrivals = []

    for _, task in route.iterrows():
        target = point(task)
        leg = math.dist(current, target)

        distance += leg
        elapsed += leg / SPEED

        arrivals.append(elapsed)

        elapsed += INSPECTION
        current = target

    return_leg = math.dist(current, DEPOT)
    distance += return_leg
    elapsed += return_leg / SPEED

    return elapsed, distance, arrivals

# Scenario: Drone 1 aborts after 2 inspections
drone1 = (
    tasks[tasks["drone"] == "Drone 1"]
    .sort_values("sequence")
    .reset_index(drop=True)
)

drone2 = (
    tasks[tasks["drone"] == "Drone 2"]
    .sort_values("sequence")
    .reset_index(drop=True)
)

completed_d1 = drone1.iloc[:2].copy()
pending_d1 = drone1.iloc[2:].copy()

abort_time, abort_distance, _ = route_metrics(
    completed_d1
)

drone2_time, drone2_distance, _ = route_metrics(
    drone2
)

print("\n=== FAILURE SCENARIO ===")
print("Failed Drone: Drone 1")
print("Failure Type: SIMULATED CONTROLLED ABORT")
print("Completed Before Abort:", len(completed_d1))
print("Pending Tasks:", len(pending_d1))
print("Drone 2 Original Tasks:", len(drone2))

print("\n=== BATTERY SAFETY ===")
print("Drone 1 Abort + Return:", round(abort_time, 3), "min")
print("Drone 2 Original Mission:", round(drone2_time, 3), "min")
print("Usable Time Per Sortie:", USABLE, "min")

assert abort_time <= USABLE
assert drone2_time <= USABLE

# Conservative recovery:
# Drone 2 finishes its current sortie first.
# Remaining tasks are planned after recharge.
print("\n=== PENDING TASKS ===")

display(
    pending_d1[
        ["location_id", "priority_rank", "risk_category"]
    ]
)

print("\n=== RECOVERY SORTIE OPTIMIZATION ===")

# Exact search for the 5 remaining tasks
# Objective: minimize priority-weighted arrival
# while keeping the battery reserve.

best_route = None
best_score = None
best_time = None

for order in permutations(range(len(pending_d1))):
    candidate = pending_d1.iloc[list(order)]

    mission_time, _, arrivals = route_metrics(
        candidate
    )

    if mission_time > USABLE + 1e-9:
        continue

    weights = [
        (16 - int(rank)) ** 2
        for rank in candidate["priority_rank"]
    ]

    weighted_arrival = (
        sum(a * w for a, w in zip(arrivals, weights))
        / sum(weights)
    )

    score = (weighted_arrival, mission_time)

    if best_score is None or score < best_score:
        best_score = score
        best_route = candidate.copy()
        best_time = mission_time

if best_route is None:
    print("STATUS: RECOVERY SORTIE NOT FEASIBLE")
    print("Remaining tasks must stay PENDING.")
else:
    recovery_time, recovery_distance, arrivals = (
        route_metrics(best_route)
    )

    recovery_route = best_route.copy()
    recovery_route["recovery_sequence"] = range(
        1, len(recovery_route) + 1
    )
    recovery_route["recovery_arrival_min"] = [
        round(a, 3) for a in arrivals
    ]

    print("Recovery Drone: Drone 2")
    print("Recovery Mode: NEW SORTIE AFTER RECHARGE")
    print("Recovery Tasks:", len(recovery_route))
    print("Recovery Time:", round(recovery_time, 3), "min")
    print(
        "Battery Remaining:",
        round(BATTERY_LIMIT - recovery_time, 3), "min"
    )

    print("\n=== RECOVERY ROUTE ===")
    display(
        recovery_route[
            [
                "recovery_sequence",
                "location_id",
                "priority_rank",
                "recovery_arrival_min"
            ]
        ]
    )

    # Validate coverage across all sorties
    completed_ids = (
        completed_d1["location_id"].tolist()
        + drone2["location_id"].tolist()
        + recovery_route["location_id"].tolist()
    )

    checks = {
        "All 15 Tasks Covered": (
            len(completed_ids) == 15
            and set(completed_ids) == set(tasks["location_id"])
        ),
        "No Duplicate Tasks": (
            len(completed_ids) == len(set(completed_ids))
        ),
        "Drone 1 Safe Abort Budget": (
            abort_time <= USABLE
        ),
        "Drone 2 Original Battery PASS": (
            drone2_time <= USABLE
        ),
        "Recovery Battery Reserve PASS": (
            recovery_time <= USABLE
        ),
        "All Recovery Tasks Assigned": (
            set(recovery_route["location_id"])
            == set(pending_d1["location_id"])
        )
    }

    print("\n=== AUTOMATED VALIDATION ===")

    for name, passed in checks.items():
        print(
            "PASS" if passed else "FAIL",
            "-", name
        )

    print(
        "\nTests Passed:",
        sum(checks.values()), "/", len(checks)
    )

    assert all(checks.values())

    print("\nSTATUS: RECOVERY PLAN VALIDATED")

print("\nSimulation Only: YES")
print("Original Files: UNCHANGED")
print("New Files Saved: NO")
print("STEP 10 COMPLETE")

````

### Original cell 77/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from itertools import permutations
import pandas as pd
import math
import hashlib

print("=== F5 DRONE FAILURE RECOVERY: STEP 11 ===")

source = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670/"
    "F5_Battery_Safe_Assignments.csv"
)

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hash = file_hash(source)

df = pd.read_csv(source)

assert len(df) == 15
assert df["location_id"].is_unique
assert set(df["drone"]) == {"Drone 1", "Drone 2"}

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY = 45.0
RESERVE = 4.5
MAX_TIME = BATTERY - RESERVE

task_info = df.set_index("location_id").to_dict("index")

routes = {
    drone: group.sort_values("sequence")[
        "location_id"
    ].tolist()
    for drone, group in df.groupby("drone")
}

all_ids = set(df["location_id"])


def calculate(route):
    """Independent route calculation, including return."""
    current = DEPOT
    elapsed = 0.0
    weighted_sum = 0.0
    total_weight = 0.0

    for location_id in route:
        task = task_info[location_id]

        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        elapsed += math.dist(current, target) / SPEED

        rank = int(task["priority_rank"])
        weight = (16 - rank) ** 2

        weighted_sum += elapsed * weight
        total_weight += weight

        elapsed += INSPECTION
        current = target

    elapsed += math.dist(current, DEPOT) / SPEED

    weighted_arrival = (
        weighted_sum / total_weight
        if total_weight else 0.0
    )

    return elapsed, weighted_arrival


def optimize_recovery(pending):
    """Find best priority-aware, battery-safe recovery."""
    if not pending:
        return [], 0.0

    best_route = None
    best_score = None
    best_time = None

    for order in permutations(pending):
        mission_time, weighted = calculate(order)

        if mission_time > MAX_TIME + 1e-9:
            continue

        score = (weighted, mission_time)

        if best_score is None or score < best_score:
            best_score = score
            best_route = list(order)
            best_time = mission_time

    return best_route, best_time


print("\n=== ORIGINAL MISSION SAFETY ===")

for drone, route in routes.items():
    mission_time, _ = calculate(route)

    print(
        drone,
        "| Tasks:", len(route),
        "| Time:", round(mission_time, 3),
        "| Reserve:", round(BATTERY - mission_time, 3)
    )

    assert mission_time <= MAX_TIME


print("\n=== STEP 10 INDEPENDENT CROSS-CHECK ===")

step10_route = [
    "SIM-AREA-006",
    "SIM-AREA-009",
    "SIM-AREA-004",
    "SIM-AREA-008",
    "SIM-AREA-005"
]

step10_time, _ = calculate(step10_route)

print(
    "Step 10 Reported Recovery Time:",
    39.477, "min"
)
print(
    "Independently Recalculated:",
    round(step10_time, 3), "min"
)

step10_match = abs(step10_time - 39.477) < 0.02

print(
    "Cross-Check:",
    "PASS" if step10_match else "NEEDS REVIEW"
)


print("\n=== MULTIPLE FAILURE SCENARIOS ===")

results = []

for failed_drone in ["Drone 1", "Drone 2"]:

    survivor = (
        "Drone 2"
        if failed_drone == "Drone 1"
        else "Drone 1"
    )

    failed_route = routes[failed_drone]
    survivor_route = routes[survivor]

    n = len(failed_route)

    # Six different failure points per drone
    failure_points = sorted(set([
        0, 1, 2, n // 2, n - 1, n
    ]))

    for completed_count in failure_points:

        completed = failed_route[:completed_count]
        pending = failed_route[completed_count:]

        # Controlled abort and return to depot
        abort_time, _ = calculate(completed)

        # Survivor completes original sortie
        survivor_time, _ = calculate(survivor_route)

        # Recovery after survivor returns and recharges
        recovery_route, recovery_time = (
            optimize_recovery(pending)
        )

        recovery_feasible = (
            recovery_route is not None
        )

        planned = (
            completed
            + survivor_route
            + (
                recovery_route
                if recovery_feasible else []
            )
        )

        coverage_ok = (
            len(planned) == 15
            and set(planned) == all_ids
        )

        duplicate_free = (
            len(planned) == len(set(planned))
        )

        battery_ok = (
            abort_time <= MAX_TIME + 1e-9
            and survivor_time <= MAX_TIME + 1e-9
            and recovery_feasible
            and recovery_time <= MAX_TIME + 1e-9
        )

        pending_ok = (
            recovery_feasible
            and set(recovery_route) == set(pending)
        )

        passed = (
            coverage_ok
            and duplicate_free
            and battery_ok
            and pending_ok
        )

        results.append({
            "Failed_Drone": failed_drone,
            "Completed_Before_Failure": completed_count,
            "Pending_Tasks": len(pending),
            "Recovery_Drone": survivor,
            "Abort_Return_min": round(abort_time, 3),
            "Recovery_Time_min": (
                round(recovery_time, 3)
                if recovery_feasible else None
            ),
            "Recovery_Reserve_min": (
                round(BATTERY - recovery_time, 3)
                if recovery_feasible and pending
                else None
            ),
            "Status": "PASS" if passed else "FAIL"
        })


scenario_df = pd.DataFrame(results)

display(scenario_df)

passed_count = scenario_df["Status"].eq("PASS").sum()

print("\n=== FINAL VALIDATION ===")
print(
    "Scenarios Passed:",
    passed_count, "/", len(scenario_df)
)
print(
    "Drone 1 Failure Scenarios:",
    len(scenario_df[
        scenario_df["Failed_Drone"] == "Drone 1"
    ])
)
print(
    "Drone 2 Failure Scenarios:",
    len(scenario_df[
        scenario_df["Failed_Drone"] == "Drone 2"
    ])
)

print(
    "Step 10 Cross-Check:",
    "PASS" if step10_match else "FAIL"
)

print(
    "Original File Unchanged:",
    file_hash(source) == original_hash
)

assert step10_match, "Step 10 time mismatch"
assert scenario_df["Status"].eq("PASS").all()
assert file_hash(source) == original_hash

print("\nSTATUS: MULTI-SCENARIO RECOVERY VALIDATED")
print("Real Drone Failure Tested: NO")
print("Recharge Duration Modeled: NO")
print("Actual Field Inspections: NO")
print("New Files Saved: NO")
print("STEP 11 COMPLETE")

````

### Original cell 78/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
from itertools import permutations
import pandas as pd
import math
import hashlib
import json

print("=== F5 DRONE FAILURE RECOVERY: STEP 12 ===")

source = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670/"
    "F5_Battery_Safe_Assignments.csv"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hash = sha256(source)

df = pd.read_csv(source)

assert len(df) == 15
assert df["location_id"].is_unique
assert set(df["drone"]) == {"Drone 1", "Drone 2"}

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY = 45.0
RESERVE = 4.5
MAX_TIME = BATTERY - RESERVE

task_info = df.set_index(
    "location_id"
).to_dict("index")

routes = {
    drone: group.sort_values("sequence")[
        "location_id"
    ].tolist()
    for drone, group in df.groupby("drone")
}

all_ids = set(df["location_id"])

def evaluate_route(route):
    current = DEPOT
    elapsed = 0.0
    weighted_sum = 0.0
    total_weight = 0.0
    arrivals = {}

    for location_id in route:
        task = task_info[location_id]

        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        elapsed += math.dist(current, target) / SPEED

        arrivals[location_id] = elapsed

        rank = int(task["priority_rank"])
        weight = (16 - rank) ** 2

        weighted_sum += elapsed * weight
        total_weight += weight

        elapsed += INSPECTION
        current = target

    # Return to simulated depot
    elapsed += math.dist(current, DEPOT) / SPEED

    weighted = (
        weighted_sum / total_weight
        if total_weight else 0.0
    )

    return elapsed, weighted, arrivals


def optimize_recovery(pending):
    if not pending:
        return [], 0.0, {}

    best = None

    for candidate in permutations(pending):
        time, weighted, arrivals = evaluate_route(candidate)

        if time > MAX_TIME + 1e-9:
            continue

        score = (weighted, time)

        if best is None or score < best[0]:
            best = (
                score,
                list(candidate),
                time,
                arrivals
            )

    if best is None:
        return None, None, None

    return best[1], best[2], best[3]


scenario_rows = []
handoff_rows = []
audit_rows = []

print("\nGenerating recovery handoff proposals...")

for failed_drone in ["Drone 1", "Drone 2"]:

    survivor = (
        "Drone 2"
        if failed_drone == "Drone 1"
        else "Drone 1"
    )

    failed_route = routes[failed_drone]
    survivor_route = routes[survivor]

    n = len(failed_route)

    failure_points = sorted(set([
        0, 1, 2, n // 2, n - 1, n
    ]))

    for completed_count in failure_points:

        scenario_id = (
            failed_drone.replace(" ", "")
            + "_AFTER_"
            + str(completed_count)
        )

        completed = failed_route[:completed_count]
        pending = failed_route[completed_count:]

        abort_time, _, _ = evaluate_route(completed)

        survivor_time, _, _ = evaluate_route(
            survivor_route
        )

        recovery_route, recovery_time, arrivals = (
            optimize_recovery(pending)
        )

        assert recovery_route is not None, (
            "Recovery infeasible: " + scenario_id
        )

        planned = (
            completed
            + survivor_route
            + recovery_route
        )

        coverage_ok = (
            len(planned) == 15
            and set(planned) == all_ids
            and len(planned) == len(set(planned))
        )

        battery_ok = (
            abort_time <= MAX_TIME + 1e-9
            and survivor_time <= MAX_TIME + 1e-9
            and recovery_time <= MAX_TIME + 1e-9
        )

        pending_ok = (
            set(recovery_route) == set(pending)
        )

        passed = (
            coverage_ok and battery_ok and pending_ok
        )

        assert passed, "Scenario failed: " + scenario_id

        scenario_rows.append({
            "scenario_id": scenario_id,
            "failed_drone": failed_drone,
            "completed_before_abort": len(completed),
            "pending_tasks": len(pending),
            "recovery_drone": survivor,
            "abort_return_min": round(abort_time, 3),
            "survivor_original_min": round(
                survivor_time, 3
            ),
            "recovery_sortie_min": round(
                recovery_time, 3
            ),
            "recovery_reserve_min": (
                round(BATTERY - recovery_time, 3)
                if pending else None
            ),
            "recovery_required": bool(pending),
            "handoff_status": (
                "AWAITING_HUMAN_APPROVAL"
                if pending else "NOT_REQUIRED"
            ),
            "recharge_status": (
                "NOT_CONFIRMED"
                if pending else "NOT_REQUIRED"
            ),
            "status": "PASS"
        })

        # Proposed handoff for each pending task
        for sequence, location_id in enumerate(
            recovery_route, 1
        ):
            task = task_info[location_id]

            handoff_rows.append({
                "scenario_id": scenario_id,
                "location_id": location_id,
                "priority_rank": int(
                    task["priority_rank"]
                ),
                "risk_category": task["risk_category"],
                "from_drone": failed_drone,
                "proposed_to_drone": survivor,
                "recovery_sequence": sequence,
                "recovery_arrival_min": round(
                    arrivals[location_id], 3
                ),
                "handoff_status": "PROPOSED_ONLY",
                "human_approval": "PENDING",
                "recharge_confirmation": "PENDING",
                "execution_status": "NOT_EXECUTED"
            })

        # All 15 tasks receive a scenario-specific state
        for location_id in df["location_id"]:

            if location_id in completed:
                task_state = (
                    "SIMULATED_COMPLETED_BEFORE_ABORT"
                )
                assigned_drone = failed_drone
                proposed_drone = None

            elif location_id in survivor_route:
                task_state = "ORIGINAL_SORTIE_PLANNED"
                assigned_drone = survivor
                proposed_drone = None

            else:
                task_state = (
                    "RECOVERY_PROPOSED_NOT_EXECUTED"
                )
                assigned_drone = failed_drone
                proposed_drone = survivor

            audit_rows.append({
                "scenario_id": scenario_id,
                "location_id": location_id,
                "task_state": task_state,
                "original_drone": assigned_drone,
                "proposed_recovery_drone": proposed_drone,
                "real_field_completion_confirmed": False
            })


scenario_results = pd.DataFrame(scenario_rows)
handoff_proposals = pd.DataFrame(handoff_rows)
task_audit = pd.DataFrame(audit_rows)

print("\n=== SCENARIO RESULTS ===")

display(scenario_results)

print("\n=== EXAMPLE: DRONE 1 AFTER 2 TASKS ===")

example = handoff_proposals[
    handoff_proposals["scenario_id"]
    == "Drone1_AFTER_2"
]

display(
    example[
        [
            "location_id",
            "priority_rank",
            "proposed_to_drone",
            "recovery_sequence",
            "recovery_arrival_min",
            "handoff_status"
        ]
    ]
)

# Final validation before saving
checks = {
    "12 Scenarios": len(scenario_results) == 12,
    "All Scenarios PASS": (
        scenario_results["status"].eq("PASS").all()
    ),
    "Both Failure Drones Tested": (
        scenario_results["failed_drone"].nunique() == 2
    ),
    "Every Scenario Tracks 15 Tasks": (
        task_audit.groupby("scenario_id")
        .size().eq(15).all()
    ),
    "No Duplicate Task States": (
        not task_audit.duplicated(
            ["scenario_id", "location_id"]
        ).any()
    ),
    "Handoff Count Matches Pending": (
        len(handoff_proposals)
        == int(scenario_results["pending_tasks"].sum())
    ),
    "No Duplicate Handoff Tasks": (
        not handoff_proposals.duplicated(
            ["scenario_id", "location_id"]
        ).any()
    ),
    "All Handoffs Require Approval": (
        handoff_proposals["human_approval"]
        .eq("PENDING").all()
    ),
    "No Handoff Marked Executed": (
        handoff_proposals["execution_status"]
        .eq("NOT_EXECUTED").all()
    ),
    "Original Source Unchanged": (
        sha256(source) == original_hash
    )
}

print("\n=== FINAL VALIDATION ===")

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/",
    len(checks)
)

assert all(checks.values()), "Validation failed"

# Save only validated outputs
parent = source.parent.parent

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = (
    parent / f"Drone_Failure_Recovery_{timestamp}"
)

output_dir.mkdir(parents=True, exist_ok=False)

scenario_path = (
    output_dir / "F5_Failure_Scenario_Results.csv"
)

handoff_path = (
    output_dir / "F5_Failure_Handoff_Proposals.csv"
)

audit_path = (
    output_dir / "F5_Failure_Task_State_Audit.csv"
)

report_path = (
    output_dir / "F5_Failure_Recovery_Report.json"
)

scenario_results.to_csv(scenario_path, index=False)
handoff_proposals.to_csv(handoff_path, index=False)
task_audit.to_csv(audit_path, index=False)

report = {
    "feature": "F5 Advanced Drone Failure Recovery",
    "status": "VALIDATED_SIMULATION",
    "scenario_count": len(scenario_results),
    "scenarios_passed": int(
        scenario_results["status"].eq("PASS").sum()
    ),
    "proposed_handoffs": len(handoff_proposals),
    "task_state_records": len(task_audit),
    "battery_limit_min": BATTERY,
    "reserve_min": RESERVE,
    "recovery_mode": (
        "Surviving drone, new sortie after recharge"
    ),
    "optimization": (
        "Exact permutation search for "
        "priority-weighted arrival"
    ),
    "validation": {
        name: bool(value)
        for name, value in checks.items()
    },
    "assumptions": {
        "synthetic_coordinates": True,
        "controlled_abort_at_completed_task": True,
        "recharge_duration_modeled": False,
        "emergency_landing_modeled": False,
        "human_approval_required": True,
        "real_drone_flight": False,
        "real_field_completion": False,
        "handoffs_executed": False
    },
    "original_source_sha256": original_hash
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

saved_files = [
    scenario_path,
    handoff_path,
    audit_path,
    report_path
]

assert all(
    p.exists() and p.stat().st_size > 0
    for p in saved_files
)

assert len(pd.read_csv(scenario_path)) == 12
assert len(pd.read_csv(audit_path)) == 180
assert sha256(source) == original_hash

print("\n=== SAVED FILES ===")

for path in saved_files:
    print("SAVED:", path.name)

print("\nOutput Folder:", output_dir)

print("\n=== FINAL STATUS ===")
print("Scenarios:", len(scenario_results))
print("Proposed Handoffs:", len(handoff_proposals))
print("Task State Records:", len(task_audit))
print("Original Files: UNCHANGED")
print("Real Drone Flight: NO")
print("Real Handoff Execution: NO")
print("STATUS: RECOVERY REPORT EXPORT COMPLETE")

````

### Original cell 79/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import numpy as np
import math
import hashlib

print("=== F5 EXPLAINABLE MISSION: STEP 13 ===")

base = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner"
)

old_file = (
    base / "F5_Optimized_20261122_055356_483143"
    / "F5_Optimized_Mission_Assignments.csv"
)

new_file = (
    base / "F5_Advanced_Enhancements"
    / "Battery_Reserve_20261122_060550_816670"
    / "F5_Battery_Safe_Assignments.csv"
)

old = pd.read_csv(old_file)
new = pd.read_csv(new_file)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

old_hash = sha256(old_file)
new_hash = sha256(new_file)

assert len(old) == len(new) == 15
assert old["location_id"].is_unique
assert new["location_id"].is_unique

old_lookup = old.set_index("location_id")

DEPOT = (250.0, 250.0)
SPEED = 60.0
INSPECTION = 3.0
BATTERY = 45.0
RESERVE = 4.5
MAX_TIME = BATTERY - RESERVE

explanation_rows = []
drone_summary = []
arrival_validations = []

for drone in ["Drone 1", "Drone 2"]:

    group = (
        new[new["drone"] == drone]
        .sort_values("sequence")
    )

    current = DEPOT
    previous_stop = "DEPOT"
    elapsed = 0.0
    distance_total = 0.0

    for _, task in group.iterrows():

        location_id = task["location_id"]

        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        leg_distance = math.dist(current, target)

        distance_total += leg_distance
        elapsed += leg_distance / SPEED

        arrival = elapsed

        arrival_validations.append(
            abs(arrival - float(task["arrival_min"]))
            < 0.002
        )

        rank = int(task["priority_rank"])
        risk = float(task["risk_score"])
        category = str(task["risk_category"])

        old_arrival = float(
            old_lookup.loc[location_id, "arrival_min"]
        )

        change = arrival - old_arrival

        weight = (16 - rank) ** 2

        if rank <= 2 and int(task["sequence"]) == 1:
            reason = (
                "Top-risk area explicitly fixed "
                "as the drone's first stop"
            )
        else:
            reason = (
                "Stop selected by priority-aware "
                "route heuristic under battery constraints"
            )

        explanation_rows.append({
            "location_id": location_id,
            "drone": drone,
            "sequence": int(task["sequence"]),
            "risk_category": category,
            "risk_score": risk,
            "priority_rank": rank,
            "priority_weight": weight,
            "previous_stop": previous_stop,
            "leg_distance_m": round(leg_distance, 2),
            "arrival_min": round(arrival, 3),
            "previous_F5_arrival_min": round(
                old_arrival, 3
            ),
            "arrival_change_min": round(change, 3),
            "planning_reason": reason,
            "data_scope": "SYNTHETIC_SIMULATION"
        })

        elapsed += INSPECTION
        current = target
        previous_stop = location_id

    return_distance = math.dist(current, DEPOT)

    distance_total += return_distance
    elapsed += return_distance / SPEED

    drone_summary.append({
        "drone": drone,
        "tasks": len(group),
        "total_distance_m": round(distance_total, 3),
        "return_distance_m": round(return_distance, 3),
        "mission_time_min": round(elapsed, 3),
        "battery_remaining_min": round(
            BATTERY - elapsed, 3
        ),
        "reserve_margin_min": round(
            BATTERY - elapsed - RESERVE, 3
        ),
        "reserve_pass": elapsed <= MAX_TIME + 1e-9
    })

explain_df = pd.DataFrame(explanation_rows)
explain_summary = pd.DataFrame(drone_summary)

# Priority-weighted arrival comparison
old_weights = (16 - old["priority_rank"]) ** 2

old_weighted = np.average(
    old["arrival_min"],
    weights=old_weights
)

new_weighted = np.average(
    explain_df["arrival_min"],
    weights=explain_df["priority_weight"]
)

print("\n=== EXPLAINABLE MISSION TABLE ===")

display(
    explain_df[
        [
            "location_id",
            "drone",
            "sequence",
            "priority_rank",
            "risk_score",
            "previous_stop",
            "arrival_min",
            "arrival_change_min",
            "planning_reason"
        ]
    ]
)

print("\n=== BATTERY EXPLANATION ===")

display(explain_summary)

print("\n=== PRIORITY PERFORMANCE ===")

print(
    "Previous F5 Weighted Arrival:",
    round(old_weighted, 3), "min"
)

print(
    "Advanced F5 Weighted Arrival:",
    round(new_weighted, 3), "min"
)

print(
    "Improvement:",
    round(
        (old_weighted - new_weighted)
        / old_weighted * 100, 2
    ), "%"
)

# Check original F4 values are preserved
matched = explain_df.set_index(
    "location_id"
).sort_index()

reference = old_lookup.sort_index()

checks = {
    "15 Areas Explained": len(explain_df) == 15,
    "No Duplicate Areas": (
        explain_df["location_id"].is_unique
    ),
    "Risk Rank Preserved": (
        matched["priority_rank"].equals(
            reference["priority_rank"]
        )
    ),
    "Risk Score Preserved": (
        np.allclose(
            matched["risk_score"],
            reference["risk_score"]
        )
    ),
    "Arrival Recalculation PASS": (
        all(arrival_validations)
    ),
    "All Areas Have Explanation": (
        explain_df["planning_reason"]
        .notna().all()
    ),
    "Battery Reserve PASS": (
        explain_summary["reserve_pass"].all()
    ),
    "Priority Improved": (
        new_weighted < old_weighted
    ),
    "Original Files Unchanged": (
        sha256(old_file) == old_hash
        and sha256(new_file) == new_hash
    )
}

print("\n=== EXPLANATION VALIDATION ===")

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values())

print("\nSTATUS: EXPLAINABLE MISSION VALIDATED")
print("Simulation Only: YES")
print("New Files Saved: NO")
print("STEP 13 COMPLETE")

````

### Original cell 80/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import numpy as np
import math
import hashlib

print("=== F5 WHAT-IF SIMULATION: STEP 14 ===")

folder = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670"
)

route_file = folder / "F5_Battery_Safe_Assignments.csv"
summary_file = folder / "F5_Battery_Safe_Summary.csv"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

route_hash = sha256(route_file)
summary_hash = sha256(summary_file)

tasks = pd.read_csv(route_file)
saved_summary = pd.read_csv(summary_file)

assert len(tasks) == 15
assert tasks["location_id"].is_unique
assert set(tasks["drone"]) == {"Drone 1", "Drone 2"}

DEPOT = (250.0, 250.0)
RESERVE_PERCENT = 10.0

# Scenario, battery minutes, speed m/min,
# inspection minutes per task
scenarios = [
    ("BASELINE", 45, 60, 3.0),
    ("BATTERY_42_MIN", 42, 60, 3.0),
    ("BATTERY_40_MIN", 40, 60, 3.0),
    ("BATTERY_50_MIN", 50, 60, 3.0),
    ("SPEED_50_MPM", 45, 50, 3.0),
    ("SPEED_70_MPM", 45, 70, 3.0),
    ("INSPECTION_3_5_MIN", 45, 60, 3.5),
    ("INSPECTION_2_5_MIN", 45, 60, 2.5),
    ("COMBINED_STRESS", 42, 55, 3.5)
]

detail_rows = []
scenario_rows = []

for name, battery, speed, inspection in scenarios:

    reserve = battery * RESERVE_PERCENT / 100
    usable = battery - reserve

    weighted_sum = 0.0
    total_weight = 0.0
    drone_results = []

    for drone in ["Drone 1", "Drone 2"]:

        group = (
            tasks[tasks["drone"] == drone]
            .sort_values("sequence")
        )

        current = DEPOT
        elapsed = 0.0
        distance_total = 0.0

        for _, task in group.iterrows():

            target = (
                float(task["x_m"]),
                float(task["y_m"])
            )

            leg = math.dist(current, target)

            distance_total += leg
            elapsed += leg / speed

            rank = int(task["priority_rank"])
            weight = (16 - rank) ** 2

            weighted_sum += elapsed * weight
            total_weight += weight

            elapsed += inspection
            current = target

        # Return to simulated depot
        return_distance = math.dist(current, DEPOT)

        distance_total += return_distance
        elapsed += return_distance / speed

        remaining = battery - elapsed
        reserve_margin = remaining - reserve

        passed = reserve_margin >= -1e-9

        drone_results.append(passed)

        detail_rows.append({
            "Scenario": name,
            "Drone": drone,
            "Tasks": len(group),
            "Battery_min": battery,
            "Speed_m_per_min": speed,
            "Inspection_min": inspection,
            "Mission_Time_min": round(elapsed, 3),
            "Required_Reserve_min": round(reserve, 3),
            "Battery_Remaining_min": round(
                remaining, 3
            ),
            "Reserve_Margin_min": round(
                reserve_margin, 3
            ),
            "Status": (
                "PASS" if passed
                else "NEEDS_REPLANNING"
            )
        })

    weighted_arrival = weighted_sum / total_weight

    scenario_rows.append({
        "Scenario": name,
        "Battery_min": battery,
        "Speed_m_per_min": speed,
        "Inspection_min": inspection,
        "Weighted_Arrival_min": round(
            weighted_arrival, 3
        ),
        "Drones_Passing": sum(drone_results),
        "Decision": (
            "CURRENT_ROUTE_SAFE"
            if all(drone_results)
            else "REPLAN_REQUIRED"
        )
    })

whatif_detail = pd.DataFrame(detail_rows)
whatif_summary = pd.DataFrame(scenario_rows)

print("\n=== WHAT-IF SCENARIO SUMMARY ===")

display(whatif_summary)

print("\n=== DRONE-WISE BATTERY ANALYSIS ===")

display(
    whatif_detail[
        [
            "Scenario",
            "Drone",
            "Mission_Time_min",
            "Required_Reserve_min",
            "Battery_Remaining_min",
            "Reserve_Margin_min",
            "Status"
        ]
    ]
)

print("\n=== BASELINE CROSS-CHECK ===")

baseline = whatif_detail[
    whatif_detail["Scenario"] == "BASELINE"
]

baseline_ok = True

for _, row in baseline.iterrows():

    reference = saved_summary[
        saved_summary["Drone"] == row["Drone"]
    ]

    assert len(reference) == 1

    saved_time = float(
        reference.iloc[0]["Mission_Time_min"]
    )

    difference = abs(
        saved_time - row["Mission_Time_min"]
    )

    print(
        row["Drone"],
        "| Saved:", saved_time,
        "| Recalculated:", row["Mission_Time_min"],
        "| Difference:", round(difference, 4)
    )

    baseline_ok = baseline_ok and difference < 0.01


print("\n=== AUTOMATED VALIDATION ===")

checks = {
    "9 What-if Scenarios": len(whatif_summary) == 9,
    "18 Drone Evaluations": len(whatif_detail) == 18,
    "No Duplicate Scenario-Drone": (
        not whatif_detail.duplicated(
            ["Scenario", "Drone"]
        ).any()
    ),
    "15 Unique Source Areas": (
        len(tasks) == 15
        and tasks["location_id"].is_unique
    ),
    "Baseline Matches Saved F5": baseline_ok,
    "Baseline Reserve PASS": (
        baseline["Status"].eq("PASS").all()
    ),
    "Baseline Weighted Arrival Matches": (
        abs(
            float(
                whatif_summary.loc[
                    whatif_summary["Scenario"]
                    == "BASELINE",
                    "Weighted_Arrival_min"
                ].iloc[0]
            ) - 10.821
        ) < 0.02
    ),
    "All Scenario Results Calculated": (
        whatif_detail[
            [
                "Mission_Time_min",
                "Battery_Remaining_min",
                "Reserve_Margin_min"
            ]
        ].notna().all().all()
    ),
    "Original Files Unchanged": (
        sha256(route_file) == route_hash
        and sha256(summary_file) == summary_hash
    )
}

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values())

print("\n=== FINAL SUMMARY ===")

print(
    "Current Route Safe Scenarios:",
    int(
        whatif_summary["Decision"]
        .eq("CURRENT_ROUTE_SAFE").sum()
    )
)

print(
    "Scenarios Requiring Replanning:",
    int(
        whatif_summary["Decision"]
        .eq("REPLAN_REQUIRED").sum()
    )
)

print("\nSTATUS: WHAT-IF SENSITIVITY TEST COMPLETE")
print("Route Reoptimization Performed: NO")
print("Original Files: UNCHANGED")
print("New Files Saved: NO")
print("Real Drone Flight: NO")
print("STEP 14 COMPLETE")

````

### Original cell 81/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from functools import lru_cache
import pandas as pd
import numpy as np
import math
import hashlib

print("=== F5 WHAT-IF REPLANNING: STEP 15 ===")

source = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670/"
    "F5_Battery_Safe_Assignments.csv"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hash = sha256(source)

df = pd.read_csv(source)

assert len(df) == 15
assert df["location_id"].is_unique

DEPOT = (250.0, 250.0)

info = df.set_index("location_id").to_dict("index")

# Keep the two highest-risk areas first
rank1 = df.loc[
    df["priority_rank"] == 1, "location_id"
].iloc[0]

rank2 = df.loc[
    df["priority_rank"] == 2, "location_id"
].iloc[0]

assert len(df[df["priority_rank"].isin([1, 2])]) == 2

original_routes = {
    drone: group.sort_values("sequence")[
        "location_id"
    ].tolist()
    for drone, group in df.groupby("drone")
}

assert original_routes["Drone 1"][0] == rank2
assert original_routes["Drone 2"][0] == rank1

# Remaining 13 tasks can be divided between drones
others = [
    loc for loc in df["location_id"]
    if loc not in (rank1, rank2)
]

# Index 0: depot
# Index 1: Drone 1's first task
# Index 2: Drone 2's first task
# Index 3 onward: remaining tasks
node_ids = [None, rank2, rank1] + others

coords = [DEPOT] + [
    (
        float(info[loc]["x_m"]),
        float(info[loc]["y_m"])
    )
    for loc in node_ids[1:]
]

N = len(coords)

dist = [
    [
        math.dist(coords[i], coords[j])
        for j in range(N)
    ]
    for i in range(N)
]

FULL = (1 << len(others)) - 1

print("Areas:", len(df))
print("Fixed First Stop Drone 1:", rank2)
print("Fixed First Stop Drone 2:", rank1)
print("Possible Task Partitions:", FULL + 1)

print("\nPreparing exact shortest-distance search...")

# Shortest path through any chosen subset,
# starting at a fixed node and returning to depot.
@lru_cache(maxsize=None)
def shortest_remaining(last, mask):

    if mask == 0:
        return dist[last][0]

    best = float("inf")
    bits = mask

    while bits:
        bit = bits & -bits
        j = bit.bit_length() - 1
        next_node = 3 + j

        candidate = (
            dist[last][next_node]
            + shortest_remaining(
                next_node, mask ^ bit
            )
        )

        best = min(best, candidate)
        bits ^= bit

    return best


# Precompute shortest routes for all partitions
distance1 = []
distance2 = []

for mask in range(FULL + 1):

    distance1.append(
        dist[0][1] + shortest_remaining(1, mask)
    )

    other_mask = FULL ^ mask

    distance2.append(
        dist[0][2]
        + shortest_remaining(2, other_mask)
    )


def rebuild_route(first_node, mask):
    route = [node_ids[first_node]]
    current = first_node

    while mask:
        best_choice = None

        bits = mask

        while bits:
            bit = bits & -bits
            j = bit.bit_length() - 1
            next_node = 3 + j

            score = (
                dist[current][next_node]
                + shortest_remaining(
                    next_node, mask ^ bit
                )
            )

            if (
                best_choice is None
                or score < best_choice[0]
            ):
                best_choice = (
                    score, bit, next_node
                )

            bits ^= bit

        _, bit, next_node = best_choice

        route.append(node_ids[next_node])
        current = next_node
        mask ^= bit

    return route


def evaluate_route(route, speed, inspection):
    current = DEPOT
    elapsed = 0.0
    arrivals = {}
    weighted_sum = 0.0
    total_weight = 0

    for location_id in route:

        task = info[location_id]

        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        elapsed += (
            math.dist(current, target) / speed
        )

        arrivals[location_id] = elapsed

        rank = int(task["priority_rank"])
        weight = (16 - rank) ** 2

        weighted_sum += elapsed * weight
        total_weight += weight

        elapsed += inspection
        current = target

    elapsed += math.dist(current, DEPOT) / speed

    return (
        elapsed,
        weighted_sum,
        total_weight,
        arrivals
    )


scenarios = [
    ("BASELINE", 45, 60, 3.0),
    ("BATTERY_42_MIN", 42, 60, 3.0),
    ("BATTERY_40_MIN", 40, 60, 3.0),
    ("BATTERY_50_MIN", 50, 60, 3.0),
    ("SPEED_50_MPM", 45, 50, 3.0),
    ("SPEED_70_MPM", 45, 70, 3.0),
    ("INSPECTION_3_5_MIN", 45, 60, 3.5),
    ("INSPECTION_2_5_MIN", 45, 60, 2.5),
    ("COMBINED_STRESS", 42, 55, 3.5)
]

summary_rows = []
route_rows = []
pending_rows = []

for name, battery, speed, inspection in scenarios:

    reserve = battery * 0.10
    usable = battery - reserve

    original_evaluations = {
        drone: evaluate_route(
            route, speed, inspection
        )
        for drone, route in original_routes.items()
    }

    original_safe = all(
        result[0] <= usable + 1e-9
        for result in original_evaluations.values()
    )

    # Minimum total time among all two-sortie
    # partitions with the top two fixed first.
    min_combined_time = min(
        (distance1[m] + distance2[m]) / speed
        + 15 * inspection
        for m in range(FULL + 1)
    )

    selected_routes = None

    if original_safe:
        decision = "KEEP_CURRENT_ROUTE"
        selected_routes = original_routes

    else:
        best = None

        for mask in range(FULL + 1):

            count1 = 1 + mask.bit_count()
            count2 = 15 - count1

            time1 = (
                distance1[mask] / speed
                + count1 * inspection
            )

            time2 = (
                distance2[mask] / speed
                + count2 * inspection
            )

            if (
                time1 > usable + 1e-9
                or time2 > usable + 1e-9
            ):
                continue

            route1 = rebuild_route(1, mask)
            route2 = rebuild_route(2, FULL ^ mask)

            eval1 = evaluate_route(
                route1, speed, inspection
            )
            eval2 = evaluate_route(
                route2, speed, inspection
            )

            weighted = (
                eval1[1] + eval2[1]
            ) / (
                eval1[2] + eval2[2]
            )

            # Choose better priority-weighted arrival
            # among shortest-distance feasible routes.
            score = (
                weighted,
                max(time1, time2),
                time1 + time2
            )

            if best is None or score < best[0]:
                best = (
                    score,
                    {
                        "Drone 1": route1,
                        "Drone 2": route2
                    }
                )

        if best is None:
            decision = "EXTRA_SORTIE_OR_DEFER"
        else:
            decision = "REPLANNED_TWO_SORTIES"
            selected_routes = best[1]

    if selected_routes is not None:

        selected_ids = (
            selected_routes["Drone 1"]
            + selected_routes["Drone 2"]
        )

        assert len(selected_ids) == 15
        assert len(set(selected_ids)) == 15
        assert set(selected_ids) == set(df["location_id"])

        assert selected_routes["Drone 1"][0] == rank2
        assert selected_routes["Drone 2"][0] == rank1

        total_weighted = 0.0
        total_weights = 0.0

        for drone, route in selected_routes.items():

            mission_time, weighted_sum, weights, arrivals = (
                evaluate_route(
                    route, speed, inspection
                )
            )

            assert mission_time <= usable + 1e-9

            total_weighted += weighted_sum
            total_weights += weights

            for sequence, location_id in enumerate(
                route, 1
            ):
                route_rows.append({
                    "Scenario": name,
                    "Drone": drone,
                    "Sequence": sequence,
                    "Location": location_id,
                    "Priority_Rank": int(
                        info[location_id]["priority_rank"]
                    ),
                    "Arrival_min": round(
                        arrivals[location_id], 3
                    ),
                    "Mission_Time_min": round(
                        mission_time, 3
                    ),
                    "Battery_Remaining_min": round(
                        battery - mission_time, 3
                    ),
                    "Decision": decision
                })

        weighted_arrival = (
            total_weighted / total_weights
        )

    else:
        weighted_arrival = None

        # No task is silently discarded.
        # All remain pending for extra sortie planning.
        for location_id in df["location_id"]:
            pending_rows.append({
                "Scenario": name,
                "Location": location_id,
                "Priority_Rank": int(
                    info[location_id]["priority_rank"]
                ),
                "Status": "PENDING_EXTRA_SORTIE_PLAN"
            })

    summary_rows.append({
        "Scenario": name,
        "Battery_min": battery,
        "Usable_min_per_drone": round(usable, 3),
        "Minimum_Combined_Time_min": round(
            min_combined_time, 3
        ),
        "Two_Drone_Total_Budget_min": round(
            2 * usable, 3
        ),
        "Weighted_Arrival_min": (
            round(weighted_arrival, 3)
            if weighted_arrival is not None
            else None
        ),
        "Decision": decision,
        "Pending_Extra_Sortie": (
            15 if selected_routes is None else 0
        )
    })


whatif_replan_summary = pd.DataFrame(summary_rows)
whatif_replan_routes = pd.DataFrame(route_rows)
whatif_pending = pd.DataFrame(pending_rows)

print("\n=== WHAT-IF REPLANNING RESULTS ===")

display(whatif_replan_summary)

print("\n=== DECISION COUNTS ===")

print(
    whatif_replan_summary["Decision"].value_counts()
)

print("\n=== AUTOMATED VALIDATION ===")

checks = {
    "9 Scenarios Classified": (
        len(whatif_replan_summary) == 9
    ),
    "15 Unique Source Tasks": (
        len(df) == 15
        and df["location_id"].is_unique
    ),
    "Every Scenario Accounted For": all(
        (
            len(whatif_replan_routes[
                whatif_replan_routes["Scenario"] == name
            ]) == 15
        )
        if decision != "EXTRA_SORTIE_OR_DEFER"
        else (
            len(whatif_pending[
                whatif_pending["Scenario"] == name
            ]) == 15
        )
        for name, decision in zip(
            whatif_replan_summary["Scenario"],
            whatif_replan_summary["Decision"]
        )
    ),
    "No Duplicate Planned Tasks": (
        not whatif_replan_routes.duplicated(
            ["Scenario", "Location"]
        ).any()
    ),
    "No Duplicate Pending Tasks": (
        not whatif_pending.duplicated(
            ["Scenario", "Location"]
        ).any()
    ),
    "All Planned Routes Battery Safe": (
        all(
            whatif_replan_routes[
                whatif_replan_routes["Scenario"] == name
            ]["Mission_Time_min"].max()
            <= battery * 0.90 + 0.002
            for name, battery, _, _ in scenarios
            if name in set(whatif_replan_routes["Scenario"])
        )
    ),
    "Original File Unchanged": (
        sha256(source) == original_hash
    )
}

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values())

print("\n=== FINAL STATUS ===")
print("Simulation Only: YES")
print("Real Drone Flights: NO")
print("New Files Saved: NO")
print("STEP 15 COMPLETE")

````

### Original cell 82/118 — PYTHON SOURCE (archived, not run)

````python

from itertools import combinations
import pandas as pd
import math
import hashlib

print("=== F5 AUTOMATIC EXTRA-SORTIE PLANNING: STEP 16 ===")

# Reuse validated Step 15 calculations
needed = [
    "df", "info", "others", "FULL",
    "distance1", "distance2",
    "shortest_remaining", "rebuild_route",
    "original_routes", "evaluate_route",
    "scenarios", "source", "rank1", "rank2"
]

missing = [x for x in needed if x not in globals()]
assert not missing, f"Run Step 15 first: {missing}"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_hash = sha256(source)
all_ids = set(df["location_id"])

assert len(all_ids) == 15

# Reconstruct shortest extra route from depot
def extra_route(mask):
    # rebuild_route(0, mask) begins with None (depot).
    return rebuild_route(0, mask)[1:]


def plan_three_sorties(battery, speed, inspection):
    usable = battery * 0.90

    # Minimize number of tasks shifted to later sortie.
    # Prefer lower-risk tasks when choosing what to shift.
    for k in range(1, len(others) + 1):

        candidates = []

        for indexes in combinations(
            range(len(others)), k
        ):
            mask3 = sum(1 << i for i in indexes)

            distance3 = shortest_remaining(0, mask3)
            time3 = distance3 / speed + k * inspection

            if time3 > usable + 1e-9:
                continue

            delayed_weight = sum(
                (16 - int(info[others[i]]["priority_rank"])) ** 2
                for i in indexes
            )

            candidates.append(
                (delayed_weight, mask3, time3)
            )

        candidates.sort()

        for penalty, mask3, time3 in candidates:

            remaining = FULL ^ mask3
            mask1 = remaining

            best_split = None

            while True:
                mask2 = remaining ^ mask1

                count1 = 1 + mask1.bit_count()
                count2 = 1 + mask2.bit_count()

                time1 = (
                    distance1[mask1] / speed
                    + count1 * inspection
                )

                time2 = (
                    distance2[mask2] / speed
                    + count2 * inspection
                )

                if (
                    time1 <= usable + 1e-9
                    and time2 <= usable + 1e-9
                ):
                    score = (
                        max(time1, time2),
                        time1 + time2
                    )

                    if (
                        best_split is None
                        or score < best_split[0]
                    ):
                        best_split = (
                            score, mask1, mask2,
                            time1, time2
                        )

                if mask1 == 0:
                    break

                mask1 = (mask1 - 1) & remaining

            if best_split is not None:

                _, mask1, mask2, time1, time2 = best_split

                route1 = rebuild_route(1, mask1)
                route2 = rebuild_route(2, mask2)
                route3 = extra_route(mask3)

                # Choose which drone takes the extra sortie.
                # Recharge completion is assumed, not timed.
                extra_drone = (
                    "Drone 1" if time1 <= time2
                    else "Drone 2"
                )

                return [
                    ("Drone 1", 1, route1),
                    ("Drone 2", 1, route2),
                    (extra_drone, 2, route3)
                ]

    return None


def fallback_multiple_sorties(speed, inspection, usable):
    """
    Conservative fallback:
    Split each original drone route into
    battery-safe consecutive chunks.
    """
    plans = []

    for drone in ["Drone 1", "Drone 2"]:
        remaining = list(original_routes[drone])
        sortie_number = 1

        while remaining:

            best_count = 0

            for count in range(1, len(remaining) + 1):
                candidate = remaining[:count]

                time, _, _, _ = evaluate_route(
                    candidate, speed, inspection
                )

                if time <= usable + 1e-9:
                    best_count = count

            if best_count == 0:
                raise RuntimeError(
                    "Even a single task is infeasible: "
                    + remaining[0]
                )

            chunk = remaining[:best_count]

            plans.append(
                (drone, sortie_number, chunk)
            )

            remaining = remaining[best_count:]
            sortie_number += 1

    return plans


summary_rows = []
sortie_rows = []
task_rows = []

for name, battery, speed, inspection in scenarios:

    usable = battery * 0.90
    reserve = battery * 0.10

    current_safe = all(
        evaluate_route(route, speed, inspection)[0]
        <= usable + 1e-9
        for route in original_routes.values()
    )

    if current_safe:
        decision = "KEEP_CURRENT_ROUTE"

        plans = [
            ("Drone 1", 1, original_routes["Drone 1"]),
            ("Drone 2", 1, original_routes["Drone 2"])
        ]

    else:
        plans = plan_three_sorties(
            battery, speed, inspection
        )

        if plans is not None:
            decision = "EXTRA_SORTIE_PLANNED"
        else:
            plans = fallback_multiple_sorties(
                speed, inspection, usable
            )
            decision = "MULTIPLE_EXTRA_SORTIES_PLANNED"

    planned_ids = []
    times = []

    for drone, sortie_number, route in plans:

        mission_time, _, _, arrivals = evaluate_route(
            route, speed, inspection
        )

        remaining_battery = battery - mission_time

        assert mission_time <= usable + 1e-9

        times.append(mission_time)
        planned_ids.extend(route)

        sortie_id = (
            drone.replace(" ", "")
            + "_S" + str(sortie_number)
        )

        sortie_rows.append({
            "Scenario": name,
            "Drone": drone,
            "Sortie": sortie_number,
            "Sortie_ID": sortie_id,
            "Tasks": len(route),
            "Mission_Time_min": round(
                mission_time, 3
            ),
            "Battery_Remaining_min": round(
                remaining_battery, 3
            ),
            "Required_Reserve_min": round(
                reserve, 3
            ),
            "Reserve_PASS": True,
            "Recharge_Required_Before": (
                sortie_number > 1
            ),
            "Execution_Status": "NOT_EXECUTED"
        })

        for sequence, location_id in enumerate(
            route, 1
        ):
            task_rows.append({
                "Scenario": name,
                "Drone": drone,
                "Sortie": sortie_number,
                "Sequence": sequence,
                "Location": location_id,
                "Priority_Rank": int(
                    info[location_id]["priority_rank"]
                ),
                "Arrival_Within_Sortie_min": round(
                    arrivals[location_id], 3
                ),
                "Handoff_Status": (
                    "AWAITING_APPROVAL_AND_RECHARGE"
                    if sortie_number > 1
                    else "PLANNED"
                ),
                "Execution_Status": "NOT_EXECUTED"
            })

    # No missing or duplicated tasks
    assert len(planned_ids) == 15
    assert len(set(planned_ids)) == 15
    assert set(planned_ids) == all_ids

    # First two highest-risk areas stay first
    first_d1 = next(
        route[0]
        for drone, s, route in plans
        if drone == "Drone 1" and s == 1
    )

    first_d2 = next(
        route[0]
        for drone, s, route in plans
        if drone == "Drone 2" and s == 1
    )

    assert first_d1 == rank2
    assert first_d2 == rank1

    summary_rows.append({
        "Scenario": name,
        "Battery_min": battery,
        "Decision": decision,
        "Total_Sorties": len(plans),
        "Extra_Sorties": len(plans) - 2,
        "Tasks_Covered": len(planned_ids),
        "Minimum_Reserve_Margin_min": round(
            min(battery - t - reserve for t in times),
            3
        ),
        "Status": "PASS"
    })


whatif_multisortie_summary = pd.DataFrame(summary_rows)
whatif_multisortie_sorties = pd.DataFrame(sortie_rows)
whatif_multisortie_tasks = pd.DataFrame(task_rows)

print("\n=== AUTOMATIC MULTI-SORTIE RESULTS ===")

display(whatif_multisortie_summary)

print("\n=== SORTIE-WISE BATTERY DETAILS ===")

display(
    whatif_multisortie_sorties[
        [
            "Scenario",
            "Drone",
            "Sortie",
            "Tasks",
            "Mission_Time_min",
            "Battery_Remaining_min",
            "Recharge_Required_Before"
        ]
    ]
)

print("\n=== COMBINED STRESS RECOVERY EXAMPLE ===")

display(
    whatif_multisortie_tasks[
        whatif_multisortie_tasks["Scenario"]
        == "COMBINED_STRESS"
    ][
        [
            "Drone",
            "Sortie",
            "Sequence",
            "Location",
            "Priority_Rank",
            "Handoff_Status"
        ]
    ]
)

print("\n=== FINAL VALIDATION ===")

checks = {
    "9 Scenarios Planned": (
        len(whatif_multisortie_summary) == 9
    ),
    "135 Task Assignments": (
        len(whatif_multisortie_tasks) == 135
    ),
    "15 Tasks Per Scenario": (
        whatif_multisortie_tasks.groupby(
            "Scenario"
        ).size().eq(15).all()
    ),
    "No Duplicate Tasks": (
        not whatif_multisortie_tasks.duplicated(
            ["Scenario", "Location"]
        ).any()
    ),
    "All Sorties Battery Safe": (
        whatif_multisortie_sorties[
            "Reserve_PASS"
        ].all()
    ),
    "No Tasks Marked Executed": (
        whatif_multisortie_tasks[
            "Execution_Status"
        ].eq("NOT_EXECUTED").all()
    ),
    "All Scenarios PASS": (
        whatif_multisortie_summary[
            "Status"
        ].eq("PASS").all()
    ),
    "Original File Unchanged": (
        sha256(source) == source_hash
    )
}

for label, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", label
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values())

print("\nSTATUS: EXTRA-SORTIE PLANNING VALIDATED")
print("Recharge Time Modeled: NO")
print("Human Approval Completed: NO")
print("Real Drone Flight: NO")
print("New Files Saved: NO")
print("STEP 16 COMPLETE")

````

### Original cell 83/118 — PYTHON SOURCE (archived, not run)

````python

from itertools import combinations
import pandas as pd
import math
import hashlib

print("=== F5 AUTOMATIC EXTRA-SORTIE PLANNING: STEP 16 ===")

# Reuse validated Step 15 calculations
needed = [
    "df", "info", "others", "FULL",
    "distance1", "distance2",
    "shortest_remaining", "rebuild_route",
    "original_routes", "evaluate_route",
    "scenarios", "source", "rank1", "rank2"
]

missing = [x for x in needed if x not in globals()]
assert not missing, f"Run Step 15 first: {missing}"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_hash = sha256(source)
all_ids = set(df["location_id"])

assert len(all_ids) == 15

# Reconstruct shortest extra route from depot
def extra_route(mask):
    # rebuild_route(0, mask) begins with None (depot).
    return rebuild_route(0, mask)[1:]


def plan_three_sorties(battery, speed, inspection):
    usable = battery * 0.90

    # Minimize number of tasks shifted to later sortie.
    # Prefer lower-risk tasks when choosing what to shift.
    for k in range(1, len(others) + 1):

        candidates = []

        for indexes in combinations(
            range(len(others)), k
        ):
            mask3 = sum(1 << i for i in indexes)

            distance3 = shortest_remaining(0, mask3)
            time3 = distance3 / speed + k * inspection

            if time3 > usable + 1e-9:
                continue

            delayed_weight = sum(
                (16 - int(info[others[i]]["priority_rank"])) ** 2
                for i in indexes
            )

            candidates.append(
                (delayed_weight, mask3, time3)
            )

        candidates.sort()

        for penalty, mask3, time3 in candidates:

            remaining = FULL ^ mask3
            mask1 = remaining

            best_split = None

            while True:
                mask2 = remaining ^ mask1

                count1 = 1 + mask1.bit_count()
                count2 = 1 + mask2.bit_count()

                time1 = (
                    distance1[mask1] / speed
                    + count1 * inspection
                )

                time2 = (
                    distance2[mask2] / speed
                    + count2 * inspection
                )

                if (
                    time1 <= usable + 1e-9
                    and time2 <= usable + 1e-9
                ):
                    score = (
                        max(time1, time2),
                        time1 + time2
                    )

                    if (
                        best_split is None
                        or score < best_split[0]
                    ):
                        best_split = (
                            score, mask1, mask2,
                            time1, time2
                        )

                if mask1 == 0:
                    break

                mask1 = (mask1 - 1) & remaining

            if best_split is not None:

                _, mask1, mask2, time1, time2 = best_split

                route1 = rebuild_route(1, mask1)
                route2 = rebuild_route(2, mask2)
                route3 = extra_route(mask3)

                # Choose which drone takes the extra sortie.
                # Recharge completion is assumed, not timed.
                extra_drone = (
                    "Drone 1" if time1 <= time2
                    else "Drone 2"
                )

                return [
                    ("Drone 1", 1, route1),
                    ("Drone 2", 1, route2),
                    (extra_drone, 2, route3)
                ]

    return None


def fallback_multiple_sorties(speed, inspection, usable):
    """
    Conservative fallback:
    Split each original drone route into
    battery-safe consecutive chunks.
    """
    plans = []

    for drone in ["Drone 1", "Drone 2"]:
        remaining = list(original_routes[drone])
        sortie_number = 1

        while remaining:

            best_count = 0

            for count in range(1, len(remaining) + 1):
                candidate = remaining[:count]

                time, _, _, _ = evaluate_route(
                    candidate, speed, inspection
                )

                if time <= usable + 1e-9:
                    best_count = count

            if best_count == 0:
                raise RuntimeError(
                    "Even a single task is infeasible: "
                    + remaining[0]
                )

            chunk = remaining[:best_count]

            plans.append(
                (drone, sortie_number, chunk)
            )

            remaining = remaining[best_count:]
            sortie_number += 1

    return plans


summary_rows = []
sortie_rows = []
task_rows = []

for name, battery, speed, inspection in scenarios:

    usable = battery * 0.90
    reserve = battery * 0.10

    current_safe = all(
        evaluate_route(route, speed, inspection)[0]
        <= usable + 1e-9
        for route in original_routes.values()
    )

    if current_safe:
        decision = "KEEP_CURRENT_ROUTE"

        plans = [
            ("Drone 1", 1, original_routes["Drone 1"]),
            ("Drone 2", 1, original_routes["Drone 2"])
        ]

    else:
        plans = plan_three_sorties(
            battery, speed, inspection
        )

        if plans is not None:
            decision = "EXTRA_SORTIE_PLANNED"
        else:
            plans = fallback_multiple_sorties(
                speed, inspection, usable
            )
            decision = "MULTIPLE_EXTRA_SORTIES_PLANNED"

    planned_ids = []
    times = []

    for drone, sortie_number, route in plans:

        mission_time, _, _, arrivals = evaluate_route(
            route, speed, inspection
        )

        remaining_battery = battery - mission_time

        assert mission_time <= usable + 1e-9

        times.append(mission_time)
        planned_ids.extend(route)

        sortie_id = (
            drone.replace(" ", "")
            + "_S" + str(sortie_number)
        )

        sortie_rows.append({
            "Scenario": name,
            "Drone": drone,
            "Sortie": sortie_number,
            "Sortie_ID": sortie_id,
            "Tasks": len(route),
            "Mission_Time_min": round(
                mission_time, 3
            ),
            "Battery_Remaining_min": round(
                remaining_battery, 3
            ),
            "Required_Reserve_min": round(
                reserve, 3
            ),
            "Reserve_PASS": True,
            "Recharge_Required_Before": (
                sortie_number > 1
            ),
            "Execution_Status": "NOT_EXECUTED"
        })

        for sequence, location_id in enumerate(
            route, 1
        ):
            task_rows.append({
                "Scenario": name,
                "Drone": drone,
                "Sortie": sortie_number,
                "Sequence": sequence,
                "Location": location_id,
                "Priority_Rank": int(
                    info[location_id]["priority_rank"]
                ),
                "Arrival_Within_Sortie_min": round(
                    arrivals[location_id], 3
                ),
                "Handoff_Status": (
                    "AWAITING_APPROVAL_AND_RECHARGE"
                    if sortie_number > 1
                    else "PLANNED"
                ),
                "Execution_Status": "NOT_EXECUTED"
            })

    # No missing or duplicated tasks
    assert len(planned_ids) == 15
    assert len(set(planned_ids)) == 15
    assert set(planned_ids) == all_ids

    # First two highest-risk areas stay first
    first_d1 = next(
        route[0]
        for drone, s, route in plans
        if drone == "Drone 1" and s == 1
    )

    first_d2 = next(
        route[0]
        for drone, s, route in plans
        if drone == "Drone 2" and s == 1
    )

    assert first_d1 == rank2
    assert first_d2 == rank1

    summary_rows.append({
        "Scenario": name,
        "Battery_min": battery,
        "Decision": decision,
        "Total_Sorties": len(plans),
        "Extra_Sorties": len(plans) - 2,
        "Tasks_Covered": len(planned_ids),
        "Minimum_Reserve_Margin_min": round(
            min(battery - t - reserve for t in times),
            3
        ),
        "Status": "PASS"
    })


whatif_multisortie_summary = pd.DataFrame(summary_rows)
whatif_multisortie_sorties = pd.DataFrame(sortie_rows)
whatif_multisortie_tasks = pd.DataFrame(task_rows)

print("\n=== AUTOMATIC MULTI-SORTIE RESULTS ===")

display(whatif_multisortie_summary)

print("\n=== SORTIE-WISE BATTERY DETAILS ===")

display(
    whatif_multisortie_sorties[
        [
            "Scenario",
            "Drone",
            "Sortie",
            "Tasks",
            "Mission_Time_min",
            "Battery_Remaining_min",
            "Recharge_Required_Before"
        ]
    ]
)

print("\n=== COMBINED STRESS RECOVERY EXAMPLE ===")

display(
    whatif_multisortie_tasks[
        whatif_multisortie_tasks["Scenario"]
        == "COMBINED_STRESS"
    ][
        [
            "Drone",
            "Sortie",
            "Sequence",
            "Location",
            "Priority_Rank",
            "Handoff_Status"
        ]
    ]
)

print("\n=== FINAL VALIDATION ===")

checks = {
    "9 Scenarios Planned": (
        len(whatif_multisortie_summary) == 9
    ),
    "135 Task Assignments": (
        len(whatif_multisortie_tasks) == 135
    ),
    "15 Tasks Per Scenario": (
        whatif_multisortie_tasks.groupby(
            "Scenario"
        ).size().eq(15).all()
    ),
    "No Duplicate Tasks": (
        not whatif_multisortie_tasks.duplicated(
            ["Scenario", "Location"]
        ).any()
    ),
    "All Sorties Battery Safe": (
        whatif_multisortie_sorties[
            "Reserve_PASS"
        ].all()
    ),
    "No Tasks Marked Executed": (
        whatif_multisortie_tasks[
            "Execution_Status"
        ].eq("NOT_EXECUTED").all()
    ),
    "All Scenarios PASS": (
        whatif_multisortie_summary[
            "Status"
        ].eq("PASS").all()
    ),
    "Original File Unchanged": (
        sha256(source) == source_hash
    )
}

for label, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", label
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values())

print("\nSTATUS: EXTRA-SORTIE PLANNING VALIDATED")
print("Recharge Time Modeled: NO")
print("Human Approval Completed: NO")
print("Real Drone Flight: NO")
print("New Files Saved: NO")
print("STEP 16 COMPLETE")

````

### Original cell 84/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from itertools import combinations
from functools import lru_cache
import pandas as pd
import math
import hashlib

print("=== F5 AUTOMATIC EXTRA-SORTIE PLANNING: STEP 16 ===")

# 1. Load original validated mission
source = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner/"
    "F5_Advanced_Enhancements/"
    "Battery_Reserve_20261122_060550_816670/"
    "F5_Battery_Safe_Assignments.csv"
)

assert source.exists(), "Mission CSV not found"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_hash = sha256(source)

df = pd.read_csv(source)

assert len(df) == 15
assert df["location_id"].is_unique
assert set(df["drone"]) == {"Drone 1", "Drone 2"}

DEPOT = (250.0, 250.0)

info = df.set_index("location_id").to_dict("index")
all_ids = set(df["location_id"])

rank1 = df.loc[
    df["priority_rank"] == 1, "location_id"
].iloc[0]

rank2 = df.loc[
    df["priority_rank"] == 2, "location_id"
].iloc[0]

original_routes = {
    drone: group.sort_values("sequence")[
        "location_id"
    ].tolist()
    for drone, group in df.groupby("drone")
}

assert original_routes["Drone 1"][0] == rank2
assert original_routes["Drone 2"][0] == rank1

others = [
    loc for loc in df["location_id"]
    if loc not in (rank1, rank2)
]

FULL = (1 << len(others)) - 1

print("Areas:", len(df))
print("Drone 1 First Stop:", rank2)
print("Drone 2 First Stop:", rank1)
print("Remaining Tasks:", len(others))


# 2. Build distance matrix
node_ids = [None, rank2, rank1] + others

coords = [DEPOT] + [
    (
        float(info[loc]["x_m"]),
        float(info[loc]["y_m"])
    )
    for loc in node_ids[1:]
]

dist = [
    [
        math.dist(coords[i], coords[j])
        for j in range(len(coords))
    ]
    for i in range(len(coords))
]


# 3. Exact shortest-distance calculation
@lru_cache(maxsize=None)
def shortest_remaining(last, mask):

    if mask == 0:
        return dist[last][0]

    best = float("inf")
    bits = mask

    while bits:
        bit = bits & -bits
        index = bit.bit_length() - 1
        next_node = 3 + index

        candidate = (
            dist[last][next_node]
            + shortest_remaining(
                next_node, mask ^ bit
            )
        )

        best = min(best, candidate)
        bits ^= bit

    return best


def build_route(start_node, mask):
    route = []
    current = start_node

    while mask:
        best = None
        bits = mask

        while bits:
            bit = bits & -bits
            index = bit.bit_length() - 1
            next_node = 3 + index

            score = (
                dist[current][next_node]
                + shortest_remaining(
                    next_node, mask ^ bit
                )
            )

            if best is None or score < best[0]:
                best = (score, bit, next_node)

            bits ^= bit

        _, bit, next_node = best

        route.append(node_ids[next_node])
        current = next_node
        mask ^= bit

    return route


# Precompute correct distances for every subset
print("\nPreparing shortest-distance calculations...")

distance_d1 = [
    dist[0][1] + shortest_remaining(1, mask)
    for mask in range(FULL + 1)
]

distance_d2 = [
    dist[0][2] + shortest_remaining(2, mask)
    for mask in range(FULL + 1)
]

distance_extra = [
    shortest_remaining(0, mask)
    for mask in range(FULL + 1)
]


# 4. Calculate full sortie time
def evaluate_route(route, speed, inspection):

    current = DEPOT
    elapsed = 0.0
    arrivals = {}

    for location_id in route:
        task = info[location_id]

        target = (
            float(task["x_m"]),
            float(task["y_m"])
        )

        elapsed += math.dist(current, target) / speed

        arrivals[location_id] = elapsed

        elapsed += inspection
        current = target

    elapsed += math.dist(current, DEPOT) / speed

    return elapsed, arrivals


# 5. Find a battery-safe three-sortie plan
def plan_three_sorties(battery, speed, inspection):

    usable = battery * 0.90

    weights = [
        (16 - int(info[loc]["priority_rank"])) ** 2
        for loc in others
    ]

    # Try delaying the fewest tasks first
    for k in range(1, len(others) + 1):

        candidates = []

        for indexes in combinations(
            range(len(others)), k
        ):
            mask3 = sum(1 << i for i in indexes)

            time3 = (
                distance_extra[mask3] / speed
                + k * inspection
            )

            if time3 > usable + 1e-9:
                continue

            delay_penalty = sum(
                weights[i] for i in indexes
            )

            candidates.append(
                (delay_penalty, time3, mask3)
            )

        candidates.sort()

        for penalty, time3, mask3 in candidates:

            remaining = FULL ^ mask3
            mask1 = remaining

            best_split = None

            while True:
                mask2 = remaining ^ mask1

                count1 = 1 + mask1.bit_count()
                count2 = 1 + mask2.bit_count()

                # CORRECTED: each drone uses its own mask
                time1 = (
                    distance_d1[mask1] / speed
                    + count1 * inspection
                )

                time2 = (
                    distance_d2[mask2] / speed
                    + count2 * inspection
                )

                if (
                    time1 <= usable + 1e-9
                    and time2 <= usable + 1e-9
                ):
                    score = (
                        max(time1, time2),
                        time1 + time2
                    )

                    if (
                        best_split is None
                        or score < best_split[0]
                    ):
                        best_split = (
                            score, mask1, mask2,
                            time1, time2
                        )

                if mask1 == 0:
                    break

                mask1 = (mask1 - 1) & remaining

            if best_split is not None:

                _, mask1, mask2, time1, time2 = best_split

                route1 = [rank2] + build_route(1, mask1)
                route2 = [rank1] + build_route(2, mask2)
                route3 = build_route(0, mask3)

                extra_drone = (
                    "Drone 1"
                    if time1 <= time2
                    else "Drone 2"
                )

                return [
                    ("Drone 1", 1, route1),
                    ("Drone 2", 1, route2),
                    (extra_drone, 2, route3)
                ]

    return None


# 6. Fallback if three sorties are insufficient
def fallback_multiple_sorties(speed, inspection, usable):

    plans = []

    for drone in ["Drone 1", "Drone 2"]:

        remaining = list(original_routes[drone])
        sortie_number = 1

        while remaining:

            best_count = 0

            for count in range(1, len(remaining) + 1):

                candidate = remaining[:count]

                mission_time, _ = evaluate_route(
                    candidate, speed, inspection
                )

                if mission_time <= usable + 1e-9:
                    best_count = count

            if best_count == 0:
                raise RuntimeError(
                    "Single task exceeds battery budget: "
                    + remaining[0]
                )

            chunk = remaining[:best_count]

            plans.append(
                (drone, sortie_number, chunk)
            )

            remaining = remaining[best_count:]
            sortie_number += 1

    return plans


# 7. What-if scenarios
scenarios = [
    ("BASELINE", 45, 60, 3.0),
    ("BATTERY_42_MIN", 42, 60, 3.0),
    ("BATTERY_40_MIN", 40, 60, 3.0),
    ("BATTERY_50_MIN", 50, 60, 3.0),
    ("SPEED_50_MPM", 45, 50, 3.0),
    ("SPEED_70_MPM", 45, 70, 3.0),
    ("INSPECTION_3_5_MIN", 45, 60, 3.5),
    ("INSPECTION_2_5_MIN", 45, 60, 2.5),
    ("COMBINED_STRESS", 42, 55, 3.5)
]

summary_rows = []
sortie_rows = []
task_rows = []

print("\n=== RUNNING SCENARIOS ===")

for name, battery, speed, inspection in scenarios:

    usable = battery * 0.90
    reserve = battery * 0.10

    current_safe = all(
        evaluate_route(route, speed, inspection)[0]
        <= usable + 1e-9
        for route in original_routes.values()
    )

    if current_safe:
        decision = "KEEP_CURRENT_ROUTE"

        plans = [
            ("Drone 1", 1, original_routes["Drone 1"]),
            ("Drone 2", 1, original_routes["Drone 2"])
        ]

    else:
        plans = plan_three_sorties(
            battery, speed, inspection
        )

        if plans is not None:
            decision = "EXTRA_SORTIE_PLANNED"
        else:
            plans = fallback_multiple_sorties(
                speed, inspection, usable
            )
            decision = "MULTIPLE_EXTRA_SORTIES_PLANNED"

    planned_ids = []
    mission_times = []

    for drone, sortie_number, route in plans:

        mission_time, arrivals = evaluate_route(
            route, speed, inspection
        )

        # Independently verify the battery constraint
        if mission_time > usable + 1e-9:
            raise RuntimeError(
                f"Battery violation: {name}, "
                f"{drone}, Sortie {sortie_number}, "
                f"Time={mission_time:.3f}, "
                f"Limit={usable:.3f}"
            )

        mission_times.append(mission_time)
        planned_ids.extend(route)

        sortie_rows.append({
            "Scenario": name,
            "Drone": drone,
            "Sortie": sortie_number,
            "Tasks": len(route),
            "Mission_Time_min": round(
                mission_time, 3
            ),
            "Battery_Remaining_min": round(
                battery - mission_time, 3
            ),
            "Required_Reserve_min": round(
                reserve, 3
            ),
            "Reserve_Margin_min": round(
                usable - mission_time, 3
            ),
            "Recharge_Required_Before": (
                sortie_number > 1
            ),
            "Execution_Status": "NOT_EXECUTED"
        })

        for sequence, location_id in enumerate(
            route, 1
        ):
            task_rows.append({
                "Scenario": name,
                "Drone": drone,
                "Sortie": sortie_number,
                "Sequence": sequence,
                "Location": location_id,
                "Priority_Rank": int(
                    info[location_id]["priority_rank"]
                ),
                "Arrival_Within_Sortie_min": round(
                    arrivals[location_id], 3
                ),
                "Handoff_Status": (
                    "AWAITING_APPROVAL_AND_RECHARGE"
                    if sortie_number > 1
                    else "PLANNED"
                ),
                "Execution_Status": "NOT_EXECUTED"
            })

    assert len(planned_ids) == 15
    assert len(set(planned_ids)) == 15
    assert set(planned_ids) == all_ids

    first_d1 = next(
        route[0]
        for drone, sortie, route in plans
        if drone == "Drone 1" and sortie == 1
    )

    first_d2 = next(
        route[0]
        for drone, sortie, route in plans
        if drone == "Drone 2" and sortie == 1
    )

    assert first_d1 == rank2
    assert first_d2 == rank1

    summary_rows.append({
        "Scenario": name,
        "Battery_min": battery,
        "Decision": decision,
        "Total_Sorties": len(plans),
        "Extra_Sorties": len(plans) - 2,
        "Tasks_Covered": len(planned_ids),
        "Minimum_Reserve_Margin_min": round(
            min(usable - t for t in mission_times),
            3
        ),
        "Status": "PASS"
    })

    print(
        name,
        "| Sorties:", len(plans),
        "|", decision
    )


# 8. Results
whatif_multisortie_summary = pd.DataFrame(summary_rows)
whatif_multisortie_sorties = pd.DataFrame(sortie_rows)
whatif_multisortie_tasks = pd.DataFrame(task_rows)

print("\n=== AUTOMATIC MULTI-SORTIE RESULTS ===")
display(whatif_multisortie_summary)

print("\n=== SORTIE-WISE BATTERY DETAILS ===")
display(
    whatif_multisortie_sorties[
        [
            "Scenario",
            "Drone",
            "Sortie",
            "Tasks",
            "Mission_Time_min",
            "Battery_Remaining_min",
            "Reserve_Margin_min",
            "Recharge_Required_Before"
        ]
    ]
)

print("\n=== COMBINED STRESS RECOVERY ===")
display(
    whatif_multisortie_tasks[
        whatif_multisortie_tasks["Scenario"]
        == "COMBINED_STRESS"
    ][
        [
            "Drone",
            "Sortie",
            "Sequence",
            "Location",
            "Priority_Rank",
            "Handoff_Status"
        ]
    ]
)


# 9. Final validation
checks = {
    "9 Scenarios Planned": (
        len(whatif_multisortie_summary) == 9
    ),
    "135 Task Assignments": (
        len(whatif_multisortie_tasks) == 135
    ),
    "15 Tasks Per Scenario": (
        whatif_multisortie_tasks.groupby(
            "Scenario"
        ).size().eq(15).all()
    ),
    "No Duplicate Tasks": (
        not whatif_multisortie_tasks.duplicated(
            ["Scenario", "Location"]
        ).any()
    ),
    "All Sorties Battery Safe": all(
        row["Mission_Time_min"]
        <= row["Battery_min"] * 0.90 + 0.002
        for _, row in whatif_multisortie_sorties.merge(
            whatif_multisortie_summary[
                ["Scenario", "Battery_min"]
            ],
            on="Scenario"
        ).iterrows()
    ),
    "Extra Sorties Require Recharge": (
        whatif_multisortie_sorties.apply(
            lambda row: (
                row["Recharge_Required_Before"]
                == (row["Sortie"] > 1)
            ),
            axis=1
        ).all()
    ),
    "No Tasks Marked Executed": (
        whatif_multisortie_tasks[
            "Execution_Status"
        ].eq("NOT_EXECUTED").all()
    ),
    "All Scenarios PASS": (
        whatif_multisortie_summary[
            "Status"
        ].eq("PASS").all()
    ),
    "Original File Unchanged": (
        sha256(source) == source_hash
    )
}

print("\n=== FINAL VALIDATION ===")

for label, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", label
    )

print(
    "\nTests Passed:",
    sum(bool(v) for v in checks.values()),
    "/", len(checks)
)

assert all(checks.values()), "Step 16 validation failed"

print("\nSTATUS: EXTRA-SORTIE PLANNING VALIDATED")
print("Original Files: UNCHANGED")
print("New Files Saved: NO")
print("Recharge Time Modeled: NO")
print("Human Approval Completed: NO")
print("Real Drone Flight: NO")
print("STEP 16 COMPLETE")

````

### Original cell 85/118 — MARKDOWN SOURCE (archived, not run)

````markdown
### Archived F5 console output (not executable)

```text
 === F5 AUTOMATIC EXTRA-SORTIE PLANNING: STEP 16 ===
Areas: 15
Drone 1 First Stop: SIM-AREA-007
Drone 2 First Stop: SIM-AREA-011
Remaining Tasks: 13

Preparing shortest-distance calculations...

=== RUNNING SCENARIOS ===
BASELINE | Sorties: 2 | KEEP_CURRENT_ROUTE
BATTERY_42_MIN | Sorties: 3 | EXTRA_SORTIE_PLANNED
BATTERY_40_MIN | Sorties: 3 | EXTRA_SORTIE_PLANNED
BATTERY_50_MIN | Sorties: 2 | KEEP_CURRENT_ROUTE
SPEED_50_MPM | Sorties: 3 | EXTRA_SORTIE_PLANNED
SPEED_70_MPM | Sorties: 2 | KEEP_CURRENT_ROUTE
INSPECTION_3_5_MIN | Sorties: 3 | EXTRA_SORTIE_PLANNED
INSPECTION_2_5_MIN | Sorties: 2 | KEEP_CURRENT_ROUTE
COMBINED_STRESS | Sorties: 3 | EXTRA_SORTIE_PLANNED

=== AUTOMATIC MULTI-SORTIE RESULTS ===




    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }





      Scenario
      Battery_min
      Decision
      Total_Sorties
      Extra_Sorties
      Tasks_Covered
      Minimum_Reserve_Margin_min
      Status




      0
      BASELINE
      45
      KEEP_CURRENT_ROUTE
      2
      0
      15
      1.023
      PASS


      1
      BATTERY_42_MIN
      42
      EXTRA_SORTIE_PLANNED
      3
      1
      15
      0.560
      PASS


      2
      BATTERY_40_MIN
      40
      EXTRA_SORTIE_PLANNED
      3
      1
      15
      0.748
      PASS


      3
      BATTERY_50_MIN
      50
      KEEP_CURRENT_ROUTE
      2
      0
      15
      5.523
      PASS


      4
      SPEED_50_MPM
      45
      EXTRA_SORTIE_PLANNED
      3
      1
      15
      0.385
      PASS


      5
      SPEED_70_MPM
      45
      KEEP_CURRENT_ROUTE
      2
      0
      15
      3.234
      PASS


      6
      INSPECTION_3_5_MIN
      45
      EXTRA_SORTIE_PLANNED
      3
      1
      15
      2.181
      PASS


      7
      INSPECTION_2_5_MIN
      45
      KEEP_CURRENT_ROUTE
      2
      0
      15
      4.587
      PASS


      8
      COMBINED_STRESS
      42
      EXTRA_SORTIE_PLANNED
      3
      1
      15
      0.187
      PASS















    .colab-df-container {
      display:flex;
      gap: 12px;
    }

    .colab-df-convert {
      background-color: #E8F0FE;
      border: none;
      border-radius: 50%;
      cursor: pointer;
      display: none;
      fill: #1967D2;
      height: 32px;
      padding: 0 0 0 0;
      width: 32px;
    }

    .colab-df-convert:hover {
      background-color: #E2EBFA;
      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
      fill: #174EA6;
    }

    .colab-df-buttons div {
      margin-bottom: 4px;
    }

    [theme=dark] .colab-df-convert {
      background-color: #3B4455;
      fill: #D2E3FC;
    }

    [theme=dark] .colab-df-convert:hover {
      background-color: #434B5C;
      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
      fill: #FFFFFF;
    }



      const buttonEl =
        document.querySelector('#df-bc57e552-3158-422a-87fb-45790822d726 button.colab-df-convert');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      async function convertToInteractive(key) {
        const element = document.querySelector('#df-bc57e552-3158-422a-87fb-45790822d726');
        const dataTable =
          await google.colab.kernel.invokeFunction('convertToInteractive',
                                                    [key], {});
        if (!dataTable) return;

        const docLinkHtml = 'Like what you see? Visit the ' +
          '<a target="_blank" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'
          + ' to learn more about interactive tables.';
        element.innerHTML = '';
        dataTable['output_type'] = 'display_data';
        await google.colab.output.renderOutput(dataTable, element);
        const docLink = document.createElement('div');
        docLink.innerHTML = docLinkHtml;
        element.appendChild(docLink);
      }






      .colab-df-generate {
        background-color: #E8F0FE;
        border: none;
        border-radius: 50%;
        cursor: pointer;
        display: none;
        fill: #1967D2;
        height: 32px;
        padding: 0 0 0 0;
        width: 32px;
      }

      .colab-df-generate:hover {
        background-color: #E2EBFA;
        box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
        fill: #174EA6;
      }

      [theme=dark] .colab-df-generate {
        background-color: #3B4455;
        fill: #D2E3FC;
      }

      [theme=dark] .colab-df-generate:hover {
        background-color: #434B5C;
        box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
        filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
        fill: #FFFFFF;
      }








      (() => {
      const buttonEl =
        document.querySelector('#id_798c5b2f-4cf2-477f-ba5c-f88873ae9f3e button.colab-df-generate');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      buttonEl.onclick = () => {
        google.colab.notebook.generateWithVariable('whatif_multisortie_summary');
      }
      })();






=== SORTIE-WISE BATTERY DETAILS ===




    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }





      Scenario
      Drone
      Sortie
      Tasks
      Mission_Time_min
      Battery_Remaining_min
      Reserve_Margin_min
      Recharge_Required_Before




      0
      BASELINE
      Drone 1
      1
      7
      39.413
      5.587
      1.087
      False


      1
      BASELINE
      Drone 2
      1
      8
      39.477
      5.523
      1.023
      False


      2
      BATTERY_42_MIN
      Drone 1
      1
      8
      37.240
      4.760
      0.560
      False


      3
      BATTERY_42_MIN
      Drone 2
      1
      6
      36.429
      5.571
      1.371
      False


      4
      BATTERY_42_MIN
      Drone 2
      2
      1
      9.257
      32.743
      28.543
      True


      5
      BATTERY_40_MIN
      Drone 1
      1
      6
      35.252
      4.748
      0.748
      False


      6
      BATTERY_40_MIN
      Drone 2
      1
      7
      34.978
      5.022
      1.022
      False


      7
      BATTERY_40_MIN
      Drone 2
      2
      2
      14.027
      25.973
      21.973
      True


      8
      BATTERY_50_MIN
      Drone 1
      1
      7
      39.413
      10.587
      5.587
      False


      9
      BATTERY_50_MIN
      Drone 2
      1
      8
      39.477
      10.523
      5.523
      False


      10
      SPEED_50_MPM
      Drone 1
      1
      8
      39.888
      5.112
      0.612
      False


      11
      SPEED_50_MPM
      Drone 2
      1
      6
      40.115
      4.885
      0.385
      False


      12
      SPEED_50_MPM
      Drone 1
      2
      1
      10.509
      34.491
      29.991
      True


      13
      SPEED_70_MPM
      Drone 1
      1
      7
      36.783
      8.217
      3.717
      False


      14
      SPEED_70_MPM
      Drone 2
      1
      8
      37.266
      7.734
      3.234
      False


      15
      INSPECTION_3_5_MIN
      Drone 1
      1
      6
      38.319
      6.681
      2.181
      False


      16
      INSPECTION_3_5_MIN
      Drone 2
      1
      7
      37.831
      7.169
      2.669
      False


      17
      INSPECTION_3_5_MIN
      Drone 2
      2
      2
      15.027
      29.973
      25.473
      True


      18
      INSPECTION_2_5_MIN
      Drone 1
      1
      7
      35.913
      9.087
      4.587
      False


      19
      INSPECTION_2_5_MIN
      Drone 2
      1
      8
      35.477
      9.523
      5.023
      False


      20
      COMBINED_STRESS
      Drone 1
      1
      7
      37.613
      4.387
      0.187
      False


      21
      COMBINED_STRESS
      Drone 2
      1
      5
      37.594
      4.406
      0.206
      False


      22
      COMBINED_STRESS
      Drone 2
      2
      3
      29.118
      12.882
      8.682
      True















    .colab-df-container {
      display:flex;
      gap: 12px;
    }

    .colab-df-convert {
      background-color: #E8F0FE;
      border: none;
      border-radius: 50%;
      cursor: pointer;
      display: none;
      fill: #1967D2;
      height: 32px;
      padding: 0 0 0 0;
      width: 32px;
    }

    .colab-df-convert:hover {
      background-color: #E2EBFA;
      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
      fill: #174EA6;
    }

    .colab-df-buttons div {
      margin-bottom: 4px;
    }

    [theme=dark] .colab-df-convert {
      background-color: #3B4455;
      fill: #D2E3FC;
    }

    [theme=dark] .colab-df-convert:hover {
      background-color: #434B5C;
      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
      fill: #FFFFFF;
    }



      const buttonEl =
        document.querySelector('#df-e6c86690-082e-4739-90e3-b05fc0ab1c3e button.colab-df-convert');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      async function convertToInteractive(key) {
        const element = document.querySelector('#df-e6c86690-082e-4739-90e3-b05fc0ab1c3e');
        const dataTable =
          await google.colab.kernel.invokeFunction('convertToInteractive',
                                                    [key], {});
        if (!dataTable) return;

        const docLinkHtml = 'Like what you see? Visit the ' +
          '<a target="_blank" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'
          + ' to learn more about interactive tables.';
        element.innerHTML = '';
        dataTable['output_type'] = 'display_data';
        await google.colab.output.renderOutput(dataTable, element);
        const docLink = document.createElement('div');
        docLink.innerHTML = docLinkHtml;
        element.appendChild(docLink);
      }







=== COMBINED STRESS RECOVERY ===




    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }





      Drone
      Sortie
      Sequence
      Location
      Priority_Rank
      Handoff_Status




      120
      Drone 1
      1
      1
      SIM-AREA-007
      2
      PLANNED


      121
      Drone 1
      1
      2
      SIM-AREA-012
      4
      PLANNED


      122
      Drone 1
      1
      3
      SIM-AREA-006
      3
      PLANNED


      123
      Drone 1
      1
      4
      SIM-AREA-004
      12
      PLANNED


      124
      Drone 1
      1
      5
      SIM-AREA-010
      15
      PLANNED


      125
      Drone 1
      1
      6
      SIM-AREA-015
      9
      PLANNED


      126
      Drone 1
      1
      7
      SIM-AREA-002
      7
      PLANNED


      127
      Drone 2
      1
      1
      SIM-AREA-011
      1
      PLANNED


      128
      Drone 2
      1
      2
      SIM-AREA-013
      6
      PLANNED


      129
      Drone 2
      1
      3
      SIM-AREA-003
      10
      PLANNED


      130
      Drone 2
      1
      4
      SIM-AREA-014
      5
      PLANNED


      131
      Drone 2
      1
      5
      SIM-AREA-009
      8
      PLANNED


      132
      Drone 2
      2
      1
      SIM-AREA-005
      14
      AWAITING_APPROVAL_AND_RECHARGE


      133
      Drone 2
      2
      2
      SIM-AREA-008
      13
      AWAITING_APPROVAL_AND_RECHARGE


      134
      Drone 2
      2
      3
      SIM-AREA-001
      11
      AWAITING_APPROVAL_AND_RECHARGE















    .colab-df-container {
      display:flex;
      gap: 12px;
    }

    .colab-df-convert {
      background-color: #E8F0FE;
      border: none;
      border-radius: 50%;
      cursor: pointer;
      display: none;
      fill: #1967D2;
      height: 32px;
      padding: 0 0 0 0;
      width: 32px;
    }

    .colab-df-convert:hover {
      background-color: #E2EBFA;
      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
      fill: #174EA6;
    }

    .colab-df-buttons div {
      margin-bottom: 4px;
    }

    [theme=dark] .colab-df-convert {
      background-color: #3B4455;
      fill: #D2E3FC;
    }

    [theme=dark] .colab-df-convert:hover {
      background-color: #434B5C;
      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
      fill: #FFFFFF;
    }



      const buttonEl =
        document.querySelector('#df-a3ebb8c8-53c2-4d31-a6e5-cd2cab79e31f button.colab-df-convert');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      async function convertToInteractive(key) {
        const element = document.querySelector('#df-a3ebb8c8-53c2-4d31-a6e5-cd2cab79e31f');
        const dataTable =
          await google.colab.kernel.invokeFunction('convertToInteractive',
                                                    [key], {});
        if (!dataTable) return;

        const docLinkHtml = 'Like what you see? Visit the ' +
          '<a target="_blank" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'
          + ' to learn more about interactive tables.';
        element.innerHTML = '';
        dataTable['output_type'] = 'display_data';
        await google.colab.output.renderOutput(dataTable, element);
        const docLink = document.createElement('div');
        docLink.innerHTML = docLinkHtml;
        element.appendChild(docLink);
      }







=== FINAL VALIDATION ===
PASS - 9 Scenarios Planned
PASS - 135 Task Assignments
PASS - 15 Tasks Per Scenario
PASS - No Duplicate Tasks
PASS - All Sorties Battery Safe
PASS - Extra Sorties Require Recharge
PASS - No Tasks Marked Executed
PASS - All Scenarios PASS
PASS - Original File Unchanged

Tests Passed: 9 / 9

STATUS: EXTRA-SORTIE PLANNING VALIDATED
Original Files: UNCHANGED
New Files Saved: NO
Recharge Time Modeled: NO
Human Approval Completed: NO
Real Drone Flight: NO
STEP 16 COMPLETE
```
````

### Original cell 86/118 — PYTHON SOURCE (archived, not run)

````python

````

### Original cell 87/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import json
import hashlib

print("=== F5 ADVANCED STEP 17: FINAL EXPORT ===")

# ==========================================
# 1. CHECK PREVIOUS STEP RESULTS
# ==========================================

required = [
    "explain_df",
    "explain_summary",
    "whatif_summary",
    "whatif_detail",
    "whatif_multisortie_summary",
    "whatif_multisortie_sorties",
    "whatif_multisortie_tasks"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Previous results missing: "
        + ", ".join(missing)
        + ". Run the relevant previous steps first."
    )

print("Previous Step Results: FOUND")


# ==========================================
# 2. SOURCE FILES
# ==========================================

base = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/Mission_Planner"
)

original_file = (
    base
    / "F5_Optimized_20261122_055356_483143"
    / "F5_Optimized_Mission_Assignments.csv"
)

battery_folder = (
    base
    / "F5_Advanced_Enhancements"
    / "Battery_Reserve_20261122_060550_816670"
)

battery_file = (
    battery_folder
    / "F5_Battery_Safe_Assignments.csv"
)

battery_summary_file = (
    battery_folder
    / "F5_Battery_Safe_Summary.csv"
)

protected_files = [
    original_file,
    battery_file,
    battery_summary_file
]

def sha256(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

for path in protected_files:
    assert path.exists(), f"File missing: {path}"

original_hashes = {
    str(path): sha256(path)
    for path in protected_files
}

source_df = pd.read_csv(battery_file)

print("Original Source Files: FOUND")


# ==========================================
# 3. PREPARE VALIDATED DATA
# ==========================================

explanations = explain_df.copy()
drone_explanations = explain_summary.copy()

sensitivity = whatif_summary.copy()
sensitivity_details = whatif_detail.copy()

multi_summary = whatif_multisortie_summary.copy()
multi_sorties = whatif_multisortie_sorties.copy()
multi_tasks = whatif_multisortie_tasks.copy()

expected_scenarios = {
    "BASELINE",
    "BATTERY_42_MIN",
    "BATTERY_40_MIN",
    "BATTERY_50_MIN",
    "SPEED_50_MPM",
    "SPEED_70_MPM",
    "INSPECTION_3_5_MIN",
    "INSPECTION_2_5_MIN",
    "COMBINED_STRESS"
}


# ==========================================
# 4. AUTOMATED VALIDATION
# ==========================================

checks = {}

checks["15 Source Areas"] = (
    len(source_df) == 15
    and source_df["location_id"].is_unique
)

checks["15 Area Explanations"] = (
    len(explanations) == 15
    and explanations["location_id"].is_unique
    and set(explanations["location_id"])
    == set(source_df["location_id"])
)

checks["2 Drone Summaries"] = (
    len(drone_explanations) == 2
)

checks["9 Sensitivity Scenarios"] = (
    len(sensitivity) == 9
    and set(sensitivity["Scenario"])
    == expected_scenarios
)

checks["18 Drone Sensitivity Records"] = (
    len(sensitivity_details) == 18
)

checks["9 Multi-Sortie Scenarios"] = (
    len(multi_summary) == 9
    and set(multi_summary["Scenario"])
    == expected_scenarios
)

checks["135 Task Assignments"] = (
    len(multi_tasks) == 135
)

checks["15 Tasks Per Scenario"] = (
    multi_tasks.groupby("Scenario")
    .size()
    .eq(15)
    .all()
)

checks["No Duplicate Tasks"] = (
    not multi_tasks.duplicated(
        ["Scenario", "Location"]
    ).any()
)

checks["All Areas Covered"] = all(
    set(group["Location"])
    == set(source_df["location_id"])
    for _, group in multi_tasks.groupby("Scenario")
)

checks["No Duplicate Sorties"] = (
    not multi_sorties.duplicated(
        ["Scenario", "Drone", "Sortie"]
    ).any()
)

checks["Correct Sortie Counts"] = all(
    len(
        multi_sorties[
            multi_sorties["Scenario"]
            == row["Scenario"]
        ]
    ) == int(row["Total_Sorties"])
    for _, row in multi_summary.iterrows()
)

# Independently check battery reserve
battery_check = multi_sorties.merge(
    multi_summary[
        ["Scenario", "Battery_min"]
    ],
    on="Scenario",
    validate="many_to_one"
)

checks["10 Percent Battery Reserve"] = (
    (
        battery_check["Mission_Time_min"]
        <= battery_check["Battery_min"] * 0.90 + 0.002
    ).all()
)

checks["Recharge Required For Later Sorties"] = (
    (
        multi_sorties["Recharge_Required_Before"]
        == (multi_sorties["Sortie"] > 1)
    ).all()
)

checks["No Real Task Execution"] = (
    multi_tasks["Execution_Status"]
    .eq("NOT_EXECUTED")
    .all()
)

checks["4 Safe Current Routes"] = (
    multi_summary["Decision"]
    .eq("KEEP_CURRENT_ROUTE")
    .sum() == 4
)

checks["5 Extra Sortie Scenarios"] = (
    multi_summary["Decision"]
    .eq("EXTRA_SORTIE_PLANNED")
    .sum() == 5
)

checks["All Scenarios PASS"] = (
    multi_summary["Status"]
    .eq("PASS")
    .all()
)

checks["Original Files Unchanged"] = all(
    sha256(path) == original_hashes[str(path)]
    for path in protected_files
)

print("\n=== PRE-EXPORT VALIDATION ===")

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

passed_count = sum(
    bool(value) for value in checks.values()
)

print(
    "\nTests Passed:",
    passed_count,
    "/",
    len(checks)
)

assert all(checks.values()), (
    "Validation failed. No export performed."
)


# ==========================================
# 5. CREATE NEW OUTPUT FOLDER
# ==========================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_folder = (
    base
    / "F5_Advanced_Enhancements"
    / ("Explainable_WhatIf_" + timestamp)
)

output_folder.mkdir(
    parents=True,
    exist_ok=False
)

print("\nNew Output Folder Created")


# ==========================================
# 6. SAVE 7 CSV FILES
# ==========================================

exports = {
    "F5_Explainable_Mission_Decisions.csv":
        explanations,

    "F5_Explainable_Drone_Summary.csv":
        drone_explanations,

    "F5_WhatIf_Sensitivity_Summary.csv":
        sensitivity,

    "F5_WhatIf_Sensitivity_Details.csv":
        sensitivity_details,

    "F5_WhatIf_MultiSortie_Summary.csv":
        multi_summary,

    "F5_WhatIf_MultiSortie_Details.csv":
        multi_sorties,

    "F5_WhatIf_Task_Assignments.csv":
        multi_tasks
}

file_manifest = {}

print("\n=== SAVING CSV FILES ===")

for filename, dataframe in exports.items():

    filepath = output_folder / filename

    dataframe.to_csv(
        filepath,
        index=False
    )

    assert filepath.exists()
    assert filepath.stat().st_size > 0

    saved_df = pd.read_csv(filepath)

    assert len(saved_df) == len(dataframe)

    file_manifest[filename] = {
        "rows": int(len(saved_df)),
        "sha256": sha256(filepath)
    }

    print(
        "SAVED:",
        filename,
        "| Rows:",
        len(saved_df)
    )


# ==========================================
# 7. CREATE FINAL JSON REPORT
# ==========================================

combined = multi_summary[
    multi_summary["Scenario"]
    == "COMBINED_STRESS"
].iloc[0]

report = {
    "project": "DENGU HUNTER",

    "feature": "F5 Advanced Mission Planner",

    "module": (
        "Explainable Mission Planning "
        "and What-if Simulation"
    ),

    "status": "SIMULATION_VALIDATED",

    "export_timestamp": timestamp,

    "total_areas": 15,

    "simulated_drones": 2,

    "whatif_scenarios": 9,

    "safe_current_route_scenarios": 4,

    "extra_sortie_scenarios": 5,

    "total_task_assignments": int(
        len(multi_tasks)
    ),

    "total_sorties": int(
        len(multi_sorties)
    ),

    "combined_stress": {
        "sorties": int(
            combined["Total_Sorties"]
        ),

        "tasks_covered": int(
            combined["Tasks_Covered"]
        ),

        "minimum_reserve_margin_min": float(
            combined["Minimum_Reserve_Margin_min"]
        )
    },

    "safety": {
        "battery_reserve_percent": 10,

        "coordinates": "SYNTHETIC",

        "risk_model": "F4_RULE_BASED_SIMULATION",

        "return_to_depot_included": True,

        "recharge_duration_modeled": False,

        "weather_modeled": False,

        "real_battery_uncertainty_modeled": False,

        "human_approval_completed": False,

        "real_drone_flight": False,

        "real_field_inspections": False,

        "execution_status": "NOT_EXECUTED"
    },

    "files": file_manifest
}

report_file = (
    output_folder
    / "F5_Explainable_WhatIf_Report.json"
)

with open(
    report_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

print(
    "SAVED:",
    report_file.name
)


# ==========================================
# 8. FINAL FILE VERIFICATION
# ==========================================

with open(
    report_file,
    "r",
    encoding="utf-8"
) as f:
    saved_report = json.load(f)

assert saved_report["whatif_scenarios"] == 9

saved_files = list(
    output_folder.iterdir()
)

assert len(saved_files) == 8

for filename, metadata in file_manifest.items():

    filepath = output_folder / filename

    assert filepath.exists()

    assert (
        sha256(filepath)
        == metadata["sha256"]
    )

assert all(
    sha256(path) == original_hashes[str(path)]
    for path in protected_files
)


# ==========================================
# 9. FINAL OUTPUT
# ==========================================

print("\n=== FINAL EXPORT VERIFICATION ===")

print("CSV Files:", len(exports))

print("JSON Reports:", 1)

print("Total Saved Files:", len(saved_files))

print("What-if Scenarios:", len(multi_summary))

print("Task Assignments:", len(multi_tasks))

print("Original Files: UNCHANGED")

print("\n=== OUTPUT FOLDER ===")

print(output_folder)

print("\n=== FINAL STATUS ===")

print("STATUS: F5 EXPLAINABLE WHAT-IF EXPORT COMPLETE")

print("Simulation Only: YES")

print("Real Drone Flight: NO")

print("Human Approval Completed: NO")

print("STEP 17 COMPLETE")

````

### Original cell 88/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
import math

print("=== DENGU HUNTER F5 ADVANCED: STEP 18 ===")
print("=== FINAL INTEGRATION & CROSS-FEATURE TEST ===")

drive = Path("/content/drive/MyDrive")

base = (
    drive / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Mission_Planner"
)

advanced = base / "F5_Advanced_Enhancements"

battery_dir = (
    advanced
    / "Battery_Reserve_20261122_060550_816670"
)

revisit_dir = (
    advanced
    / "Smart_Revisit_20261122_060131_387121"
)

failure_dir = (
    advanced
    / "Drone_Failure_Recovery_20261122_060911_054114"
)

whatif_dir = (
    advanced
    / "Explainable_WhatIf_20261122_061824_836010"
)

files = {
    "F4": (
        drive / "DENGU_HUNTER_RESULTS"
        / "F4_Hackathon_Runs"
        / "Corrected_20261122_052657"
        / "F4_To_F5_Priority.csv"
    ),

    "ORIGINAL": (
        base / "F5_Optimized_20261122_055356_483143"
        / "F5_Optimized_Mission_Assignments.csv"
    ),

    "BATTERY": (
        battery_dir / "F5_Battery_Safe_Assignments.csv"
    ),

    "BATTERY_SUMMARY": (
        battery_dir / "F5_Battery_Safe_Summary.csv"
    ),

    "REVISIT": (
        revisit_dir / "F5_Smart_Revisit_Decisions.csv"
    ),

    "FAILURE": (
        failure_dir / "F5_Failure_Scenario_Results.csv"
    ),

    "HANDOFF": (
        failure_dir / "F5_Failure_Handoff_Proposals.csv"
    ),

    "AUDIT": (
        failure_dir / "F5_Failure_Task_State_Audit.csv"
    ),

    "EXPLAIN": (
        whatif_dir / "F5_Explainable_Mission_Decisions.csv"
    ),

    "EXPLAIN_SUMMARY": (
        whatif_dir / "F5_Explainable_Drone_Summary.csv"
    ),

    "SENSITIVITY": (
        whatif_dir / "F5_WhatIf_Sensitivity_Summary.csv"
    ),

    "SENSITIVITY_DETAILS": (
        whatif_dir / "F5_WhatIf_Sensitivity_Details.csv"
    ),

    "MULTI_SUMMARY": (
        whatif_dir / "F5_WhatIf_MultiSortie_Summary.csv"
    ),

    "MULTI_SORTIES": (
        whatif_dir / "F5_WhatIf_MultiSortie_Details.csv"
    ),

    "MULTI_TASKS": (
        whatif_dir / "F5_WhatIf_Task_Assignments.csv"
    ),

    "REPORT": (
        whatif_dir / "F5_Explainable_WhatIf_Report.json"
    )
}

# 1. Confirm all files exist
missing = [
    name for name, path in files.items()
    if not path.is_file()
]

assert not missing, f"Missing files: {missing}"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

hashes_before = {
    name: sha256(path)
    for name, path in files.items()
}

data = {
    name: pd.read_csv(path)
    for name, path in files.items()
    if path.suffix == ".csv"
}

with open(files["REPORT"], encoding="utf-8") as f:
    report = json.load(f)

print("\nSource Files Found:", len(files))

for name in files:
    print("FOUND:", name)


# 2. Cross-feature checks
checks = {}

f4 = data["F4"]
old = data["ORIGINAL"]
battery = data["BATTERY"]
battery_summary = data["BATTERY_SUMMARY"]
revisit = data["REVISIT"]
failure = data["FAILURE"]
handoff = data["HANDOFF"]
audit = data["AUDIT"]
explain = data["EXPLAIN"]
sensitivity = data["SENSITIVITY"]
multi = data["MULTI_SUMMARY"]
sorties = data["MULTI_SORTIES"]
tasks = data["MULTI_TASKS"]

checks["F4 Has 15 Unique Areas"] = (
    len(f4) == 15
    and f4["location_id"].is_unique
)

checks["Original F5 Has 15 Unique Areas"] = (
    len(old) == 15
    and old["location_id"].is_unique
)

checks["Battery F5 Has 15 Unique Areas"] = (
    len(battery) == 15
    and battery["location_id"].is_unique
)

checks["Same 15 Areas Across F4 and F5"] = (
    set(f4["location_id"])
    == set(old["location_id"])
    == set(battery["location_id"])
)

# Compare F4 risk data with battery-safe F5
a = f4.set_index("location_id").sort_index()
b = battery.set_index("location_id").sort_index()

checks["F4 Risk Scores Preserved"] = np.allclose(
    a["risk_score"].astype(float),
    b["risk_score"].astype(float),
    atol=1e-6
)

checks["F4 Priority Ranks Preserved"] = (
    a["priority_rank"].astype(int).tolist()
    == b["priority_rank"].astype(int).tolist()
)

checks["F4 Risk Categories Preserved"] = (
    a["risk_category"].astype(str).tolist()
    == b["risk_category"].astype(str).tolist()
)

checks["Battery Has Two Drones"] = (
    set(battery["drone"])
    == {"Drone 1", "Drone 2"}
)

checks["Smart Revisit Has 15 Decisions"] = (
    len(revisit) == 15
)

checks["Failure Has 12 Scenarios"] = (
    len(failure) == 12
    and failure["scenario_id"].is_unique
)

checks["All Failure Scenarios PASS"] = (
    failure["status"].eq("PASS").all()
)

checks["Failure Handoffs Match Pending Tasks"] = (
    len(handoff) == int(failure["pending_tasks"].sum())
    and (
        handoff.groupby("scenario_id")
        .size()
        .reindex(
            failure["scenario_id"],
            fill_value=0
        )
        .to_numpy()
        == failure["pending_tasks"].to_numpy()
    ).all()
)

checks["Failure Audit Has 180 Records"] = (
    len(audit) == 180
)

checks["Failure Handoffs Not Executed"] = (
    handoff["execution_status"]
    .eq("NOT_EXECUTED").all()
)

checks["Failure Handoffs Await Approval"] = (
    handoff["human_approval"]
    .eq("PENDING").all()
)

checks["Explainable Mission Has 15 Areas"] = (
    len(explain) == 15
    and explain["location_id"].is_unique
    and set(explain["location_id"])
    == set(battery["location_id"])
)

checks["9 What-if Sensitivity Scenarios"] = (
    len(sensitivity) == 9
    and sensitivity["Scenario"].is_unique
)

checks["9 Multi-Sortie Scenarios"] = (
    len(multi) == 9
    and multi["Scenario"].is_unique
)

checks["Sensitivity and Multi-Sortie Match"] = (
    set(sensitivity["Scenario"])
    == set(multi["Scenario"])
)

checks["135 What-if Task Assignments"] = (
    len(tasks) == 135
)

checks["All Scenarios Cover Same 15 Areas"] = all(
    len(group) == 15
    and set(group["Location"])
    == set(battery["location_id"])
    for _, group in tasks.groupby("Scenario")
)

checks["No Duplicate Scenario Tasks"] = (
    not tasks.duplicated(
        ["Scenario", "Location"]
    ).any()
)

checks["23 Total Planned Sorties"] = (
    len(sorties) == 23
)

checks["No Real Sortie Executed"] = (
    sorties["Execution_Status"]
    .eq("NOT_EXECUTED").all()
    and tasks["Execution_Status"]
    .eq("NOT_EXECUTED").all()
)

checks["JSON Report Matches CSV"] = (
    report["whatif_scenarios"] == len(multi)
    and report["total_task_assignments"] == len(tasks)
    and report["total_sorties"] == len(sorties)
)


# 3. Independently recalculate all 23 sorties
scenario_settings = {
    "BASELINE": (45, 60, 3.0),
    "BATTERY_42_MIN": (42, 60, 3.0),
    "BATTERY_40_MIN": (40, 60, 3.0),
    "BATTERY_50_MIN": (50, 60, 3.0),
    "SPEED_50_MPM": (45, 50, 3.0),
    "SPEED_70_MPM": (45, 70, 3.0),
    "INSPECTION_3_5_MIN": (45, 60, 3.5),
    "INSPECTION_2_5_MIN": (45, 60, 2.5),
    "COMBINED_STRESS": (42, 55, 3.5)
}

lookup = battery.set_index("location_id")

independent_results = []

for _, sortie in sorties.iterrows():

    name = sortie["Scenario"]
    drone = sortie["Drone"]
    sortie_number = int(sortie["Sortie"])

    battery_minutes, speed, inspection = (
        scenario_settings[name]
    )

    route = tasks[
        (tasks["Scenario"] == name)
        & (tasks["Drone"] == drone)
        & (tasks["Sortie"] == sortie_number)
    ].sort_values("Sequence")

    current = (250.0, 250.0)
    elapsed = 0.0

    for _, task in route.iterrows():

        location = lookup.loc[task["Location"]]

        target = (
            float(location["x_m"]),
            float(location["y_m"])
        )

        elapsed += math.dist(
            current, target
        ) / speed

        elapsed += inspection
        current = target

    elapsed += math.dist(
        current, (250.0, 250.0)
    ) / speed

    reported = float(sortie["Mission_Time_min"])
    usable = battery_minutes * 0.90

    independent_results.append({
        "Scenario": name,
        "Drone": drone,
        "Sortie": sortie_number,
        "Tasks": len(route),
        "Recalculated_Time": round(elapsed, 3),
        "Reported_Time": reported,
        "Time_Match": abs(elapsed - reported) < 0.002,
        "Reserve_PASS": elapsed <= usable + 1e-9
    })

independent_df = pd.DataFrame(independent_results)

checks["All 23 Sortie Times Recalculated"] = (
    len(independent_df) == 23
    and independent_df["Time_Match"].all()
)

checks["All 23 Sorties Meet Battery Reserve"] = (
    independent_df["Reserve_PASS"].all()
)

checks["Original Files Unchanged"] = all(
    sha256(path) == hashes_before[name]
    for name, path in files.items()
)


# 4. Final results
print("\n=== INDEPENDENT SORTIE VERIFICATION ===")

display(independent_df)

print("\n=== CROSS-FEATURE VALIDATION ===")

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

passed_count = sum(
    bool(value) for value in checks.values()
)

print(
    "\nTests Passed:",
    passed_count, "/", len(checks)
)

if not all(checks.values()):
    print("\nSTATUS: INTEGRATION NEEDS REVIEW")
    raise AssertionError(
        "One or more integration checks failed"
    )

print("\n=== FINAL STATUS ===")
print("STATUS: F5 ADVANCED INTEGRATION VALIDATED")
print("Source Files: UNCHANGED")
print("New Files Saved: NO")
print("Real Drone Flight: NO")
print("Real Field Verification: NO")
print("STEP 18 COMPLETE")

````

### Original cell 89/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import hashlib
import json

print("=== DENGU HUNTER F5 ADVANCED: STEP 19 ===")
print("=== FINAL INTEGRATION REPORT EXPORT ===")

# 1. Verify Step 18 results
required = [
    "files", "checks", "independent_df",
    "hashes_before", "data"
]

missing = [x for x in required if x not in globals()]

assert not missing, (
    "Step 18 data missing: " + ", ".join(missing)
)

assert len(checks) == 28
assert all(bool(v) for v in checks.values())
assert len(files) == 16
assert len(independent_df) == 23
assert independent_df["Time_Match"].all()
assert independent_df["Reserve_PASS"].all()

def sha256(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

for name, path in files.items():
    assert Path(path).exists()
    assert sha256(path) == hashes_before[name]

print("Step 18 Validation: 28/28 PASS")
print("Independent Sorties: 23/23 PASS")
print("Source Files: 16/16 UNCHANGED")

# 2. Create unique Google Drive folder
base = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/"
    "Mission_Planner/F5_Advanced_Enhancements"
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output = base / ("F5_Final_Integration_" + timestamp)
output.mkdir(parents=True, exist_ok=False)

# 3. Save integration tests
test_df = pd.DataFrame([
    {
        "Test": name,
        "Status": "PASS" if passed else "FAIL"
    }
    for name, passed in checks.items()
])

test_path = output / "F5_Advanced_Integration_Tests.csv"
test_df.to_csv(test_path, index=False)

# 4. Save source manifest
manifest = pd.DataFrame([
    {
        "Module": name,
        "Path": str(path),
        "SHA256": sha256(path),
        "Status": "VERIFIED"
    }
    for name, path in files.items()
])

manifest_path = output / "F5_Advanced_Source_Manifest.csv"
manifest.to_csv(manifest_path, index=False)

# 5. Prepare final report
multi = data["MULTI_SUMMARY"]
tasks = data["MULTI_TASKS"]
sorties = data["MULTI_SORTIES"]

report = {
    "project": "DENGU HUNTER",
    "feature": "F5 Advanced Drone Mission Planner",
    "status": "SIMULATION_COMPLETE",
    "completion_scope": "HACKATHON_PROTOTYPE",
    "timestamp": timestamp,

    "modules": {
        "Smart Revisit": "SIMULATION_COMPLETE",
        "Battery Reserve Planning": "SIMULATION_COMPLETE",
        "Drone Failure Recovery": "SIMULATION_COMPLETE",
        "Explainable Mission Planning": "SIMULATION_COMPLETE",
        "What-if Replanning": "SIMULATION_COMPLETE",
        "Final Integration": "PASS"
    },

    "validation": {
        "integration_tests": 28,
        "tests_passed": 28,
        "independently_verified_sorties": 23,
        "source_files_verified": 16,
        "original_files_modified": 0
    },

    "mission": {
        "areas": 15,
        "simulated_drones": 2,
        "failure_scenarios": len(data["FAILURE"]),
        "revisit_decisions": len(data["REVISIT"]),
        "whatif_scenarios": len(multi),
        "task_assignments": len(tasks),
        "planned_sorties": len(sorties),
        "extra_sortie_scenarios": int(
            multi["Decision"].eq(
                "EXTRA_SORTIE_PLANNED"
            ).sum()
        )
    },

    "limitations": {
        "synthetic_coordinates": True,
        "rule_based_risk_simulation": True,
        "real_drone_flight": False,
        "real_field_verification": False,
        "recharge_duration_modeled": False,
        "weather_modeled": False,
        "human_approval_completed": False,
        "real_flight_safety_certified": False
    },

    "minimum_reserve_margin_min": float(
        multi["Minimum_Reserve_Margin_min"].min()
    ),

    "next_step": "F7 Closed-Loop Verification"
}

report_path = output / "F5_Advanced_Final_Report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# 6. Dashboard integration index
index = {
    "project": "DENGU HUNTER",
    "feature": "F5",
    "status": "SIMULATION_COMPLETE",
    "dashboard_integrated": False,
    "simulation_only": True,
    "data_sources": {
        name: str(path)
        for name, path in files.items()
    }
}

index_path = output / "F5_Dashboard_Data_Index.json"

with open(index_path, "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2, ensure_ascii=False)

# 7. Verify exported files
saved_files = [
    test_path,
    manifest_path,
    report_path,
    index_path
]

assert len(saved_files) == 4
assert all(p.exists() and p.stat().st_size > 0
           for p in saved_files)

assert len(pd.read_csv(test_path)) == 28
assert len(pd.read_csv(manifest_path)) == 16

with open(report_path, encoding="utf-8") as f:
    saved_report = json.load(f)

assert saved_report["status"] == "SIMULATION_COMPLETE"
assert saved_report["validation"]["tests_passed"] == 28

assert all(
    sha256(path) == hashes_before[name]
    for name, path in files.items()
)

print("\n=== FINAL VERIFICATION ===")
print("Integration Tests: 28/28 PASS")
print("Independent Sorties: 23/23 PASS")
print("Source Files: 16/16 UNCHANGED")
print("Saved Files: 4/4 VERIFIED")

print("\n=== SAVED FILES ===")
for path in saved_files:
    print("SAVED:", path.name)

print("\n=== OUTPUT FOLDER ===")
print(output)

print("\n=== FINAL STATUS ===")
print("F5 ADVANCED: 100% SIMULATION COMPLETE")
print("Real Drone Flight: NO")
print("Final Website Integration: PENDING")
print("STEP 19 COMPLETE")

````

### Original cell 90/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json
import hashlib

print("=== DENGU HUNTER F7: STEP 1 ===")
print("=== EXISTING DATA INSPECTION ===")

# Google Drive connection
drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")

root = (
    drive_root
    / "DENGU_HUNTER_FINAL_SHOWCASE"
)

assert root.exists(), (
    "DENGU_HUNTER_FINAL_SHOWCASE folder not found!"
)

integration = root / "Unified_AI_Integration"
f7_folder = root / "Feature_7_Closed_Loop"
f5_folder = root / "Mission_Planner"

# Existing files - READ ONLY
sources = {
    "F7_BEFORE_AFTER": (
        integration
        / "before_after_verification_intake.csv"
    ),

    "F7_INTERVENTION": (
        integration
        / "human_approved_intervention_workflow.csv"
    ),

    "FEATURE_STATUS": (
        integration
        / "eight_feature_status.json"
    ),

    "F5_SMART_REVISIT": (
        f5_folder
        / "F5_Advanced_Enhancements"
        / "Smart_Revisit_20261122_060131_387121"
        / "F5_Smart_Revisit_Decisions.csv"
    ),

    "F5_BATTERY_MISSION": (
        f5_folder
        / "F5_Advanced_Enhancements"
        / "Battery_Reserve_20261122_060550_816670"
        / "F5_Battery_Safe_Assignments.csv"
    ),

    "F4_PRIORITY": (
        drive_root
        / "DENGU_HUNTER_RESULTS"
        / "F4_Hackathon_Runs"
        / "Corrected_20261122_052657"
        / "F4_To_F5_Priority.csv"
    )
}

def file_hash(path):
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

original_hashes = {
    name: file_hash(path)
    for name, path in sources.items()
    if path.is_file()
}

loaded = {}

print("\n=== SOURCE FILE INSPECTION ===")

for name, path in sources.items():

    print("\n" + "=" * 50)
    print("SOURCE:", name)
    print("FILE:", path.name)

    if not path.is_file():
        print("STATUS: NOT FOUND")
        continue

    print("STATUS: FOUND")

    if path.suffix.lower() == ".csv":

        try:
            df = pd.read_csv(path)
        except Exception as e:
            print("READ ERROR:", str(e))
            continue

        loaded[name] = df

        print("ROWS:", len(df))
        print("COLUMNS:", len(df.columns))
        print("COLUMN NAMES:")
        print(df.columns.tolist())

        print("\nFIRST 3 RECORDS:")
        if len(df):
            display(df.head(3))
        else:
            print("EMPTY DATAFRAME")

        print("\nMISSING VALUE COUNT:")
        print(df.isna().sum().to_string())

        # Identify location and evidence columns
        keywords = [
            "location", "area", "image",
            "before", "after", "status",
            "approval", "verification",
            "revisit", "water"
        ]

        important_cols = [
            col for col in df.columns
            if any(
                word in col.lower()
                for word in keywords
            )
        ]

        print("\nIMPORTANT FIELDS:")
        print(important_cols)

        # Check available after-image evidence
        after_cols = [
            col for col in df.columns
            if "after" in col.lower()
            and (
                "image" in col.lower()
                or "photo" in col.lower()
                or "path" in col.lower()
            )
        ]

        for col in after_cols:
            valid = (
                df[col].dropna()
                .astype(str).str.strip()
            )

            valid = valid[
                ~valid.str.lower().isin(
                    ["", "none", "null", "nan"]
                )
            ]

            print(
                "AFTER EVIDENCE:",
                col,
                "->",
                len(valid),
                "non-empty records"
            )

    elif path.suffix.lower() == ".json":

        try:
            with open(
                path,
                encoding="utf-8"
            ) as f:
                obj = json.load(f)

            loaded[name] = obj

            print("JSON CONTENT:")
            print(
                json.dumps(
                    obj,
                    indent=2,
                    ensure_ascii=False
                )[:4000]
            )

        except Exception as e:
            print("JSON READ ERROR:", str(e))


# Inspect existing F7 files
print("\n" + "=" * 50)
print("=== F7 FOLDER INVENTORY ===")

if f7_folder.exists():

    all_files = sorted(
        p for p in f7_folder.rglob("*")
        if p.is_file()
    )

    image_ext = {
        ".jpg", ".jpeg", ".png",
        ".webp", ".bmp"
    }

    image_files = [
        p for p in all_files
        if p.suffix.lower() in image_ext
    ]

    print("F7 Folder: FOUND")
    print("Total Files:", len(all_files))
    print("Image Files:", len(image_files))

    print("\nFIRST 30 FILES:")

    for p in all_files[:30]:
        print(p.relative_to(f7_folder))

else:
    print("F7 Folder: NOT FOUND")


# Inspect location ID formats
print("\n" + "=" * 50)
print("=== LOCATION ID FORMAT CHECK ===")

for name in [
    "F7_BEFORE_AFTER",
    "F7_INTERVENTION",
    "F5_SMART_REVISIT",
    "F5_BATTERY_MISSION",
    "F4_PRIORITY"
]:

    if name not in loaded:
        continue

    df = loaded[name]

    id_cols = [
        col for col in df.columns
        if (
            "location" in col.lower()
            or "area_id" in col.lower()
        )
    ]

    print("\nSOURCE:", name)

    if not id_cols:
        print("No obvious location ID column")
        continue

    for col in id_cols[:3]:
        values = (
            df[col].dropna()
            .astype(str).unique()[:5]
        )

        print(col, ":", list(values))


# Verify no source files changed
print("\n" + "=" * 50)
print("=== SOURCE INTEGRITY ===")

unchanged = all(
    file_hash(sources[name]) == old_hash
    for name, old_hash in original_hashes.items()
)

assert unchanged, "Original source file changed!"

print("Original Files Unchanged: YES")
print("New Files Created: NO")

print("\n=== F7 STEP 1 STATUS ===")
print("Sources Found:", len(original_hashes), "/ 6")
print("Sources Loaded:", len(loaded))
print("Real Field Verification: NO")
print("STATUS: INSPECTION COMPLETE")
print("F7 STEP 1 COMPLETE")

````

### Original cell 91/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json

print("=== DENGU HUNTER F7: STEP 2 ===")
print("=== EXISTING F7 FILES AND LOCATION LINKING ===")

root = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE"
)

integration = root / "Unified_AI_Integration"
f7_dir = root / "Feature_7_Closed_Loop"

f5_path = (
    root / "Mission_Planner"
    / "F5_Advanced_Enhancements"
    / "Smart_Revisit_20261122_060131_387121"
    / "F5_Smart_Revisit_Decisions.csv"
)

intake_path = (
    integration
    / "before_after_verification_intake.csv"
)

workflow_path = (
    integration
    / "human_approved_intervention_workflow.csv"
)

# 1. Read existing sources
for path in [f5_path, intake_path, workflow_path]:
    assert path.is_file(), f"Missing: {path}"

f5 = pd.read_csv(f5_path)
intake = pd.read_csv(intake_path)
workflow = pd.read_csv(workflow_path)

print("\n=== SOURCE COUNTS ===")
print("F5 Revisit:", len(f5))
print("F7 Before/After:", len(intake))
print("Intervention Tasks:", len(workflow))

# 2. Inspect five existing F7 files
print("\n=== EXISTING F7 FILE INSPECTION ===")

existing_files = [
    f7_dir / "Comparison_Results"
    / "before_after_comparison.csv",

    f7_dir / "Comparison_Results"
    / "comparison_summary.json",

    f7_dir / "closed_loop_summary.json",

    f7_dir / "intervention_register.csv",

    f7_dir / "post_intervention_image_template.csv"
]

for path in existing_files:

    print("\n" + "-" * 50)
    print("FILE:", path.name)

    if not path.is_file():
        print("STATUS: NOT FOUND")
        continue

    print("STATUS: FOUND")

    if path.suffix.lower() == ".csv":

        try:
            df = pd.read_csv(path)

            print("ROWS:", len(df))
            print("COLUMNS:", df.columns.tolist())

            if len(df):
                print("FIRST 3 ROWS:")
                display(df.head(3))

            status_cols = [
                col for col in df.columns
                if any(
                    word in col.lower()
                    for word in [
                        "status", "result",
                        "decision", "approval"
                    ]
                )
            ]

            for col in status_cols[:6]:
                print(
                    col,
                    "VALUES:",
                    df[col].value_counts(
                        dropna=False
                    ).to_dict()
                )

        except Exception as e:
            print("READ ERROR:", str(e))

    elif path.suffix.lower() == ".json":

        try:
            with open(
                path, encoding="utf-8"
            ) as f:
                obj = json.load(f)

            print(
                json.dumps(
                    obj,
                    indent=2,
                    ensure_ascii=False
                )[:3000]
            )

        except Exception as e:
            print("READ ERROR:", str(e))


# 3. Validate unique identifiers
print("\n=== LOCATION MAPPING VALIDATION ===")

checks = {}

checks["15 F5 Records"] = len(f5) == 15
checks["15 F7 Records"] = len(intake) == 15
checks["15 Intervention Tasks"] = len(workflow) == 15

checks["Unique F5 Locations"] = (
    f5["location_id"].is_unique
)

checks["Unique F7 Locations"] = (
    intake["location_id"].is_unique
)

checks["Unique Intervention Task IDs"] = (
    workflow["task_id"].is_unique
)

checks["Unique F5-to-F7 Mapping"] = (
    f5["f7_location_id"].is_unique
)

checks["Unique F5 Task Mapping"] = (
    f5["task_id"].is_unique
)

# 4. Build an in-memory linked registry
# No files will be written
f5_part = f5[
    [
        "location_id",
        "f7_location_id",
        "task_id",
        "image",
        "priority_rank",
        "risk_score",
        "risk_category",
        "revisit_decision"
    ]
].copy()

f5_part = f5_part.rename(
    columns={"image": "f5_image"}
)

intake_part = intake.rename(
    columns={
        "location_id": "f7_location_id",
        "before_image": "intake_before_image"
    }
)

workflow_part = workflow.rename(
    columns={
        "image": "workflow_image"
    }
)

f7_step2_registry = f5_part.merge(
    intake_part,
    on="f7_location_id",
    how="left",
    validate="one_to_one"
)

f7_step2_registry = f7_step2_registry.merge(
    workflow_part,
    on="task_id",
    how="left",
    validate="one_to_one"
)

# Compare image basenames
def normalize_image(value):
    if pd.isna(value):
        return ""
    return str(value).replace(
        "\\", "/"
    ).split("/")[-1].strip().lower()

f7_step2_registry["f5_image_key"] = (
    f7_step2_registry["f5_image"]
    .map(normalize_image)
)

f7_step2_registry["before_image_key"] = (
    f7_step2_registry["intake_before_image"]
    .map(normalize_image)
)

f7_step2_registry["workflow_image_key"] = (
    f7_step2_registry["workflow_image"]
    .map(normalize_image)
)

before_match = (
    f7_step2_registry["f5_image_key"]
    == f7_step2_registry["before_image_key"]
)

workflow_match = (
    f7_step2_registry["f5_image_key"]
    == f7_step2_registry["workflow_image_key"]
)

checks["All 15 F7 Locations Linked"] = (
    f7_step2_registry["intake_before_image"]
    .notna().sum() == 15
)

checks["All 15 Intervention Tasks Linked"] = (
    f7_step2_registry["workflow_image"]
    .notna().sum() == 15
)

checks["Before Image Names Match"] = (
    before_match.all()
)

checks["Intervention Image Names Match"] = (
    workflow_match.all()
)

# 5. Check real evidence availability
after_values = (
    f7_step2_registry["after_image"]
    .fillna("")
    .astype(str)
    .str.strip()
)

after_available = (
    ~after_values.str.lower().isin(
        ["", "nan", "none", "null"]
    )
)

print("\n=== EVIDENCE READINESS ===")
print("Linked Locations:", len(f7_step2_registry))
print("Matching Before Images:", int(before_match.sum()))
print("Matching Intervention Images:", int(workflow_match.sum()))
print("After Images Available:", int(after_available.sum()))

print("\n=== LOCATION LINKING PREVIEW ===")

display(
    f7_step2_registry[
        [
            "location_id",
            "f7_location_id",
            "task_id",
            "priority_rank",
            "revisit_decision",
            "field_action_status",
            "verification_status"
        ]
    ].head(10)
)

# 6. Final test results
print("\n=== STEP 2 TEST RESULTS ===")

for name, result in checks.items():
    print(
        "PASS" if result else "FAIL",
        "-", name
    )

passed = sum(bool(x) for x in checks.values())

print("\nTests Passed:", passed, "/", len(checks))

print("\n=== FINAL STATUS ===")
print("Original Files Modified: NO")
print("New Files Saved: NO")
print("Real Field Verification: NO")

if all(checks.values()):
    print("STATUS: LOCATION LINKING VALIDATED")
    print("F7 STEP 2 COMPLETE")
else:
    print("STATUS: LOCATION LINKING NEEDS REVIEW")

````

### Original cell 92/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import hashlib
import os

print("=== DENGU HUNTER F7: STEP 3 ===")
print("=== F2 ASSETS AND IMAGE VERIFICATION ===")

drive = Path("/content/drive/MyDrive")
root = drive / "DENGU_HUNTER_FINAL_SHOWCASE"

f7_dir = root / "Feature_7_Closed_Loop"
integration = root / "Unified_AI_Integration"

f5_file = (
    root / "Mission_Planner"
    / "F5_Advanced_Enhancements"
    / "Smart_Revisit_20261122_060131_387121"
    / "F5_Smart_Revisit_Decisions.csv"
)

paths = {
    "F5": f5_file,
    "INTAKE": integration / "before_after_verification_intake.csv",
    "COMPARISON": (
        f7_dir / "Comparison_Results"
        / "before_after_comparison.csv"
    ),
    "REGISTER": f7_dir / "intervention_register.csv"
}

assert all(p.is_file() for p in paths.values()), (
    "One or more required files missing"
)

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

hashes = {
    name: file_hash(path)
    for name, path in paths.items()
}

df = {
    name: pd.read_csv(path)
    for name, path in paths.items()
}

def image_key(value):
    if pd.isna(value):
        return ""
    return str(value).replace(
        "\\", "/"
    ).split("/")[-1].strip().lower()

f5 = df["F5"]
intake = df["INTAKE"]
comparison = df["COMPARISON"]
register = df["REGISTER"]

print("\n=== RECORD COUNTS ===")
for name, table in df.items():
    print(name, ":", len(table))

# 1. Verify F5 and F7 main mapping
linked = f5[
    ["location_id", "f7_location_id", "image"]
].merge(
    intake[["location_id", "before_image"]],
    left_on="f7_location_id",
    right_on="location_id",
    how="left",
    validate="one_to_one",
    suffixes=("_f5", "_f7")
)

main_match = (
    linked["image"].map(image_key)
    == linked["before_image"].map(image_key)
)

print("\n=== MAIN LOCATION LINK ===")
print("Linked Records:", len(linked))
print("Matching Images:", int(main_match.sum()))

# 2. Compare legacy F7 image references
intake_keys = set(
    intake["before_image"].map(image_key)
)

legacy_keys = set(
    comparison["location_image"].map(image_key)
)

register_keys = set(
    register["image"].map(image_key)
)

legacy_overlap = intake_keys & legacy_keys
register_overlap = intake_keys & register_keys

print("\n=== LEGACY IMAGE COMPATIBILITY ===")
print("Intake Unique Images:", len(intake_keys))
print("Legacy Comparison Images:", len(legacy_keys))
print("Legacy Register Images:", len(register_keys))

print(
    "Comparison-to-Intake Matches:",
    len(legacy_overlap)
)

print(
    "Register-to-Intake Matches:",
    len(register_overlap)
)

print("\nLEGACY COMPARISON PREVIEW:")

preview = comparison[
    ["verification_id", "location_image", "status"]
].copy()

preview["matches_main_intake"] = (
    preview["location_image"]
    .map(image_key)
    .isin(intake_keys)
)

display(preview.head(8))

if len(legacy_overlap) < 15:
    print(
        "\nNOTICE: Some legacy comparison images "
        "do not match the main intake."
    )
    print(
        "Do not join these records by row number "
        "or numeric ID suffix."
    )

# 3. Check existing before-image paths
print("\n=== BEFORE IMAGE FILE CHECK ===")

image_paths = comparison["before_image"]

exists = []

for value in image_paths:

    if pd.isna(value):
        exists.append(False)
        continue

    path = Path(str(value).strip())
    exists.append(path.is_file())

comparison_check = comparison[
    ["verification_id", "before_image"]
].copy()

comparison_check["file_exists"] = exists

print("Referenced Images:", len(comparison_check))
print("Existing Image Files:", sum(exists))
print("Missing Image Files:", len(exists) - sum(exists))

display(comparison_check.head(8))

# 4. Locate reusable F2 assets
print("\n=== F2 AI ASSET DISCOVERY ===")

search_roots = [
    drive / "DENGUE_HUNTER_RESULTS",
    drive / "DENGU_HUNTER_RESULTS",
    root / "Surveillance_Integration"
]

model_candidates = []
pipeline_candidates = []
water_candidates = []

for search_root in search_roots:

    print("\nSEARCH ROOT:", search_root)

    if not search_root.is_dir():
        print("NOT FOUND")
        continue

    visited = 0
    root_depth = len(search_root.parts)

    for current, dirs, filenames in os.walk(search_root):

        visited += 1

        if visited > 600:
            print("Directory scan limit reached.")
            break

        depth = len(Path(current).parts) - root_depth

        if depth >= 5:
            dirs[:] = []

        # Skip raw image datasets during asset discovery
        dirs[:] = [
            d for d in dirs
            if d.lower() not in {
                "train", "valid", "test",
                "images", ".ipynb_checkpoints"
            }
        ]

        for filename in filenames:

            path = Path(current) / filename
            low = filename.lower()
            full_low = str(path).lower()

            if path.suffix.lower() in {
                ".pt", ".onnx"
            }:
                model_candidates.append(str(path))

            if path.suffix.lower() in {
                ".py", ".ipynb"
            }:
                pipeline_candidates.append(str(path))

            if any(
                word in full_low
                for word in [
                    "water_mask",
                    "segmentation",
                    "water_segment",
                    "f2_evidence"
                ]
            ):
                if path.suffix.lower() in {
                    ".csv", ".json", ".png",
                    ".jpg", ".npy"
                }:
                    water_candidates.append(str(path))

    print("Directories Inspected:", visited)

# Remove duplicates
model_candidates = sorted(set(model_candidates))
pipeline_candidates = sorted(set(pipeline_candidates))
water_candidates = sorted(set(water_candidates))

print("\n=== DISCOVERED AI ASSETS ===")

print("\nModel Candidates:", len(model_candidates))
for p in model_candidates[:25]:
    print("MODEL:", p)

print("\nPython/Notebook Candidates:", len(pipeline_candidates))
for p in pipeline_candidates[:20]:
    print("CODE:", p)

print("\nWater/Segmentation Artifacts:", len(water_candidates))
for p in water_candidates[:20]:
    print("WATER:", p)

# 5. Integrity verification
unchanged = all(
    file_hash(paths[name]) == original
    for name, original in hashes.items()
)

assert unchanged, "Source file changed!"

print("\n=== STEP 3 FINAL STATUS ===")
print("Main F5-F7 Image Matches:", int(main_match.sum()), "/ 15")
print("Legacy Comparison Matches:", len(legacy_overlap), "/ 15")
print("Before Image Files Found:", sum(exists), "/ 15")
print("Model Candidates:", len(model_candidates))
print("Original Files Unchanged: YES")
print("New Files Saved: NO")
print("Real After Images Verified: NO")
print("F7 STEP 3 INSPECTION COMPLETE")

````

### Original cell 93/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import hashlib

print("=== DENGU HUNTER F7: STEP 4 ===")
print("=== REAL AI WATER SEGMENTATION TEST ===")

# 1. Import existing YOLO framework
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "ultralytics"
    ])
    from ultralytics import YOLO

drive = Path("/content/drive/MyDrive")

model_path = (
    drive / "DENGUE_HUNTER_RESULTS"
    / "Water_Segmentation"
    / "DengueHunter_Water_v1"
    / "weights"
    / "best.pt"
)

comparison_path = (
    drive / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Feature_7_Closed_Loop"
    / "Comparison_Results"
    / "before_after_comparison.csv"
)

assert model_path.is_file(), "Water model missing!"
assert comparison_path.is_file(), "F7 comparison CSV missing!"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

original_hashes = {
    model_path: sha256(model_path),
    comparison_path: sha256(comparison_path)
}

# 2. Load trained water model
print("\n=== LOADING WATER MODEL ===")

model = YOLO(str(model_path))

print("MODEL LOADED: YES")
print("MODEL TASK:", model.task)
print("MODEL CLASSES:", model.names)

names = {
    int(k): str(v)
    for k, v in model.names.items()
}

water_ids = [
    i for i, name in names.items()
    if "water" in name.lower()
    and not any(
        word in name.lower()
        for word in [
            "non-water", "nonwater",
            "no_water", "dry"
        ]
    )
]

print("WATER CLASS IDs:", water_ids)

if model.task != "segment":
    print("WARNING: Model is not a segmentation model.")

if not water_ids:
    print("WARNING: Water class not identified.")
    print("Class names need manual review.")

# 3. Load three existing F7 before images
df = pd.read_csv(comparison_path)

sample = df.head(3).copy()

image_paths = [
    Path(str(p))
    for p in sample["before_image"]
]

for path in image_paths:
    assert path.is_file(), f"Image missing: {path}"

print("\n=== INPUT IMAGES ===")

for i, path in enumerate(image_paths):
    print(
        sample.iloc[i]["verification_id"],
        "->", path.name
    )

# 4. Run actual trained model
print("\n=== RUNNING AI INFERENCE ===")

results = model.predict(
    source=[str(p) for p in image_paths],
    imgsz=640,
    conf=0.25,
    retina_masks=True,
    save=False,
    verbose=False
)

assert len(results) == 3, (
    "Expected 3 inference results"
)

records = []
first_overlay = None
first_original = None

for i, result in enumerate(results):

    verification_id = sample.iloc[i]["verification_id"]

    original = cv2.cvtColor(
        result.orig_img,
        cv2.COLOR_BGR2RGB
    )

    h, w = original.shape[:2]

    water_mask = np.zeros(
        (h, w), dtype=np.uint8
    )

    classes = (
        result.boxes.cls.int().cpu().tolist()
        if result.boxes is not None
        else []
    )

    water_detection_indices = [
        j for j, class_id in enumerate(classes)
        if class_id in water_ids
    ]

    mask_available = result.masks is not None

    if mask_available and water_ids:

        polygons = result.masks.xy

        for j in water_detection_indices:

            if j >= len(polygons):
                continue

            polygon = polygons[j]

            if len(polygon) < 3:
                continue

            points = np.round(
                polygon
            ).astype(np.int32)

            cv2.fillPoly(
                water_mask,
                [points],
                1
            )

    can_measure = (
        model.task == "segment"
        and bool(water_ids)
    )

    coverage = (
        round(
            100 * np.count_nonzero(water_mask)
            / (h * w),
            3
        )
        if can_measure
        else None
    )

    records.append({
        "verification_id": verification_id,
        "total_detections": len(classes),
        "water_detections": len(
            water_detection_indices
        ),
        "mask_available": mask_available,
        "water_mask_pct": coverage
    })

    if i == 0:

        overlay = original.copy()

        overlay[water_mask > 0] = (
            0.45 * overlay[water_mask > 0]
            + 0.55 * np.array([0, 255, 255])
        ).astype(np.uint8)

        first_original = original
        first_overlay = overlay

# 5. Show inference summary
print("\n=== AI INFERENCE RESULTS ===")

result_df = pd.DataFrame(records)

display(result_df)

# 6. Visualize water segmentation
print("\n=== WATER MASK VISUALIZATION ===")

fig, axes = plt.subplots(
    1, 2, figsize=(11, 5)
)

axes[0].imshow(first_original)
axes[0].set_title("Original Before Image")
axes[0].axis("off")

axes[1].imshow(first_overlay)
axes[1].set_title(
    "AI Water Mask (Cyan Overlay)"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

# 7. Check source integrity
unchanged = all(
    sha256(path) == old_hash
    for path, old_hash in original_hashes.items()
)

assert unchanged, "Original file changed!"

print("\n=== STEP 4 FINAL STATUS ===")

print("Model Task:", model.task)
print("Water Classes Found:", len(water_ids))
print("Images Tested:", len(results))
print(
    "Images With Water Detections:",
    int(
        (result_df["water_detections"] > 0).sum()
    )
)
print("Original Files Unchanged: YES")
print("Results Saved To Drive: NO")
print("Real After Images Tested: NO")

if model.task == "segment" and water_ids:
    print("F7 STEP 4 BASELINE TEST COMPLETE")
else:
    print("F7 STEP 4 NEEDS MODEL REVIEW")

````

### Original cell 94/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import cv2
import json
import hashlib

from ultralytics import YOLO
from IPython.display import display, Image

print("=== DENGU HUNTER F7: STEP 5 ===")
print("=== ALL 15 BEFORE IMAGE BASELINE ===")

drive = Path("/content/drive/MyDrive")

root = (
    drive / "DENGU_HUNTER_FINAL_SHOWCASE"
)

f7 = root / "Feature_7_Closed_Loop"

model_path = (
    drive / "DENGUE_HUNTER_RESULTS"
    / "Water_Segmentation"
    / "DengueHunter_Water_v1"
    / "weights"
    / "best.pt"
)

comparison_path = (
    f7 / "Comparison_Results"
    / "before_after_comparison.csv"
)

intake_path = (
    root / "Unified_AI_Integration"
    / "before_after_verification_intake.csv"
)

f5_path = (
    root / "Mission_Planner"
    / "F5_Advanced_Enhancements"
    / "Smart_Revisit_20261122_060131_387121"
    / "F5_Smart_Revisit_Decisions.csv"
)

sources = [
    model_path,
    comparison_path,
    intake_path,
    f5_path
]

assert all(p.is_file() for p in sources), (
    "Required source file missing"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

original_hashes = {
    str(p): sha256(p)
    for p in sources
}

def image_key(value):
    return Path(
        str(value).replace("\\", "/")
    ).name.strip().lower()

# 1. Load data
comparison = pd.read_csv(comparison_path)
intake = pd.read_csv(intake_path)
f5 = pd.read_csv(f5_path)

comparison["image_key"] = (
    comparison["location_image"]
    .map(image_key)
)

intake["image_key"] = (
    intake["before_image"]
    .map(image_key)
)

f5["image_key"] = (
    f5["image"].map(image_key)
)

# 2. Match using image identity, not row number
registry = comparison.merge(
    intake[["location_id", "image_key"]],
    on="image_key",
    how="left",
    validate="one_to_one"
)

f5_link = f5[
    [
        "f7_location_id",
        "location_id",
        "image_key",
        "priority_rank",
        "risk_score",
        "risk_category"
    ]
].rename(
    columns={"location_id": "f5_location_id"}
)

registry = registry.merge(
    f5_link,
    left_on=["location_id", "image_key"],
    right_on=["f7_location_id", "image_key"],
    how="left",
    validate="one_to_one"
)

assert len(registry) == 15
assert registry["location_id"].notna().all()
assert registry["f5_location_id"].notna().all()
assert registry["verification_id"].is_unique

image_paths = [
    Path(str(p))
    for p in registry["before_image"]
]

assert all(p.is_file() for p in image_paths)

print("\nLocation Mapping: 15/15 VALID")
print("Before Images: 15/15 FOUND")

# 3. Load existing segmentation model
model = YOLO(str(model_path))

assert model.task == "segment"

names = {
    int(k): str(v)
    for k, v in model.names.items()
}

target_ids = [
    k for k, v in names.items()
    if v in {
        "tire_with_water",
        "vase_with_water"
    }
]

assert target_ids, "Expected classes not found"

print("\nMODEL:", model_path.name)
print("TASK:", model.task)
print("CLASSES:", names)

# 4. Create a NEW versioned output folder
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = (
    f7 / "F7_AI_Baseline"
    / f"Baseline_{timestamp}"
)

mask_dir = output_dir / "Masks"
overlay_dir = output_dir / "Overlays"

mask_dir.mkdir(parents=True, exist_ok=False)
overlay_dir.mkdir(parents=True, exist_ok=True)

# 5. Run trained model on all images
print("\n=== RUNNING 15 IMAGE INFERENCES ===")

results = model.predict(
    source=[str(p) for p in image_paths],
    imgsz=640,
    conf=0.25,
    retina_masks=True,
    save=False,
    verbose=False
)

assert len(results) == 15

records = []

for i, result in enumerate(results):

    row = registry.iloc[i]

    verification_id = row["verification_id"]

    original = result.orig_img.copy()
    h, w = original.shape[:2]

    mask = np.zeros(
        (h, w), dtype=np.uint8
    )

    classes = (
        result.boxes.cls.int().cpu().tolist()
        if result.boxes is not None
        else []
    )

    confidences = (
        result.boxes.conf.cpu().tolist()
        if result.boxes is not None
        else []
    )

    selected_indices = [
        j for j, class_id in enumerate(classes)
        if class_id in target_ids
    ]

    if result.masks is not None:

        polygons = result.masks.xy

        for j in selected_indices:

            if j >= len(polygons):
                continue

            polygon = polygons[j]

            if len(polygon) < 3:
                continue

            points = np.round(
                polygon
            ).astype(np.int32)

            cv2.fillPoly(
                mask, [points], 255
            )

    coverage = round(
        100 * np.count_nonzero(mask)
        / (h * w),
        4
    )

    detected_classes = sorted(set(
        names[classes[j]]
        for j in selected_indices
    ))

    max_conf = (
        round(
            max(
                confidences[j]
                for j in selected_indices
            ),
            4
        )
        if selected_indices
        else None
    )

    status = (
        "WATER_ASSOCIATED_OBJECT_DETECTED"
        if selected_indices
        else "NOT_DETECTED"
    )

    mask_path = (
        mask_dir
        / f"{verification_id}_mask.png"
    )

    overlay_path = (
        overlay_dir
        / f"{verification_id}_overlay.jpg"
    )

    # Binary mask
    assert cv2.imwrite(
        str(mask_path), mask
    )

    # Visual overlay
    overlay = original.copy()

    selected_pixels = mask > 0

    if np.any(selected_pixels):

        tint = np.array(
            [255, 255, 0],
            dtype=np.float32
        )

        overlay[selected_pixels] = (
            0.50 * overlay[selected_pixels]
            + 0.50 * tint
        ).astype(np.uint8)

    cv2.putText(
        overlay,
        verification_id,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    assert cv2.imwrite(
        str(overlay_path), overlay
    )

    records.append({
        "verification_id": verification_id,
        "f7_location_id": row["location_id"],
        "f5_location_id": row["f5_location_id"],
        "priority_rank": int(row["priority_rank"]),
        "risk_score": float(row["risk_score"]),
        "risk_category": row["risk_category"],
        "before_image": str(image_paths[i]),
        "before_image_sha256": sha256(image_paths[i]),
        "model_classes": ",".join(detected_classes),
        "object_detections": len(selected_indices),
        "max_confidence": max_conf,
        "object_mask_pct": coverage,
        "detection_status": status,
        "mask_path": str(mask_path),
        "overlay_path": str(overlay_path),
        "after_image_available": False,
        "verification_status": "AWAITING_AFTER_IMAGE",
        "evidence_scope": "EXISTING_BEFORE_IMAGE"
    })

# 6. Save results
baseline = pd.DataFrame(records)

csv_path = (
    output_dir / "F7_Before_AI_Baseline.csv"
)

baseline.to_csv(
    csv_path, index=False
)

summary = {
    "feature": "F7 Closed-Loop Verification",
    "stage": "BEFORE_IMAGE_BASELINE",
    "total_images": len(baseline),
    "images_with_detection": int(
        (baseline["object_detections"] > 0).sum()
    ),
    "images_without_detection": int(
        (baseline["object_detections"] == 0).sum()
    ),
    "model_classes": names,
    "confidence_threshold": 0.25,
    "measurement": "water-associated object mask area",
    "not_a_direct_water_pixel_measurement": True,
    "after_images_tested": 0,
    "real_intervention_verified": False,
    "human_approval": "PENDING",
    "source_hashes": original_hashes
}

json_path = (
    output_dir / "F7_Baseline_Summary.json"
)

with open(
    json_path, "w", encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# 7. Validation
tests = {
    "15 baseline records": len(baseline) == 15,
    "15 unique locations": (
        baseline["f7_location_id"].is_unique
    ),
    "15 mask files": all(
        Path(p).is_file()
        for p in baseline["mask_path"]
    ),
    "15 overlay files": all(
        Path(p).is_file()
        for p in baseline["overlay_path"]
    ),
    "CSV saved": csv_path.is_file(),
    "JSON saved": json_path.is_file(),
    "Sources unchanged": all(
        sha256(p) == original_hashes[str(p)]
        for p in sources
    )
}

print("\n=== STEP 5 TEST RESULTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTESTS:",
    sum(tests.values()),
    "/", len(tests)
)

print("\n=== BASELINE SUMMARY ===")
print("Images Analyzed:", len(baseline))
print(
    "Water-associated Object Detected:",
    summary["images_with_detection"]
)
print(
    "No Detection:",
    summary["images_without_detection"]
)

print("\n=== FIRST 10 RESULTS ===")

display(
    baseline[
        [
            "verification_id",
            "f7_location_id",
            "object_detections",
            "object_mask_pct",
            "detection_status"
        ]
    ].head(10)
)

print("\n=== SAMPLE AI OVERLAY ===")

display(
    Image(
        filename=baseline.iloc[0]["overlay_path"],
        width=550
    )
)

print("\n=== OUTPUT FOLDER ===")
print(output_dir)

print("\n=== FINAL STATUS ===")
print("Original Sources Modified: NO")
print("Real After Images Tested: NO")
print("Real Field Verification: NO")

if all(tests.values()):
    print("F7 STEP 5 COMPLETE")
else:
    print("F7 STEP 5 NEEDS REVIEW")

````

### Original cell 95/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import importlib.util
import hashlib
import json
import cv2
import numpy as np
import pandas as pd

print("=== DENGU HUNTER F7: STEP 6 ===")
print("=== CLOSED-LOOP MASK COMPARISON ENGINE ===")

# 1. Existing Step 5 baseline
f7 = Path(
    "/content/drive/MyDrive/"
    "DENGU_HUNTER_FINAL_SHOWCASE/"
    "Feature_7_Closed_Loop"
)

baseline_dir = (
    f7 / "F7_AI_Baseline"
    / "Baseline_20261122_063424_731302"
)

baseline_path = (
    baseline_dir / "F7_Before_AI_Baseline.csv"
)

assert baseline_path.is_file(), "Step 5 baseline missing!"

baseline = pd.read_csv(baseline_path)

assert len(baseline) == 15
assert baseline["verification_id"].is_unique

mask_paths = [
    Path(p) for p in baseline["mask_path"]
]

assert all(p.is_file() for p in mask_paths)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

source_files = [baseline_path] + mask_paths

source_hashes = {
    str(p): sha256(p)
    for p in source_files
}

# 2. Create versioned output folder
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = (
    f7 / "F7_Comparison_Engine"
    / f"Engine_Test_{timestamp}"
)

output_dir.mkdir(
    parents=True,
    exist_ok=False
)

# 3. Save a reusable comparison engine
engine_path = (
    output_dir / "f7_mask_comparison_engine.py"
)

engine_code = '''
import numpy as np

def compare_object_masks(
    before,
    after,
    revisit=None,
    aligned=False
):
    """
    Compare masks of water-associated objects.
    Does NOT directly measure water pixels.
    Does NOT verify real-world elimination.
    """

    if before is None:
        return {
            "status": "BEFORE_MASK_REQUIRED",
            "human_review": "REQUIRED"
        }

    if after is None:
        return {
            "status": "AWAITING_AFTER_IMAGE",
            "human_review": "REQUIRED"
        }

    b = np.asarray(before) > 0
    a = np.asarray(after) > 0

    if b.ndim != 2 or a.ndim != 2:
        raise ValueError("Masks must be 2D")

    if b.shape != a.shape:
        raise ValueError("Mask dimensions do not match")

    r = None

    if revisit is not None:
        r = np.asarray(revisit) > 0

        if r.shape != b.shape:
            raise ValueError("Revisit dimensions mismatch")

    if not aligned:
        return {
            "status": "ALIGNMENT_REQUIRED",
            "human_review": "REQUIRED"
        }

    pixels = b.size

    minimum = max(
        10,
        int(np.ceil(pixels * 0.0005))
    )

    bc = int(b.sum())
    ac = int(a.sum())
    rc = int(r.sum()) if r is not None else None

    b_found = bc >= minimum
    a_found = ac >= minimum

    before_pct = 100 * bc / pixels
    after_pct = 100 * ac / pixels

    relative_change = (
        100 * (ac - bc) / bc
        if bc > 0 else None
    )

    union = int(np.logical_or(b, a).sum())
    intersection = int(
        np.logical_and(b, a).sum()
    )

    iou = (
        intersection / union
        if union > 0 else None
    )

    status = "INCONCLUSIVE"

    # A third observation is necessary
    # to flag possible recurrence.
    if r is not None and b_found and not a_found:

        overlap = int(
            np.logical_and(b, r).sum()
        )

        revisit_overlap = (
            overlap / bc if bc else 0
        )

        if rc >= minimum and revisit_overlap >= 0.25:
            status = "REAPPEARANCE_CANDIDATE"

    if status == "INCONCLUSIVE":

        if b_found and not a_found:
            status = "TARGET_NOT_DETECTED_AFTER"

        elif not b_found and a_found:
            status = "NEW_TARGET_CANDIDATE"

        elif not b_found and not a_found:
            status = "NO_TARGET_DETECTED"

        elif relative_change <= -20:
            status = "OBJECT_MASK_REDUCED"

        elif relative_change >= 20:
            status = "OBJECT_MASK_EXPANDED"

        else:
            status = "OBJECT_MASK_STABLE"

    return {
        "status": status,
        "before_object_mask_pct": round(
            before_pct, 4
        ),
        "after_object_mask_pct": round(
            after_pct, 4
        ),
        "relative_change_pct": (
            round(relative_change, 2)
            if relative_change is not None
            else None
        ),
        "mask_iou": (
            round(iou, 4)
            if iou is not None
            else None
        ),
        "revisit_object_mask_pct": (
            round(100 * rc / pixels, 4)
            if rc is not None
            else None
        ),
        "human_review": "REQUIRED",
        "real_world_verified": False,
        "measurement": "WATER_ASSOCIATED_OBJECT_MASK"
    }
'''

engine_path.write_text(
    engine_code,
    encoding="utf-8"
)

# Import the saved engine
spec = importlib.util.spec_from_file_location(
    "f7_engine_step6",
    str(engine_path)
)

engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

print("\nComparison Engine: CREATED")
print("Engine File:", engine_path.name)

# 4. Select real AI baseline masks
positive_rows = baseline[
    baseline["object_detections"] > 0
]

negative_rows = baseline[
    baseline["object_detections"] == 0
]

assert len(positive_rows) > 0
assert len(negative_rows) > 0

positive_path = Path(
    positive_rows.iloc[0]["mask_path"]
)

negative_path = Path(
    negative_rows.iloc[0]["mask_path"]
)

positive = (
    cv2.imread(
        str(positive_path),
        cv2.IMREAD_GRAYSCALE
    ) > 0
)

negative = (
    cv2.imread(
        str(negative_path),
        cv2.IMREAD_GRAYSCALE
    ) > 0
)

assert positive.any()
assert not negative.any()

# 5. Build controlled synthetic masks
removed = np.zeros_like(positive)

reduced = positive.copy()

ys, xs = np.where(reduced)

# Remove approximately half the mask pixels
reduced[ys[::2], xs[::2]] = False

expanded = cv2.dilate(
    positive.astype(np.uint8),
    np.ones((41, 41), np.uint8),
    iterations=2
) > 0

new_target = cv2.resize(
    positive.astype(np.uint8),
    (negative.shape[1], negative.shape[0]),
    interpolation=cv2.INTER_NEAREST
) > 0

# 6. Test seven comparison scenarios
scenarios = [
    (
        "Object No Longer Detected",
        positive, removed, None,
        "TARGET_NOT_DETECTED_AFTER"
    ),
    (
        "Object Mask Reduced",
        positive, reduced, None,
        "OBJECT_MASK_REDUCED"
    ),
    (
        "Object Mask Expanded",
        positive, expanded, None,
        "OBJECT_MASK_EXPANDED"
    ),
    (
        "Object Mask Stable",
        positive, positive.copy(), None,
        "OBJECT_MASK_STABLE"
    ),
    (
        "New Target Candidate",
        negative, new_target, None,
        "NEW_TARGET_CANDIDATE"
    ),
    (
        "No Target Detected",
        negative, negative.copy(), None,
        "NO_TARGET_DETECTED"
    ),
    (
        "Possible Recurrence",
        positive, removed, positive.copy(),
        "REAPPEARANCE_CANDIDATE"
    )
]

records = []
tests = []

print("\n=== SYNTHETIC MASK TESTS ===")

for name, before, after, revisit, expected in scenarios:

    result = engine.compare_object_masks(
        before,
        after,
        revisit=revisit,
        aligned=True
    )

    passed = result["status"] == expected

    tests.append({
        "test": name,
        "passed": bool(passed)
    })

    records.append({
        "scenario": name,
        "expected": expected,
        **result,
        "evidence_scope": "SYNTHETIC_MASK_TEST"
    })

    print(
        "PASS" if passed else "FAIL",
        "-", name,
        "->", result["status"]
    )

# 7. Safety tests
pending = engine.compare_object_masks(
    positive,
    None
)

tests.append({
    "test": "Missing After Evidence",
    "passed": (
        pending["status"]
        == "AWAITING_AFTER_IMAGE"
    )
})

alignment = engine.compare_object_masks(
    positive,
    positive,
    aligned=False
)

tests.append({
    "test": "Alignment Required",
    "passed": (
        alignment["status"]
        == "ALIGNMENT_REQUIRED"
    )
})

dimension_rejected = False

try:
    engine.compare_object_masks(
        positive,
        np.zeros((10, 10)),
        aligned=True
    )
except ValueError:
    dimension_rejected = True

tests.append({
    "test": "Invalid Dimensions Rejected",
    "passed": dimension_rejected
})

# 8. Preserve real cases as PENDING
real_cases = baseline[
    [
        "verification_id",
        "f7_location_id",
        "f5_location_id",
        "before_image",
        "mask_path",
        "object_mask_pct"
    ]
].copy()

real_cases["after_image"] = ""
real_cases["comparison_status"] = (
    "AWAITING_AFTER_IMAGE"
)
real_cases["human_review"] = "PENDING"
real_cases["real_world_verified"] = False

tests.append({
    "test": "15 Real Cases Remain Pending",
    "passed": (
        len(real_cases) == 15
        and real_cases["comparison_status"]
        .eq("AWAITING_AFTER_IMAGE").all()
    )
})

# 9. Save new results
simulation_path = (
    output_dir / "F7_Synthetic_Comparison_Tests.csv"
)

pending_path = (
    output_dir / "F7_Real_Cases_Pending.csv"
)

tests_path = (
    output_dir / "F7_Engine_Test_Results.csv"
)

pd.DataFrame(records).to_csv(
    simulation_path,
    index=False
)

real_cases.to_csv(
    pending_path,
    index=False
)

# 10. Check original files
unchanged = all(
    sha256(Path(p)) == original_hash
    for p, original_hash in source_hashes.items()
)

tests.append({
    "test": "Original Sources Unchanged",
    "passed": unchanged
})

pd.DataFrame(tests).to_csv(
    tests_path,
    index=False
)

summary = {
    "feature": "F7 Closed-Loop Verification",
    "step": 6,
    "engine_type": "OBJECT_MASK_COMPARISON",
    "synthetic_scenarios": len(scenarios),
    "tests_passed": sum(
        t["passed"] for t in tests
    ),
    "tests_total": len(tests),
    "real_cases_pending": len(real_cases),
    "real_after_images": 0,
    "real_interventions_verified": 0,
    "water_pixels_directly_measured": False,
    "synthetic_masks_are_field_evidence": False,
    "human_review_required": True,
    "source_hashes": source_hashes
}

summary_path = (
    output_dir / "F7_Engine_Summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

# 11. Final verification
print("\n=== STEP 6 TEST RESULTS ===")

test_df = pd.DataFrame(tests)
display(test_df)

print(
    "TESTS:",
    int(test_df["passed"].sum()),
    "/",
    len(test_df)
)

print("\n=== COMPARISON PREVIEW ===")

display(
    pd.DataFrame(records)[
        [
            "scenario",
            "status",
            "before_object_mask_pct",
            "after_object_mask_pct",
            "relative_change_pct"
        ]
    ]
)

print("\n=== SAVED OUTPUT ===")
print(output_dir)

print("\n=== FINAL STATUS ===")
print("Original Files Unchanged:", unchanged)
print("Real Cases Pending:", len(real_cases))
print("Real After Images:", 0)
print("Real Field Verification: NO")

if test_df["passed"].all():
    print("F7 STEP 6 COMPLETE")
else:
    print("F7 STEP 6 NEEDS REVIEW")

````

### Original cell 96/118 — PYTHON SOURCE (archived, not run)

````python

import json
import numpy as np
import pandas as pd
from pathlib import Path

print("=== F7 STEP 6: JSON FIX ===")

# Previous Step 6 variables must exist
assert "summary" in globals(), "Previous runtime not found"
assert "output_dir" in globals(), "Output folder not found"

# Convert NumPy values into JSON-compatible types
def json_safe(value):

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"Unsupported type: {type(value).__name__}"
    )

# Fix summary values
summary["tests_passed"] = int(
    sum(bool(t["passed"]) for t in tests)
)

summary["tests_total"] = int(len(tests))

summary["real_cases_pending"] = int(
    len(real_cases)
)

summary["synthetic_scenarios"] = int(
    len(scenarios)
)

# Repair ONLY the Step 6 summary file
summary_path = (
    output_dir / "F7_Engine_Summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
        default=json_safe
    )

# Verify JSON can be read again
with open(
    summary_path,
    encoding="utf-8"
) as f:
    verified_summary = json.load(f)

print("JSON SAVE: SUCCESS")

# Verify all saved files
required_files = [
    engine_path,
    simulation_path,
    pending_path,
    tests_path,
    summary_path
]

saved_ok = all(
    Path(p).is_file()
    for p in required_files
)

# Verify original files remain unchanged
sources_ok = all(
    sha256(Path(path)) == old_hash
    for path, old_hash in source_hashes.items()
)

# Read saved test results
saved_tests = pd.read_csv(tests_path)

passed = int(
    saved_tests["passed"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

total = len(saved_tests)

pending_cases = pd.read_csv(pending_path)

pending_ok = (
    len(pending_cases) == 15
    and pending_cases["comparison_status"]
    .eq("AWAITING_AFTER_IMAGE").all()
)

print("\n=== FINAL VERIFICATION ===")
print("Tests Passed:", passed, "/", total)
print("Files Saved:", sum(
    Path(p).is_file()
    for p in required_files
), "/", len(required_files))

print("Original Sources Unchanged:", sources_ok)
print("Real Cases Pending:", len(pending_cases))
print("Pending Status Correct:", pending_ok)
print("JSON Readback: PASS")

print("\n=== OUTPUT FOLDER ===")
print(output_dir)

print("\n=== FINAL STATUS ===")

if (
    passed == total
    and total == 12
    and saved_ok
    and sources_ok
    and pending_ok
):
    print("F7 STEP 6 COMPLETE")
else:
    print("F7 STEP 6 NEEDS REVIEW")

````

### Original cell 97/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import importlib.util
import hashlib
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

print("=== DENGU HUNTER F7: STEP 7 ===")
print("=== AI BEFORE-AFTER IMAGE COMPARISON DEMO ===")

drive = Path("/content/drive/MyDrive")
f7 = (
    drive / "DENGU_HUNTER_FINAL_SHOWCASE"
    / "Feature_7_Closed_Loop"
)

baseline_path = (
    f7 / "F7_AI_Baseline"
    / "Baseline_20261122_063424_731302"
    / "F7_Before_AI_Baseline.csv"
)

engine_path = (
    f7 / "F7_Comparison_Engine"
    / "Engine_Test_20261122_063559_244263"
    / "f7_mask_comparison_engine.py"
)

model_path = (
    drive / "DENGUE_HUNTER_RESULTS"
    / "Water_Segmentation"
    / "DengueHunter_Water_v1"
    / "weights"
    / "best.pt"
)

assert all(
    p.is_file()
    for p in [baseline_path, engine_path, model_path]
), "Required Step 5/6 files missing"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

# 1. Load baseline
baseline = pd.read_csv(baseline_path)

assert len(baseline) == 15

positive = baseline[
    baseline["object_detections"] > 0
]

assert len(positive) > 0, (
    "No positive baseline image found"
)

selected = positive.iloc[0]

verification_id = str(
    selected["verification_id"]
)

before_path = Path(
    selected["before_image"]
)

baseline_mask_path = Path(
    selected["mask_path"]
)

assert before_path.is_file()
assert baseline_mask_path.is_file()

source_paths = [
    baseline_path,
    engine_path,
    model_path,
    before_path,
    baseline_mask_path
]

source_hashes = {
    str(p): sha256(p)
    for p in source_paths
}

assert (
    sha256(before_path)
    == selected["before_image_sha256"]
), "Before image hash mismatch"

print("\nSELECTED LOCATION:", selected["f7_location_id"])
print("VERIFICATION ID:", verification_id)
print("BEFORE IMAGE:", before_path.name)

# 2. Load Step 6 comparison engine
spec = importlib.util.spec_from_file_location(
    "f7_step7_engine",
    str(engine_path)
)

engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)

print("\nCOMPARISON ENGINE: LOADED")

# 3. Load trained segmentation model
model = YOLO(str(model_path))

assert model.task == "segment"

names = {
    int(k): str(v)
    for k, v in model.names.items()
}

target_ids = {
    k for k, name in names.items()
    if name in {
        "tire_with_water",
        "vase_with_water"
    }
}

assert target_ids

print("AI MODEL: LOADED")
print("MODEL CLASSES:", names)

# 4. Create versioned demo folder
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = (
    f7 / "F7_Image_Comparison_Demo"
    / f"Demo_{timestamp}"
)

image_dir = output_dir / "Images"
mask_dir = output_dir / "Masks"
overlay_dir = output_dir / "Overlays"

for folder in [image_dir, mask_dir, overlay_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# 5. Read original image and AI baseline mask
before_img = cv2.imread(str(before_path))

baseline_mask = cv2.imread(
    str(baseline_mask_path),
    cv2.IMREAD_GRAYSCALE
)

assert before_img is not None
assert baseline_mask is not None

assert (
    before_img.shape[:2]
    == baseline_mask.shape
)

assert np.count_nonzero(baseline_mask) > 0

# 6. Create synthetic cleanup image
# This edits the detected OBJECT region,
# not necessarily the water pixels.
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (17, 17)
)

edit_region = cv2.dilate(
    baseline_mask,
    kernel,
    iterations=1
)

synthetic_after = cv2.inpaint(
    before_img,
    edit_region,
    9,
    cv2.INPAINT_TELEA
)

cleanup_path = (
    image_dir / "SYNTHETIC_CLEANUP.jpg"
)

unchanged_path = (
    image_dir / "UNCHANGED_CONTROL.jpg"
)

revisit_path = (
    image_dir / "SIMULATED_REVISIT.jpg"
)

assert cv2.imwrite(
    str(cleanup_path), synthetic_after
)

# Identical-image controls
shutil.copy2(before_path, unchanged_path)
shutil.copy2(before_path, revisit_path)

print("\n=== DEMO IMAGES PREPARED ===")
print("Synthetic Cleanup: CREATED")
print("Unchanged Control: CREATED")
print("Simulated Revisit: CREATED")
print("Real After Image: NO")

# 7. Actual AI inference on all images
input_paths = [
    before_path,
    cleanup_path,
    unchanged_path,
    revisit_path
]

labels = [
    "BEFORE",
    "SYNTHETIC_CLEANUP",
    "UNCHANGED_CONTROL",
    "SIMULATED_REVISIT"
]

print("\n=== RUNNING TRAINED AI MODEL ===")

results = model.predict(
    source=[str(p) for p in input_paths],
    imgsz=640,
    conf=0.25,
    retina_masks=True,
    save=False,
    verbose=False
)

assert len(results) == 4

def extract_target_mask(result):
    h, w = result.orig_img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    if result.boxes is None:
        return mask, 0

    classes = (
        result.boxes.cls.int().cpu().tolist()
    )

    indices = [
        i for i, class_id in enumerate(classes)
        if class_id in target_ids
    ]

    if result.masks is not None:
        polygons = result.masks.xy

        for i in indices:
            if i >= len(polygons):
                continue

            polygon = polygons[i]

            if len(polygon) < 3:
                continue

            points = np.round(
                polygon
            ).astype(np.int32)

            cv2.fillPoly(
                mask, [points], 255
            )

    return mask, len(indices)

masks = {}
inference_records = {}
overlay_images = {}

for label, result in zip(labels, results):

    mask, detections = extract_target_mask(result)

    masks[label] = mask

    mask_path = mask_dir / f"{label}_mask.png"
    overlay_path = overlay_dir / f"{label}_overlay.jpg"

    assert cv2.imwrite(str(mask_path), mask)

    overlay = result.orig_img.copy()

    pixels = mask > 0

    if np.any(pixels):
        tint = np.array(
            [255, 255, 0],
            dtype=np.float32
        )

        overlay[pixels] = (
            0.5 * overlay[pixels]
            + 0.5 * tint
        ).astype(np.uint8)

    assert cv2.imwrite(
        str(overlay_path), overlay
    )

    overlay_images[label] = overlay

    coverage = (
        100 * np.count_nonzero(mask)
        / mask.size
    )

    inference_records[label] = {
        "detections": int(detections),
        "object_mask_pct": round(
            float(coverage), 4
        ),
        "mask_path": str(mask_path),
        "overlay_path": str(overlay_path)
    }

# 8. Compare masks using Step 6 engine
# Synthetic edits preserve the original
# image coordinates, so aligned=True
# is valid ONLY for these controlled demos.

cleanup_result = engine.compare_object_masks(
    masks["BEFORE"],
    masks["SYNTHETIC_CLEANUP"],
    aligned=True
)

control_result = engine.compare_object_masks(
    masks["BEFORE"],
    masks["UNCHANGED_CONTROL"],
    aligned=True
)

revisit_result = engine.compare_object_masks(
    masks["BEFORE"],
    masks["SYNTHETIC_CLEANUP"],
    revisit=masks["SIMULATED_REVISIT"],
    aligned=True
)

comparisons = [
    {
        "scenario": "SYNTHETIC_CLEANUP",
        **cleanup_result
    },
    {
        "scenario": "UNCHANGED_CONTROL",
        **control_result
    },
    {
        "scenario": "SIMULATED_REVISIT",
        **revisit_result
    }
]

comparison_df = pd.DataFrame(comparisons)

# 9. Save AI results
inference_df = pd.DataFrame([
    {
        "image_type": label,
        **record
    }
    for label, record in inference_records.items()
])

inference_csv = (
    output_dir / "F7_Image_AI_Inference.csv"
)

comparison_csv = (
    output_dir / "F7_Image_Comparison_Results.csv"
)

inference_df.to_csv(
    inference_csv, index=False
)

comparison_df.to_csv(
    comparison_csv, index=False
)

# 10. Save provenance and limitations
manifest = {
    "feature": "F7 Closed-Loop Verification",
    "step": 7,
    "verification_id": verification_id,
    "f7_location_id": str(
        selected["f7_location_id"]
    ),
    "source_before_image": str(before_path),
    "synthetic_after_generation": (
        "OpenCV inpainting of AI-detected "
        "water-associated object mask"
    ),
    "synthetic_revisit_generation": (
        "Copy of original before image"
    ),
    "model_classes": names,
    "ai_inference_performed": True,
    "real_after_image": False,
    "real_drone_revisit": False,
    "real_field_intervention": False,
    "real_world_verified": False,
    "human_review_required": True,
    "aligned_only_because_synthetic_edits": True,
    "direct_water_pixel_measurement": False,
    "source_hashes": source_hashes
}

manifest_path = (
    output_dir / "F7_Demo_Provenance.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# 11. Final tests
sources_unchanged = all(
    sha256(Path(p)) == old_hash
    for p, old_hash in source_hashes.items()
)

tests = {
    "4 Images Inferred": len(results) == 4,
    "4 Masks Saved": all(
        (mask_dir / f"{label}_mask.png").is_file()
        for label in labels
    ),
    "4 Overlays Saved": all(
        (overlay_dir / f"{label}_overlay.jpg").is_file()
        for label in labels
    ),
    "3 Comparisons Completed": (
        len(comparison_df) == 3
    ),
    "Control Is Stable": (
        control_result["status"]
        == "OBJECT_MASK_STABLE"
    ),
    "Human Review Required": (
        comparison_df["human_review"]
        .eq("REQUIRED").all()
    ),
    "No Real Verification Claimed": (
        comparison_df["real_world_verified"]
        .eq(False).all()
    ),
    "Output Files Saved": all(
        p.is_file()
        for p in [
            inference_csv,
            comparison_csv,
            manifest_path
        ]
    ),
    "Original Sources Unchanged": sources_unchanged
}

print("\n=== AI INFERENCE RESULTS ===")
display(inference_df[
    ["image_type", "detections", "object_mask_pct"]
])

print("\n=== IMAGE COMPARISON RESULTS ===")
display(comparison_df[
    [
        "scenario",
        "status",
        "before_object_mask_pct",
        "after_object_mask_pct",
        "relative_change_pct"
    ]
])

print("\n=== STEP 7 TEST RESULTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTESTS:",
    sum(tests.values()),
    "/", len(tests)
)

# 12. Visual inspection
fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 5)
)

for ax, label, title in zip(
    axes,
    [
        "BEFORE",
        "SYNTHETIC_CLEANUP",
        "SIMULATED_REVISIT"
    ],
    [
        "Original Before",
        "Synthetic After (AI Overlay)",
        "Simulated Revisit"
    ]
):
    rgb = cv2.cvtColor(
        overlay_images[label],
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\n=== OUTPUT FOLDER ===")
print(output_dir)

print("\n=== FINAL STATUS ===")
print("Real After Image: NO")
print("Real Drone Revisit: NO")
print("Real Field Verification: NO")
print("Original Sources Unchanged:", sources_unchanged)

if all(tests.values()):
    print("F7 STEP 7 COMPLETE")
else:
    print("F7 STEP 7 NEEDS REVIEW")

````

### Original cell 98/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import hashlib
import importlib.util
import json

print("=== DENGU HUNTER F7: STEP 8 ===")
print("=== F7 + F5 SMART REVISIT INTEGRATION ===")

drive = Path("/content/drive/MyDrive")
root = drive / "DENGU_HUNTER_FINAL_SHOWCASE"
f7 = root / "Feature_7_Closed_Loop"

baseline_path = (
    f7 / "F7_AI_Baseline"
    / "Baseline_20261122_063424_731302"
    / "F7_Before_AI_Baseline.csv"
)

f5_path = (
    root / "Mission_Planner"
    / "F5_Advanced_Enhancements"
    / "Smart_Revisit_20261122_060131_387121"
    / "F5_Smart_Revisit_Decisions.csv"
)

intervention_path = (
    root / "Unified_AI_Integration"
    / "human_approved_intervention_workflow.csv"
)

demo_dir = (
    f7 / "F7_Image_Comparison_Demo"
    / "Demo_20261122_063751_108660"
)

demo_path = (
    demo_dir / "F7_Image_Comparison_Results.csv"
)

manifest_path = (
    demo_dir / "F7_Demo_Provenance.json"
)

sources = [
    baseline_path,
    f5_path,
    intervention_path,
    demo_path,
    manifest_path
]

assert all(p.is_file() for p in sources), (
    "Required Step 5, F5, or Step 7 file missing"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

source_hashes = {
    str(p): sha256(p)
    for p in sources
}

# 1. Load existing sources
baseline = pd.read_csv(baseline_path)
f5_data = pd.read_csv(f5_path)
interventions = pd.read_csv(intervention_path)
demo = pd.read_csv(demo_path)

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

print("\n=== SOURCE COUNTS ===")
print("F7 Baseline:", len(baseline))
print("F5 Smart Revisit:", len(f5_data))
print("Intervention Tasks:", len(interventions))
print("Demo Comparisons:", len(demo))

assert len(baseline) == 15
assert len(f5_data) == 15
assert len(interventions) == 15
assert len(demo) == 3

# 2. Create a reusable decision router
router_code = '''
def route_f7_to_f5(
    ai_status,
    risk_category="UNKNOWN",
    evidence_scope="REAL_FIELD",
    after_image_available=False,
    alignment_confirmed=False,
    human_approval="PENDING",
    field_action_status="NOT_STARTED",
    human_review="PENDING"
):
    """
    Recommend F5 follow-up from F7 AI results.

    This function does NOT dispatch drones,
    approve field actions, or prove elimination.
    """

    actions = {
        "TARGET_NOT_DETECTED_AFTER":
            "REQUEST_HUMAN_VISUAL_REVIEW",

        "OBJECT_MASK_REDUCED":
            "REVIEW_REMAINING_TARGET",

        "OBJECT_MASK_STABLE":
            "REINSPECT_LOCATION",

        "OBJECT_MASK_EXPANDED":
            "REQUEST_PRIORITY_REINSPECTION",

        "NEW_TARGET_CANDIDATE":
            "REQUEST_PRIORITY_REINSPECTION",

        "REAPPEARANCE_CANDIDATE":
            "REQUEST_PRIORITY_REVISIT_APPROVAL",

        "NO_TARGET_DETECTED":
            "MONITOR_WITH_HUMAN_REVIEW"
    }

    action = actions.get(
        ai_status,
        "MANUAL_REVIEW"
    )

    if not after_image_available:
        return {
            "workflow_status": "AWAITING_AFTER_IMAGE",
            "f5_revisit_action":
                "KEEP_EXISTING_F5_PLAN_PENDING",
            "human_review_required": True,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    if not alignment_confirmed:
        return {
            "workflow_status":
                "ALIGNMENT_REVIEW_REQUIRED",
            "f5_revisit_action":
                "VERIFY_SAME_LOCATION_AND_VIEW",
            "human_review_required": True,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    # Synthetic evidence must never close
    # or verify a real field case.
    if evidence_scope != "REAL_FIELD":
        return {
            "workflow_status":
                "DEMO_ONLY_REVIEW_REQUIRED",
            "f5_revisit_action": action,
            "human_review_required": True,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    if (
        human_approval != "APPROVED"
        or field_action_status != "COMPLETED"
    ):
        return {
            "workflow_status":
                "FIELD_APPROVAL_OR_ACTION_PENDING",
            "f5_revisit_action": action,
            "human_review_required": True,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    if human_review == "DISAGREE":
        return {
            "workflow_status":
                "HUMAN_REINSPECTION_REQUIRED",
            "f5_revisit_action":
                "REQUEST_PRIORITY_REINSPECTION",
            "human_review_required": False,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    if (
        human_review == "CONFIRMED_CLEAR"
        and ai_status == "TARGET_NOT_DETECTED_AFTER"
    ):
        return {
            "workflow_status":
                "VISUAL_TARGET_CLEARANCE_REVIEWED",
            "f5_revisit_action":
                "SCHEDULE_MONITORING",
            "human_review_required": False,
            "real_world_elimination_verified": False,
            "drone_dispatched": False
        }

    return {
        "workflow_status":
            "HUMAN_REVIEW_REQUIRED",
        "f5_revisit_action": action,
        "human_review_required": True,
        "real_world_elimination_verified": False,
        "drone_dispatched": False
    }
'''

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

output_dir = (
    f7 / "F7_F5_Integration"
    / f"Integration_{timestamp}"
)

output_dir.mkdir(
    parents=True,
    exist_ok=False
)

router_path = (
    output_dir / "f7_f5_workflow_router.py"
)

router_path.write_text(
    router_code,
    encoding="utf-8"
)

spec = importlib.util.spec_from_file_location(
    "f7_f5_router_step8",
    str(router_path)
)

router = importlib.util.module_from_spec(spec)
spec.loader.exec_module(router)

print("\nWORKFLOW ROUTER: CREATED")

# 3. Join F7 baseline with F5 locations
baseline_fields = baseline[
    [
        "verification_id",
        "f7_location_id",
        "f5_location_id",
        "before_image",
        "mask_path",
        "object_detections",
        "object_mask_pct"
    ]
].copy()

f5_fields = f5_data[
    [
        "location_id",
        "f7_location_id",
        "task_id",
        "priority_rank",
        "risk_score",
        "risk_category",
        "revisit_decision",
        "recommended_next_action",
        "human_approval",
        "field_action_status",
        "verification_status"
    ]
].rename(columns={
    "location_id": "f5_location_id",
    "human_approval": "f5_human_approval",
    "field_action_status": "f5_field_action_status",
    "verification_status": "f5_verification_status"
})

registry = baseline_fields.merge(
    f5_fields,
    on=["f7_location_id", "f5_location_id"],
    how="left",
    validate="one_to_one"
)

# 4. Attach existing intervention tasks
task_fields = interventions[
    [
        "task_id",
        "human_approval",
        "field_action_status",
        "completion_evidence",
        "verification_status"
    ]
].rename(columns={
    "human_approval": "task_human_approval",
    "field_action_status": "task_field_action_status",
    "completion_evidence": "task_completion_evidence",
    "verification_status": "task_verification_status"
})

registry = registry.merge(
    task_fields,
    on="task_id",
    how="left",
    validate="one_to_one"
)

assert len(registry) == 15
assert registry["task_id"].notna().all()
assert registry["verification_id"].is_unique
assert registry["f7_location_id"].is_unique
assert registry["f5_location_id"].is_unique
assert registry["task_id"].is_unique

print("\n=== LOCATION AND TASK LINKING ===")
print("F7 ↔ F5 ↔ Task:", len(registry), "/ 15")

# 5. Route all real cases
real_routes = []

for _, row in registry.iterrows():

    decision = router.route_f7_to_f5(
        ai_status="AWAITING_AFTER_IMAGE",
        risk_category=row["risk_category"],
        evidence_scope="REAL_FIELD",
        after_image_available=False,
        alignment_confirmed=False,
        human_approval=row["task_human_approval"],
        field_action_status=row["task_field_action_status"],
        human_review="PENDING"
    )

    real_routes.append(decision)

real_registry = pd.concat(
    [
        registry.reset_index(drop=True),
        pd.DataFrame(real_routes)
    ],
    axis=1
)

real_registry["after_image"] = ""
real_registry["human_review"] = "PENDING"
real_registry["evidence_scope"] = (
    "REAL_FIELD_EVIDENCE_PENDING"
)

# 6. Route three existing demo outcomes
demo_verification_id = manifest["verification_id"]

matching = registry[
    registry["verification_id"]
    == demo_verification_id
]

assert len(matching) == 1

selected = matching.iloc[0]

demo_records = []

for _, row in demo.iterrows():

    decision = router.route_f7_to_f5(
        ai_status=row["status"],
        risk_category=selected["risk_category"],
        evidence_scope="SYNTHETIC_DEMO",
        after_image_available=True,
        alignment_confirmed=True,
        human_approval="PENDING",
        field_action_status="NOT_STARTED",
        human_review="PENDING"
    )

    demo_records.append({
        "scenario": row["scenario"],
        "verification_id": demo_verification_id,
        "f7_location_id": selected["f7_location_id"],
        "f5_location_id": selected["f5_location_id"],
        "task_id": selected["task_id"],
        "priority_rank": int(
            selected["priority_rank"]
        ),
        "risk_category": selected["risk_category"],
        "ai_status": row["status"],
        "before_object_mask_pct":
            row["before_object_mask_pct"],
        "after_object_mask_pct":
            row["after_object_mask_pct"],
        "evidence_scope": "SYNTHETIC_DEMO",
        "human_approval": "PENDING",
        "field_action_status": "NOT_STARTED",
        "human_review": "PENDING",
        **decision
    })

demo_decisions = pd.DataFrame(demo_records)

print("\n=== DEMO FOLLOW-UP DECISIONS ===")

display(
    demo_decisions[
        [
            "scenario",
            "ai_status",
            "f5_revisit_action",
            "workflow_status"
        ]
    ]
)

# 7. Safety and integration tests
tests = {}

tests["15 Real Cases Linked"] = (
    len(real_registry) == 15
)

tests["15 Unique F7 Locations"] = (
    real_registry["f7_location_id"].is_unique
)

tests["15 Unique F5 Locations"] = (
    real_registry["f5_location_id"].is_unique
)

tests["15 Unique Task IDs"] = (
    real_registry["task_id"].is_unique
)

tests["All Task Links Present"] = (
    real_registry["task_id"].notna().all()
)

tests["Before Images Exist"] = all(
    Path(p).is_file()
    for p in real_registry["before_image"]
)

tests["15 Real Cases Pending"] = (
    real_registry["workflow_status"]
    .eq("AWAITING_AFTER_IMAGE").all()
)

tests["Real Field Actions Not Completed"] = (
    real_registry["task_field_action_status"]
    .ne("COMPLETED").all()
)

tests["No Automatic Human Approval"] = (
    real_registry["task_human_approval"]
    .ne("APPROVED").all()
)

tests["3 Demo Decisions"] = (
    len(demo_decisions) == 3
)

tests["Demo Cases Clearly Labeled"] = (
    demo_decisions["evidence_scope"]
    .eq("SYNTHETIC_DEMO").all()
)

tests["No Demo Case Closed"] = (
    demo_decisions["workflow_status"]
    .eq("DEMO_ONLY_REVIEW_REQUIRED").all()
)

tests["Recurrence Requests Revisit"] = (
    demo_decisions.loc[
        demo_decisions["ai_status"]
        == "REAPPEARANCE_CANDIDATE",
        "f5_revisit_action"
    ].eq(
        "REQUEST_PRIORITY_REVISIT_APPROVAL"
    ).all()
    and (
        demo_decisions["ai_status"]
        == "REAPPEARANCE_CANDIDATE"
    ).any()
)

tests["No Drone Dispatch Claimed"] = (
    not real_registry["drone_dispatched"].any()
    and not demo_decisions["drone_dispatched"].any()
)

tests["No Elimination Verification Claimed"] = (
    not real_registry[
        "real_world_elimination_verified"
    ].any()
    and not demo_decisions[
        "real_world_elimination_verified"
    ].any()
)

# 8. Additional approval-gate test
gate_test = router.route_f7_to_f5(
    ai_status="TARGET_NOT_DETECTED_AFTER",
    evidence_scope="REAL_FIELD",
    after_image_available=True,
    alignment_confirmed=True,
    human_approval="PENDING",
    field_action_status="NOT_STARTED",
    human_review="CONFIRMED_CLEAR"
)

tests["Human Approval Gate Works"] = (
    gate_test["workflow_status"]
    == "FIELD_APPROVAL_OR_ACTION_PENDING"
)

# 9. Save integration results
real_path = (
    output_dir / "F7_F5_Real_Pending_Registry.csv"
)

demo_output_path = (
    output_dir / "F7_F5_Demo_Decisions.csv"
)

test_path = (
    output_dir / "F7_F5_Integration_Tests.csv"
)

real_registry.to_csv(
    real_path,
    index=False
)

demo_decisions.to_csv(
    demo_output_path,
    index=False
)

# 10. Verify original source integrity
sources_unchanged = all(
    sha256(Path(path)) == original_hash
    for path, original_hash in source_hashes.items()
)

tests["Original Sources Unchanged"] = (
    sources_unchanged
)

test_df = pd.DataFrame([
    {
        "test": name,
        "passed": bool(passed)
    }
    for name, passed in tests.items()
])

test_df.to_csv(
    test_path,
    index=False
)

summary = {
    "feature": "F7 Closed-Loop Verification",
    "step": 8,
    "integration": "F7_AI_TO_F5_SMART_REVISIT",
    "linked_real_cases": int(len(real_registry)),
    "synthetic_demo_cases": int(len(demo_decisions)),
    "real_after_images": 0,
    "real_cases_closed": 0,
    "real_drone_dispatches": 0,
    "human_approval_automated": False,
    "real_elimination_verified": False,
    "tests_passed": int(test_df["passed"].sum()),
    "tests_total": int(len(test_df)),
    "source_hashes": source_hashes
}

summary_path = (
    output_dir / "F7_F5_Integration_Summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# 11. Final output verification
saved_files = [
    router_path,
    real_path,
    demo_output_path,
    test_path,
    summary_path
]

saved_ok = all(
    p.is_file()
    for p in saved_files
)

print("\n=== STEP 8 TEST RESULTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTESTS:",
    int(test_df["passed"].sum()),
    "/", len(test_df)
)

print("\n=== REAL CASES PREVIEW ===")

display(
    real_registry[
        [
            "verification_id",
            "f7_location_id",
            "task_id",
            "priority_rank",
            "workflow_status",
            "f5_revisit_action"
        ]
    ].head(10)
)

print("\n=== SAVED FILES ===")

for path in saved_files:
    print(
        "SAVED" if path.is_file() else "MISSING",
        "-", path.name
    )

print("\n=== OUTPUT FOLDER ===")
print(output_dir)

print("\n=== FINAL STATUS ===")
print("Files Saved:", sum(
    p.is_file() for p in saved_files
), "/", len(saved_files))
print("Original Sources Unchanged:", sources_unchanged)
print("Real After Images: 0")
print("Real Field Verification: NO")

if test_df["passed"].all() and saved_ok:
    print("F7 STEP 8 COMPLETE")
else:
    print("F7 STEP 8 NEEDS REVIEW")

````

### Original cell 99/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import os

print("=== F7 STEP 9: SOURCE DISCOVERY ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F7_ROOT = ROOT / "Feature_7_Closed_Loop"

assert ROOT.exists(), "Google Drive folder not found!"
assert F7_ROOT.exists(), "F7 folder not found!"

print("\n=== F7 MODULES ===")

for folder in sorted(F7_ROOT.iterdir()):
    if folder.is_dir():
        print("\n", folder.name)
        for f in list(folder.iterdir())[:8]:
            print("  ", f.name)

print("\n=== SEARCHING TRAINED MODEL ===")

models = list(ROOT.rglob("best.pt"))

for model in models[:10]:
    print("MODEL:", model)

print("\n=== SEARCHING F7 BASELINE CSV ===")

csv_files = list(F7_ROOT.rglob("*.csv"))

for f in csv_files:
    print("CSV:", f)

print("\n=== STEP 9 SOURCE DISCOVERY COMPLETE ===")
print("Models found:", len(models))
print("CSV files found:", len(csv_files))

````

### Original cell 100/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F7 = ROOT / "Feature_7_Closed_Loop"

baseline_path = next(F7.rglob("F7_Before_AI_Baseline.csv"))
registry_path = next(F7.rglob("F7_F5_Real_Pending_Registry.csv"))

intervention_path = F7 / "intervention_register.csv"
template_path = F7 / "post_intervention_image_template.csv"

print("=== STEP 9.2: SOURCE DATA INSPECTION ===")

baseline = pd.read_csv(baseline_path)
registry = pd.read_csv(registry_path)
interventions = pd.read_csv(intervention_path)
template = pd.read_csv(template_path)

for name, df in [
    ("BASELINE", baseline),
    ("F5 REGISTRY", registry),
    ("INTERVENTION REGISTER", interventions),
    ("AFTER IMAGE TEMPLATE", template)
]:
    print(f"\n=== {name} ===")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print(df.head(3).to_string(index=False))

print("\n=== AFTER IMAGE FOLDER ===")

after_dir = F7 / "post_intervention_images"

files = [
    p for p in after_dir.rglob("*")
    if p.is_file()
]

print("Folder:", after_dir)
print("Existing files:", len(files))

for p in files[:20]:
    print(p.name)

print("\n=== MODEL CHECK ===")

model_path = ROOT / "Models/Water_Segmentation_Model/best.pt"

print("Model exists:", model_path.exists())
print("Model size MB:",
      round(model_path.stat().st_size / (1024*1024), 2)
      if model_path.exists() else "MISSING")

print("\nSTEP 9.2 COMPLETE")

````

### Original cell 101/118 — PYTHON SOURCE (archived, not run)

````python

import sys
import subprocess
import importlib.util

# Install only if missing
for package in ["gradio", "ultralytics"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip",
            "install", "-q", package
        ])

from pathlib import Path
from datetime import datetime
import hashlib
import importlib.util
import json
import secrets
import uuid

import cv2
import numpy as np
import pandas as pd
import gradio as gr

from ultralytics import YOLO

print("=== DENGU HUNTER F7: STEP 9.3 ===")
print("=== INTERACTIVE AFTER IMAGE VERIFICATION ===")

# ---------------------------------
# 1. Existing project files
# ---------------------------------

DRIVE = Path("/content/drive/MyDrive")

ROOT = (
    DRIVE / "DENGU_HUNTER_FINAL_SHOWCASE"
)

F7 = ROOT / "Feature_7_Closed_Loop"

BASELINE_PATH = (
    F7 / "F7_AI_Baseline"
    / "Baseline_20261122_063424_731302"
    / "F7_Before_AI_Baseline.csv"
)

REGISTRY_PATH = (
    F7 / "F7_F5_Integration"
    / "Integration_20261122_063918_927395"
    / "F7_F5_Real_Pending_Registry.csv"
)

ENGINE_PATH = (
    F7 / "F7_Comparison_Engine"
    / "Engine_Test_20261122_063559_244263"
    / "f7_mask_comparison_engine.py"
)

ROUTER_PATH = (
    F7 / "F7_F5_Integration"
    / "Integration_20261122_063918_927395"
    / "f7_f5_workflow_router.py"
)

MODEL_PATH = (
    DRIVE / "DENGUE_HUNTER_RESULTS"
    / "Water_Segmentation"
    / "DengueHunter_Water_v1"
    / "weights" / "best.pt"
)

DEMO_IMAGE_PATH = (
    F7 / "F7_Image_Comparison_Demo"
    / "Demo_20261122_063751_108660"
    / "Images" / "SYNTHETIC_CLEANUP.jpg"
)

required = [
    BASELINE_PATH,
    REGISTRY_PATH,
    ENGINE_PATH,
    ROUTER_PATH,
    MODEL_PATH,
    DEMO_IMAGE_PATH
]

assert all(p.is_file() for p in required), (
    "A required source file is missing"
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

source_hashes = {
    str(p): sha256(p)
    for p in required
}

# ---------------------------------
# 2. Load and link 15 locations
# ---------------------------------

baseline = pd.read_csv(BASELINE_PATH)
registry = pd.read_csv(REGISTRY_PATH)

task_columns = [
    "verification_id",
    "f7_location_id",
    "f5_location_id",
    "task_id",
    "task_human_approval",
    "task_field_action_status"
]

locations = baseline.merge(
    registry[task_columns],
    on=[
        "verification_id",
        "f7_location_id",
        "f5_location_id"
    ],
    how="left",
    validate="one_to_one"
)

assert len(locations) == 15
assert locations["verification_id"].is_unique
assert locations["task_id"].notna().all()

assert all(
    Path(p).is_file()
    for p in locations["before_image"]
)

assert all(
    Path(p).is_file()
    for p in locations["mask_path"]
)

location_map = {}

for _, row in locations.iterrows():

    label = (
        f"{row['f7_location_id']} | "
        f"{row['verification_id']} | "
        f"{row['task_id']}"
    )

    location_map[label] = row.to_dict()

choices = list(location_map.keys())

# ---------------------------------
# 3. Load previous tested engines
# ---------------------------------

def load_python_module(name, path):
    spec = importlib.util.spec_from_file_location(
        name, str(path)
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

engine = load_python_module(
    "f7_step9_engine", ENGINE_PATH
)

router = load_python_module(
    "f7_step9_router", ROUTER_PATH
)

model = YOLO(str(MODEL_PATH))

assert model.task == "segment"

model_names = {
    int(k): str(v)
    for k, v in model.names.items()
}

target_ids = {
    k for k, name in model_names.items()
    if name in {
        "tire_with_water",
        "vase_with_water"
    }
}

assert target_ids

print("Locations:", len(locations))
print("Model:", model_names)
print("Comparison Engine: LOADED")
print("F5 Router: LOADED")

# ---------------------------------
# 4. Create new output workspace
# ---------------------------------

run_id = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

RUN_DIR = (
    F7 / "F7_Interactive_Uploads"
    / f"Run_{run_id}"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False
)

# ---------------------------------
# 5. Image helpers
# ---------------------------------

def read_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError("Cannot read image")
    return cv2.cvtColor(
        img, cv2.COLOR_BGR2RGB
    )

def extract_mask(result):

    h, w = result.orig_img.shape[:2]

    mask = np.zeros(
        (h, w), dtype=np.uint8
    )

    if result.boxes is None:
        return mask, 0, []

    classes = (
        result.boxes.cls.int().cpu().tolist()
    )

    indices = [
        i for i, c in enumerate(classes)
        if c in target_ids
    ]

    if result.masks is not None:

        for i in indices:

            if i >= len(result.masks.xy):
                continue

            polygon = result.masks.xy[i]

            if len(polygon) < 3:
                continue

            points = np.round(
                polygon
            ).astype(np.int32)

            cv2.fillPoly(
                mask, [points], 255
            )

    labels = [
        model_names[classes[i]]
        for i in indices
    ]

    return mask, len(indices), labels

def make_overlay(image_bgr, mask):

    overlay = image_bgr.copy()

    pixels = mask > 0

    if np.any(pixels):

        tint = np.array(
            [255, 255, 0],
            dtype=np.float32
        )

        overlay[pixels] = (
            0.5 * overlay[pixels]
            + 0.5 * tint
        ).astype(np.uint8)

    return overlay

# ---------------------------------
# 6. Location preview
# ---------------------------------

def show_location(label):

    row = location_map[label]

    before = read_rgb(
        row["before_image"]
    )

    information = (
        f"Location: {row['f7_location_id']}\n"
        f"Verification: {row['verification_id']}\n"
        f"Field Task: {row['task_id']}\n"
        f"Risk: {row['risk_category']}\n"
        f"Before AI Detections: "
        f"{row['object_detections']}\n"
        f"Before Object Mask: "
        f"{row['object_mask_pct']}%\n"
        f"Human Approval: "
        f"{row['task_human_approval']}"
    )

    return before, information

# ---------------------------------
# 7. Load saved synthetic demo
# ---------------------------------

demo_row = locations[
    locations["verification_id"]
    == "DH-CL-001"
].iloc[0]

DEMO_LABEL = next(
    key for key, row in location_map.items()
    if row["verification_id"] == "DH-CL-001"
)

def load_saved_demo():

    demo_image = read_rgb(
        DEMO_IMAGE_PATH
    )

    return (
        DEMO_LABEL,
        demo_image,
        "SYNTHETIC_DEMO",
        True,
        "Saved synthetic cleanup loaded. "
        "This is NOT real field evidence."
    )

# ---------------------------------
# 8. Main AI verification function
# ---------------------------------

def analyze_after_image(
    selected_location,
    after_rgb,
    evidence_scope,
    same_view,
    notes
):

    if after_rgb is None:
        raise gr.Error(
            "Upload an After Image or load the demo."
        )

    row = location_map[selected_location]

    before_path = Path(
        row["before_image"]
    )

    before_mask_path = Path(
        row["mask_path"]
    )

    before_bgr = cv2.imread(
        str(before_path)
    )

    before_mask = cv2.imread(
        str(before_mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if before_bgr is None or before_mask is None:
        raise gr.Error(
            "Before image or baseline mask missing."
        )

    if (
        sha256(before_path)
        != row["before_image_sha256"]
    ):
        raise gr.Error(
            "Original Before Image hash mismatch."
        )

    after_rgb = np.asarray(after_rgb)

    if (
        after_rgb.ndim != 3
        or after_rgb.shape[2] not in (3, 4)
    ):
        raise gr.Error(
            "Please upload a color image."
        )

    if after_rgb.dtype != np.uint8:
        after_rgb = np.clip(
            after_rgb, 0, 255
        ).astype(np.uint8)

    if after_rgb.shape[2] == 4:
        after_bgr = cv2.cvtColor(
            after_rgb, cv2.COLOR_RGBA2BGR
        )
    else:
        after_bgr = cv2.cvtColor(
            after_rgb, cv2.COLOR_RGB2BGR
        )

    if (
        max(after_bgr.shape[:2]) > 4096
        or after_bgr.shape[0]
        * after_bgr.shape[1] > 12000000
    ):
        raise gr.Error(
            "Image too large. Use an image "
            "under 4096 px and 12 megapixels."
        )

    # Run the real trained YOLO model
    prediction = model.predict(
        source=after_bgr,
        imgsz=640,
        conf=0.25,
        retina_masks=True,
        save=False,
        verbose=False
    )[0]

    after_mask, detections, classes = (
        extract_mask(prediction)
    )

    after_pct = round(
        float(
            100 * np.count_nonzero(after_mask)
            / after_mask.size
        ),
        4
    )

    # Human assertion, not automatic
    # geometric alignment verification.
    dimensions_match = (
        before_mask.shape == after_mask.shape
    )

    alignment_asserted = (
        bool(same_view)
        and dimensions_match
    )

    if dimensions_match:

        comparison = engine.compare_object_masks(
            before_mask,
            after_mask,
            aligned=alignment_asserted
        )

    else:

        comparison = {
            "status": "ALIGNMENT_REQUIRED",
            "human_review": "REQUIRED",
            "real_world_verified": False
        }

    # Generate F5 recommendation only.
    # No automatic approval or dispatch.
    route = router.route_f7_to_f5(
        ai_status=comparison["status"],
        risk_category=row["risk_category"],
        evidence_scope=evidence_scope,
        after_image_available=True,
        alignment_confirmed=alignment_asserted,
        human_approval="PENDING",
        field_action_status="NOT_STARTED",
        human_review="PENDING"
    )

    if (
        evidence_scope == "UNVERIFIED_UPLOAD"
        and alignment_asserted
    ):
        route["workflow_status"] = (
            "UNVERIFIED_UPLOAD_REVIEW_REQUIRED"
        )

    # Unique output folder per submission
    submission_id = (
        datetime.now().strftime(
            "%Y%m%d_%H%M%S_%f"
        )
        + "_"
        + uuid.uuid4().hex[:8]
    )

    submission_dir = (
        RUN_DIR / f"Submission_{submission_id}"
    )

    submission_dir.mkdir(
        parents=True,
        exist_ok=False
    )

    after_path = (
        submission_dir / "after_uploaded.png"
    )

    mask_path = (
        submission_dir / "after_ai_mask.png"
    )

    overlay_path = (
        submission_dir / "after_ai_overlay.png"
    )

    cv2.imwrite(
        str(after_path),
        after_bgr
    )

    cv2.imwrite(
        str(mask_path),
        after_mask
    )

    overlay = make_overlay(
        after_bgr,
        after_mask
    )

    cv2.imwrite(
        str(overlay_path),
        overlay
    )

    # Human review remains required
    summary = {
        "verification_id":
            str(row["verification_id"]),
        "f7_location_id":
            str(row["f7_location_id"]),
        "f5_location_id":
            str(row["f5_location_id"]),
        "task_id":
            str(row["task_id"]),
        "risk_category":
            str(row["risk_category"]),
        "evidence_scope":
            evidence_scope,
        "before_object_mask_pct":
            float(row["object_mask_pct"]),
        "after_object_mask_pct":
            after_pct,
        "after_ai_detections":
            int(detections),
        "after_detected_classes":
            classes,
        "comparison_status":
            comparison["status"],
        "comparison_details":
            comparison,
        "same_view_user_asserted":
            bool(same_view),
        "image_dimensions_match":
            bool(dimensions_match),
        "geometric_alignment_verified":
            False,
        "f5_revisit_action":
            route["f5_revisit_action"],
        "workflow_status":
            route["workflow_status"],
        "human_review":
            "PENDING",
        "field_action":
            "NOT_EXECUTED_BY_THIS_APP",
        "drone_dispatched":
            False,
        "real_elimination_verified":
            False,
        "direct_water_pixel_measurement":
            False,
        "after_image_sha256":
            sha256(after_path),
        "notes":
            str(notes or ""),
        "submission_id":
            submission_id
    }

    summary_path = (
        submission_dir / "verification_result.json"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False
        )

    pd.DataFrame([{
        "verification_id":
            summary["verification_id"],
        "f7_location_id":
            summary["f7_location_id"],
        "task_id":
            summary["task_id"],
        "evidence_scope":
            evidence_scope,
        "before_object_mask_pct":
            summary["before_object_mask_pct"],
        "after_object_mask_pct":
            after_pct,
        "comparison_status":
            comparison["status"],
        "f5_revisit_action":
            route["f5_revisit_action"],
        "human_review":
            "PENDING",
        "real_elimination_verified":
            False
    }]).to_csv(
        submission_dir / "verification_result.csv",
        index=False
    )

    sources_unchanged = all(
        sha256(Path(p)) == original_hash
        for p, original_hash
        in source_hashes.items()
    )

    if not sources_unchanged:
        raise gr.Error(
            "Source integrity check failed."
        )

    message = (
        f"AI INFERENCE COMPLETE\n"
        f"Location: {row['f7_location_id']}\n"
        f"Before Mask: "
        f"{row['object_mask_pct']}%\n"
        f"After Mask: {after_pct}%\n"
        f"AI Result: {comparison['status']}\n"
        f"F5 Recommendation: "
        f"{route['f5_revisit_action']}\n"
        f"Human Review: REQUIRED\n"
        f"Real Elimination Verified: NO\n"
        f"Saved Submission: {submission_id}"
    )

    print(
        "STEP 9.3 SUBMISSION SAVED:",
        submission_dir
    )

    return (
        cv2.cvtColor(
            before_bgr,
            cv2.COLOR_BGR2RGB
        ),
        cv2.cvtColor(
            overlay,
            cv2.COLOR_BGR2RGB
        ),
        summary,
        message
    )

# ---------------------------------
# 9. Gradio interactive interface
# ---------------------------------

initial_before, initial_info = (
    show_location(choices[0])
)

with gr.Blocks(
    title="DENGU HUNTER | F7 Verification"
) as app:

    gr.Markdown(
        "# 🦟 DENGU HUNTER — F7"
        "\n### AI Before–After Verification"
        "\n**Prototype:** Human review required. "
        "No automatic field verification."
    )

    location_input = gr.Dropdown(
        choices=choices,
        value=choices[0],
        label="Select Location"
    )

    with gr.Row():

        before_view = gr.Image(
            value=initial_before,
            label="Original Before Image",
            interactive=False
        )

        after_input = gr.Image(
            label="Upload After Image",
            type="numpy",
            sources=["upload"]
        )

    before_info = gr.Textbox(
        value=initial_info,
        label="Location Information",
        lines=7,
        interactive=False
    )

    evidence_input = gr.Radio(
        choices=[
            "SYNTHETIC_DEMO",
            "UNVERIFIED_UPLOAD"
        ],
        value="UNVERIFIED_UPLOAD",
        label="Evidence Type"
    )

    same_view_input = gr.Checkbox(
        value=False,
        label=(
            "I confirm both images show the "
            "same scene with comparable framing "
            "(manual assertion only)"
        )
    )

    notes_input = gr.Textbox(
        label="Notes (optional)",
        placeholder="Describe the uploaded image"
    )

    demo_button = gr.Button(
        "Load Saved Synthetic Cleanup Demo"
    )

    demo_message = gr.Textbox(
        label="Demo Status",
        interactive=False
    )

    analyze_button = gr.Button(
        "Run AI Verification",
        variant="primary"
    )

    with gr.Row():

        result_before = gr.Image(
            label="Before Image",
            interactive=False
        )

        result_after = gr.Image(
            label="After Image + AI Mask",
            interactive=False
        )

    result_json = gr.JSON(
        label="AI Verification + F5 Decision"
    )

    result_text = gr.Textbox(
        label="Verification Summary",
        lines=9,
        interactive=False
    )

    location_input.change(
        fn=show_location,
        inputs=location_input,
        outputs=[
            before_view,
            before_info
        ]
    )

    demo_button.click(
        fn=load_saved_demo,
        outputs=[
            location_input,
            after_input,
            evidence_input,
            same_view_input,
            demo_message
        ]
    )

    analyze_button.click(
        fn=analyze_after_image,
        inputs=[
            location_input,
            after_input,
            evidence_input,
            same_view_input,
            notes_input
        ],
        outputs=[
            result_before,
            result_after,
            result_json,
            result_text
        ]
    )

# ---------------------------------
# 10. Launch protected demo
# ---------------------------------

password = secrets.token_urlsafe(12)

print("\n=== STEP 9.3 INITIAL CHECK ===")
print("15 Locations Linked:", len(locations) == 15)
print("YOLO Model Loaded:", model.task == "segment")
print("Comparison Engine: READY")
print("F5 Router: READY")
print("Original Files Modified: NO")
print("Real Field Verification: NO")
print("New Upload Folder:", RUN_DIR)

print("\n=== GRADIO LOGIN ===")
print("Username: boss")
print("Password:", password)
print("Keep this temporary link and password private.")

app.queue().launch(
    share=True,
    inline=False,
    auth=("boss", password),
    show_error=True
)

````

### Original cell 102/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json

print("=== DENGU HUNTER F7: STEP 9.4 ===")
print("=== LIVE AI PIPELINE VALIDATION ===")

# Reuse Step 9.3 functions and loaded model
required_names = [
    "analyze_after_image",
    "read_rgb",
    "DEMO_IMAGE_PATH",
    "DEMO_LABEL",
    "RUN_DIR",
    "locations",
    "model",
    "source_hashes",
    "sha256"
]

missing = [
    name for name in required_names
    if name not in globals()
]

assert not missing, (
    "Run Step 9.3 first. Missing: "
    + ", ".join(missing)
)

print("\n=== RUNNING SAVED SYNTHETIC DEMO ===")

demo_image = read_rgb(DEMO_IMAGE_PATH)

before_image, after_overlay, result, message = (
    analyze_after_image(
        selected_location=DEMO_LABEL,
        after_rgb=demo_image,
        evidence_scope="SYNTHETIC_DEMO",
        same_view=True,
        notes="Automated synthetic demo test"
    )
)

print("\n=== AI RESULT ===")
print(message)

print("\n=== F5 DECISION ===")
print("AI Status:", result["comparison_status"])
print("F5 Action:", result["f5_revisit_action"])
print("Workflow:", result["workflow_status"])
print("Human Review:", result["human_review"])

# Verify saved output
submission_dir = (
    RUN_DIR
    / f"Submission_{result['submission_id']}"
)

required_outputs = [
    submission_dir / "after_uploaded.png",
    submission_dir / "after_ai_mask.png",
    submission_dir / "after_ai_overlay.png",
    submission_dir / "verification_result.json",
    submission_dir / "verification_result.csv"
]

with open(
    submission_dir / "verification_result.json",
    encoding="utf-8"
) as f:
    saved_result = json.load(f)

tests = {
    "15 Locations Available":
        len(locations) == 15,

    "YOLO Segmentation Loaded":
        model.task == "segment",

    "Synthetic Demo Correctly Labeled":
        result["evidence_scope"]
        == "SYNTHETIC_DEMO",

    "Before Object Detected":
        result["before_object_mask_pct"] > 0,

    "After Image Analyzed":
        "after_ai_detections" in result,

    "AI Comparison Completed":
        result["comparison_status"]
        == "TARGET_NOT_DETECTED_AFTER",

    "F5 Recommendation Generated":
        result["f5_revisit_action"]
        == "REQUEST_HUMAN_VISUAL_REVIEW",

    "Human Review Pending":
        result["human_review"] == "PENDING",

    "No Real Elimination Claimed":
        result["real_elimination_verified"] is False,

    "No Drone Dispatch Claimed":
        result["drone_dispatched"] is False,

    "Five Result Files Saved":
        all(p.is_file() for p in required_outputs),

    "Saved JSON Matches Result":
        saved_result["submission_id"]
        == result["submission_id"],

    "Original Sources Unchanged":
        all(
            sha256(Path(p)) == original_hash
            for p, original_hash
            in source_hashes.items()
        )
}

print("\n=== TEST RESULTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

passed_count = sum(bool(v) for v in tests.values())

print("\nTESTS:", passed_count, "/", len(tests))
print("OUTPUT:", submission_dir)

if all(tests.values()):
    print("\nF7 STEP 9.4 COMPLETE")
else:
    print("\nF7 STEP 9.4 NEEDS REVIEW")

````

### Original cell 103/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime, timezone
import json
import uuid
import secrets
import gradio as gr

print("=== DENGU HUNTER F7: STEP 10 ===")
print("=== HUMAN REVIEW INTEGRATION ===")

# Reuse the tested Step 9.3 and 9.4 pipeline.
required_names = [
    "analyze_after_image",
    "load_saved_demo",
    "show_location",
    "read_rgb",
    "locations",
    "choices",
    "source_hashes",
    "sha256",
    "F7",
    "RUN_DIR",
    "app"
]

missing = [
    name for name in required_names
    if name not in globals()
]

assert not missing, (
    "Run Step 9.3 and 9.4 first. Missing: "
    + ", ".join(missing)
)

UPLOAD_ROOT = (
    F7 / "F7_Interactive_Uploads"
).resolve()

REVIEW_DIR = (
    RUN_DIR / "Human_Review_Events"
)

REVIEW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------
# 1. Find saved AI submissions
# ---------------------------------

def get_submissions():

    files = sorted(
        UPLOAD_ROOT.glob(
            "Run_*/Submission_*/verification_result.json"
        ),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    options = []

    for path in files:

        try:
            with open(
                path, encoding="utf-8"
            ) as f:
                data = json.load(f)

            label = (
                f"{data['f7_location_id']} | "
                f"{data['evidence_scope']} | "
                f"{path.parent.name}"
            )

            options.append(
                (label, str(path))
            )

        except (OSError, ValueError, KeyError):
            continue

    return options


def refresh_submissions():

    options = get_submissions()

    return gr.update(
        choices=options,
        value=(
            options[0][1]
            if options else None
        )
    )


# ---------------------------------
# 2. Validate saved submission
# ---------------------------------

def validated_submission(selected_path):

    if not selected_path:
        raise gr.Error(
            "Select an AI submission first."
        )

    path = Path(selected_path).resolve()

    if (
        not path.is_relative_to(UPLOAD_ROOT)
        or path.name != "verification_result.json"
        or not path.parent.name.startswith("Submission_")
        or not path.parent.parent.name.startswith("Run_")
        or not path.is_file()
    ):
        raise gr.Error(
            "Invalid submission location."
        )

    with open(
        path, encoding="utf-8"
    ) as f:
        data = json.load(f)

    matching = locations[
        locations["verification_id"]
        == data["verification_id"]
    ]

    if len(matching) != 1:
        raise gr.Error(
            "Verification ID not found."
        )

    row = matching.iloc[0]

    if (
        data["f7_location_id"]
        != row["f7_location_id"]
        or data["task_id"]
        != row["task_id"]
    ):
        raise gr.Error(
            "Location or task mismatch."
        )

    before_path = Path(
        row["before_image"]
    )

    after_path = (
        path.parent / "after_uploaded.png"
    )

    overlay_path = (
        path.parent / "after_ai_overlay.png"
    )

    if not all(
        p.is_file()
        for p in [
            before_path,
            after_path,
            overlay_path
        ]
    ):
        raise gr.Error(
            "Submission image files missing."
        )

    if (
        sha256(after_path)
        != data["after_image_sha256"]
    ):
        raise gr.Error(
            "After Image integrity check failed."
        )

    if (
        sha256(before_path)
        != row["before_image_sha256"]
    ):
        raise gr.Error(
            "Before Image integrity check failed."
        )

    return path, data, row, (
        before_path,
        after_path,
        overlay_path
    )


# ---------------------------------
# 3. Display AI evidence
# ---------------------------------

def preview_submission(selected_path):

    path, data, row, images = (
        validated_submission(selected_path)
    )

    message = (
        f"Location: {data['f7_location_id']}\n"
        f"Task: {data['task_id']}\n"
        f"Evidence: {data['evidence_scope']}\n"
        f"AI Result: {data['comparison_status']}\n"
        f"F5 Action: {data['f5_revisit_action']}\n"
        f"Human Review: PENDING\n"
        f"Field Verification: NOT CLAIMED"
    )

    return (
        read_rgb(images[0]),
        read_rgb(images[1]),
        read_rgb(images[2]),
        data,
        message
    )


# ---------------------------------
# 4. Human review decisions
# ---------------------------------

REVIEW_DECISIONS = [
    "CONFIRM_AI_RESULT",
    "REQUEST_REINSPECTION",
    "REJECT_AI_RESULT"
]


def save_human_review(
    selected_path,
    decision,
    reviewed_checkbox,
    notes
):

    if not reviewed_checkbox:
        raise gr.Error(
            "Review the images and check the "
            "confirmation box first."
        )

    if decision not in REVIEW_DECISIONS:
        raise gr.Error(
            "Choose a valid review decision."
        )

    if (
        decision != "CONFIRM_AI_RESULT"
        and not str(notes or "").strip()
    ):
        raise gr.Error(
            "Please explain why you are "
            "rejecting or requesting reinspection."
        )

    path, data, row, images = (
        validated_submission(selected_path)
    )

    if decision == "CONFIRM_AI_RESULT":

        action = data["f5_revisit_action"]

        review_status = (
            "AI_RESULT_REVIEWED"
        )

    elif decision == "REQUEST_REINSPECTION":

        action = (
            "REQUEST_PRIORITY_REVISIT_APPROVAL"
        )

        review_status = (
            "REINSPECTION_REQUESTED"
        )

    else:

        action = (
            "MANUAL_REASSESSMENT_REQUIRED"
        )

        review_status = (
            "AI_RESULT_REJECTED"
        )

    if data["evidence_scope"] == "SYNTHETIC_DEMO":
        workflow_status = (
            "DEMO_HUMAN_REVIEW_RECORDED"
        )
    else:
        workflow_status = (
            "UNVERIFIED_EVIDENCE_REVIEW_RECORDED"
        )

    review_id = (
        datetime.now(timezone.utc).strftime(
            "%Y%m%d_%H%M%S_%f"
        )
        + "_"
        + uuid.uuid4().hex[:8]
    )

    event = {
        "review_id": review_id,
        "reviewed_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "verification_id":
            data["verification_id"],
        "f7_location_id":
            data["f7_location_id"],
        "f5_location_id":
            data["f5_location_id"],
        "task_id":
            data["task_id"],
        "submission_id":
            data["submission_id"],
        "source_submission":
            str(path),
        "source_submission_sha256":
            sha256(path),
        "after_image_sha256":
            data["after_image_sha256"],
        "evidence_scope":
            data["evidence_scope"],
        "ai_comparison_status":
            data["comparison_status"],
        "review_decision":
            decision,
        "review_status":
            review_status,
        "workflow_status":
            workflow_status,
        "f5_revisit_recommendation":
            action,
        "reviewer_role":
            "DEMO_OPERATOR",
        "reviewer_notes":
            str(notes or ""),
        "ai_result_human_reviewed":
            True,
        "real_field_elimination_verified":
            False,
        "field_action_approved":
            False,
        "drone_dispatched":
            False,
        "original_submission_modified":
            False
    }

    event_path = (
        REVIEW_DIR
        / f"Review_{review_id}.json"
    )

    # Exclusive creation: no overwriting.
    with open(
        event_path,
        "x",
        encoding="utf-8"
    ) as f:
        json.dump(
            event,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        "HUMAN REVIEW EVENT SAVED:",
        event_path
    )

    summary = (
        f"Review Saved: YES\n"
        f"Location: {data['f7_location_id']}\n"
        f"Decision: {decision}\n"
        f"Workflow: {workflow_status}\n"
        f"F5 Recommendation: {action}\n"
        f"Real Field Verification: NO\n"
        f"Drone Dispatched: NO\n"
        f"Review ID: {review_id}"
    )

    return event, summary


# ---------------------------------
# 5. Build one combined F7 dashboard
# ---------------------------------

options = get_submissions()

assert len(options) >= 1, (
    "No Step 9.4 AI submission found."
)

new_password = secrets.token_urlsafe(12)

with gr.Blocks(
    title="DENGU HUNTER — F7 Complete Workflow"
) as app_v2:

    gr.Markdown(
        "# 🦟 DENGU HUNTER — F7"
        "\n### Closed-Loop AI Verification"
        "\nAI + Human Review + F5 Revisit"
        "\n\n**Prototype:** No automatic "
        "field approval or drone dispatch."
    )

    with gr.Tab("1. AI Verification"):

        location_input = gr.Dropdown(
            choices=choices,
            value=choices[0],
            label="Select Location"
        )

        with gr.Row():

            before_view = gr.Image(
                label="Before Image",
                interactive=False
            )

            after_input = gr.Image(
                label="Upload After Image",
                type="numpy",
                sources=["upload"]
            )

        before_info = gr.Textbox(
            label="Location Information",
            lines=7,
            interactive=False
        )

        evidence_input = gr.Radio(
            choices=[
                "SYNTHETIC_DEMO",
                "UNVERIFIED_UPLOAD"
            ],
            value="UNVERIFIED_UPLOAD",
            label="Evidence Type"
        )

        same_view_input = gr.Checkbox(
            value=False,
            label=(
                "I confirm comparable framing "
                "(manual assertion only)"
            )
        )

        notes_input = gr.Textbox(
            label="Image Notes"
        )

        demo_button = gr.Button(
            "Load Saved Synthetic Cleanup Demo"
        )

        demo_message = gr.Textbox(
            label="Demo Status",
            interactive=False
        )

        analyze_button = gr.Button(
            "Run AI Verification",
            variant="primary"
        )

        with gr.Row():

            result_before = gr.Image(
                label="Before",
                interactive=False
            )

            result_after = gr.Image(
                label="After AI Overlay",
                interactive=False
            )

        result_json = gr.JSON(
            label="AI + F5 Decision"
        )

        result_text = gr.Textbox(
            label="AI Verification Summary",
            lines=9
        )

        location_input.change(
            fn=show_location,
            inputs=location_input,
            outputs=[
                before_view,
                before_info
            ]
        )

        demo_button.click(
            fn=load_saved_demo,
            outputs=[
                location_input,
                after_input,
                evidence_input,
                same_view_input,
                demo_message
            ]
        )

        analyze_button.click(
            fn=analyze_after_image,
            inputs=[
                location_input,
                after_input,
                evidence_input,
                same_view_input,
                notes_input
            ],
            outputs=[
                result_before,
                result_after,
                result_json,
                result_text
            ]
        )

    with gr.Tab("2. Human Review"):

        gr.Markdown(
            "### Review saved AI results"
            "\nConfirming an AI result does "
            "**not** verify real elimination."
        )

        refresh_button = gr.Button(
            "Refresh AI Submissions"
        )

        review_selection = gr.Dropdown(
            choices=options,
            value=options[0][1],
            label="Select Saved AI Submission"
        )

        preview_button = gr.Button(
            "Load AI Evidence"
        )

        with gr.Row():

            review_before = gr.Image(
                label="Before",
                interactive=False
            )

            review_after = gr.Image(
                label="Uploaded After",
                interactive=False
            )

            review_overlay = gr.Image(
                label="AI Overlay",
                interactive=False
            )

        review_data = gr.JSON(
            label="Original AI Result"
        )

        review_info = gr.Textbox(
            label="Review Information",
            lines=8
        )

        decision_input = gr.Radio(
            choices=REVIEW_DECISIONS,
            value="CONFIRM_AI_RESULT",
            label="Human Review Decision"
        )

        review_notes = gr.Textbox(
            label="Reviewer Notes"
        )

        reviewed_checkbox = gr.Checkbox(
            value=False,
            label=(
                "I have inspected the images "
                "and AI comparison result."
            )
        )

        save_button = gr.Button(
            "Save Human Review",
            variant="primary"
        )

        review_result = gr.JSON(
            label="Saved Review Event"
        )

        review_summary = gr.Textbox(
            label="Review Summary",
            lines=8
        )

        refresh_button.click(
            fn=refresh_submissions,
            outputs=review_selection
        )

        preview_button.click(
            fn=preview_submission,
            inputs=review_selection,
            outputs=[
                review_before,
                review_after,
                review_overlay,
                review_data,
                review_info
            ]
        )

        save_button.click(
            fn=save_human_review,
            inputs=[
                review_selection,
                decision_input,
                reviewed_checkbox,
                review_notes
            ],
            outputs=[
                review_result,
                review_summary
            ]
        )

# ---------------------------------
# 6. Structural validation
# ---------------------------------

test_path, test_data, test_row, test_images = (
    validated_submission(options[0][1])
)

tests = {
    "15 Locations Linked":
        len(locations) == 15,

    "AI Submission Available":
        len(options) >= 1,

    "Saved Before Image Exists":
        test_images[0].is_file(),

    "Saved After Image Exists":
        test_images[1].is_file(),

    "AI Overlay Exists":
        test_images[2].is_file(),

    "Submission Hash Valid":
        sha256(test_images[1])
        == test_data["after_image_sha256"],

    "Human Review Initially Pending":
        test_data["human_review"] == "PENDING",

    "No Real Elimination Claimed":
        test_data["real_elimination_verified"] is False,

    "No Drone Dispatch Claimed":
        test_data["drone_dispatched"] is False,

    "Original Sources Unchanged":
        all(
            sha256(Path(p)) == original_hash
            for p, original_hash
            in source_hashes.items()
        )
}

print("\n=== STEP 10 STRUCTURAL TESTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "TESTS:",
    sum(tests.values()),
    "/", len(tests)
)

assert all(tests.values()), (
    "Step 10 structural tests failed."
)

print("\n=== NEW DASHBOARD LOGIN ===")
print("Username: boss")
print("Password:", new_password)
print("Keep the login private.")
print("\nReview Events Folder:", REVIEW_DIR)

# Launch new combined dashboard
app_v2.queue().launch(
    share=True,
    inline=False,
    auth=("boss", new_password),
    show_error=True
)

# Close the older dashboard only
# after the new one launches.
try:
    app.close()
except Exception:
    pass

print("\nF7 STEP 10 DASHBOARD READY")
print("Human Review Save Test: PENDING")

````

### Original cell 104/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import json
import uuid

print("=== DENGU HUNTER F7: STEP 11 ===")
print("=== FINAL STANDALONE VALIDATION ===")

# Reuse completed F7 work
required = [
    "F7",
    "locations",
    "model",
    "source_hashes",
    "sha256",
    "REVIEW_DIR",
    "get_submissions",
    "validated_submission"
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Missing previous F7 variables: "
    + ", ".join(missing)
)

F7_ROOT = Path(F7)
REVIEW_ROOT = Path(REVIEW_DIR)

# 1. Load saved human review
review_files = sorted(
    REVIEW_ROOT.glob("Review_*.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

assert review_files, "No saved Human Review found"

review_path = review_files[0]

with open(review_path, encoding="utf-8") as f:
    review = json.load(f)

# 2. Validate linked AI submission
submission_path, ai, location, images = (
    validated_submission(
        review["source_submission"]
    )
)

# 3. Load F5 pending registry
registry_files = sorted(
    (
        F7_ROOT / "F7_F5_Integration"
    ).glob(
        "Integration_*/F7_F5_Real_Pending_Registry.csv"
    )
)

assert registry_files, (
    "F5 integration registry not found"
)

registry = pd.read_csv(registry_files[-1])

def all_false(series):
    return series.astype(str).str.lower().isin(
        ["false", "0", "no"]
    ).all()

# 4. Final regression tests
tests = {
    "15 Locations Linked":
        len(locations) == 15,

    "Unique Verification IDs":
        locations["verification_id"].nunique() == 15,

    "Unique Location IDs":
        locations["f7_location_id"].nunique() == 15,

    "Segmentation Model Loaded":
        model.task == "segment",

    "AI Submission Available":
        len(get_submissions()) >= 1,

    "Saved Review Exists":
        review_path.is_file(),

    "Review Decision Correct":
        review["review_decision"]
        == "CONFIRM_AI_RESULT",

    "Review Status Correct":
        review["review_status"]
        == "AI_RESULT_REVIEWED",

    "Synthetic Evidence Preserved":
        review["evidence_scope"]
        == ai["evidence_scope"]
        == "SYNTHETIC_DEMO",

    "Verification ID Matches":
        review["verification_id"]
        == ai["verification_id"],

    "Location ID Matches":
        review["f7_location_id"]
        == ai["f7_location_id"],

    "F5 Location Matches":
        review["f5_location_id"]
        == ai["f5_location_id"],

    "Task ID Matches":
        review["task_id"]
        == ai["task_id"],

    "Submission ID Matches":
        review["submission_id"]
        == ai["submission_id"],

    "Original AI JSON Unchanged":
        sha256(submission_path)
        == review["source_submission_sha256"],

    "After Image Hash Matches":
        review["after_image_sha256"]
        == ai["after_image_sha256"],

    "AI Comparison Preserved":
        review["ai_comparison_status"]
        == ai["comparison_status"],

    "Human Review Recorded":
        review["ai_result_human_reviewed"] is True,

    "Demo Workflow Correct":
        review["workflow_status"]
        == "DEMO_HUMAN_REVIEW_RECORDED",

    "No Real Elimination Claimed":
        review["real_field_elimination_verified"]
        is False,

    "No Field Approval Claimed":
        review["field_action_approved"]
        is False,

    "No Drone Dispatch Claimed":
        review["drone_dispatched"]
        is False,

    "No Direct Water Pixel Claim":
        ai["direct_water_pixel_measurement"]
        is False,

    "No Geometric Alignment Claim":
        ai["geometric_alignment_verified"]
        is False,

    "F5 Registry Has 15 Tasks":
        len(registry) == 15,

    "F5 Registry No Real Verification":
        all_false(
            registry["real_world_elimination_verified"]
        ),

    "F5 Registry No Drone Dispatch":
        all_false(
            registry["drone_dispatched"]
        ),

    "Original Source Files Unchanged":
        all(
            Path(p).is_file()
            and sha256(Path(p)) == original_hash
            for p, original_hash
            in source_hashes.items()
        )
}

print("\n=== FINAL TEST RESULTS ===")

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

passed_count = sum(bool(x) for x in tests.values())
total_count = len(tests)

print("\nTOTAL:", passed_count, "/", total_count)

# 5. Save final report only if all tests pass
if all(tests.values()):

    final_id = (
        datetime.now(timezone.utc).strftime(
            "%Y%m%d_%H%M%S_%f"
        )
        + "_"
        + uuid.uuid4().hex[:6]
    )

    output_dir = (
        F7_ROOT
        / "F7_Final_Validation"
        / f"Final_{final_id}"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=False
    )

    summary = {
        "feature": "F7_Closed_Loop_Verification",
        "prototype_status":
            "READY_FOR_HACKATHON_DEMO",
        "scope":
            "SIMULATION_AND_SYNTHETIC_DEMO",
        "tests_passed": passed_count,
        "tests_total": total_count,
        "human_review_saved": True,
        "f5_integration": "VALIDATED",
        "real_field_verification": False,
        "real_drone_operation": False,
        "direct_water_pixel_verification": False,
        "source_files_unchanged": True,
        "tests": {
            name: bool(value)
            for name, value in tests.items()
        },
        "review_file": str(review_path),
        "created_at_utc":
            datetime.now(timezone.utc).isoformat()
    }

    report_path = (
        output_dir / "F7_Final_Report.json"
    )

    with open(
        report_path,
        "x",
        encoding="utf-8"
    ) as f:
        json.dump(
            summary,
            f,
            indent=2,
            ensure_ascii=False
        )

    with open(
        report_path,
        encoding="utf-8"
    ) as f:
        saved = json.load(f)

    assert saved["tests_passed"] == total_count

    print("\nFINAL REPORT SAVED:", report_path)
    print("\nF7 STANDALONE PROTOTYPE: COMPLETE")
    print("HACKATHON DEMO: READY")
    print("REAL FIELD VERIFICATION: NO")
    print("NEXT FEATURE: F8")

else:
    print("\nF7 FINAL VALIDATION: NEEDS REVIEW")
    print("Fix failed tests before marking complete.")

````

### Original cell 105/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import os
import pandas as pd

print("=== DENGU HUNTER F8: STEP 1 ===")
print("=== EXISTING ASSET DISCOVERY ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

assert ROOT.exists(), "Mount Google Drive first"

keywords = [
    "feature_8", "f8_", "community",
    "government", "dashboard", "citizen",
    "worker", "feature_4", "f4_",
    "feature_6", "f6_", "hotspot",
    "elimination", "risk_prediction"
]

skip = {
    "images", "masks", "overlays",
    "__pycache__", ".ipynb_checkpoints",
    "f7_interactive_uploads",
    "f5_advanced_enhancements"
}

found = []

for current, dirs, files in os.walk(ROOT):
    path = Path(current)
    depth = len(path.relative_to(ROOT).parts)

    dirs[:] = [
        d for d in dirs
        if d.lower() not in skip
    ]

    if depth >= 6:
        dirs[:] = []

    for filename in files:
        file = path / filename

        if file.suffix.lower() not in {
            ".csv", ".json", ".html", ".py"
        }:
            continue

        relative = str(file.relative_to(ROOT)).lower()

        if any(k in relative for k in keywords):
            found.append(file)

print("\n=== EXISTING F4 / F6 / F8 FILES ===")

for file in sorted(found)[:60]:
    print(file.relative_to(ROOT))

print("\nTotal Matching Files:", len(found))

print("\n=== CSV STRUCTURES ===")

csv_files = [
    p for p in found
    if p.suffix.lower() == ".csv"
]

for file in sorted(csv_files)[:15]:
    try:
        df = pd.read_csv(file, nrows=2)

        print("\nFILE:", file.relative_to(ROOT))
        print("COLUMNS:", list(df.columns))

    except Exception as e:
        print("READ ERROR:", str(e)[:100])

print("\n=== F7 TASK REGISTRY ===")

registries = sorted(
    (
        ROOT / "Feature_7_Closed_Loop"
        / "F7_F5_Integration"
    ).glob(
        "Integration_*/F7_F5_Real_Pending_Registry.csv"
    )
)

if registries:
    df = pd.read_csv(registries[-1])
    print("Found:", len(df), "tasks")
    print("Columns:", list(df.columns))
else:
    print("Not found")

print("\n=== F7 HUMAN REVIEWS ===")

reviews = list(
    (
        ROOT / "Feature_7_Closed_Loop"
        / "F7_Interactive_Uploads"
    ).glob(
        "Run_*/Human_Review_Events/Review_*.json"
    )
)

print("Saved Review Events:", len(reviews))

print("\nOriginal Files Modified: NO")
print("F8 STEP 1 DISCOVERY COMPLETE")

````

### Original cell 106/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
import pandas as pd
import json
import re

print("=== DENGU HUNTER F8: STEP 2 ===")
print("=== EXISTING DATA & INTEGRATION CHECK ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F8 = ROOT / "Feature_8_Community_Government"
BATCH = ROOT / "Development_Batch_Features_3_4_6_8"

registry_files = sorted(
    (ROOT / "Feature_7_Closed_Loop"
     / "F7_F5_Integration").glob(
        "Integration_*/F7_F5_Real_Pending_Registry.csv"
    )
)

assert registry_files, "F7 Registry missing"

paths = {
    "F8 Citizen Reports":
        F8 / "community_reporting_template.csv",

    "F8 Government Tasks":
        F8 / "government_task_register.csv",

    "F4 Risk Prediction":
        BATCH / "predictive_hotspot_demo.csv",

    "F6 Elimination":
        BATCH / "smart_elimination_recommendations.csv",

    "F7 Verified Registry":
        registry_files[-1]
}

data = {}

print("\n=== 1. DATA FILES ===")

for name, path in paths.items():

    if not path.is_file():
        print("MISSING:", name)
        continue

    df = pd.read_csv(path)
    data[name] = df

    print("\nFILE:", name)
    print("ROWS:", len(df))
    print("COLUMNS:", list(df.columns))

    if len(df):
        print("SAMPLE:")
        print(
            df.head(2).to_string(
                index=False,
                max_cols=12,
                max_colwidth=28
            )
        )

print("\n=== 2. EXISTING GOVERNMENT DASHBOARD ===")

html_path = (
    F8 / "DenguHunter_Government_Dashboard.html"
)

if html_path.is_file():

    html = html_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    title = re.search(
        r"<title[^>]*>(.*?)</title>",
        html,
        flags=re.I | re.S
    )

    print(
        "HTML TITLE:",
        title.group(1).strip()
        if title else "Not specified"
    )

    print("HTML SIZE:", len(html), "characters")

    for keyword in [
        "citizen",
        "worker",
        "government",
        "upload",
        "risk",
        "task",
        "verification",
        "localStorage",
        "type=\"file\""
    ]:
        print(
            keyword,
            "FOUND" if keyword.lower() in html.lower()
            else "NOT FOUND"
        )

else:
    print("Government HTML not found")

print("\n=== 3. EXISTING METRICS ===")

metrics_path = F8 / "integrated_metrics.json"

if metrics_path.is_file():

    with open(
        metrics_path,
        encoding="utf-8"
    ) as f:
        metrics = json.load(f)

    print(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
            default=str
        )[:3500]
    )

else:
    print("Metrics JSON not found")

print("\n=== 4. CROSS-FEATURE ID MATCHING ===")

def id_set(series):
    return set(
        series.dropna()
        .astype(str)
        .str.strip()
    ) - {""}

def image_set(series):
    return {
        Path(str(x)).name.lower()
        for x in series.dropna()
        if str(x).strip()
    }

f7 = data.get("F7 Verified Registry")
f4 = data.get("F4 Risk Prediction")
f6 = data.get("F6 Elimination")
f8_tasks = data.get("F8 Government Tasks")

if f7 is not None:

    print("F7 Task IDs:", f7["task_id"].nunique())

    if f8_tasks is not None:

        common_tasks = (
            id_set(f7["task_id"])
            & id_set(f8_tasks["task_id"])
        )

        print(
            "F8 ↔ F7 Matching Task IDs:",
            len(common_tasks)
        )

    if f4 is not None:

        f4_ids = id_set(f4["location_id"])

        for column in [
            "f5_location_id",
            "f7_location_id"
        ]:

            common = (
                f4_ids & id_set(f7[column])
            )

            print(
                "F4 ↔ F7",
                column,
                "matches:",
                len(common)
            )

        common_images = (
            image_set(f4["associated_image"])
            & image_set(f7["before_image"])
        )

        print(
            "F4 ↔ F7 Matching Image Names:",
            len(common_images)
        )

    if f6 is not None:

        common_images = (
            image_set(f6["image"])
            & image_set(f7["before_image"])
        )

        print(
            "F6 ↔ F7 Matching Image Names:",
            len(common_images)
        )

print("\n=== 5. EXISTING AI MODEL ===")

model_candidates = [
    ROOT / "Models/Detection_Model/best.pt",
    ROOT / "Models/Water_Segmentation_Model/best.pt",
    Path(
        "/content/drive/MyDrive/"
        "DENGUE_HUNTER_RESULTS/"
        "Larvae_Model/DengueHunter_Larvae_v1/"
        "weights/best.pt"
    )
]

for path in model_candidates:
    print(
        "FOUND" if path.is_file() else "MISSING",
        "-",
        path.name,
        "-",
        str(path.parent)
    )

print("\n=== 6. HUMAN REVIEW STATUS ===")

review_files = sorted(
    (ROOT / "Feature_7_Closed_Loop"
     / "F7_Interactive_Uploads").glob(
        "Run_*/Human_Review_Events/Review_*.json"
    )
)

print("Human Review Events:", len(review_files))

if review_files:

    with open(
        review_files[-1],
        encoding="utf-8"
    ) as f:
        review = json.load(f)

    print(
        "Latest Decision:",
        review.get("review_decision")
    )

    print(
        "Evidence:",
        review.get("evidence_scope")
    )

print("\n=== STEP 2 SUMMARY ===")
print("Original Files Modified: NO")
print("Data Mapping Inspected: YES")
print("F8 STEP 2 COMPLETE")

````

### Original cell 107/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json
import uuid

print("=== DENGU HUNTER F8: STEP 3 ===")
print("=== SAFE CROSS-FEATURE DATA INTEGRATION ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F8 = ROOT / "Feature_8_Community_Government"
BATCH = ROOT / "Development_Batch_Features_3_4_6_8"

registries = sorted(
    (ROOT / "Feature_7_Closed_Loop"
     / "F7_F5_Integration").glob(
        "Integration_*/F7_F5_Real_Pending_Registry.csv"
    )
)

assert registries, "F7 registry missing"

sources = {
    "F8": F8 / "government_task_register.csv",
    "F4": BATCH / "predictive_hotspot_demo.csv",
    "F6": BATCH / "smart_elimination_recommendations.csv",
    "F7": registries[-1],
    "Citizen": F8 / "community_reporting_template.csv"
}

assert all(p.is_file() for p in sources.values())

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

original_hashes = {
    name: sha256(path)
    for name, path in sources.items()
}

def image_key(value):
    if pd.isna(value):
        return ""
    return Path(str(value).strip()).name.lower()

# Load source files
gov = pd.read_csv(sources["F8"])
f4 = pd.read_csv(sources["F4"])
f6 = pd.read_csv(sources["F6"])
f7 = pd.read_csv(sources["F7"])
citizens = pd.read_csv(sources["Citizen"])

# Keep the two task-ID systems separate
gov = gov.rename(columns={
    "task_id": "government_task_id",
    "image": "government_image"
})

f7 = f7.rename(columns={
    "task_id": "f7_task_id"
})

f4 = f4.rename(columns={
    "associated_image": "f4_image",
    "location_id": "f4_location_id"
})

f6 = f6.rename(columns={
    "image": "f6_image"
})

gov["image_key"] = gov["government_image"].map(image_key)
f4["image_key"] = f4["f4_image"].map(image_key)
f6["image_key"] = f6["f6_image"].map(image_key)
f7["image_key"] = f7["before_image"].map(image_key)

print("\n=== SOURCE COUNTS ===")

for name, df in [
    ("F8 Government", gov),
    ("F4 Risk", f4),
    ("F6 Elimination", f6),
    ("F7 Verification", f7)
]:
    print(
        name,
        "| Rows:", len(df),
        "| Unique Images:", df["image_key"].nunique()
    )

# Check exact image matching before joining
sets = {
    "F8": set(gov["image_key"]),
    "F4": set(f4["image_key"]),
    "F6": set(f6["image_key"]),
    "F7": set(f7["image_key"])
}

print("\n=== IMAGE MATCH AUDIT ===")

for name in ["F4", "F6", "F7"]:
    common = sets["F8"] & sets[name]
    missing = sets["F8"] - sets[name]

    print(
        f"F8 ↔ {name}:",
        len(common), "/",
        len(sets["F8"])
    )

    if missing:
        print("Unmatched Images:", sorted(missing)[:5])

valid_keys = all(
    len(df) == 15
    and df["image_key"].nunique() == 15
    and df["image_key"].ne("").all()
    for df in [gov, f4, f6, f7]
)

full_match = (
    valid_keys
    and sets["F8"] == sets["F4"]
    == sets["F6"] == sets["F7"]
)

if not full_match:
    print("\nMAPPING NEEDS REVIEW")
    print("No integrated file created.")
    print("Original Files Modified: NO")

else:
    # One-to-one mapping by exact image filename
    merged = gov.merge(
        f4,
        on="image_key",
        how="left",
        validate="one_to_one"
    ).merge(
        f6,
        on="image_key",
        how="left",
        validate="one_to_one"
    ).merge(
        f7,
        on="image_key",
        how="left",
        validate="one_to_one"
    )

    # Independently confirm F4 location ID
    # against F7's mapped F5 location.
    location_match = (
        merged["f4_location_id"].astype(str)
        == merged["f5_location_id"].astype(str)
    ).all()

    print("\n=== LOCATION CROSS-CHECK ===")
    print("F4 ↔ F7 Location IDs:", location_match)

    if not location_match:
        print("LOCATION MISMATCH — NO FILE SAVED")

    else:
        merged["record_scope"] = (
            "HACKATHON_PROTOTYPE_SIMULATION"
        )

        merged["real_field_verified"] = False
        merged["citizen_report_linked"] = False

        # These are independent identifiers.
        # Do not replace one with another.
        print("\n=== LINKED TASK EXAMPLES ===")

        columns = [
            "government_task_id",
            "f7_task_id",
            "f7_location_id",
            "f5_location_id",
            "f4_location_id",
            "demonstration_risk_category",
            "recommended_action"
        ]

        print(
            merged[columns].head(5)
            .to_string(index=False)
        )

        def all_false(series):
            return series.astype(str).str.lower().isin(
                ["false", "0", "no"]
            ).all()

        tests = {
            "15 Unified Tasks": len(merged) == 15,
            "15 Unique Government Tasks":
                merged["government_task_id"].nunique() == 15,
            "15 Unique F7 Tasks":
                merged["f7_task_id"].nunique() == 15,
            "15 Unique Locations":
                merged["f7_location_id"].nunique() == 15,
            "F4 Location Mapping Correct":
                location_match,
            "F4 Risk Scores Available":
                merged["demonstration_risk_score"].notna().all(),
            "F6 Recommendations Available":
                merged["recommended_action"].notna().all(),
            "F7 Verification IDs Available":
                merged["verification_id"].notna().all(),
            "No Real Verification Claimed":
                all_false(
                    merged["real_world_elimination_verified"]
                ),
            "No Drone Dispatch Claimed":
                all_false(merged["drone_dispatched"]),
            "Citizen Reports Initially Empty":
                len(citizens) == 0,
            "Original Sources Unchanged":
                all(
                    sha256(sources[name]) == old_hash
                    for name, old_hash in original_hashes.items()
                )
        }

        print("\n=== INTEGRATION TESTS ===")

        for name, passed in tests.items():
            print(
                "PASS" if passed else "FAIL",
                "-", name
            )

        print(
            "\nTESTS:",
            sum(bool(v) for v in tests.values()),
            "/", len(tests)
        )

        if all(tests.values()):

            run_id = (
                datetime.now(timezone.utc).strftime(
                    "%Y%m%d_%H%M%S_%f"
                )
                + "_"
                + uuid.uuid4().hex[:6]
            )

            output = (
                F8 / "F8_Unified_Data"
                / f"Integration_{run_id}"
            )

            output.mkdir(
                parents=True,
                exist_ok=False
            )

            csv_path = (
                output / "F8_Unified_Task_Registry.csv"
            )

            merged.to_csv(csv_path, index=False)

            metrics = {
                "project": "DENGU HUNTER",
                "feature": "F8",
                "scope": "HACKATHON_SIMULATION",
                "unified_tasks": len(merged),
                "citizen_reports": len(citizens),
                "f4_location_matches": 15,
                "f6_image_matches": 15,
                "f7_image_matches": 15,
                "real_field_verifications": 0,
                "real_drone_dispatches": 0,
                "source_files_unchanged": True,
                "test_results": {
                    k: bool(v)
                    for k, v in tests.items()
                },
                "source_sha256": original_hashes
            }

            json_path = (
                output / "F8_Integration_Metrics.json"
            )

            with open(
                json_path, "x",
                encoding="utf-8"
            ) as f:
                json.dump(
                    metrics, f,
                    indent=2
                )

            assert len(pd.read_csv(csv_path)) == 15

            with open(json_path, encoding="utf-8") as f:
                saved_metrics = json.load(f)

            assert saved_metrics["unified_tasks"] == 15

            assert all(
                sha256(sources[name]) == old_hash
                for name, old_hash in original_hashes.items()
            )

            print("\nOUTPUT:", output)
            print("Saved Files: 2/2")
            print("Original Files Modified: NO")
            print("F8 STEP 3 COMPLETE")

        else:
            print("\nTESTS FAILED — NO FILE SAVED")

````

### Original cell 108/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime, timezone
from ultralytics import YOLO
from PIL import Image
import gradio as gr
import pandas as pd
import numpy as np
import cv2
import json
import uuid
import secrets
import hashlib

print("=== DENGU HUNTER F8: STEP 4 ===")
print("=== CITIZEN PHOTO UPLOAD + YOLO AI ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F8_DIR = ROOT / "Feature_8_Community_Government"

integrations = sorted(
    (F8_DIR / "F8_Unified_Data").glob(
        "Integration_*/F8_Unified_Task_Registry.csv"
    )
)

assert integrations, "Run F8 Step 3 first"

UNIFIED_PATH = integrations[-1]

MODEL_PATH = (
    ROOT / "Models/Detection_Model/best.pt"
)

CITIZEN_TEMPLATE = (
    F8_DIR / "community_reporting_template.csv"
)

assert MODEL_PATH.is_file(), "F2 model missing"
assert CITIZEN_TEMPLATE.is_file(), "Citizen template missing"

unified = pd.read_csv(UNIFIED_PATH)

assert len(unified) == 15
assert unified["f4_location_id"].nunique() == 15

def f8_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

protected_sources = [
    UNIFIED_PATH,
    MODEL_PATH,
    CITIZEN_TEMPLATE
]

source_hashes_f8 = {
    str(p): f8_hash(p)
    for p in protected_sources
}

print("\n=== LOADING TRAINED F2 MODEL ===")

f8_detection_model = YOLO(str(MODEL_PATH))

print("Model Task:", f8_detection_model.task)
print("Model Classes:", f8_detection_model.names)

assert f8_detection_model.task == "detect"

# Create a new output folder.
# Original CSV and model files remain untouched.

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    + "_"
    + uuid.uuid4().hex[:6]
)

F8_CITIZEN_RUN = (
    F8_DIR
    / "F8_Citizen_Portal"
    / f"Run_{run_id}"
)

F8_CITIZEN_RUN.mkdir(
    parents=True,
    exist_ok=False
)

REPORT_DIR = F8_CITIZEN_RUN / "Reports"
REPORT_DIR.mkdir()

area_choices = (
    ["UNLINKED"]
    + sorted(
        unified["f4_location_id"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
)

def analyze_citizen_photo(
    image,
    selected_area,
    area_name,
    site_type,
    description
):

    if image is None:
        raise gr.Error("Upload a photo first.")

    rgb = np.asarray(image)

    if rgb.dtype != np.uint8:
        rgb = np.clip(rgb, 0, 255).astype(np.uint8)

    rgb = np.asarray(
        Image.fromarray(rgb).convert("RGB")
    )

    if rgb.shape[0] * rgb.shape[1] > 16000000:
        raise gr.Error(
            "Image is too large. Use an image "
            "under 16 megapixels."
        )

    bgr = cv2.cvtColor(
        rgb, cv2.COLOR_RGB2BGR
    )

    # Real model inference on uploaded image
    prediction = f8_detection_model.predict(
        source=bgr,
        conf=0.25,
        imgsz=640,
        verbose=False
    )[0]

    detections = []

    if prediction.boxes is not None:

        for box in prediction.boxes:

            class_id = int(
                box.cls[0].item()
            )

            class_name = (
                f8_detection_model.names[class_id]
            )

            detections.append({
                "class_name": str(class_name),
                "confidence": round(
                    float(box.conf[0].item()), 4
                ),
                "bounding_box_xyxy": [
                    round(float(x), 2)
                    for x in box.xyxy[0].tolist()
                ]
            })

    overlay_bgr = prediction.plot()

    overlay_rgb = cv2.cvtColor(
        overlay_bgr,
        cv2.COLOR_BGR2RGB
    )

    count = len(detections)

    # Prototype triage heuristic.
    # NOT a trained dengue risk prediction.
    if count >= 2:
        inspection_priority = "HIGH_REVIEW_PRIORITY"
    elif count == 1:
        inspection_priority = "MEDIUM_REVIEW_PRIORITY"
    else:
        inspection_priority = "MANUAL_REVIEW_REQUIRED"

    # Optional independent F4 simulated context
    f4_context = None

    if selected_area != "UNLINKED":

        match = unified[
            unified["f4_location_id"]
            == selected_area
        ]

        if len(match) == 1:

            row = match.iloc[0]

            f4_context = {
                "simulated_location":
                    selected_area,
                "demonstration_risk_score":
                    float(
                        row["demonstration_risk_score"]
                    ),
                "demonstration_risk_category":
                    str(
                        row["demonstration_risk_category"]
                    ),
                "data_status":
                    str(row["data_status"]),
                "note":
                    "Existing simulated F4 context; "
                    "not calculated from this photo."
            }

    report_id = (
        "DH-CIT-"
        + datetime.now(timezone.utc).strftime(
            "%Y%m%d%H%M%S%f"
        )
        + "-"
        + uuid.uuid4().hex[:6]
    )

    report_folder = REPORT_DIR / report_id

    report_folder.mkdir(
        parents=False,
        exist_ok=False
    )

    photo_path = (
        report_folder / "submitted_image.png"
    )

    overlay_path = (
        report_folder / "ai_detection_overlay.png"
    )

    Image.fromarray(rgb).save(photo_path)

    Image.fromarray(
        overlay_rgb
    ).save(overlay_path)

    record = {
        "report_id": report_id,
        "submission_date_utc":
            datetime.now(timezone.utc).isoformat(),
        "area_name":
            str(area_name or "").strip(),
        "selected_simulated_area":
            selected_area,
        "area_link_verified": False,
        "latitude": None,
        "longitude": None,
        "gps_validated": False,
        "suspected_site_type": site_type,
        "description":
            str(description or "").strip(),
        "image_path": str(photo_path),
        "image_sha256": f8_hash(photo_path),
        "ai_overlay_path": str(overlay_path),
        "ai_model":
            "F2_EXISTING_TRAINED_YOLO_DETECTION",
        "detected_object_count": count,
        "detections": detections,
        "inspection_priority":
            inspection_priority,
        "inspection_priority_method":
            "DEMO_OBJECT_COUNT_HEURISTIC",
        "f4_simulated_context":
            f4_context,
        "review_status":
            "AWAITING_HUMAN_REVIEW",
        "assigned_task_id": None,
        "field_action_completed": False,
        "real_world_verified": False,
        "evidence_scope":
            "USER_UPLOADED_UNVERIFIED",
        "water_presence_verified": False,
        "aedes_larvae_verified": False
    }

    json_path = (
        report_folder / "citizen_report.json"
    )

    with open(
        json_path,
        "x",
        encoding="utf-8"
    ) as f:
        json.dump(
            record,
            f,
            indent=2,
            ensure_ascii=False
        )

    with open(
        json_path,
        encoding="utf-8"
    ) as f:
        saved = json.load(f)

    assert saved["report_id"] == report_id
    assert f8_hash(photo_path) == saved["image_sha256"]

    assert all(
        f8_hash(Path(path)) == original_hash
        for path, original_hash
        in source_hashes_f8.items()
    )

    print(
        "CITIZEN REPORT SAVED:",
        report_id
    )

    summary = (
        f"Report Saved: YES\n"
        f"Report ID: {report_id}\n"
        f"Detected Objects: {count}\n"
        f"Inspection Priority: {inspection_priority}\n"
        f"Human Review: PENDING\n"
        f"Government Task Assigned: NO\n"
        f"Real Field Verification: NO\n"
        f"Source Files Modified: NO"
    )

    return overlay_rgb, record, summary


# --------------------------------
# Automated inference smoke test
# --------------------------------

print("\n=== STEP 4 INITIAL TESTS ===")

sample_path = Path(
    unified.iloc[0]["before_image"]
)

sample_bgr = cv2.imread(str(sample_path))

assert sample_bgr is not None, (
    "Existing F7 sample image not readable"
)

sample_result = f8_detection_model.predict(
    source=sample_bgr,
    imgsz=640,
    conf=0.25,
    verbose=False
)

tests = {
    "15 Integrated Locations":
        len(unified) == 15,

    "Unique Area IDs":
        unified["f4_location_id"].nunique() == 15,

    "F2 Detection Model Loaded":
        f8_detection_model.task == "detect",

    "Existing Sample Image Readable":
        sample_bgr is not None,

    "Real YOLO Inference Runs":
        len(sample_result) == 1,

    "YOLO Detection Output Available":
        sample_result[0].boxes is not None,

    "Citizen Output Folder Created":
        REPORT_DIR.is_dir(),

    "Original Sources Unchanged":
        all(
            f8_hash(Path(path)) == original_hash
            for path, original_hash
            in source_hashes_f8.items()
        )
}

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTESTS:",
    sum(bool(v) for v in tests.values()),
    "/", len(tests)
)

assert all(tests.values())

# --------------------------------
# Interactive Citizen Portal
# --------------------------------

f8_password = secrets.token_urlsafe(12)

with gr.Blocks(
    title="DENGU HUNTER — Citizen Portal"
) as f8_citizen_app:

    gr.Markdown(
        "# 🦟 DENGU HUNTER"
        "\n## Citizen Reporting Portal"
        "\nUpload a suspected breeding-site photo "
        "for AI-assisted inspection triage."
        "\n\n**Prototype only:** AI object detection "
        "does not confirm standing water, "
        "Aedes larvae, or actual dengue risk."
        "\nUse demo images without identifiable "
        "people or sensitive information."
    )

    with gr.Row():

        citizen_photo = gr.Image(
            label="Upload Suspected Site Photo",
            type="numpy",
            sources=["upload"]
        )

        ai_overlay = gr.Image(
            label="AI Detection Result",
            interactive=False
        )

    selected_area = gr.Dropdown(
        choices=area_choices,
        value="UNLINKED",
        label="Optional Simulated Area",
        info=(
            "Selecting an area does not "
            "verify the photo's location."
        )
    )

    area_name_input = gr.Textbox(
        label="Reported Area Name (Optional)",
        placeholder="Example: Community Zone A"
    )

    site_type_input = gr.Dropdown(
        choices=[
            "Unknown",
            "Tire",
            "Bottle",
            "Vase",
            "Drain",
            "Rooftop Tank",
            "Other"
        ],
        value="Unknown",
        label="Suspected Site Type"
    )

    description_input = gr.Textbox(
        label="Description",
        lines=2
    )

    analyze_button = gr.Button(
        "Analyze Photo & Save Citizen Report",
        variant="primary"
    )

    ai_result = gr.JSON(
        label="AI + Report Details"
    )

    report_summary = gr.Textbox(
        label="Submission Summary",
        lines=8
    )

    analyze_button.click(
        fn=analyze_citizen_photo,
        inputs=[
            citizen_photo,
            selected_area,
            area_name_input,
            site_type_input,
            description_input
        ],
        outputs=[
            ai_overlay,
            ai_result,
            report_summary
        ]
    )

print("\n=== CITIZEN PORTAL LOGIN ===")
print("Username: boss")
print("Password:", f8_password)
print("Keep this login private.")

print("\nSaved Reports Folder:", REPORT_DIR)

f8_citizen_app.queue().launch(
    share=True,
    inline=False,
    auth=("boss", f8_password),
    show_error=True
)

print("\nF8 STEP 4 PORTAL READY")
print("Citizen Upload Save Test: PENDING")

````

### Original cell 109/118 — PYTHON SOURCE (archived, not run)

````python

from google.colab import files
from pathlib import Path

image_path = Path(unified.iloc[0]["before_image"])

print("Image:", image_path.name)
print("Found:", image_path.is_file())

if image_path.is_file():
    files.download(str(image_path))

````

### Original cell 110/118 — PYTHON SOURCE (archived, not run)

````python

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import pandas as pd
import numpy as np
import gradio as gr
import hashlib
import secrets
import uuid
import json

print("=== DENGU HUNTER F8: STEP 5 ===")
print("=== WORKER TASK MANAGEMENT ===")

ROOT = Path(
    "/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE"
)

F8 = ROOT / "Feature_8_Community_Government"

registries = sorted(
    (F8 / "F8_Unified_Data").glob(
        "Integration_*/F8_Unified_Task_Registry.csv"
    )
)

assert registries, "F8 Step 3 registry missing"

REGISTRY_PATH = registries[-1]
tasks = pd.read_csv(REGISTRY_PATH)

assert len(tasks) == 15
assert tasks["government_task_id"].nunique() == 15

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

original_hash = file_hash(REGISTRY_PATH)

run_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    + "_"
    + uuid.uuid4().hex[:6]
)

WORKER_ROOT = (
    F8 / "F8_Worker_Portal" / f"Run_{run_id}"
)

EVENTS = WORKER_ROOT / "Task_Events"
EVIDENCE = WORKER_ROOT / "Demo_Evidence"

EVENTS.mkdir(parents=True, exist_ok=False)
EVIDENCE.mkdir()

task_ids = (
    tasks["government_task_id"]
    .astype(str)
    .tolist()
)

# --------------------------------
# Task state from saved events
# --------------------------------

def get_task_states():

    states = {}

    for _, row in tasks.iterrows():

        task_id = str(row["government_task_id"])

        states[task_id] = {
            "government_task_id": task_id,
            "f7_task_id": str(row["f7_task_id"]),
            "location_id": str(row["f7_location_id"]),
            "risk_category": str(row["task_priority"]),
            "recommended_action":
                str(row["recommended_action"]),
            "demo_worker": "UNASSIGNED",
            "demo_status": "AWAITING_ASSIGNMENT",
            "demo_completion_photo": None,
            "real_field_status": "NOT_VERIFIED",
            "real_elimination_verified": False,
            "drone_dispatched": False
        }

    for path in sorted(EVENTS.glob("Event_*.json")):

        with open(path, encoding="utf-8") as f:
            event = json.load(f)

        task_id = event["government_task_id"]

        if task_id not in states:
            continue

        state = states[task_id]
        action = event["action"]

        if action == "ASSIGN_DEMO_WORKER":

            state["demo_worker"] = event["demo_worker"]
            state["demo_status"] = "DEMO_ASSIGNED"

        elif action == "START_DEMO_TASK":

            state["demo_status"] = "DEMO_IN_PROGRESS"

        elif action == "SUBMIT_DEMO_COMPLETION":

            state["demo_status"] = (
                "DEMO_COMPLETION_PHOTO_SUBMITTED"
            )

            state["demo_completion_photo"] = (
                event["photo_path"]
            )

    return states


def task_table():

    states = get_task_states()

    return pd.DataFrame([
        {
            "Task ID": s["government_task_id"],
            "Location": s["location_id"],
            "Priority": s["risk_category"],
            "Demo Worker": s["demo_worker"],
            "Demo Status": s["demo_status"]
        }
        for s in states.values()
    ])


def show_task(task_id):

    return get_task_states()[task_id]


# --------------------------------
# Save append-only demo events
# --------------------------------

def save_event(
    task_id,
    action,
    worker,
    notes="",
    photo=None
):

    event_id = (
        datetime.now(timezone.utc).strftime(
            "%Y%m%d_%H%M%S_%f"
        )
        + "_"
        + uuid.uuid4().hex[:8]
    )

    photo_path = None
    photo_sha256 = None

    if photo is not None:

        image = np.asarray(photo)

        image = np.clip(
            image, 0, 255
        ).astype(np.uint8)

        photo_file = (
            EVIDENCE / f"Demo_{event_id}.png"
        )

        Image.fromarray(image).convert(
            "RGB"
        ).save(photo_file)

        photo_path = str(photo_file)
        photo_sha256 = file_hash(photo_file)

    event = {
        "event_id": event_id,
        "created_at_utc":
            datetime.now(timezone.utc).isoformat(),
        "government_task_id": task_id,
        "action": action,
        "demo_worker": worker,
        "notes": str(notes or ""),
        "photo_path": photo_path,
        "photo_sha256": photo_sha256,
        "evidence_scope": "DEMO_ONLY",
        "actual_worker_identity_verified": False,
        "real_field_action_verified": False,
        "real_elimination_verified": False,
        "drone_dispatched": False
    }

    path = EVENTS / f"Event_{event_id}.json"

    with open(
        path, "x", encoding="utf-8"
    ) as f:
        json.dump(event, f, indent=2)

    assert file_hash(REGISTRY_PATH) == original_hash

    return event


# --------------------------------
# Demo worker assignment
# --------------------------------

def assign_worker(
    task_id,
    worker_name,
    approval,
    notes
):

    if not approval:
        raise gr.Error(
            "Confirm demo assignment approval first."
        )

    worker_name = str(worker_name or "").strip()

    if not worker_name:
        raise gr.Error("Enter a demo worker name.")

    if len(worker_name) > 80:
        raise gr.Error("Worker name is too long.")

    state = get_task_states()[task_id]

    if state["demo_status"] != "AWAITING_ASSIGNMENT":
        raise gr.Error(
            "This demo task is already assigned."
        )

    event = save_event(
        task_id,
        "ASSIGN_DEMO_WORKER",
        worker_name,
        notes
    )

    return (
        "Demo Worker Assigned: YES\n"
        f"Task: {task_id}\n"
        f"Worker: {worker_name}\n"
        "Real Government Assignment: NO",
        show_task(task_id),
        task_table()
    )


# --------------------------------
# Demo task progress + photo
# --------------------------------

def update_worker_task(
    task_id,
    action,
    photo,
    notes,
    demo_confirmed
):

    if not demo_confirmed:
        raise gr.Error(
            "Confirm this is a demo action."
        )

    state = get_task_states()[task_id]

    worker = state["demo_worker"]
    status = state["demo_status"]

    if worker == "UNASSIGNED":
        raise gr.Error(
            "Assign a demo worker first."
        )

    if action == "START_DEMO_TASK":

        if status != "DEMO_ASSIGNED":
            raise gr.Error(
                "Task must be in DEMO_ASSIGNED status."
            )

        event = save_event(
            task_id,
            action,
            worker,
            notes
        )

    elif action == "SUBMIT_DEMO_COMPLETION":

        if status != "DEMO_IN_PROGRESS":
            raise gr.Error(
                "Start the demo task first."
            )

        if photo is None:
            raise gr.Error(
                "Upload a demo completion photo."
            )

        event = save_event(
            task_id,
            action,
            worker,
            notes,
            photo
        )

    else:
        raise gr.Error("Invalid action.")

    return (
        "Demo Event Saved: YES\n"
        f"Task: {task_id}\n"
        f"Action: {action}\n"
        "Evidence: DEMO_ONLY\n"
        "Real Field Verification: NO",
        show_task(task_id),
        task_table()
    )


# --------------------------------
# Initial tests
# --------------------------------

print("\n=== STEP 5 INITIAL TESTS ===")

states = get_task_states()

tests = {
    "15 Government Tasks": len(tasks) == 15,
    "15 Unique Task IDs":
        tasks["government_task_id"].nunique() == 15,
    "15 F7 Task Links":
        tasks["f7_task_id"].nunique() == 15,
    "All Tasks Initially Unassigned":
        all(
            s["demo_worker"] == "UNASSIGNED"
            for s in states.values()
        ),
    "All Tasks Await Assignment":
        all(
            s["demo_status"] == "AWAITING_ASSIGNMENT"
            for s in states.values()
        ),
    "No Real Verification Claimed":
        all(
            s["real_elimination_verified"] is False
            for s in states.values()
        ),
    "No Drone Dispatch Claimed":
        all(
            s["drone_dispatched"] is False
            for s in states.values()
        ),
    "Event Folder Ready": EVENTS.is_dir(),
    "Evidence Folder Ready": EVIDENCE.is_dir(),
    "Original Registry Unchanged":
        file_hash(REGISTRY_PATH) == original_hash
}

for name, passed in tests.items():
    print(
        "PASS" if passed else "FAIL",
        "-", name
    )

print(
    "\nTESTS:",
    sum(tests.values()), "/", len(tests)
)

assert all(tests.values())

# --------------------------------
# Interactive worker dashboard
# --------------------------------

worker_password = secrets.token_urlsafe(12)

with gr.Blocks(
    title="DENGU HUNTER — Worker Portal"
) as worker_app:

    gr.Markdown(
        "# 🦟 DENGU HUNTER"
        "\n## Municipal Worker Task Portal — Demo"
        "\n**Simulation only.** Assignments and "
        "completion evidence do not represent "
        "actual government operations."
    )

    task_list = gr.Dataframe(
        value=task_table(),
        label="15 Government Demo Tasks",
        interactive=False
    )

    refresh = gr.Button("Refresh Task List")

    with gr.Tab("1. Assign Demo Worker"):

        assign_task_id = gr.Dropdown(
            choices=task_ids,
            value=task_ids[0],
            label="Select Government Task"
        )

        task_details = gr.JSON(
            value=show_task(task_ids[0]),
            label="Task Details"
        )

        worker_name_input = gr.Textbox(
            label="Demo Worker Name",
            value="Demo Worker 01"
        )

        assignment_notes = gr.Textbox(
            label="Assignment Notes"
        )

        approval_box = gr.Checkbox(
            label="Approve DEMO assignment only"
        )

        assign_button = gr.Button(
            "Assign Demo Worker",
            variant="primary"
        )

        assignment_result = gr.Textbox(
            label="Assignment Result"
        )

        assign_task_id.change(
            fn=show_task,
            inputs=assign_task_id,
            outputs=task_details
        )

        assign_button.click(
            fn=assign_worker,
            inputs=[
                assign_task_id,
                worker_name_input,
                approval_box,
                assignment_notes
            ],
            outputs=[
                assignment_result,
                task_details,
                task_list
            ]
        )

    with gr.Tab("2. Worker Task Update"):

        update_task_id = gr.Dropdown(
            choices=task_ids,
            value=task_ids[0],
            label="Select Task"
        )

        update_details = gr.JSON(
            value=show_task(task_ids[0]),
            label="Current Task State"
        )

        action_input = gr.Radio(
            choices=[
                "START_DEMO_TASK",
                "SUBMIT_DEMO_COMPLETION"
            ],
            value="START_DEMO_TASK",
            label="Demo Action"
        )

        completion_photo = gr.Image(
            label="Demo Completion Photo",
            type="numpy",
            sources=["upload"]
        )

        update_notes = gr.Textbox(
            label="Worker Notes"
        )

        demo_checkbox = gr.Checkbox(
            label="This is a DEMO action, not real field work"
        )

        update_button = gr.Button(
            "Save Demo Task Update",
            variant="primary"
        )

        update_result = gr.Textbox(
            label="Update Result"
        )

        update_task_id.change(
            fn=show_task,
            inputs=update_task_id,
            outputs=update_details
        )

        update_button.click(
            fn=update_worker_task,
            inputs=[
                update_task_id,
                action_input,
                completion_photo,
                update_notes,
                demo_checkbox
            ],
            outputs=[
                update_result,
                update_details,
                task_list
            ]
        )

    refresh.click(
        fn=task_table,
        outputs=task_list
    )

print("\n=== WORKER PORTAL LOGIN ===")
print("Username: boss")
print("Password:", worker_password)
print("Keep credentials private.")
print("\nEvent Folder:", EVENTS)

worker_app.queue().launch(
    share=True,
    inline=False,
    auth=("boss", worker_password),
    show_error=True
)

print("\nF8 STEP 5 PORTAL READY")
print("Worker Assignment Live Test: PENDING")
print("Worker Completion Live Test: PENDING")

````

### Original cell 111/118 — PYTHON SOURCE (archived, not run)

````python
from google.colab import files
files.download(str(sample_path))
````

### Original cell 112/118 — PYTHON SOURCE (archived, not run)

````python

# DENGU HUNTER — F8 STEP 6
# GOVERNMENT DASHBOARD

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import gradio as gr
import hashlib
import secrets
import json
import uuid

ROOT = Path("/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE")
F8 = ROOT / "Feature_8_Community_Government"

# 1. Load existing F8 registry
registry_files = sorted(
    (F8 / "F8_Unified_Data").glob(
        "Integration_*/F8_Unified_Task_Registry.csv"
    )
)

assert registry_files, "F8 Unified Registry not found!"
REGISTRY_PATH = registry_files[-1]
registry = pd.read_csv(REGISTRY_PATH)

assert len(registry) == 15, "Expected 15 government tasks"
assert registry["government_task_id"].is_unique

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

original_hash = sha256(REGISTRY_PATH)

# 2. Read saved Worker Portal events
EVENT_ROOT = F8 / "F8_Worker_Portal"

def worker_states():
    states = {}

    for _, row in registry.iterrows():
        tid = str(row["government_task_id"])
        states[tid] = {
            "worker": "UNASSIGNED",
            "status": "AWAITING_ASSIGNMENT",
            "completion_photo": None
        }

    event_files = sorted(
        EVENT_ROOT.glob("Run_*/Task_Events/Event_*.json"),
        key=lambda p: p.name
    )

    for path in event_files:
        try:
            with open(path, encoding="utf-8") as f:
                event = json.load(f)
        except (OSError, ValueError):
            continue

        tid = str(event.get("government_task_id", ""))
        if tid not in states:
            continue

        action = event.get("action")
        state = states[tid]

        if action == "ASSIGN_DEMO_WORKER":
            state["worker"] = event.get(
                "demo_worker", "DEMO_WORKER"
            )
            state["status"] = "DEMO_ASSIGNED"

        elif action == "START_DEMO_TASK":
            state["status"] = "DEMO_IN_PROGRESS"

        elif action == "SUBMIT_DEMO_COMPLETION":
            state["status"] = "DEMO_COMPLETION_PHOTO_SUBMITTED"
            state["completion_photo"] = event.get("photo_path")

    return states, len(event_files)

# 3. Count saved citizen reports
CITIZEN_ROOT = F8 / "F8_Citizen_Portal"

def citizen_count():
    return len(list(
        CITIZEN_ROOT.glob(
            "Run_*/Reports/*/citizen_report.json"
        )
    ))

# 4. Build Government Task Table
def build_table():
    states, _ = worker_states()
    rows = []

    for _, r in registry.iterrows():
        tid = str(r["government_task_id"])
        state = states[tid]

        rows.append({
            "Government Task": tid,
            "F7 Task": str(r["f7_task_id"]),
            "Location": str(r["f7_location_id"]),
            "Priority": str(r["task_priority"]),
            "Risk": str(r["demonstration_risk_category"]),
            "Recommended Action": str(r["recommended_action"]),
            "Demo Worker": state["worker"],
            "Demo Status": state["status"],
            "Real Field Verified": "NO"
        })

    return pd.DataFrame(rows)

# 5. Dashboard refresh function
def refresh_dashboard(priority):
    df = build_table()
    states, event_count = worker_states()

    completed = sum(
        s["status"] == "DEMO_COMPLETION_PHOTO_SUBMITTED"
        for s in states.values()
    )

    assigned = sum(
        s["status"] != "AWAITING_ASSIGNMENT"
        for s in states.values()
    )

    summary = f"""
# DENGU HUNTER — Government Command Center

**Government Tasks:** {len(df)}

**Assigned / Started / Demo Submitted:** {assigned}

**Demo Completion Photos Submitted:** {completed}

**Saved Worker Events:** {event_count}

**Citizen Reports:** {citizen_count()}

**Real Field Verification:** NOT PERFORMED

**Important:** This is a hackathon prototype.
Drone operations, worker activities and cleanup are simulated.
Risk categories are demonstration estimates, not validated forecasts.
"""

    if priority != "ALL":
        df = df[df["Priority"] == priority]

    return summary, df.reset_index(drop=True)

# 6. Save new dashboard snapshot
run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    + "_" + uuid.uuid4().hex[:6]
)

OUTPUT = (
    F8 / "F8_Government_Dashboard" / f"Run_{run_id}"
)
OUTPUT.mkdir(parents=True, exist_ok=False)

table = build_table()
table.to_csv(
    OUTPUT / "Government_Task_Dashboard.csv",
    index=False
)

with open(
    OUTPUT / "Dashboard_Metadata.json",
    "x",
    encoding="utf-8"
) as f:
    json.dump({
        "source_registry": str(REGISTRY_PATH),
        "total_tasks": len(table),
        "citizen_reports": citizen_count(),
        "simulation_only": True,
        "real_field_verified": False
    }, f, indent=2)

# 7. Initial checks
states, event_count = worker_states()

checks = {
    "15 Government Tasks": len(table) == 15,
    "Unique Government IDs":
        table["Government Task"].is_unique,
    "F7 Task Links Present":
        table["F7 Task"].notna().all(),
    "Recommendations Present":
        table["Recommended Action"].notna().all(),
    "Original Registry Unchanged":
        sha256(REGISTRY_PATH) == original_hash
}

print("=== F8 GOVERNMENT DASHBOARD TESTS ===")
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "|", name)

print("\nWorker Events Found:", event_count)
print("Citizen Reports Found:", citizen_count())
print("DH-TASK-001:", states.get("DH-TASK-001"))
print("Saved To:", OUTPUT)

assert all(checks.values()), "Dashboard checks failed!"

# 8. Launch interactive Government Dashboard
priorities = sorted(table["Priority"].unique().tolist())

with gr.Blocks(
    title="DENGU HUNTER — Government Dashboard"
) as govt_app:

    gr.Markdown(
        "# DENGU HUNTER — Government Dashboard"
    )
    gr.Markdown(
        "AI-assisted task monitoring | Simulation only"
    )

    priority_filter = gr.Dropdown(
        choices=["ALL"] + priorities,
        value="ALL",
        label="Filter by Priority"
    )

    refresh_btn = gr.Button("Refresh Dashboard")
    summary = gr.Markdown()

    task_display = gr.Dataframe(
        label="Government Task Registry",
        interactive=False
    )

    priority_filter.change(
        refresh_dashboard,
        inputs=priority_filter,
        outputs=[summary, task_display]
    )

    refresh_btn.click(
        refresh_dashboard,
        inputs=priority_filter,
        outputs=[summary, task_display]
    )

    govt_app.load(
        refresh_dashboard,
        inputs=priority_filter,
        outputs=[summary, task_display]
    )

login_user = "government_demo"
login_password = secrets.token_urlsafe(12)

print("\nDashboard Username:", login_user)
print("Dashboard Password:", login_password)
print("Do not share these credentials publicly.")

govt_app.launch(
    share=True,
    auth=(login_user, login_password),
    debug=False
)

````

### Original cell 113/118 — PYTHON SOURCE (archived, not run)

````python

# DENGU HUNTER — F8 STEP 7: FINAL END-TO-END AUDIT
# READ-ONLY on existing files. Saves a new audit run only.
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib, json, uuid

ROOT = Path('/content/drive/MyDrive/DENGU_HUNTER_FINAL_SHOWCASE')
F8 = ROOT / 'Feature_8_Community_Government'

def latest(pattern):
    files = sorted(F8.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'Missing: {pattern}')
    return files[-1]

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def exists_and_hash(path_text, expected):
    if not path_text or not expected:
        return False
    p = Path(path_text)
    return p.is_file() and digest(p) == expected

def all_false(series):
    return series.astype(str).str.strip().str.lower().isin(['false','0','no']).all()

registry_path = latest('F8_Unified_Data/Integration_*/F8_Unified_Task_Registry.csv')
gov_path = latest('F8_Government_Dashboard/Run_*/Government_Task_Dashboard.csv')
report_files = sorted(F8.glob('F8_Citizen_Portal/Run_*/Reports/*/citizen_report.json'))
event_files = sorted(F8.glob('F8_Worker_Portal/Run_*/Task_Events/Event_*.json'))

if not report_files or not event_files:
    raise FileNotFoundError('Citizen report or worker event not found')

protected = [registry_path, gov_path] + report_files + event_files
before = {str(p): digest(p) for p in protected}

registry = pd.read_csv(registry_path)
gov = pd.read_csv(gov_path)
reports = [load_json(p) for p in report_files]
events = [load_json(p) for p in event_files]
events.sort(key=lambda e: e.get('created_at_utc', ''))

task_id = 'DH-TASK-001'
task_events = [
    e for e in events
    if e.get('government_task_id') == task_id
]

actions = [e.get('action') for e in task_events]

expected_actions = [
    'ASSIGN_DEMO_WORKER',
    'START_DEMO_TASK',
    'SUBMIT_DEMO_COMPLETION'
]

completion = next(
    (e for e in reversed(task_events)
     if e.get('action') == 'SUBMIT_DEMO_COMPLETION'),
    {}
)

checks = {
    '01 | 15 unified government tasks':
        len(registry) == 15,

    '02 | unique government task IDs':
        registry['government_task_id'].is_unique,

    '03 | unique F7 task IDs':
        registry['f7_task_id'].is_unique,

    '04 | F4-F5 location mapping':
        (
            registry['f4_location_id'].astype(str)
            == registry['f5_location_id'].astype(str)
        ).all(),

    '05 | F4 risk scores present':
        registry['demonstration_risk_score'].notna().all(),

    '06 | F6 recommendations present':
        registry['recommended_action'].notna().all()
        and registry['recommended_action'].astype(str)
        .str.strip().ne('').all(),

    '07 | F7 verification IDs present':
        registry['verification_id'].notna().all(),

    '08 | no real field verification claimed':
        all_false(registry['real_field_verified']),

    '09 | government dashboard has 15 tasks':
        len(gov) == 15,

    '10 | government IDs match dashboard':
        (
            set(registry['government_task_id'].astype(str))
            == set(gov['Government Task'].astype(str))
        ),

    '11 | at least one citizen report saved':
        len(reports) >= 1,

    '12 | citizen original images verified':
        all(
            exists_and_hash(
                r.get('image_path'),
                r.get('image_sha256')
            ) for r in reports
        ),

    '13 | citizen AI overlays saved':
        all(
            Path(r.get('ai_overlay_path') or '/nonexistent').is_file()
            for r in reports
        ),

    '14 | citizen report contains genuine F2 inference record':
        any(
            r.get('ai_model') == 'F2_EXISTING_TRAINED_YOLO_DETECTION'
            and int(r.get('detected_object_count') or 0) > 0
            and len(r.get('detections') or []) > 0
            for r in reports
        ),

    '15 | citizen review still correctly pending':
        all(
            r.get('review_status') == 'AWAITING_HUMAN_REVIEW'
            for r in reports
        ),

    '16 | no automatic citizen-to-task assignment':
        all(
            r.get('assigned_task_id') is None
            for r in reports
        ),

    '17 | citizen field verification not claimed':
        all(
            r.get('real_world_verified') is False
            and r.get('field_action_completed') is False
            for r in reports
        ),

    '18 | worker event sequence valid':
        actions == expected_actions,

    '19 | completion photo SHA256 valid':
        exists_and_hash(
            completion.get('photo_path'),
            completion.get('photo_sha256')
        ),

    '20 | worker events explicitly demo only':
        all(
            e.get('evidence_scope') == 'DEMO_ONLY'
            and e.get('real_field_action_verified') is False
            and e.get('real_elimination_verified') is False
            and e.get('drone_dispatched') is False
            for e in events
        ),

    '21 | dashboard reflects completion':
        (
            gov.loc[
                gov['Government Task'].astype(str) == task_id,
                'Demo Status'
            ].astype(str)
            == 'DEMO_COMPLETION_PHOTO_SUBMITTED'
        ).any(),

    '22 | source files unchanged':
        all(
            digest(Path(p)) == h
            for p, h in before.items()
        )
}

run_id = (
    datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    + '_' + uuid.uuid4().hex[:6]
)

out = F8 / 'F8_Final_Integration' / ('Run_' + run_id)
out.mkdir(parents=True, exist_ok=False)

queue = pd.DataFrame([{
    'report_id': r.get('report_id'),
    'AI_object_count': r.get('detected_object_count'),
    'inspection_priority': r.get('inspection_priority'),
    'human_review': r.get('review_status'),
    'government_task_assigned': 'NO',
    'real_world_verified': 'NO'
} for r in reports])

queue.to_csv(out / 'Citizen_Review_Queue.csv', index=False)
gov.to_csv(out / 'Government_Tasks_Snapshot.csv', index=False)

result = {
    'audit_utc': datetime.now(timezone.utc).isoformat(),
    'passed': sum(bool(v) for v in checks.values()),
    'total': len(checks),
    'checks': {k: bool(v) for k, v in checks.items()},
    'citizen_reports': len(reports),
    'worker_events': len(events),
    'government_tasks': len(registry),
    'limitations': [
        'Citizen report awaiting human review',
        'Worker completion is demo evidence, not real cleanup',
        'Drone missions and risk forecasts are simulations'
    ]
}

with open(out / 'F8_Final_Audit.json', 'x', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

print('=== F8 FINAL END-TO-END TEST ===')

for name, passed in checks.items():
    print(('PASS' if passed else 'FAIL'), '|', name)

print(f'\nFINAL RESULT: {result["passed"]}/{result["total"]} PASS')
print('Citizen Reports:', len(reports))
print('Worker Events:', len(events))
print('Saved Audit:', out)
print('Existing Files Modified: NO')

if not all(checks.values()):
    print('ACTION: Send FAIL lines. Do not mark F8 complete yet.')
else:
    print('F8 STANDALONE DATA WORKFLOW: VERIFIED')
    print('Human review and real-world operations: NOT COMPLETED')

````

### Original cell 114/118 — MARKDOWN SOURCE (archived, not run)

````markdown
## DENGU HUNTER — Quality Upgrade Pack (F3, F4, F7)

**Run the following four cells, in order, after mounting Google Drive.** They use your existing files and create only new audit/analysis files. Do **not** use *Runtime > Run all* on this historical development notebook; previous cells include exploratory training and old demo launches.

- **F3:** conservative confidence-based review triage on the existing larvae-model results; no Aedes-specific diagnosis.
- **F4:** 500 assumed-weather scenario sensitivity checks and robust priority ranking; **not** a trained or validated future dengue predictor.
- **F7:** SHA-256 evidence integrity and ORB-based image comparability audit; **not** real-world cleanup verification.
- **Final audit:** checks the three new outputs and reports pass/fail. The 76% proposal coverage is a **target**, not a test score.

Original notebook cells are retained. Saved outputs were cleared in this shareable copy to avoid exposing temporary Gradio credentials. Original Drive source data are not overwritten.

````

### Original cell 115/118 — PYTHON SOURCE (archived, not run)

````python
# DENGU HUNTER — QUALITY UPGRADE 1/3: F3 conservative larvae review triage
# Independent of earlier notebook variables. Uses existing REAL model predictions.
# No model retraining, no Aedes-species claim, no original file modifications.
from pathlib import Path
from datetime import datetime, timezone
import os, json, uuid
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get('DH_DRIVE_ROOT', '/content/drive/MyDrive'))
if not DRIVE_ROOT.is_dir() and 'google.colab' in __import__('sys').modules:
    from google.colab import drive
    drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), 'Mount Google Drive first (Colab Files > Mount Drive).'

F3_SOURCE = DRIVE_ROOT / 'DENGUE_HUNTER_RESULTS/Larvae_Model/F3_Dashboard_Evidence'
F3_PREDICTIONS = F3_SOURCE / 'F3_Larvae_Predictions.csv'
F3_REVIEWS = F3_SOURCE / 'F3_Human_Review.csv'
assert F3_PREDICTIONS.is_file(), f'Existing F3 predictions missing: {F3_PREDICTIONS}'

# Confidence is a model score, NOT a calibrated probability of larvae/Aedes.
# These thresholds ONLY prioritize review; no automatic positive/negative verdict.
def f3_review_triage(max_conf, boxes, human_decision='PENDING'):
    decision = str(human_decision or 'PENDING').strip().upper()
    if decision not in {'AGREE','DISAGREE','UNCERTAIN','PENDING'}:
        raise ValueError(f'Unexpected human decision: {decision}')
    if boxes < 0:
        raise ValueError('Negative box count')
    if boxes == 0:
        triage = 'NO_BOX_MANUAL_REVIEW_NOT_NEGATIVE_PROOF'
    elif max_conf is None or not np.isfinite(float(max_conf)) or not (0 <= float(max_conf) <= 1):
        raise ValueError('Invalid detection confidence')
    elif float(max_conf) < 0.50:
        triage = 'LOW_CONFIDENCE_REQUEST_CLEARER_CLOSEUP'
    elif float(max_conf) < 0.70:
        triage = 'UNCERTAIN_PRIORITY_HUMAN_REVIEW'
    else:
        triage = 'HIGH_SCORE_STILL_REQUIRES_HUMAN_REVIEW'
    if decision == 'DISAGREE':
        next_step = 'MANUAL_REASSESSMENT'
    elif decision == 'UNCERTAIN':
        next_step = 'COLLECT_ADDITIONAL_CLOSEUP'
    elif decision == 'AGREE':
        next_step = 'HUMAN_AGREES_NOT_SPECIES_CONFIRMED'
    else:
        next_step = 'HUMAN_REVIEW_PENDING'
    return triage, next_step

# Built-in policy tests (do not require the YOLO weights).
assert f3_review_triage(None,0)[0].startswith('NO_BOX')
assert f3_review_triage(.30,1)[0].startswith('LOW_CONFIDENCE')
assert f3_review_triage(.60,1)[0].startswith('UNCERTAIN')
assert f3_review_triage(.90,1,'DISAGREE')[1] == 'MANUAL_REASSESSMENT'
assert f3_review_triage(.90,1,'AGREE')[1] == 'HUMAN_AGREES_NOT_SPECIES_CONFIRMED'

pred = pd.read_csv(F3_PREDICTIONS, dtype=str).fillna('')
required_cols = {'image','confidence','class_name'}
assert required_cols.issubset(pred.columns), f'F3 prediction columns missing: {required_cols - set(pred.columns)}'
pred['image'] = pred['image'].str.strip()
assert pred['image'].ne('').all(), 'Blank image name in predictions'
pred['confidence_num'] = pd.to_numeric(pred['confidence'], errors='coerce')
assert pred.loc[pred['confidence'].ne(''), 'confidence_num'].notna().all(), 'Invalid confidence value'
assert pred['confidence_num'].dropna().between(0,1).all(), 'Confidence out of range'

review_map = {}
if F3_REVIEWS.is_file():
    reviews = pd.read_csv(F3_REVIEWS, dtype=str).fillna('')
    assert {'image','human_decision'}.issubset(reviews.columns)
    assert reviews['image'].is_unique, 'Duplicate F3 review records'
    review_map = dict(zip(reviews['image'].str.strip(), reviews['human_decision'].str.strip().str.upper()))

rows=[]
for name, group in pred.groupby('image', sort=True):
    valid = group[group['confidence_num'].notna() & group['class_name'].str.strip().str.lower().eq('mosquito-larvae')]
    n = len(valid)
    confidence = float(valid['confidence_num'].max()) if n else None
    human = review_map.get(name,'PENDING') or 'PENDING'
    triage, action = f3_review_triage(confidence, n, human)
    rows.append({'image':name, 'ai_boxes':n, 'max_model_confidence':confidence,
                 'review_triage':triage, 'human_decision':human, 'next_step':action,
                 'aedes_species_verified':False, 'same_site_f2_pairing_verified':False,
                 'automatic_elimination_approved':False})

result = pd.DataFrame(rows)
assert len(result) == pred['image'].nunique()
assert not result['aedes_species_verified'].any()
assert not result['automatic_elimination_approved'].any()

run = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6]
OUT = DRIVE_ROOT / 'DENGU_HUNTER_FINAL_SHOWCASE/Quality_Upgrades/F3_Review_Triage' / f'Run_{run}'
OUT.mkdir(parents=True, exist_ok=False)
result.to_csv(OUT/'F3_Confidence_Human_Review_Queue.csv', index=False)
summary = {'feature':'F3', 'images':len(result), 'review_triage_counts':result['review_triage'].value_counts().to_dict(),
           'human_review_counts':result['human_decision'].value_counts().to_dict(),
           'confidence_thresholds':'0.50 and 0.70 are heuristic triage cutoffs, NOT calibrated accuracy',
           'aedes_species_verified':False, 'same_site_f2_pairing_verified':False,
           'existing_predictions_reused':True, 'new_model_training':False, 'original_files_modified':False,
           'tests_passed':5}
(OUT/'F3_Review_Triage_Summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('F3 REVIEW QUALITY UPGRADE: PASS | images:',len(result))
print('Review queue saved:',OUT)
print('NOTE: This is decision support, not Aedes diagnosis.')

````

### Original cell 116/118 — PYTHON SOURCE (archived, not run)

````python
# DENGU HUNTER — QUALITY UPGRADE 2/3: F4 uncertainty-aware hotspot prioritization
# Uses existing synthetic F4 weather/evidence. No trained forecast or real disease prediction.
from pathlib import Path
from datetime import datetime, timezone
import os, json, uuid
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get('DH_DRIVE_ROOT', '/content/drive/MyDrive'))
assert DRIVE_ROOT.is_dir(), 'Mount Google Drive first.'
SOURCE = DRIVE_ROOT / 'DENGU_HUNTER_FINAL_SHOWCASE/Development_Batch_Features_3_4_6_8/predictive_hotspot_demo.csv'
assert SOURCE.is_file(), f'F4 source missing: {SOURCE}'

# What-if formula matches the existing F4 scenario design.
def f4_what_if(current, rain, water, rain_delta, temp_delta):
    current = np.asarray(current, dtype=float)
    rain = np.asarray(rain, dtype=float)
    water = np.asarray(water, dtype=float)
    actual_delta = np.maximum(0, rain + rain_delta) - rain
    score = current + 0.35*(actual_delta/100.0)*(0.4+0.6*water) + 0.25*(temp_delta/20.0)
    return np.clip(score,0,1)

# Built-in regression tests: bounds, dry/wet direction, no negative rainfall, reproducibility.
assert np.isclose(f4_what_if(.4,20,1,0,0), .4)
assert f4_what_if(.4,20,1,30,1) >= .4
assert f4_what_if(.4,20,1,-20,-1) <= .4
assert np.isclose(f4_what_if(.4,0,1,-50,0), .4)
assert np.all((f4_what_if(np.array([0,1]),10,1,100,5)>=0) & (f4_what_if(np.array([0,1]),10,1,100,5)<=1))

f4 = pd.read_csv(SOURCE)
cols = ['location_id','rainfall_mm','temperature_c','water_holding_evidence','demonstration_risk_score']
assert set(cols).issubset(f4.columns), f'Missing F4 columns: {set(cols)-set(f4.columns)}'
assert f4['location_id'].notna().all() and f4['location_id'].is_unique
for col in cols[1:]:
    f4[col] = pd.to_numeric(f4[col], errors='raise')
assert f4[cols[1:]].notna().all().all()
assert f4['water_holding_evidence'].isin([0,1]).all()
assert f4['rainfall_mm'].ge(0).all()
assert f4['demonstration_risk_score'].between(0,1).all()

# Weather ranges are EXPLICIT ASSUMPTIONS, not forecast distributions.
# Shared weather shift per draw preserves cross-area scenario coherence.
N = 500
rng = np.random.default_rng(20261122)
rain_delta = rng.uniform(-20,40,size=(N,1))  # assumed mm change
Temp_delta = rng.uniform(-1.5,2.5,size=(N,1)) # assumed Celsius change
base = f4['demonstration_risk_score'].to_numpy()[None,:]
rain = f4['rainfall_mm'].to_numpy()[None,:]
water = f4['water_holding_evidence'].to_numpy()[None,:]
scores = f4_what_if(base,rain,water,rain_delta,Temp_delta)
assert scores.shape == (N,len(f4))
assert np.isfinite(scores).all() and np.all((scores>=0)&(scores<=1))

# Deterministic rank tie-break by original location order.
rank_order = np.argsort(-scores, axis=1, kind='stable')
ranks = np.argsort(rank_order,axis=1,kind='stable')+1
k = min(5,len(f4))
result = pd.DataFrame({
    'location_id':f4['location_id'],
    'current_demo_risk':f4['demonstration_risk_score'],
    'scenario_p10':np.quantile(scores,.10,axis=0),
    'scenario_median':np.quantile(scores,.50,axis=0),
    'scenario_p90':np.quantile(scores,.90,axis=0),
    'fraction_of_assumed_scenarios_high':np.mean(scores>=.65,axis=0),
    'fraction_of_assumed_scenarios_medium_or_high':np.mean(scores>=.35,axis=0),
    'fraction_of_assumed_scenarios_top5':np.mean(ranks<=k,axis=0),
    'average_scenario_rank':np.mean(ranks,axis=0),
    'uncertainty_note':'ASSUMED_WEATHER_SCENARIOS_NOT_REAL_FORECAST'
}).sort_values(['fraction_of_assumed_scenarios_top5','scenario_median','location_id'],ascending=[False,False,True])
assert result['location_id'].is_unique
assert result['scenario_p10'].le(result['scenario_median']).all()
assert result['scenario_median'].le(result['scenario_p90']).all()
assert result['fraction_of_assumed_scenarios_top5'].between(0,1).all()
assert np.isclose(result['fraction_of_assumed_scenarios_top5'].sum(),k)

run = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6]
OUT = DRIVE_ROOT / 'DENGU_HUNTER_FINAL_SHOWCASE/Quality_Upgrades/F4_Scenario_Robustness' / f'Run_{run}'
OUT.mkdir(parents=True, exist_ok=False)
result.to_csv(OUT/'F4_Weather_Sensitivity_Ranking.csv',index=False)
summary = {'feature':'F4','areas':len(result),'simulated_scenarios':N,
           'assumed_rain_change_mm':[-20,40],'assumed_temperature_change_c':[-1.5,2.5],
           'what_probabilities_mean':'Fractions across hypothetical sampled scenarios, NOT calibrated real-world probabilities',
           'trained_predictive_model':False,'real_dengue_case_forecast':False,
           'weather_inputs':'synthetic demonstration data','original_files_modified':False,
           'tests_passed':10}
(OUT/'F4_Scenario_Interpretation.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('F4 ROBUSTNESS UPGRADE: PASS | areas:',len(result),'| scenarios:',N)
print('Top scenario-stable priorities:')
print(result[['location_id','fraction_of_assumed_scenarios_top5','scenario_p10','scenario_p90']].head(5).to_string(index=False))
print('Saved:',OUT)
print('NOTE: scenario sensitivity, NOT validated dengue prediction.')

````

### Original cell 117/118 — PYTHON SOURCE (archived, not run)

````python
# DENGU HUNTER — QUALITY UPGRADE 3/3: F7 evidence integrity + alignment audit
# Audits existing BEFORE/AFTER submissions; no new AI inference or real cleanup claim.
from pathlib import Path
from datetime import datetime, timezone
import os, json, uuid, hashlib
import numpy as np
import pandas as pd
import cv2

DRIVE_ROOT = Path(os.environ.get('DH_DRIVE_ROOT', '/content/drive/MyDrive'))
assert DRIVE_ROOT.is_dir(), 'Mount Google Drive first.'
F7_ROOT = DRIVE_ROOT / 'DENGU_HUNTER_FINAL_SHOWCASE/Feature_7_Closed_Loop'
BASELINES = sorted((F7_ROOT/'F7_AI_Baseline').glob('Baseline_*/F7_Before_AI_Baseline.csv'))
assert BASELINES, 'F7 baseline missing. Existing F7 Step 5 must be completed.'
BASELINE = pd.read_csv(BASELINES[-1]).set_index('verification_id',verify_integrity=True)

# ORB homography is a geometric consistency HEURISTIC, never proof of same physical site.
def f7_alignment_quality(before_bgr, after_bgr):
    if before_bgr is None or after_bgr is None:
        return {'geometry_candidate':False,'reason':'UNREADABLE_IMAGE','good_matches':0,'inlier_fraction':0.0}
    if before_bgr.shape[:2] != after_bgr.shape[:2]:
        return {'geometry_candidate':False,'reason':'DIFFERENT_DIMENSIONS','good_matches':0,'inlier_fraction':0.0}
    h,w=before_bgr.shape[:2]
    if min(h,w)<80 or max(h,w)>4096:
        return {'geometry_candidate':False,'reason':'UNSUPPORTED_IMAGE_SIZE','good_matches':0,'inlier_fraction':0.0}
    gray1=cv2.cvtColor(before_bgr,cv2.COLOR_BGR2GRAY)
    gray2=cv2.cvtColor(after_bgr,cv2.COLOR_BGR2GRAY)
    orb=cv2.ORB_create(nfeatures=1400)
    kp1,des1=orb.detectAndCompute(gray1,None)
    kp2,des2=orb.detectAndCompute(gray2,None)
    if des1 is None or des2 is None or len(kp1)<12 or len(kp2)<12:
        return {'geometry_candidate':False,'reason':'INSUFFICIENT_VISUAL_FEATURES','good_matches':0,'inlier_fraction':0.0}
    pairs=cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des1,des2,k=2)
    good=[a for pair in pairs if len(pair)==2 for a,b in [pair] if a.distance<0.75*b.distance]
    if len(good)<12:
        return {'geometry_candidate':False,'reason':'INSUFFICIENT_FEATURE_MATCHES','good_matches':len(good),'inlier_fraction':0.0}
    src=np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst=np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H,inliers=cv2.findHomography(src,dst,cv2.RANSAC,4.0)
    ratio=float(np.mean(inliers)) if inliers is not None else 0.0
    ok=H is not None and ratio>=0.45
    return {'geometry_candidate':bool(ok),'reason':'GEOMETRY_CANDIDATE_NOT_SITE_PROOF' if ok else 'GEOMETRY_UNCERTAIN',
            'good_matches':len(good),'inlier_fraction':round(ratio,4)}

def f7_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

# Built-in geometric regression tests (synthetic pictures, not real field data).
_test_rng=np.random.default_rng(7)
_test_img=_test_rng.integers(0,256,size=(320,400,3),dtype=np.uint8)
_test_shift=cv2.warpAffine(_test_img,np.float32([[1,0,3],[0,1,2]]),(400,320))
assert f7_alignment_quality(_test_img,_test_shift)['geometry_candidate']
assert not f7_alignment_quality(_test_img,np.zeros_like(_test_img))['geometry_candidate']
assert f7_alignment_quality(_test_img,_test_img[:200])['reason']=='DIFFERENT_DIMENSIONS'

SUBMISSIONS = sorted((F7_ROOT/'F7_Interactive_Uploads').glob('Run_*/Submission_*/verification_result.json'))
assert SUBMISSIONS, 'No existing F7 AI submissions. Run the previous F7 upload/verification demo first.'
records=[]
for path in SUBMISSIONS:
    entry={'submission_file':str(path),'integrity':'FAIL','geometry_candidate':False,
           'real_field_verified':False,'human_review_required':True,'original_files_modified':False}
    try:
        d=json.loads(path.read_text(encoding='utf-8'))
        key=str(d['verification_id'])
        if key not in BASELINE.index: raise ValueError('VERIFICATION_ID_NOT_IN_BASELINE')
        row=BASELINE.loc[key]
        before=Path(row['before_image'])
        after=path.parent/'after_uploaded.png'
        if not before.is_file() or not after.is_file(): raise ValueError('IMAGE_MISSING')
        before_ok=f7_sha256(before)==str(row['before_image_sha256'])
        after_ok=f7_sha256(after)==str(d['after_image_sha256'])
        if not (before_ok and after_ok): raise ValueError('SHA256_MISMATCH')
        entry['integrity']='PASS'
        entry['verification_id']=key
        entry['evidence_scope']=str(d.get('evidence_scope','UNKNOWN'))
        entry['same_view_user_asserted']=bool(d.get('same_view_user_asserted',False))
        if f7_sha256(before)==f7_sha256(after):
            entry['quality_status']='POSSIBLE_REUSED_IMAGE_REVIEW_REQUIRED'
            entry['geometry_reason']='SAME_IMAGE_HASH'
        else:
            geom=f7_alignment_quality(cv2.imread(str(before)),cv2.imread(str(after)))
            entry['geometry_candidate']=geom['geometry_candidate']
            entry['geometry_reason']=geom['reason']
            entry['feature_matches']=geom['good_matches']
            entry['inlier_fraction']=geom['inlier_fraction']
            if entry['evidence_scope']=='SYNTHETIC_DEMO':
                entry['quality_status']='SYNTHETIC_DEMO_NOT_FIELD_EVIDENCE'
            elif geom['geometry_candidate'] and entry['same_view_user_asserted']:
                entry['quality_status']='COMPARABLE_CANDIDATE_STILL_HUMAN_REVIEW'
            else:
                entry['quality_status']='RECAPTURE_OR_MANUAL_ALIGNMENT_REVIEW'
    except Exception as exc:
        entry['quality_status']='EVIDENCE_AUDIT_FAILED'
        entry['audit_error']=str(exc)
    records.append(entry)

run = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6]
OUT = F7_ROOT / 'F7_Evidence_Quality_Audit' / f'Run_{run}'
OUT.mkdir(parents=True,exist_ok=False)
result=pd.DataFrame(records,columns=['submission_file','verification_id','evidence_scope','integrity',
    'same_view_user_asserted','geometry_candidate','geometry_reason','feature_matches','inlier_fraction',
    'quality_status','audit_error','human_review_required','real_field_verified','original_files_modified'])
result.to_csv(OUT/'F7_Existing_Submission_Quality_Audit.csv',index=False)
summary={'feature':'F7','existing_submissions_checked':len(records),
         'integrity_passed':sum(r['integrity']=='PASS' for r in records),
         'geometry_candidates':sum(bool(r['geometry_candidate']) for r in records),
         'all_require_human_review':True,'real_field_verified':False,
         'geometric_match_is_not_site_proof':True,
         'real_cleanup_evidence_provided_by_this_cell':False,
         'source_files_modified':False,'tests_passed':3}
(OUT/'F7_Evidence_Quality_Summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('F7 EVIDENCE AUDIT: PASS | submissions checked:',len(records))
print('Integrity passes:',summary['integrity_passed'],'| geometry candidates:',summary['geometry_candidates'])
print('Saved:',OUT)
print('NOTE: Human review required. No real field elimination verified.')

````

### Original cell 118/118 — PYTHON SOURCE (archived, not run)

````python
# DENGU HUNTER — QUALITY UPGRADE 4/4: final upgrade audit
# Verifies NEW F3/F4/F7 evidence files. Does not change existing F1-F8 assets.
from pathlib import Path
from datetime import datetime, timezone
import os, json, uuid
import pandas as pd

DRIVE_ROOT=Path(os.environ.get('DH_DRIVE_ROOT','/content/drive/MyDrive'))
ROOT=DRIVE_ROOT/'DENGU_HUNTER_FINAL_SHOWCASE'
def latest_one(pattern):
    matches=sorted(ROOT.glob(pattern))
    assert matches, f'Missing upgrade output: {pattern}. Run the F3/F4/F7 upgrade cells first.'
    return matches[-1]

f3_path=latest_one('Quality_Upgrades/F3_Review_Triage/Run_*/F3_Confidence_Human_Review_Queue.csv')
f4_path=latest_one('Quality_Upgrades/F4_Scenario_Robustness/Run_*/F4_Weather_Sensitivity_Ranking.csv')
f7_path=latest_one('Feature_7_Closed_Loop/F7_Evidence_Quality_Audit/Run_*/F7_Existing_Submission_Quality_Audit.csv')
f3=pd.read_csv(f3_path)
f4=pd.read_csv(f4_path)
f7=pd.read_csv(f7_path)
checks={
    'F3 review triage produced':len(f3)>0 and f3['review_triage'].notna().all(),
    'F3 species not falsely verified':not f3['aedes_species_verified'].astype(bool).any(),
    'F3 automatic intervention not approved':not f3['automatic_elimination_approved'].astype(bool).any(),
    'F4 scenario analysis produced':len(f4)>0 and f4['location_id'].is_unique,
    'F4 scenario fractions valid':f4['fraction_of_assumed_scenarios_top5'].between(0,1).all(),
    'F4 uncertainty intervals ordered':(f4['scenario_p10']<=f4['scenario_median']).all() and (f4['scenario_median']<=f4['scenario_p90']).all(),
    'F7 evidence audit produced':f7_path.is_file() and len(f7)>0,
    'F7 existing image integrity':f7['integrity'].eq('PASS').all() if len(f7) else False,
    'F7 no false field verification':not f7['real_field_verified'].astype(bool).any(),
    'F7 human review required':f7['human_review_required'].astype(bool).all() if len(f7) else True,
}
checks={key:bool(value) for key,value in checks.items()}
assert all(checks.values()),f'Upgrade audit failed: {[k for k,v in checks.items() if not v]}'
run=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:6]
out=ROOT/'Quality_Upgrades/Final_Audit'/f'Run_{run}'
out.mkdir(parents=True,exist_ok=False)
summary={'quality_upgrade_cells':'3/3 data workflows checked','quality_checks_passed':sum(checks.values()),
         'quality_checks_total':len(checks),'checks':checks,
         'f3_images':len(f3),'f4_areas':len(f4),'f7_submissions':len(f7),
         'proposal_coverage_target_percent':76,
         'proposal_coverage_actual_percent':None,
         'why_not_76_yet':'Local tests and scenario robustness are not proof of trained forecasting or real field cleanup; real Drive run and live website integration remain necessary',
         'f4_trained_forecast':False,'f7_real_field_verification':False,'original_files_modified':False}
(out/'Upgrade_Audit.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(f"QUALITY UPGRADE AUDIT: {sum(checks.values())}/{len(checks)} PASS")
print('F3 images:',len(f3),'| F4 areas:',len(f4),'| F7 submissions:',len(f7))
print('Saved:',out)
print('Proposal coverage target: 76% | actual: not yet independently established')

````

In [35]:





summary, table, _ = engine.dashboard()

print("GPS:", summary["gps_completeness_percent"])
print("Summary:", summary)
print(table[["Case ID", "GPS", "Stage"]].to_string(index=False))

GPS: 0.0
Summary: {'citizen_reports': 2, 'staged_demo_cases': 2, 'unverified_field_reports': 0, 'verified_cleanup_successes': 0, 'human_review_pending': 1, 'human_approved': 1, 'worker_tasks_created': 1, 'after_photos_submitted': 1, 'evidence_reviewed_resolved_not_field_verified': 0, 'gps_completeness_percent': 0.0, 'task_followup_evidence_percent': 100.0, 'median_submission_to_task_seconds': 108.0, 'false_alert_rate': {'status': 'NOT EVALUATED', 'false_alert_rate': None, 'note': 'Upload independent human-labelled positive AND negative images.'}, 'real_field_verification': 'NOT PERFORMED', 'risk_method': 'Explainable rule-based inspection priority; not a trained outbreak forecast', 'drone_status': 'Simulation/available imagery only; no autonomous flight claimed'}
        Case ID          GPS                 Stage
DH-000F5E6FE089 NOT PROVIDED REINSPECTION_REQUIRED
DH-DB3214D849DB NOT PROVIDED AWAITING_HUMAN_REVIEW
